# SAP ↔ PLM Material Entity Resolution & Duplicate Detection

Bu notebook temizlenmiş sürümdür. Her hücrenin `metadata.original_index`
alanında orijinal hücre numarası saklanır.

## 0. Veri yükleme ve ilk keşif


In [1]:
#Upload dataset
import os

import pandas as pd
import numpy as np
from pathlib import Path

#define the folder containing the project files
#data_folder = Path(".") choses the current path
#override with the DATA_FOLDER environment variable when running elsewhere
data_folder = Path(
    os.environ.get(
        "DATA_FOLDER",
        r"C:\Users\EMRE.YILMAZ\Desktop\Kumaş PLM Tespit"
    )
)

if not data_folder.is_dir():
    raise FileNotFoundError(
        f"Data folder not found: {data_folder}. "
        "Set the DATA_FOLDER environment variable or edit this cell."
    )

#list all excel and csv files in the data folder
files = list(data_folder.glob("*.xlsx")) + list(data_folder.glob("*.csv"))

for file in files:
    print(file.name)



Fabric_Mandatory_Fields.xlsx
Kum.xlsx
MaterialAttributes_full.xlsx
PLM Çalışması - Mara.xlsx
PLM_Codes.xlsx
PLM_MULTI_VAL_CHAR.xlsx
SATNR.xlsx
MaterialAttributes_full.csv


In [2]:
material_attributes_path = data_folder / "MaterialAttributes_full.xlsx"
sap_plm_mapping_path = data_folder / "PLM Çalışması - Mara.xlsx"
plm_codes_path = data_folder / "PLM_Codes.xlsx"

#check sheet names in each excel file
print(pd.ExcelFile(material_attributes_path).sheet_names)
print(pd.ExcelFile(sap_plm_mapping_path).sheet_names)
print(pd.ExcelFile(plm_codes_path).sheet_names)

['Data']
['GenericArticle']
['Data']


In [3]:
#Load datasets
material_attributes = pd.read_excel(
    material_attributes_path,
    sheet_name="Data"
)
sap_plm_mapping = pd.read_excel(
    sap_plm_mapping_path,
    sheet_name="GenericArticle"
)
plm_codes = pd.read_excel(
    plm_codes_path,
    sheet_name="Data"
)

In [4]:
#Check data set dimensions
print("Material attributes:", material_attributes.shape)
print("SAP-PLM Mapping:", sap_plm_mapping.shape)
print("PLM Codes :", plm_codes.shape)

Material attributes: (98248, 4)
SAP-PLM Mapping: (21205, 6)
PLM Codes : (32361, 27)


In [5]:
#display column names
print("\nMaterial Attributes columns:")
print(material_attributes.columns.tolist())

print("\nSAP-PLM Mapping columns:")
print(sap_plm_mapping.columns.tolist())

print("\nPLM Codes columns:")
print(plm_codes.columns.tolist())


Material Attributes columns:
['Malzeme', 'Dahili krkt.no.', 'Sayaç', 'Karakteristik değeri']

SAP-PLM Mapping columns:
['Malzeme', 'Mal grubu', 'Temel ölçü birimi', 'Türkçe malzeme açıklaması', 'Türkçe malzeme Uzun açıklaması', 'PLM Kodu']

PLM Codes columns:
['PLM Kodu', 'Örme alt tipi', 'İlmek uzunluğu/50 iğne', 'Malzeme statüs', '1.İplik numarası örme', '2.İplik numarası örm', '3.İplik numarası örme', 'Kumaş ağırlığı', 'Kumaş ağırlığı birimi', 'Kumaş eni', 'Kumaş eni birimi', 'Pus', 'Fine', 'Mal grubu', 'Türkçe malzeme açıklaması', 'Kumaş tipi', 'Çözgü sıklığı (tel /in', 'Atkı sıklığı (tel /inç)', 'Dokuma tipi', '1.Çözgü iplik numarası', '1.Atkı iplik numarası', '2.Çözgü iplik numarası', '2.Atkı iplik numarası', '3.Çözgü iplik numarası', '3.Atkı iplik numaras', 'Malzeme ingilizce adı', 'Malzeme Türkçe Adı']


In [6]:
display(material_attributes.head())
display(sap_plm_mapping.head())
display(plm_codes.head())

,Malzeme,Dahili krkt.no.,Sayaç,Karakteristik değeri
0,1020000012,FABRICSTRUCTURETNAME,1,TWILL
1,1020000012,WEAVETYPE,1,LCWWEAVETYPELIST8
2,1020000012,WARPDENSITY,1,44
3,1020000012,WEFTDENSITY,1,33
4,1020000012,WARPYARN1TYPEID,1,RING


,Malzeme,Mal grubu,Temel ölçü birimi,Türkçe malzeme açıklaması,Türkçe malzeme Uzun açıklaması,PLM Kodu
0,1020000000,DOKUMA,M,GABARD.30/20 HAM,NaN,NaN
1,1020000001,DOKUMA,M,40/1 80 TEL HAM POPLIN,NaN,NaN
2,1020000002,ORME,KG,30/1 SUPREM FU YA,30/1 SUPREM 180CM 60PAM 40PES 1LS-PETROL,NaN
3,1020000003,ORME,KG,30/1 PENYE INTERLOK HAM,30/1 PENYE INTERLOK HAM,NaN
4,1020000004,ORME,KG,30/1 LYC RIBANA HAM,30/1 LYC RIBANA HAM,NaN


,PLM Kodu,Örme alt tipi,İlmek uzunluğu/50 iğne,Malzeme statüs,1.İplik numarası örme,2.İplik numarası örm,3.İplik numarası örme,Kumaş ağırlığı,Kumaş ağırlığı birimi,Kumaş eni,...,Atkı sıklığı (tel /inç),Dokuma tipi,1.Çözgü iplik numarası,1.Atkı iplik numarası,2.Çözgü iplik numarası,2.Atkı iplik numarası,3.Çözgü iplik numarası,3.Atkı iplik numaras,Malzeme ingilizce adı,Malzeme Türkçe Adı
0,300001,NaN,NaN,INCONCEPT,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,300011,CIRCULAR,NaN,INCONCEPT,NaN,NaN,NaN,0.0,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,043021 MATERIAL NAME 1-1 (FOR,FLAT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,LCWWEAVETYPELIST1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,043021 MATERIAL NUMBER 1-1 (FO,FLAT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,LCWWEAVETYPELIST1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100000000,NaN,NaN,INCONCEPT,8,NaN,NaN,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# Check unique key counts
print("Unique SAP materials in attributes:",
      material_attributes["Malzeme"].nunique())

print("Unique SAP materials in mapping:",
      sap_plm_mapping["Malzeme"].nunique())

print("Unique PLM codes in mapping:",
      sap_plm_mapping["PLM Kodu"].nunique())

print("Unique PLM codes in PLM master:",
      plm_codes["PLM Kodu"].nunique())

Unique SAP materials in attributes: 9544
Unique SAP materials in mapping: 21205
Unique PLM codes in mapping: 4012
Unique PLM codes in PLM master: 32361


In [8]:
#calculate PLM mapping coverage
total_materials = sap_plm_mapping["Malzeme"].nunique()
materials_with_plm = (
    sap_plm_mapping
    .dropna(subset=["PLM Kodu"])["Malzeme"]
    .nunique()
)

materials_without_plm = total_materials - materials_with_plm
print("Total SAP materials:", total_materials)
print("SAP materials with PLM:", materials_with_plm)
print("SAP materials without PLM:", materials_without_plm)

print("PLM coverage:",
      round(materials_with_plm / total_materials * 100, 2),"%")

Total SAP materials: 21205
SAP materials with PLM: 5646
SAP materials without PLM: 15559
PLM coverage: 26.63 %


In [9]:
#Count how many SAP materials are linked to same PLM code
sap_count_per_plm = (
    sap_plm_mapping
    .dropna(subset=["PLM Kodu"])
    .groupby("PLM Kodu")["Malzeme"]
    .nunique()
    .sort_values(ascending=False)
)

display(sap_count_per_plm.head(20))

plm_sap_distribution = (
    sap_count_per_plm
    .value_counts()
    .sort_index()
    .rename_axis("sap_material_count")
    .reset_index(name="plm_count")
)

display(plm_sap_distribution.head(20))

PLM Kodu
12075.0     130
4667.0       78
2822.0       52
4823.0       45
2529.0       40
2533.0       32
2574.0       32
3859.0       25
171694.0     24
352974.0     23
26823.0      21
16678.0      20
21683.0      18
354310.0     17
123575.0     17
23061.0      15
4641.0       14
29694.0      14
12063.0      13
172235.0     12
Name: Malzeme, dtype: int64

,sap_material_count,plm_count
0,1,3541
1,2,226
2,3,102
3,4,53
4,5,22
5,6,14
6,7,4
7,8,15
8,9,6
9,10,4


In [10]:
# Standardize SAP material IDs before comparing datasets
sap_plm_mapping["material_id"] = (
    pd.to_numeric(sap_plm_mapping["Malzeme"],errors="coerce")
    .astype("Int64")
    .astype("string")
)

material_attributes["material_id"] = (
    pd.to_numeric(material_attributes["Malzeme"], errors="coerce")
    .astype("Int64")
    .astype("string")
)

# Compare SAP material coverage between datasets
mapping_materials = set(
    sap_plm_mapping["material_id"].dropna()
)

attribute_materials = set(
    material_attributes["material_id"].dropna()
)

common_materials = mapping_materials & attribute_materials

print("SAP materials in mapping:", len(mapping_materials))
print("SAP materials with attributes:", len(attribute_materials))
print("Common SAP materials:", len(common_materials))

print(
    "Attribute coverage:",
    round(len(common_materials)/len(mapping_materials)*100,2),"%"
)

SAP materials in mapping: 21205
SAP materials with attributes: 9544
Common SAP materials: 9510
Attribute coverage: 44.85 %


In [11]:
#check attribute coverage separately for materials with and without PLM codes
sap_plm_mapping["has_plm"] = sap_plm_mapping["PLM Kodu"].notna()
attribute_coverage = (
    sap_plm_mapping
    .assign(
        has_attributes=lambda df:
        df["material_id"].isin(attribute_materials)
    )
    .groupby("has_plm")
    .agg(
        material_count=("material_id", "nunique"),
        materials_with_attributes=("has_attributes", "sum")
    )
)

attribute_coverage["attribute_coverage_pct"] = (
    attribute_coverage["materials_with_attributes"]
    / attribute_coverage["material_count"]
    *100
).round(2)

display(attribute_coverage)

,material_count,materials_with_attributes,attribute_coverage_pct
has_plm,,,
False,15559,5990,38.50
True,5646,3520,62.35


## 1. Kapsam belirleme — saya / trim malzemelerinin ayıklanması


In [12]:
#Identify SAP materials that do not exist in the attribute dataset
missing_attribute_materials = (
    sap_plm_mapping[
        ~sap_plm_mapping["material_id"].isin(attribute_materials)
    ].copy()
)

print(
    "SAP materials without attribute records:",
    missing_attribute_materials["material_id"].nunique()
)

display(missing_attribute_materials.sort_values("Malzeme", ascending=False).head(20))

SAP materials without attribute records: 11695


,Malzeme,Mal grubu,Temel ölçü birimi,Türkçe malzeme açıklaması,Türkçe malzeme Uzun açıklaması,PLM Kodu,material_id,has_plm
16526,1020017322,DOKUMA,M,FLİP FLOP SAYA,NaN,NaN,1020017322,False
16512,1020017308,DOKUMA,M,4MM SUNGER + POLIBOND LAMINASYON,NaN,NaN,1020017308,False
16502,1020017297,DOKUMA,M,110 GR KUM BEJİ +4 MM EVA,NaN,NaN,1020017297,False
16499,1020017294,DENIM,M,90 GR KIRIK BEYAZ LACOSTE+4 MM EVA,NaN,NaN,1020017294,False
16490,1020017285,DOKUMA,M,500 GR PELUŞ,NaN,NaN,1020017285,False
16489,1020017284,DOKUMA,M,"110GR LACOSTE KUM BEJİ+3,26 DNS SNG+TELA",NaN,NaN,1020017284,False
16485,1020017280,DOKUMA,M,PVC ASTAR,NaN,NaN,1020017280,False
16458,1020017243,DOKUMA,M,FLİP FLOP SAYA,NaN,NaN,1020017243,False
16456,1020017241,DOKUMA,M,6815 YEŞİL VELAR VİTAŞ,NaN,NaN,1020017241,False
16445,1020017230,DENIM,M,90 GR BEYAZ LACOSTE+4 MM EVA,NaN,NaN,1020017230,False


In [13]:
from sklearn.feature_extraction.text import CountVectorizer

#combine short and long descriptions

descriptions = (
    missing_attribute_materials["Türkçe malzeme açıklaması"]
    .fillna("")
    .astype(str)
    + " "
    + missing_attribute_materials["Türkçe malzeme Uzun açıklaması"]
    .fillna("")
    .astype(str)
).str.upper()

# extract the most frequent 1-3 word expressions
vectorizer = CountVectorizer(
    ngram_range=(1,3),
    min_df=5
)

ngram_matrix = vectorizer.fit_transform(descriptions)

ngram_counts = ngram_matrix.sum(axis=0).A1
ngram_terms = vectorizer.get_feature_names_out()

frequent_expressions = (
    pd.DataFrame(
        {
            "expression": ngram_terms,
            "count": ngram_counts
        }
    )
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

display(frequent_expressions.head(100))

,expression,count
0,30,7778
1,pes,7511
2,pam,6161
3,100,6128
4,duz,4466
...,...,...
95,vual,394
96,popl,392
97,pam sup,391
98,gabard,391


In [14]:
# Remove expressions that contain only numbers
frequent_expressions_clean = frequent_expressions[
    ~frequent_expressions["expression"].str.fullmatch(r"[\d\s]+")
].copy()

# Separate single-word terms
top_unigrams = frequent_expressions_clean[
    frequent_expressions_clean["expression"].str.split().str.len() == 1
].head(100)
# Show up to 100 rows without truncation
pd.set_option("display.max_rows", 100)

display(top_unigrams)


,expression,count
1,pes,7511
2,pam,6161
4,duz,4466
5,penye,2998
6,lyc,2584
10,co,1984
11,gr,1925
13,sup,1727
15,50d,1440
16,75d,1421


In [15]:
# Extract two- and three-word expressions
top_phrases = frequent_expressions_clean[
    frequent_expressions_clean["expression"].str.split().str.len().between(2, 3)
].head(100)
# Show up to 100 rows without truncation
pd.set_option("display.max_rows", 100)

display(top_phrases)

,expression,count
7,100 pes,2533
8,30 penye,2283
14,pam pes,1630
17,100 pam,1307
19,duz bya,1293
21,sup duz,1072
24,100 co,985
25,pam lyc,976
31,30 pny,857
32,pes lyc,812


In [16]:
import re

# Define keywords that may indicate non-fabric materials
non_fabric_search_terms = [
    "SAYA",
    "AYAKKABI",
    "DERI",
    "DERİ",
    "TABAN",
    "TOKA",
    "FERMUAR",
    "DUGME",
    "DÜĞME",
    "AKSESUAR",
    "LASTIK",
    "LASTİK",
    "KORDON",
    "IP",
    "İP",
    "BANT"
]

# Count descriptions containing each keyword
keyword_counts = []

for term in non_fabric_search_terms:
    # BUGFIX: regex=False did substring matching, so short keywords such as
    # "IP" also matched inside longer words. Match whole words instead.
    pattern = r"\b" + re.escape(term) + r"\b"

    count = descriptions.str.contains(
        pattern,
        case=False,
        regex=True,
        na=False
    ).sum()

    keyword_counts.append({
        "keyword": term,
        "material_count": count
    })

keyword_summary = (
    pd.DataFrame(keyword_counts)
    .sort_values("material_count", ascending=False)
)

display(keyword_summary)

,keyword,material_count
3,DERİ,112
2,DERI,112
15,BANT,86
4,TABAN,51
13,IP,49
14,İP,49
0,SAYA,46
1,AYAKKABI,3
10,LASTIK,2
11,LASTİK,2


In [17]:
# Inspect materials containing "SAYA"
display(
    missing_attribute_materials[
        descriptions.str.contains(
            "SAYA",
            case=False,
            regex=False
        )
    ][
        [
            "Malzeme",
            "Mal grubu",
            "Türkçe malzeme açıklaması",
            "Türkçe malzeme Uzun açıklaması",
            "PLM Kodu"
        ]
    ].head(50)
)

,Malzeme,Mal grubu,Türkçe malzeme açıklaması,Türkçe malzeme Uzun açıklaması,PLM Kodu
10237,1020010496,DOKUMA,COLOMBES BEIGE SAYA TABAN MOSTRA SALPA,NaN,NaN
10239,1020010498,DOKUMA,BARBAROS SAYA TABAN MOSTRA SALPA,NaN,NaN
10240,1020010499,DOKUMA,GORAY SAYA TABAN MOSTRA SALPA,NaN,NaN
10249,1020010510,DOKUMA,PESTEN SAYA TABAN MOSTRA SALPA,NaN,NaN
10250,1020010511,DOKUMA,FORTE NAVY SAYA TABAN MOSTRA SALPA,NaN,NaN
10251,1020010512,DOKUMA,FORTE GREY SAYA TABAN MOSTRA SALPA,NaN,NaN
10252,1020010513,DOKUMA,NİCE SAYA TABAN MOSTRA SALPA,NaN,NaN
10253,1020010514,DOKUMA,KANO SAYA TABAN MOSTRA SALPA,NaN,NaN
10254,1020010515,DOKUMA,OLİVER SAYA TABAN MOSTRA SALPA,NaN,NaN
10255,1020010516,DOKUMA,SCOLA SAYA TABAN MOSTRA SALPA,NaN,NaN


In [18]:
import re
import unicodedata

# Combine material descriptions
missing_attribute_materials["combined_description"] = (
    missing_attribute_materials["Türkçe malzeme açıklaması"]
    .fillna("")
    .astype(str)
    + " "
    + missing_attribute_materials["Türkçe malzeme Uzun açıklaması"]
    .fillna("")
    .astype(str)
).str.upper()

In [19]:
# Define strong footwear upper indicators
saya_keywords = [
    "SAYA",
    "AYAKKABI",
    "MOSTRA",
    "SALPA"
]

saya_pattern = r"\b(?:" + "|".join(saya_keywords) + r")\b"

saya_candidates = missing_attribute_materials[
    missing_attribute_materials["combined_description"]
    .str.contains(saya_pattern, regex=True, na=False)
].copy()

print("Saya candidates:", len(saya_candidates))

display(
    saya_candidates[
        [
            "Malzeme",
            "Mal grubu",
            "Türkçe malzeme açıklaması",
            "PLM Kodu"
        ]
    ].head(50)
)

Saya candidates: 66


,Malzeme,Mal grubu,Türkçe malzeme açıklaması,PLM Kodu
10237,1020010496,DOKUMA,COLOMBES BEIGE SAYA TABAN MOSTRA SALPA,NaN
10239,1020010498,DOKUMA,BARBAROS SAYA TABAN MOSTRA SALPA,NaN
10240,1020010499,DOKUMA,GORAY SAYA TABAN MOSTRA SALPA,NaN
10249,1020010510,DOKUMA,PESTEN SAYA TABAN MOSTRA SALPA,NaN
10250,1020010511,DOKUMA,FORTE NAVY SAYA TABAN MOSTRA SALPA,NaN
10251,1020010512,DOKUMA,FORTE GREY SAYA TABAN MOSTRA SALPA,NaN
10252,1020010513,DOKUMA,NİCE SAYA TABAN MOSTRA SALPA,NaN
10253,1020010514,DOKUMA,KANO SAYA TABAN MOSTRA SALPA,NaN
10254,1020010515,DOKUMA,OLİVER SAYA TABAN MOSTRA SALPA,NaN
10255,1020010516,DOKUMA,SCOLA SAYA TABAN MOSTRA SALPA,NaN


In [20]:
# Assign preliminary scope category
saya_candidates["scope_category"] = "saya"

In [21]:
# Define strong trim indicators
trim_keywords = [
    "FERMUAR",
    "DÜĞME",
    "DUGME",
    "TOKA",
    "KORDON",
    "ŞERİT",
    "SERIT",
    "BİYE",
    "BIYE",
    "LASTİK",
    "LASTIK",
    "AKSESUAR"
]

# Create a regex pattern using whole words
trim_pattern = r"\b(?:" + "|".join(trim_keywords) + r")\b"

# Identify potential trim materials
trim_candidates = missing_attribute_materials[
    missing_attribute_materials["combined_description"]
    .str.contains(trim_pattern, regex=True, na=False)
].copy()

print("Trim candidates:", len(trim_candidates))

display(
    trim_candidates[
        [
            "Malzeme",
            "Mal grubu",
            "Türkçe malzeme açıklaması",
            "Türkçe malzeme Uzun açıklaması",
            "PLM Kodu"
        ]
    ].head(50)
)

Trim candidates: 11


,Malzeme,Mal grubu,Türkçe malzeme açıklaması,Türkçe malzeme Uzun açıklaması,PLM Kodu
8734,1020008939,ORME,ŞERİT DNTL DUZ BYA,P3S PINK 100 PES DNTL DUZ BYA,98492.0
9177,1020009406,DOKUMA,BRD E5X BYZ 60/1 60/1 %65 PAM %3,PAMUK VUAL ÜZERI ŞERİT BRODE(300,NaN
10330,1020010594,DOKUMA,SERIT-ANIMAL KUMAŞ,NaN,NaN
10493,1020010761,DOKUMA,RABAT HI- 16 MM SERIT,NaN,NaN
10494,1020010762,DOKUMA,RABAT HI- 10 MM SERIT,NaN,NaN
11683,1020012047,DENIM,PATARA BEJ ŞERİT,NaN,NaN
11684,1020012048,DENIM,PATARA BEJ LASTİK,NaN,NaN
11696,1020012060,DENIM,İDA SİYAH ŞERİT,NaN,NaN
11697,1020012061,DENIM,İDA SİYAH LASTİK,NaN,NaN
11915,1020012319,ORME,İNTERLOK BİYE KUMAŞ KIRMIZI,NaN,NaN


In [22]:
# Combine short and long material descriptions
sap_plm_mapping["combined_description"] = (
    sap_plm_mapping["Türkçe malzeme açıklaması"]
    .fillna("")
    .astype(str)
    + " "
    + sap_plm_mapping["Türkçe malzeme Uzun açıklaması"]
    .fillna("")
    .astype(str)
).str.upper()

# Define strong footwear-related indicators
saya_keywords = [
    "SAYA",
    "AYAKKABI",
    "MOSTRA",
    "SALPA"
]

saya_pattern = r"\b(?:" + "|".join(saya_keywords) + r")\b"

# Flag footwear-related materials
sap_plm_mapping["is_saya"] = (
    sap_plm_mapping["combined_description"]
    .str.contains(saya_pattern, regex=True, na=False)
)

print(sap_plm_mapping["is_saya"].value_counts())

is_saya
False    21094
True       111
Name: count, dtype: int64


In [23]:
# Create a separate list for potential material group corrections
material_group_update_candidates = (
    sap_plm_mapping[
        sap_plm_mapping["is_saya"]
    ]
    .copy()
)

display(
    material_group_update_candidates[
        [
            "Malzeme",
            "Mal grubu",
            "Türkçe malzeme açıklaması",
            "PLM Kodu"
        ]
    ]
)

,Malzeme,Mal grubu,Türkçe malzeme açıklaması,PLM Kodu
10237,1020010496,DOKUMA,COLOMBES BEIGE SAYA TABAN MOSTRA SALPA,NaN
10239,1020010498,DOKUMA,BARBAROS SAYA TABAN MOSTRA SALPA,NaN
10240,1020010499,DOKUMA,GORAY SAYA TABAN MOSTRA SALPA,NaN
10249,1020010510,DOKUMA,PESTEN SAYA TABAN MOSTRA SALPA,NaN
10250,1020010511,DOKUMA,FORTE NAVY SAYA TABAN MOSTRA SALPA,NaN
...,...,...,...,...
19958,1020020938,DOKUMA,"1,5 MM SALPA + SÜNGER + LAMİNASYON",NaN
20405,1020021400,DOKUMA,90 GR ALKANTRA+2.5 MM EVA+1.5 MM SALPA,NaN
20704,1020021704,DOKUMA,VANDA DANA ASTAR+MOSTRA,NaN
20963,1020021975,DOKUMA,"ALKANTRA + 2,5MM EVA + 1,5 MM SALPA",NaN


In [24]:
# Collect material IDs that should be excluded from the fabric modeling scope
excluded_materials = set(
    material_group_update_candidates["material_id"]
) | set(
    trim_candidates["material_id"]
)

# Keep saya candidates separately
saya_materials = material_group_update_candidates.copy()

# Keep trim candidates separately
trim_materials = trim_candidates.copy()

# Create the final fabric modeling dataset
fabric_materials = (
    sap_plm_mapping[
        ~sap_plm_mapping["material_id"].isin(excluded_materials)
    ]
    .copy()
)

print("Original SAP materials:", sap_plm_mapping["material_id"].nunique())
print("Fabric materials:", fabric_materials["material_id"].nunique())
print("Saya materials:", saya_materials["material_id"].nunique())
print("Trim materials:", trim_materials["material_id"].nunique())

Original SAP materials: 21205
Fabric materials: 21083
Saya materials: 111
Trim materials: 11


In [25]:
#Calculate PLM coverage within the fabric modeling scope
total_fabric_materials = fabric_materials["material_id"].nunique()
fabric_materials_with_plm = (
    fabric_materials
    .dropna(subset=["PLM Kodu"])["material_id"]
    .nunique()
)

fabric_materials_without_plm = (
    total_fabric_materials - fabric_materials_with_plm
)
print("Total fabric materials:", total_fabric_materials)
print("Fabric materials with PLM:", fabric_materials_with_plm)
print("Fabric materials without PLM:", fabric_materials_without_plm)
print(
    "PLM coverage:",
    round(fabric_materials_with_plm/total_fabric_materials*100,2),"%"

)

Total fabric materials: 21083
Fabric materials with PLM: 5645
Fabric materials without PLM: 15438
PLM coverage: 26.78 %


In [26]:
# Check attribute coverage for fabric materials
fabric_materials["has_attributes"] = (
    fabric_materials["material_id"].isin(attribute_materials)
)

fabric_attribute_coverage = (
    fabric_materials
    .groupby("has_plm")
    .agg(
        materials_count = ("material_id", "nunique"),
        materials_with_attributes = ("has_attributes", "sum")        
    )
)

fabric_attribute_coverage["attribute_coverage_pct"] = (
    fabric_attribute_coverage["materials_with_attributes"]
    / fabric_attribute_coverage["materials_count"] * 100
).round(2)

display(fabric_attribute_coverage)

,materials_count,materials_with_attributes,attribute_coverage_pct
has_plm,,,
False,15438,5945,38.51
True,5645,3520,62.36


In [27]:
#keep attribute records only for materials in the fabric modeling scope
fabric_attributes = material_attributes[
    material_attributes["material_id"].isin(
        set(fabric_materials["material_id"])
    )
].copy()

#summarize attribute availability
attribute_summary = (
    fabric_attributes
    .groupby("Dahili krkt.no.")
    .agg(
        material_count = ("material_id", "nunique"),
        unique_value_count = ("Karakteristik değeri", "nunique"),
        row_count = ("material_id", "size")
    ).sort_values("material_count", ascending=False)
)

attribute_summary["coverage_pct"] = (
    attribute_summary["material_count"] / total_fabric_materials * 100
).round(2)
print("Unique attribute types:", len(attribute_summary))

display(attribute_summary.head(40))

Unique attribute types: 75


,material_count,unique_value_count,row_count,coverage_pct
Dahili krkt.no.,,,,
WEIGHT,9178,635,9178,43.53
FABRICSTRUCTURETNAME,9076,150,9076,43.05
WIDTH,9055,155,9055,42.95
FIBERCONTENTLISTID,8940,846,13180,42.40
COLORINGID,8767,42,9417,41.58
WEIGHTUOMNAME,5737,98,5737,27.21
WEAVETYPE,4137,219,4137,19.62
DYETYPEID,3888,43,4394,18.44
YARNCOUNT1KNITSID,3746,206,3746,17.77


In [28]:
display(
    pd.DataFrame({
        "attribute_name": sorted(
            fabric_attributes["Dahili krkt.no."]
            .dropna()
            .unique()
        )
    })
)

,attribute_name
0,COLORINGID
1,DYETYPEID
2,FABRICREFERENCENO
3,FABRICSTRUCTURETNAME
4,FIBERCONTENTLISTID
5,FINE
6,KNITSUBTYPEID
7,KNITTYPEID
8,LOOPLENGTH50NEEDLE
9,PHYSICALFINISHFABRICSIDEID


## 2. SAP karakteristikleri ile PLM alanlarının eşleştirilmesi


In [29]:
# Define candidate mappings between SAP attributes and PLM master fields
attribute_mapping = {
    "WEIGHT": "Kumaş ağırlığı",
    "WEIGHTUOMNAME": "Kumaş ağırlığı birimi",
    "WIDTH": "Kumaş eni",
    "WIDTHUOMNAME": "Kumaş eni birimi",

    # Knitted fabric attributes
    "KNITSUBTYPEID": "Örme alt tipi",
    "LOOPLENGTH50NEEDLE": "İlmek uzunluğu/50 iğne",
    "YARNCOUNT1KNITSID": "1.İplik numarası örme",
    "YARNCOUNT2KNITSID": "2.İplik numarası örm",
    "YARNCOUNT3KNITSID": "3.İplik numarası örme",
    "PUS": "Pus",
    "FINE": "Fine",

    # Woven fabric attributes
    "WEAVETYPE": "Dokuma tipi",
    "WARPDENSITY": "Çözgü sıklığı (tel /in",
    "WEFTDENSITY": "Atkı sıklığı (tel /inç)",
    "WARPYARNCOUNT1ID": "1.Çözgü iplik numarası",
    "WEFTYARNCOUNT1ID": "1.Atkı iplik numarası",
    "WARPYARNCOUNT2ID": "2.Çözgü iplik numarası",
    "WEFTYARNCOUNT2ID": "2.Atkı iplik numarası",
    "WARPYARNCOUNT3ID": "3.Çözgü iplik numarası",
    "WEFTYARNCOUNT3ID": "3.Atkı iplik numaras"
}

In [30]:
# Keep only SAP materials with an existing PLM code
known_plm_materials = set(
    fabric_materials.loc[
        fabric_materials["has_plm"],
        "material_id"
    ]
)

# Calculate coverage of candidate attributes within known PLM materials
candidate_attribute_coverage = (
    fabric_attributes[
        fabric_attributes["Dahili krkt.no."].isin(attribute_mapping.keys())
        & fabric_attributes["material_id"].isin(known_plm_materials)
    ]
    .groupby("Dahili krkt.no.")["material_id"]
    .nunique()
    .to_frame("material_count")
)

candidate_attribute_coverage["coverage_pct"] = (
    candidate_attribute_coverage["material_count"]
    / len(known_plm_materials)
    * 100
).round(2)

candidate_attribute_coverage["plm_field"] = (
    candidate_attribute_coverage.index.map(attribute_mapping)
)

display(
    candidate_attribute_coverage
    .sort_values("coverage_pct", ascending=False)
)

,material_count,coverage_pct,plm_field
Dahili krkt.no.,,,
WEIGHT,3517,62.30,Kumaş ağırlığı
WIDTH,3404,60.30,Kumaş eni
WEIGHTUOMNAME,2041,36.16,Kumaş ağırlığı birimi
FINE,1639,29.03,Fine
KNITSUBTYPEID,1637,29.00,Örme alt tipi
WEAVETYPE,1474,26.11,Dokuma tipi
WARPDENSITY,1327,23.51,Çözgü sıklığı (tel /in
WEFTDENSITY,1324,23.45,Atkı sıklığı (tel /inç)
LOOPLENGTH50NEEDLE,1164,20.62,İlmek uzunluğu/50 iğne


In [31]:
# Compare sample values between SAP attributes and PLM master fields
for sap_attribute, plm_field in {
    "WEIGHT": "Kumaş ağırlığı",
    "WIDTH": "Kumaş eni",
    "WEAVETYPE": "Dokuma tipi",
    "KNITSUBTYPEID": "Örme alt tipi"
}.items():

    print(f"\n--- {sap_attribute} <-> {plm_field} ---")

    print("\nSAP sample values:")
    print(
        fabric_attributes.loc[
            fabric_attributes["Dahili krkt.no."] == sap_attribute,
            "Karakteristik değeri"
        ]
        .dropna()
        .drop_duplicates()
        .head(15)
        .tolist()
    )

    print("\nPLM sample values:")
    print(
        plm_codes[plm_field]
        .dropna()
        .drop_duplicates()
        .head(15)
        .tolist()
    )


--- WEIGHT <-> Kumaş ağırlığı ---

SAP sample values:
['115.000', '100000.0', '114000.0', '135000.0', '270000.0', '280.0', '120000.0', '240.0', '285.0', '270.0', '75000.0', '125.0', '93000.0', '145000.0', '146000.0']

PLM sample values:
[0.0, 310.0, 330.0, 130.0, 170.0, 140.0, 175.0, 10.5, 240.0, 145.0, 180.0, 270.0, 11.5, 230.0, 95.0]

--- WIDTH <-> Kumaş eni ---

SAP sample values:
['150.000', '0.0', '0.000', '180', '185', '145', '125', '150', '172', '190-195', 'CM', '135', '190', '140', '105']

PLM sample values:
[0.0, 150.0, 145.0, 146.0, 180.0, 147.0, 153.0, 170.0, 140.0, 162.0, 143.0, 155.0, 132.0, 160.0, 167.0]

--- WEAVETYPE <-> Dokuma tipi ---

SAP sample values:
['LCWWEAVETYPELIST8', 'LCWWEAVETYPELIST1', 'LCWWEAVETYPELIST5', 'LCWWEAVETYPELIST29', 'LCWWEAVETYPELIST40', 'LCWWEAVETYPELIST10', 'LCWWEAVETYPELIST57', 'LCWWEAVETYPELIST66', 'LCWWEAVETYPELIST23', 'LCWWEAVETYPELIST4', 'LCWWEAVETYPELIST31', 'LCWWEAVETYPELIST24', 'LCWWEAVETYPELIST9', 'LCWWEAVETYPELIST69', 'LCWWEAVETYPEL

In [32]:
# Standardize PLM codes while preserving alphanumeric values
def normalize_plm_code(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().upper()

    # Remove Excel-style ".0" suffix from numeric codes
    if value.endswith(".0") and value[:-2].isdigit():
        value = value[:-2]

    return value


fabric_materials["plm_code"] = (
    fabric_materials["PLM Kodu"]
    .apply(normalize_plm_code)
)

plm_codes["plm_code"] = (
    plm_codes["PLM Kodu"]
    .apply(normalize_plm_code)
)

In [33]:
# Compare existing SAP-PLM mappings with the PLM master
mapped_plm_codes = set(
    fabric_materials["plm_code"].dropna()
)

master_plm_codes = set(
    plm_codes["plm_code"].dropna()
)

common_plm_codes = mapped_plm_codes & master_plm_codes
missing_from_master = mapped_plm_codes - master_plm_codes

print("PLM codes used in SAP mapping:", len(mapped_plm_codes))
print("PLM codes available in PLM master:", len(master_plm_codes))
print("Common PLM codes:", len(common_plm_codes))
print("Mapped PLM codes missing from master:", len(missing_from_master))

PLM codes used in SAP mapping: 4012
PLM codes available in PLM master: 32360
Common PLM codes: 4012
Mapped PLM codes missing from master: 0


## 3. Ağırlık (WEIGHT) doğrulaması ve ölçek normalizasyonu


In [34]:
#Inspect weight values togetger with their units
weight_values = (
    fabric_attributes[
        fabric_attributes["Dahili krkt.no."].isin(
            ["WEIGHT", "WEIGHTUOMNAME"]
        )
    ].pivot_table(
        index = "material_id",
        columns = "Dahili krkt.no.",
        values = "Karakteristik değeri",
        aggfunc= "first"
    ).reset_index()
)

display(weight_values.head(30))

display(
    weight_values["WEIGHTUOMNAME"]
    .value_counts(dropna=False).head(30)
)

Dahili krkt.no.,material_id,WEIGHT,WEIGHTUOMNAME
0,1020000012,115.000,NaN
1,1020000016,100000.0,NaN
2,1020000019,114000.0,NaN
3,1020000020,135000.0,GSM
4,1020000021,270000.0,GSM
5,1020000022,280.0,GSM
6,1020000023,120000.0,NaN
7,1020000026,240.0,GSM
8,1020000028,285.0,NaN
9,1020000029,270.0,GSM


WEIGHTUOMNAME
NaN               3448
GSM               2679
KG                1066
GR                 789
VRDOZPERSQYARD     282
111                155
1                  134
G/M2               105
G                   81
GRAM                60
ONZ                 32
GRM                 24
123                 22
GR/M2               20
150                 17
250                 16
M                   16
MT                  11
280                 10
110                 10
CM                  10
200                  9
180                  9
210                  8
100                  8
320                  7
ADT                  7
-                    7
240                  7
260                  7
Name: count, dtype: int64

In [35]:
weight_values["sap_weight_raw"] = pd.to_numeric(
    weight_values["WEIGHT"],
    errors="coerce"
)

plm_weight = plm_codes[
    ["plm_code","Kumaş ağırlığı"]
].copy()

plm_weight["plm_weight"] = pd.to_numeric(
    plm_weight["Kumaş ağırlığı"],
    errors="coerce"
)

known_weight_pairs = (
    fabric_materials[
        fabric_materials["plm_code"].notna()
    ][["material_id", "plm_code"]]
    .merge(
        weight_values[
            ["material_id","sap_weight_raw","WEIGHTUOMNAME"]
        ],
        on="material_id",
        how="inner"
    ).merge(
        plm_weight[
            ["plm_code","plm_weight"]
             ],
             on="plm_code",
             how="inner"
        )
)

#keep valid positive values only
known_weight_pairs = known_weight_pairs[
    (known_weight_pairs["sap_weight_raw"]>0)&
    (known_weight_pairs["plm_weight"]>0)
].copy()

display(known_weight_pairs.head(30))

,material_id,plm_code,sap_weight_raw,WEIGHTUOMNAME,plm_weight
0,1020000012,100407,115.0,NaN,115.0
1,1020000016,100689,100000.0,NaN,100.0
2,1020000019,100774,114000.0,NaN,114.0
3,1020000020,101139,135000.0,GSM,135.0
4,1020000021,101143,270000.0,GSM,270.0
5,1020000022,101666,280.0,GSM,280.0
6,1020000023,101739,120000.0,NaN,120.0
7,1020000026,102617,240.0,GSM,240.0
8,1020000028,102794,285.0,NaN,285.0
9,1020000029,102972,270.0,GSM,270.0


In [36]:
# Calculate the scale ratio between SAP and PLM weight values
known_weight_pairs["weight_ratio"] = (
    known_weight_pairs["sap_weight_raw"]
    / known_weight_pairs["plm_weight"]
)

display(
    known_weight_pairs[
        [
            "material_id",
            "plm_code",
            "sap_weight_raw",
            "plm_weight",
            "weight_ratio",
            "WEIGHTUOMNAME"
        ]
    ]
    .sort_values("weight_ratio", ascending=False)
    .head(50)
)

,material_id,plm_code,sap_weight_raw,plm_weight,weight_ratio,WEIGHTUOMNAME
3196,1020021161,370680,210000.0,9.3,22580.645161,GSM
2031,1020016486,355426,260000.0,220.0,1181.818182,GSM
682,1020001234,89680,68000.0,60.0,1133.333333,NaN
2518,1020018262,366571,340000.0,310.0,1096.774194,GSM
560,1020001011,356028,260000.0,240.0,1083.333333,GSM
524,1020000934,353991,150000.0,140.0,1071.428571,NaN
189,1020000321,171704,165000.0,155.0,1064.516129,NaN
2197,1020017169,366199,220000.0,210.0,1047.619048,GSM
2198,1020017170,366200,220000.0,210.0,1047.619048,GSM
514,1020000912,353467,155000.0,150.0,1033.333333,NaN


In [37]:
# Classify common scaling patterns
def classify_weight_scale(ratio):
    if 0.95 <= ratio <= 1.05:
        return "same_scale"
    elif 950 <= ratio <= 1050:
        return "x1000"
    else:
        return "other"


known_weight_pairs["scale_pattern"] = (
    known_weight_pairs["weight_ratio"]
    .apply(classify_weight_scale)
)

display(
    known_weight_pairs["scale_pattern"]
    .value_counts()
    .to_frame("record_count")
)

,record_count
scale_pattern,
same_scale,2186
x1000,1247
other,58


In [38]:
# Normalize SAP weight values
def normalize_weight(value):
    if pd.isna(value):
        return np.nan

    # Values above 1000 appear to be stored with a x1000 scale
    if value >= 1000:
        return value / 1000

    return value


known_weight_pairs["sap_weight_normalized"] = (
    known_weight_pairs["sap_weight_raw"]
    .apply(normalize_weight)
)

# Calculate the difference after normalization
known_weight_pairs["weight_difference"] = abs(
    known_weight_pairs["sap_weight_normalized"]
    - known_weight_pairs["plm_weight"]
)

known_weight_pairs["weight_difference_pct"] = (
    known_weight_pairs["weight_difference"]
    / known_weight_pairs["plm_weight"]
    * 100
)
# Evaluate weight normalization quality
print(
    "Exact matches:",
    (known_weight_pairs["weight_difference"] == 0).sum()
)

print(
    "Within ±5%:",
    (known_weight_pairs["weight_difference_pct"] <= 5).sum()
)

print(
    "Within ±10%:",
    (known_weight_pairs["weight_difference_pct"] <= 10).sum()
)

print(
    "Total pairs:",
    len(known_weight_pairs)
)

# Calculate match percentages after normalization
weight_validation = pd.Series({
    "exact_match_pct":
        (known_weight_pairs["weight_difference"] == 0).mean() * 100,

    "within_5_pct":
        (known_weight_pairs["weight_difference_pct"] <= 5).mean() * 100,

    "within_10_pct":
        (known_weight_pairs["weight_difference_pct"] <= 10).mean() * 100
}).round(2)

display(weight_validation)

Exact matches: 3410
Within ±5%: 3431
Within ±10%: 3452
Total pairs: 3491


exact_match_pct    97.68
within_5_pct       98.28
within_10_pct      98.88
dtype: float64

In [39]:
# Inspect large differences after normalization
weight_outliers = (
    known_weight_pairs[
        known_weight_pairs["weight_difference_pct"] > 10
    ]
    .sort_values("weight_difference_pct", ascending=False)
)

display(
    weight_outliers[
        [
            "material_id",
            "plm_code",
            "sap_weight_raw",
            "sap_weight_normalized",
            "plm_weight",
            "weight_difference_pct",
            "WEIGHTUOMNAME"
        ]
    ].head(30)
)

,material_id,plm_code,sap_weight_raw,sap_weight_normalized,plm_weight,weight_difference_pct,WEIGHTUOMNAME
2976,1020020036,999999,260.0,260.000,1.0,25900.000000,GSM
3196,1020021161,370680,210000.0,210.000,9.3,2158.064516,GSM
1742,1020015246,323,320.0,320.000,150.0,113.333333,NaN
3335,1020021549,100000063,1275.0,1.275,1275.0,99.900000,VRDOZPERSQYARD
3283,1020021394,100000063,1275.0,1.275,1275.0,99.900000,VRDOZPERSQYARD
3336,1020021550,100000073,1275.0,1.275,10.5,87.857143,VRDOZPERSQYARD
3337,1020021551,100000088,1275.0,1.275,10.5,87.857143,VRDOZPERSQYARD
701,1020007299,197690,200.0,200.000,115.0,73.913043,NaN
923,1020010918,117830,250.0,250.000,150.0,66.666667,250
1188,1020012195,19113,150.0,150.000,100.0,50.000000,GSM


In [40]:
# Prepare PLM weight values together with their units
plm_weight = plm_codes[
    [
        "plm_code",
        "Kumaş ağırlığı",
        "Kumaş ağırlığı birimi"
    ]
].copy()

plm_weight["plm_weight"] = pd.to_numeric(
    plm_weight["Kumaş ağırlığı"],
    errors="coerce"
)

plm_weight = plm_weight.rename(
    columns={
        "Kumaş ağırlığı birimi": "plm_weight_unit"
    }
)

In [41]:
# Add the PLM weight unit to the validation dataset
known_weight_pairs = known_weight_pairs.drop(
    columns=["plm_weight"],
    errors="ignore"
)

known_weight_pairs = known_weight_pairs.merge(
    plm_weight[
        [
            "plm_code",
            "plm_weight",
            "plm_weight_unit"
        ]
    ],
    on="plm_code",
    how="left"
)

In [42]:
# Normalize SAP weight values
known_weight_pairs["sap_weight_normalized"] = (
    known_weight_pairs["sap_weight_raw"]
    .apply(normalize_weight)
)

known_weight_pairs["weight_difference_pct"] = (
    abs(
        known_weight_pairs["sap_weight_normalized"]
        - known_weight_pairs["plm_weight"]
    )
    / known_weight_pairs["plm_weight"]
    * 100
)

In [43]:
# Inspect remaining weight outliers together with measurement units
weight_outliers = (
    known_weight_pairs[
        known_weight_pairs["weight_difference_pct"] > 10
    ]
    .sort_values(
        "weight_difference_pct",
        ascending=False
    )
)

display(
    weight_outliers[
        [
            "material_id",
            "plm_code",
            "sap_weight_raw",
            "sap_weight_normalized",
            "WEIGHTUOMNAME",
            "plm_weight",
            "plm_weight_unit",
            "weight_difference_pct"
        ]
    ].head(40)
)

,material_id,plm_code,sap_weight_raw,sap_weight_normalized,WEIGHTUOMNAME,plm_weight,plm_weight_unit,weight_difference_pct
2952,1020020036,999999,260.0,260.000,GSM,1.0,GSM,25900.000000
3170,1020021161,370680,210000.0,210.000,GSM,9.3,VRDOZPERSQYARD,2158.064516
1720,1020015246,323,320.0,320.000,NaN,150.0,GSM,113.333333
3309,1020021549,100000063,1275.0,1.275,VRDOZPERSQYARD,1275.0,VRDOZPERSQYARD,99.900000
3257,1020021394,100000063,1275.0,1.275,VRDOZPERSQYARD,1275.0,VRDOZPERSQYARD,99.900000
3310,1020021550,100000073,1275.0,1.275,VRDOZPERSQYARD,10.5,VRDOZPERSQYARD,87.857143
3311,1020021551,100000088,1275.0,1.275,VRDOZPERSQYARD,10.5,VRDOZPERSQYARD,87.857143
685,1020007299,197690,200.0,200.000,NaN,115.0,GSM,73.913043
905,1020010918,117830,250.0,250.000,250,150.0,NE,66.666667
1170,1020012195,19113,150.0,150.000,GSM,100.0,GSM,50.000000


In [44]:
# Summarize weight outliers by SAP and PLM unit combinations
weight_outlier_units = (
    weight_outliers
    .groupby(
        ["WEIGHTUOMNAME", "plm_weight_unit"],
        dropna=False
    )
    .size()
    .reset_index(name="record_count")
    .sort_values("record_count", ascending=False)
)

display(weight_outlier_units)

,WEIGHTUOMNAME,plm_weight_unit,record_count
8,NaN,GSM,15
4,GSM,GSM,13
7,VRDOZPERSQYARD,VRDOZPERSQYARD,4
6,KG,GSM,2
0,210,GSM,1
3,GR,GSM,1
2,280,GSM,1
1,250,NE,1
5,GSM,VRDOZPERSQYARD,1


In [45]:
# Identify weight discrepancies where both systems use the same unit
same_unit_outliers = weight_outliers[
    weight_outliers["WEIGHTUOMNAME"].fillna("UNKNOWN")
    ==
    weight_outliers["plm_weight_unit"].fillna("UNKNOWN")
].copy()

display(
    same_unit_outliers[
        [
            "material_id",
            "plm_code",
            "sap_weight_raw",
            "WEIGHTUOMNAME",
            "plm_weight",
            "plm_weight_unit",
            "weight_difference_pct"
        ]
    ]
)

,material_id,plm_code,sap_weight_raw,WEIGHTUOMNAME,plm_weight,plm_weight_unit,weight_difference_pct
2952,1020020036,999999,260.0,GSM,1.0,GSM,25900.000000
3309,1020021549,100000063,1275.0,VRDOZPERSQYARD,1275.0,VRDOZPERSQYARD,99.900000
3257,1020021394,100000063,1275.0,VRDOZPERSQYARD,1275.0,VRDOZPERSQYARD,99.900000
3310,1020021550,100000073,1275.0,VRDOZPERSQYARD,10.5,VRDOZPERSQYARD,87.857143
3311,1020021551,100000088,1275.0,VRDOZPERSQYARD,10.5,VRDOZPERSQYARD,87.857143
1170,1020012195,19113,150.0,GSM,100.0,GSM,50.000000
2698,1020019033,100000026,280.0,GSM,200.0,GSM,40.000000
682,1020006776,365707,170.0,GSM,230.0,GSM,26.086957
2334,1020017703,354065,300.0,GSM,240.0,GSM,25.000000
2057,1020016658,353616,230000.0,GSM,290.0,GSM,20.689655


## 4. En (WIDTH) doğrulaması


In [46]:
# Prepare SAP width values
width_values = (
    fabric_attributes[
        fabric_attributes["Dahili krkt.no."].isin(
            ["WIDTH", "WIDTHUOMNAME"]
        )
    ]
    .pivot_table(
        index="material_id",
        columns="Dahili krkt.no.",
        values="Karakteristik değeri",
        aggfunc="first"
    )
    .reset_index()
)

display(width_values.head(30))

Dahili krkt.no.,material_id,WIDTH,WIDTHUOMNAME
0,1020000012,150.000,NaN
1,1020000016,0.0,NaN
2,1020000019,0.0,NaN
3,1020000020,0.0,NaN
4,1020000021,0.0,NaN
5,1020000022,0.0,NaN
6,1020000023,0.0,NaN
7,1020000026,0.0,NaN
8,1020000028,0.0,NaN
9,1020000029,0.0,NaN


In [47]:
# Check the most common SAP width units
display(
    width_values["WIDTHUOMNAME"]
    .value_counts(dropna=False)
    .head(30)
)

WIDTHUOMNAME
NaN      6640
CM       1958
111       128
1          87
MT         70
M          25
CMS        25
110        14
160         9
150         9
GR          8
GSM         7
185         6
170         5
250         5
KG          4
180         4
190         3
0.0         3
195         3
22          3
175         3
METRE       3
145         2
163         2
30/1        2
155         2
ADT         2
200         1
140         1
Name: count, dtype: int64

In [48]:
# Prepare PLM width data
plm_width = plm_codes[
    [
        "plm_code",
        "Kumaş eni",
        "Kumaş eni birimi"
    ]
].copy()

plm_width["plm_width"] = pd.to_numeric(
    plm_width["Kumaş eni"],
    errors="coerce"
)

plm_width = plm_width.rename(
    columns={
        "Kumaş eni birimi": "plm_width_unit"
    }
)

In [49]:
# Convert SAP width values to numeric
width_values["sap_width"] = pd.to_numeric(
    width_values["WIDTH"],
    errors="coerce"
)

In [50]:
# Create known SAP-PLM width pairs
known_width_pairs = (
    fabric_materials[
        fabric_materials["plm_code"].notna()
    ][["material_id", "plm_code"]]
    .merge(
        width_values[
            ["material_id", "sap_width", "WIDTHUOMNAME"]
        ],
        on="material_id",
        how="inner"
    )
    .merge(
        plm_width[
            ["plm_code", "plm_width", "plm_width_unit"]
        ],
        on="plm_code",
        how="inner"
    )
)

# Keep valid positive width values
known_width_pairs = known_width_pairs[
    (known_width_pairs["sap_width"] > 0) &
    (known_width_pairs["plm_width"] > 0)
].copy()

known_width_pairs["width_difference_pct"] = (
    abs(
        known_width_pairs["sap_width"]
        - known_width_pairs["plm_width"]
    )
    / known_width_pairs["plm_width"]
    * 100
)

print("Comparable width pairs:", len(known_width_pairs))

print(
    "Exact matches:",
    (known_width_pairs["width_difference_pct"] == 0).sum()
)

print(
    "Within ±5%:",
    (known_width_pairs["width_difference_pct"] <= 5).sum()
)

print(
    "Within ±10%:",
    (known_width_pairs["width_difference_pct"] <= 10).sum()
)

Comparable width pairs: 95
Exact matches: 79
Within ±5%: 90
Within ±10%: 92


In [51]:
# Evaluate SAP-PLM width consistency
width_validation = pd.Series({
    "exact_match_pct":
        (known_width_pairs["width_difference_pct"] == 0).mean() * 100,

    "within_5_pct":
        (known_width_pairs["width_difference_pct"] <= 5).mean() * 100,

    "within_10_pct":
        (known_width_pairs["width_difference_pct"] <= 10).mean() * 100
}).round(2)

display(width_validation)

exact_match_pct    83.16
within_5_pct       94.74
within_10_pct      96.84
dtype: float64

In [52]:
# Check width availability on both sides
print(
    "SAP materials with numeric width:",
    width_values["sap_width"].gt(0).sum()
)

print(
    "PLM records with numeric width:",
    plm_width["plm_width"].gt(0).sum()
)

print(
    "Known SAP-PLM materials with comparable width:",
    len(known_width_pairs)
)

SAP materials with numeric width: 5279
PLM records with numeric width: 12316
Known SAP-PLM materials with comparable width: 95


In [53]:
# Define categorical SAP-PLM attribute pairs
categorical_mapping = {
    "WEAVETYPE": "Dokuma tipi",
    "KNITSUBTYPEID": "Örme alt tipi"
}

categorical_results = []

for sap_attribute, plm_field in categorical_mapping.items():

    sap_values = (
        fabric_attributes[
            fabric_attributes["Dahili krkt.no."] == sap_attribute
        ][["material_id", "Karakteristik değeri"]]
        .drop_duplicates(subset=["material_id"])
        .rename(columns={"Karakteristik değeri": "sap_value"})
    )

    comparison = (
        fabric_materials[
            fabric_materials["plm_code"].notna()
        ][["material_id", "plm_code"]]
        .merge(sap_values, on="material_id", how="inner")
        .merge(
            plm_codes[["plm_code", plm_field]],
            on="plm_code",
            how="inner"
        )
        .dropna(subset=["sap_value", plm_field])
    )

    comparison["is_match"] = (
        comparison["sap_value"]
        .astype(str)
        .str.strip()
        .str.upper()
        ==
        comparison[plm_field]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    categorical_results.append({
        "attribute": sap_attribute,
        "comparable_pairs": len(comparison),
        "exact_matches": comparison["is_match"].sum(),
        "match_pct": round(comparison["is_match"].mean() * 100, 2)
    })

categorical_validation = pd.DataFrame(categorical_results)

display(categorical_validation)

,attribute,comparable_pairs,exact_matches,match_pct
0,WEAVETYPE,1440,1399,97.15
1,KNITSUBTYPEID,1636,185,11.31


In [54]:
# Check width availability only for PLM codes currently linked to SAP materials
mapped_plm_width = (
    fabric_materials[
        fabric_materials["plm_code"].notna()
    ][["plm_code"]]
    .drop_duplicates()
    .merge(
        plm_width[["plm_code", "plm_width", "plm_width_unit"]],
        on="plm_code",
        how="left"
    )
)

print(
    "Mapped PLM codes:",
    len(mapped_plm_width)
)

print(
    "Mapped PLM codes with positive width:",
    mapped_plm_width["plm_width"].gt(0).sum()
)

Mapped PLM codes: 4012
Mapped PLM codes with positive width: 1558


In [55]:
# Compare SAP and PLM knit subtype values
knit_subtype_pairs = (
    fabric_attributes[
        fabric_attributes["Dahili krkt.no."] == "KNITSUBTYPEID"
    ][["material_id", "Karakteristik değeri"]]
    .drop_duplicates(subset=["material_id"])
    .rename(columns={"Karakteristik değeri": "sap_knit_subtype"})
    .merge(
        fabric_materials[
            fabric_materials["plm_code"].notna()
        ][["material_id", "plm_code"]],
        on="material_id",
        how="inner"
    )
    .merge(
        plm_codes[
            ["plm_code", "Örme alt tipi"]
        ],
        on="plm_code",
        how="inner"
    )
    .dropna(
        subset=["sap_knit_subtype", "Örme alt tipi"]
    )
)

# Show the most common SAP-PLM value combinations
knit_subtype_combinations = (
    knit_subtype_pairs
    .groupby(
        ["sap_knit_subtype", "Örme alt tipi"]
    )
    .size()
    .reset_index(name="record_count")
    .sort_values("record_count", ascending=False)
)

display(knit_subtype_combinations.head(30))

,sap_knit_subtype,Örme alt tipi,record_count
1,C,CIRCULAR,1411
2,CIRCULAR,CIRCULAR,109
5,WARP,WARP,73
4,F,FLAT,38
0,C,C,3
3,F,CIRCULAR,1
6,İNTERLOK,CIRCULAR,1


In [56]:
# Normalize SAP knit subtype values
knit_subtype_mapping = {
    "C": "CIRCULAR",
    "F": "FLAT",
    "WARP": "WARP",
    "CIRCULAR": "CIRCULAR"
}

knit_subtype_pairs["sap_knit_subtype_normalized"] = (
    knit_subtype_pairs["sap_knit_subtype"]
    .replace(knit_subtype_mapping)
)

knit_subtype_pairs["is_match_normalized"] = (
    knit_subtype_pairs["sap_knit_subtype_normalized"]
    ==
    knit_subtype_pairs["Örme alt tipi"]
)

print(
    "Normalized match rate:",
    round(
        knit_subtype_pairs["is_match_normalized"].mean() * 100,
        2
    ),
    "%"
)

Normalized match rate: 99.69 %


In [57]:
# Check width availability step by step for known PLM materials
known_materials = fabric_materials[
    fabric_materials["plm_code"].notna()
][["material_id", "plm_code"]].copy()

width_check = (
    known_materials
    .merge(
        width_values[["material_id", "sap_width"]],
        on="material_id",
        how="left"
    )
    .merge(
        plm_width[["plm_code", "plm_width"]],
        on="plm_code",
        how="left"
    )
)

print("Known SAP-PLM materials:", len(width_check))
print("With SAP numeric width:", width_check["sap_width"].gt(0).sum())
print("With PLM numeric width:", width_check["plm_width"].gt(0).sum())
print(
    "With width on both sides:",
    (
        width_check["sap_width"].gt(0)
        & width_check["plm_width"].gt(0)
    ).sum()
)

Known SAP-PLM materials: 5645
With SAP numeric width: 124
With PLM numeric width: 2432
With width on both sides: 95


In [58]:
# Inspect raw WIDTH values for SAP materials with known PLM codes
known_width_raw = (
    fabric_materials[
        fabric_materials["plm_code"].notna()
    ][["material_id", "plm_code"]]
    .merge(
        width_values[
            ["material_id", "WIDTH", "sap_width", "WIDTHUOMNAME"]
        ],
        on="material_id",
        how="inner"
    )
)

print("Known materials with WIDTH:", len(known_width_raw))
print("Numeric WIDTH values:", known_width_raw["sap_width"].notna().sum())
print(
    "Non-numeric WIDTH values:",
    known_width_raw["sap_width"].isna().sum()
)

# Show the most common raw WIDTH values that could not be converted to numeric
display(
    known_width_raw[
        known_width_raw["sap_width"].isna()
    ]["WIDTH"]
    .value_counts()
    .head(30)
)

Known materials with WIDTH: 3407
Numeric WIDTH values: 3404
Non-numeric WIDTH values: 3


Series([], Name: count, dtype: int64)

In [59]:
# Check WIDTH data type
print(width_values["sap_width"].dtype)

# Rebuild the width comparison dataset
width_check = (
    fabric_materials[
        fabric_materials["plm_code"].notna()
    ][["material_id", "plm_code"]]
    .merge(
        width_values[["material_id", "sap_width"]],
        on="material_id",
        how="left"
    )
    .merge(
        plm_width[["plm_code", "plm_width"]],
        on="plm_code",
        how="left"
    )
)

print("Known SAP-PLM materials:", len(width_check))
print("With SAP numeric width:", width_check["sap_width"].notna().sum())
print("With positive SAP width:", width_check["sap_width"].gt(0).sum())
print("With positive PLM width:", width_check["plm_width"].gt(0).sum())

print(
    "With positive width on both sides:",
    (
        width_check["sap_width"].gt(0) &
        width_check["plm_width"].gt(0)
    ).sum()
)

float64
Known SAP-PLM materials: 5645
With SAP numeric width: 3404
With positive SAP width: 124
With positive PLM width: 2432
With positive width on both sides: 95


In [60]:
# Inspect the distribution of SAP width values for known PLM materials
known_sap_widths = width_check["sap_width"].dropna()

print("Numeric width records:", len(known_sap_widths))
print("Zero width records:", (known_sap_widths == 0).sum())
print("Positive width records:", (known_sap_widths > 0).sum())

display(
    known_sap_widths
    .value_counts()
    .head(20)
)

Numeric width records: 3404
Zero width records: 3280
Positive width records: 124


sap_width
0.0      3280
150.0      34
145.0      15
140.0      11
185.0      10
160.0       6
180.0       4
142.0       4
147.0       3
170.0       3
162.0       3
165.0       2
130.0       2
148.0       2
151.0       2
190.0       2
110.0       2
175.0       2
149.0       2
161.0       1
Name: count, dtype: int64

In [61]:
# Display summary statistics for SAP width values
display(
    known_sap_widths.describe()
)

count    3404.000000
mean        5.554935
std        28.801788
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       200.000000
Name: sap_width, dtype: float64

In [62]:
# Treat zero width values as missing
width_values["sap_width_clean"] = (
    width_values["sap_width"]
    .replace(0, np.nan)
)

## 5. Elyaf kompozisyonu (FIBERCONTENTLISTID) karşılaştırması


In [63]:
# Inspect fiber content records
fiber_content = fabric_attributes[
    fabric_attributes["Dahili krkt.no."] == "FIBERCONTENTLISTID"
].copy()

print("Materials with fiber content:",
      fiber_content["material_id"].nunique())

print("Total fiber content rows:",
      len(fiber_content))

display(fiber_content.head(30))

Materials with fiber content: 8940
Total fiber content rows: 13180


,Malzeme,Dahili krkt.no.,Sayaç,Karakteristik değeri,material_id
9,1020000012,FIBERCONTENTLISTID,2,100.00 COTTON,1020000012
22,1020000016,FIBERCONTENTLISTID,1,100.0 COTTON,1020000016
34,1020000019,FIBERCONTENTLISTID,1,100.0 COTTON,1020000019
43,1020000020,FIBERCONTENTLISTID,1,19.0 VISCOSE,1020000020
44,1020000020,FIBERCONTENTLISTID,2,3.0 ELASTHANE,1020000020
45,1020000020,FIBERCONTENTLISTID,3,6.0 METALLICFIBER,1020000020
46,1020000020,FIBERCONTENTLISTID,4,6.0 POLIAMID6,1020000020
47,1020000020,FIBERCONTENTLISTID,5,6.0 POLYAMIDE,1020000020
48,1020000020,FIBERCONTENTLISTID,6,66.0 POLYESTER,1020000020
69,1020000021,FIBERCONTENTLISTID,1,100.0 POLYESTER,1020000021


In [64]:
# Count fiber content records per material
fiber_count_per_material = (
    fiber_content
    .groupby("material_id")
    .size()
)

display(
    fiber_count_per_material
    .value_counts()
    .sort_index()
    .to_frame("material_count")
)
# Display the most common fiber content values
display(
    fiber_content["Karakteristik değeri"]
    .value_counts()
    .head(40)
)

,material_count
1,5892
2,2138
3,790
4,67
5,12
6,40
74,1


Karakteristik değeri
COTTON             2222
POLYESTER          1128
100.0 COTTON        965
ACRYLIC             749
ELASTHANE           463
2.0 ELASTHANE       306
1                   285
ACETAT              278
3.0 ELASTHANE       277
100.0 POLYESTER     239
VISCOSE             230
5.0 ELASTHANE       195
4.0 ELASTHANE       189
1.0 ELASTHANE       142
98.0 COTTON         124
70.0 COTTON         117
50.0 COTTON         109
97.0 COTTON         104
50.0 POLYESTER      100
30.0 POLYESTER       98
100.00 COTTON        89
35.0 POLYESTER       85
65.0 COTTON          82
40.0 POLYESTER       79
60.0 COTTON          78
95.0 COTTON          77
96.0 COTTON          70
6.0 ELASTHANE        58
100.0 VISCOSE        54
LACE                 52
99.0 COTTON          51
65.0 POLYESTER       48
7.0 ELASTHANE        48
20.0 POLYESTER       45
25.0 POLYESTER       44
80.0 COTTON          41
POLYVISCOSE          39
45.0 POLYESTER       37
93.0 COTTON          37
33.0 VISCOSE         35
Name: count, dtype:

In [65]:
# Define the PLM multi-value characteristic file path
plm_multi_value_char_path = (
    data_folder / "PLM_MULTI_VAL_CHAR.xlsx"
)

# Check sheet names
print(
    pd.ExcelFile(plm_multi_value_char_path).sheet_names
)

['Data']


In [66]:
plm_multi_value_char = pd.read_excel(
    plm_multi_value_char_path,
    sheet_name="Data"
)

print("Shape:", plm_multi_value_char.shape)
print("\nColumns:")
print(plm_multi_value_char.columns.tolist())

display(plm_multi_value_char.head(20))

Shape: (210661, 4)

Columns:
['plm_code', 'version', 'PLM Karakteristik Tanımı', 'PLM Karakteristik Değeri']


,plm_code,version,PLM Karakteristik Tanımı,PLM Karakteristik Değeri
0,test-mt,4,PHYSICALFINISHFABRICSIDEID,AMMONIAFINISHING
1,SH212120B,4,FIBERCONTENTLISTID,40.00 VISCOSE
2,SH212120B,4,FIBERCONTENTLISTID,36.00 COTTON
3,SH212120B,4,FIBERCONTENTLISTID,22.00 POLIESTER
4,SH212120B,4,FIBERCONTENTLISTID,2.00 ELASTHANE
5,SH212120B,4,COLORINGID,YARNDYED
6,"Dobby, Woven-Others, 75 D X 75",311,KNITTYPEID,LCWKNITTYPE11
7,"Dobby, Woven-Others, 75 D X 75",311,FIBERCONTENTLISTID,92.00 VISCOSE
8,"Dobby, Woven-Others, 75 D X 75",311,FIBERCONTENTLISTID,8.00 POLIESTER
9,"Dobby, Woven-Others, 75 D X 75",311,COLORINGID,ALLOVERPRINT


In [67]:
#Normalize PLM codes
plm_multi_value_char["plm_code"] = (
    plm_multi_value_char["plm_code"]
    .astype(str)
    .str.strip()
    .str.upper()
)

#keep only plm codes that exist in the plm master

valid_plm_codes = set(
    plm_codes["plm_code"].dropna()
)

plm_multi_value_valid = (
    plm_multi_value_char[
        plm_multi_value_char["plm_code"].isin(valid_plm_codes)
    ].copy()
)
print("Total rows:", len(plm_multi_value_char))
print("Valid PLM characteristic rows:", len(plm_multi_value_valid))
print(
    "Unique valid PLM codes:",
    plm_multi_value_valid["plm_code"].nunique()
)

Total rows: 210661
Valid PLM characteristic rows: 210451
Unique valid PLM codes: 31953


In [68]:
# Inspect available multi-value PLM charachteristics
plm_characteristic_summary = (
    plm_multi_value_valid
    .groupby("PLM Karakteristik Tanımı")
    .agg(
        plm_count = ("plm_code", "nunique"),
        row_count = ("plm_code", "size"),
        unique_value_count = ("PLM Karakteristik Değeri", "nunique")
    ).sort_values("plm_count", ascending=False)
)

display(plm_characteristic_summary.head(30))

# Keep PLM fiber content records
plm_fiber_content = (
    plm_multi_value_valid[
        plm_multi_value_valid["PLM Karakteristik Tanımı"]
        == "FIBERCONTENTLISTID"
    ].copy()
)

print(
    "PLM codes with fiber content:",
    plm_fiber_content["plm_code"].nunique()
)

print(
    "Total fiber content rows:",
    len(plm_fiber_content)
)

display(plm_fiber_content.head(20))

,plm_count,row_count,unique_value_count
PLM Karakteristik Tanımı,,,
FIBERCONTENTLISTID,31834,66612,1982
COLORINGID,24233,26278,42
PHYSICALFINISHFABRICSIDEID,13571,19571,87
YARNCOMPOSITION1LISTID,11799,17698,637
YARNCOMPOSITION2LISTID,8195,10475,310
DYETYPEID,7696,8161,35
WARPCOMPOSITION1LISTID,6108,7860,504
WEFTCOMPOSITION1LISTID,6029,8828,523
YARN1TYPEKNITSID,5591,5591,24


PLM codes with fiber content: 31834
Total fiber content rows: 66612


,plm_code,version,PLM Karakteristik Tanımı,PLM Karakteristik Değeri
1,SH212120B,4,FIBERCONTENTLISTID,40.00 VISCOSE
2,SH212120B,4,FIBERCONTENTLISTID,36.00 COTTON
3,SH212120B,4,FIBERCONTENTLISTID,22.00 POLIESTER
4,SH212120B,4,FIBERCONTENTLISTID,2.00 ELASTHANE
7,"DOBBY, WOVEN-OTHERS, 75 D X 75",311,FIBERCONTENTLISTID,92.00 VISCOSE
8,"DOBBY, WOVEN-OTHERS, 75 D X 75",311,FIBERCONTENTLISTID,8.00 POLIESTER
27,ARGE108919,3,FIBERCONTENTLISTID,7.00 ELASTANE/SPANDEX
28,ARGE108919,3,FIBERCONTENTLISTID,51.00 VISCOSE
29,ARGE108919,3,FIBERCONTENTLISTID,42.00 NYLON
30,ARGE108916,9,FIBERCONTENTLISTID,56.00 COTTON


In [69]:
# Check whether PLM codes have multiple versions
version_summary = ( #version summary
    plm_multi_value_valid
    .groupby("plm_code")["version"] #plm_code değerine göre grupluyor. sonra her plm'in version değerine bakıyor
    .nunique() #benzersiz versiyon sayısını buluyor.
    .value_counts() #kaç plm kodunda kaç verisyon var bunu hesaplıyor, bunu koymazsak plm kodu bazında benzersiz versiyon sayısı döner
    .sort_index() #sonucu index'e göre küçükten büyüğe sıralıyor.
)

display(version_summary)

version
1    31953
Name: count, dtype: int64

In [70]:
# Check the version column
print("Version data type:", plm_multi_value_valid["version"].dtype)

display(version_summary)

# Show PLM codes that have more than one version
multi_version_plm = (
    plm_multi_value_valid
    .groupby("plm_code")["version"]
    .nunique()
    .sort_values(ascending=False)
)

display(
    multi_version_plm[
        multi_version_plm > 1
    ].head(20)
)

Version data type: int64


version
1    31953
Name: count, dtype: int64

Series([], Name: version, dtype: int64)

In [71]:
import re

#Normalize obvious fiber name variations
def normalize_fiber_name(fiber_name):
    if pd.isna(fiber_name):
        return pd.NA
    fiber_name = str(fiber_name).strip().upper()

    fiber_name_mapping = {
        "POLIESTER": "POLYESTER",
        "ELASTHANE": "ELASTANE",
        "ELASTANE/SPANDEX": "ELASTANE",
        "SPANDEX": "ELASTANE",
        "POLIAMID6": "POLYAMIDE",
        "NYLON": "POLYAMIDE",
        "ACETAT": "ACETATE",
        "POLYAMIDE6": "POLYAMIDE",
        "TENCEL": "LYOCELL",
    }

    return fiber_name_mapping.get(fiber_name, fiber_name)

#Extract percentage and fiber name from a composition value
def parse_fiber_content(value):
    if pd.isna(value):
        return pd.Series([np.nan, pd.NA])
    value = str(value).strip().upper()
    match = re.match(r"^(\d+(?:\.\d+)?)\s+(.+)$", value)

    if match:
        percentage = float(match.group(1))
        fiber_name = normalize_fiber_name(match.group(2))
        return pd.Series([percentage, fiber_name])

    return pd.Series(
        [
            np.nan,
            normalize_fiber_name(value)
        ]
    )

# Parse SAP fiber content values
fiber_content[
    ["fiber_percentage", "fiber_name"]
] = fiber_content["Karakteristik değeri"].apply(
    parse_fiber_content
)

# Parse PLM fiber content values
plm_fiber_content[
    ["fiber_percentage", "fiber_name"]
] = plm_fiber_content[
    "PLM Karakteristik Değeri"
].apply(
    parse_fiber_content
)

# Check parsing quality
print(
    "SAP rows with percentage:",
    fiber_content["fiber_percentage"].notna().sum(),
    "/",
    len(fiber_content)
)

print(
    "PLM rows with percentage:",
    plm_fiber_content["fiber_percentage"].notna().sum(),
    "/",
    len(plm_fiber_content)
)

# Inspect the most common values without percentage information
print("SAP values without percentage:")
display(
    fiber_content[
        fiber_content["fiber_percentage"].isna()
    ]["Karakteristik değeri"]
    .value_counts()
    .head(30)
)

print("PLM values without percentage:")
display(
    plm_fiber_content[
        plm_fiber_content["fiber_percentage"].isna()
    ]["PLM Karakteristik Değeri"]
    .value_counts()
    .head(30)
)

# Calculate total composition percentage per SAP material
sap_fiber_totals = (
    fiber_content
    .groupby("material_id")["fiber_percentage"]
    .sum(min_count=1)
)

# Calculate total composition percentage per PLM code
plm_fiber_totals = (
    plm_fiber_content
    .groupby("plm_code")["fiber_percentage"]
    .sum(min_count=1)
)

print("SAP composition totals:")
display(sap_fiber_totals.describe())

print("\nPLM composition totals:")
display(plm_fiber_totals.describe())

SAP rows with percentage: 7336 / 13180
PLM rows with percentage: 66612 / 66612
SAP values without percentage:


Karakteristik değeri
COTTON           2222
POLYESTER        1128
ACRYLIC           749
ELASTHANE         463
1                 285
ACETAT            278
VISCOSE           230
LACE               52
POLYVISCOSE        39
POLYAMIDE          34
OTHER              30
MODAL              28
LINEN              19
POLYURETHANE       16
-                  15
VISKON             13
ANGORA             11
ALPACA             11
COTON              11
METALLICFIBER      10
111                10
TENCEL              9
BAMBOO              8
LYOCELL             8
RCYCLE PLSR         6
SÜPREM              6
VİSKON              6
LEATHER             6
SUPREM              6
PAMPES              5
Name: count, dtype: int64

PLM values without percentage:


Series([], Name: count, dtype: int64)

SAP composition totals:


count    3885.000000
mean       99.646332
std         5.955769
min         0.000000
25%       100.000000
50%       100.000000
75%       100.000000
max       200.000000
Name: fiber_percentage, dtype: float64


PLM composition totals:


count    31834.000000
mean       100.021518
std          2.806321
min         42.000000
25%        100.000000
50%        100.000000
75%        100.000000
max        536.500000
Name: fiber_percentage, dtype: float64

In [72]:
# Remove exact duplicate fiber records
sap_fiber_clean = (
    fiber_content[
        ["material_id", "fiber_name", "fiber_percentage"]
    ]
    .dropna(subset=["fiber_name"])
    .drop_duplicates()
    .copy()
)

plm_fiber_clean = (
    plm_fiber_content[
        ["plm_code", "fiber_name", "fiber_percentage"]
    ]
    .dropna(subset=["fiber_name"])
    .drop_duplicates()
    .copy()
)

In [73]:
# Create fiber sets for SAP materials
sap_fiber_sets = (
    sap_fiber_clean
    .groupby("material_id")["fiber_name"]
    .apply(set)
    .rename("sap_fiber_set")
    .reset_index()
)

# Create fiber sets for PLM codes
plm_fiber_sets = (
    plm_fiber_clean
    .groupby("plm_code")["fiber_name"]
    .apply(set)
    .rename("plm_fiber_set")
    .reset_index()
)

In [74]:
# Build known SAP-PLM pairs
known_fiber_pairs = (
    fabric_materials[
        fabric_materials["plm_code"].notna()
    ][["material_id", "plm_code"]]
    .drop_duplicates()
    .merge(
        sap_fiber_sets,
        on="material_id",
        how="left"
    )
    .merge(
        plm_fiber_sets,
        on="plm_code",
        how="left"
    )
)

In [75]:
# Calculate Jaccard similarity between two fiber sets
def calculate_jaccard_similarity(set_a, set_b):
    if not isinstance(set_a, set) or not isinstance(set_b, set):
        return np.nan

    union = set_a | set_b

    if len(union) == 0:
        return np.nan

    return len(set_a & set_b) / len(union)


known_fiber_pairs["fiber_set_similarity"] = (
    known_fiber_pairs.apply(
        lambda row: calculate_jaccard_similarity(
            row["sap_fiber_set"],
            row["plm_fiber_set"]
        ),
        axis=1
    )
)

In [76]:
print(
    "Comparable SAP-PLM pairs:",
    known_fiber_pairs["fiber_set_similarity"].notna().sum()
)

display(
    known_fiber_pairs[
        "fiber_set_similarity"
    ].describe()
)

display(
    known_fiber_pairs[
        "fiber_set_similarity"
    ].value_counts()
    .sort_index(ascending=False)
    .head(20)
)

Comparable SAP-PLM pairs: 3512


count    3512.000000
mean        0.992878
std         0.072121
min         0.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000000
Name: fiber_set_similarity, dtype: float64

fiber_set_similarity
1.000000    3473
0.833333       1
0.666667       4
0.571429       1
0.500000      16
0.333333       5
0.250000       1
0.000000      11
Name: count, dtype: int64

In [77]:
comparable_fiber_pairs = known_fiber_pairs[
    known_fiber_pairs["fiber_set_similarity"].notna()
]

exact_fiber_set_match_rate = (
    comparable_fiber_pairs["fiber_set_similarity"].eq(1).mean()
)

print(
    f"Exact fiber-set match rate: "
    f"{exact_fiber_set_match_rate:.2%}"
)

Exact fiber-set match rate: 98.89%


In [78]:
# Identify SAP materials with reliable percentage compositions
sap_percentage_quality = (
    fiber_content
    .groupby("material_id")
    .agg(
        total_rows=("fiber_name", "size"),
        parsed_rows=("fiber_percentage", "count"),
        composition_total=(
            "fiber_percentage",
            lambda x: x.sum(min_count=1)
        )
    )
)

sap_percentage_quality["is_reliable"] = (
    (sap_percentage_quality["total_rows"] ==
     sap_percentage_quality["parsed_rows"])
    &
    sap_percentage_quality["composition_total"].between(98, 102)
)

reliable_sap_materials = set(
    sap_percentage_quality[
        sap_percentage_quality["is_reliable"]
    ].index
)

print(
    "SAP materials with reliable percentage composition:",
    len(reliable_sap_materials)
)

SAP materials with reliable percentage composition: 3864


In [79]:
# Identify PLM codes with reliable percentage compositions
plm_percentage_quality = (
    plm_fiber_content
    .groupby("plm_code")["fiber_percentage"]
    .sum()
)

reliable_plm_codes = set(
    plm_percentage_quality[
        plm_percentage_quality.between(98, 102)
    ].index
)

print(
    "PLM codes with reliable percentage composition:",
    len(reliable_plm_codes)
)

PLM codes with reliable percentage composition: 31819


In [80]:
# Build SAP percentage profiles
sap_composition = (
    fiber_content[
        fiber_content["material_id"].isin(reliable_sap_materials)
    ]
    .groupby(
        ["material_id", "fiber_name"],
        as_index=False
    )["fiber_percentage"]
    .sum()
)

# Build PLM percentage profiles
plm_composition = (
    plm_fiber_content[
        plm_fiber_content["plm_code"].isin(reliable_plm_codes)
    ]
    .groupby(
        ["plm_code", "fiber_name"],
        as_index=False
    )["fiber_percentage"]
    .sum()
)

In [81]:
# Normalize compositions to 100%
sap_composition["fiber_percentage_normalized"] = (
    sap_composition["fiber_percentage"]
    /
    sap_composition.groupby("material_id")[
        "fiber_percentage"
    ].transform("sum")
    * 100
)

plm_composition["fiber_percentage_normalized"] = (
    plm_composition["fiber_percentage"]
    /
    plm_composition.groupby("plm_code")[
        "fiber_percentage"
    ].transform("sum")
    * 100
)

In [82]:
# Convert compositions to dictionaries
sap_composition_profiles = (
    sap_composition
    .groupby("material_id")
    .apply(
        lambda group: dict(
            zip(
                group["fiber_name"],
                group["fiber_percentage_normalized"]
            )
        ),
        include_groups=False
    )
    .rename("sap_composition")
    .reset_index()
)

plm_composition_profiles = (
    plm_composition
    .groupby("plm_code")
    .apply(
        lambda group: dict(
            zip(
                group["fiber_name"],
                group["fiber_percentage_normalized"]
            )
        ),
        include_groups=False
    )
    .rename("plm_composition")
    .reset_index()
)

In [83]:
# Calculate percentage-based composition similarity
def calculate_composition_similarity(sap_profile, plm_profile):
    if not isinstance(sap_profile, dict) or not isinstance(plm_profile, dict):
        return np.nan

    fibers = set(sap_profile) | set(plm_profile)

    total_difference = sum(
        abs(
            sap_profile.get(fiber, 0)
            - plm_profile.get(fiber, 0)
        )
        for fiber in fibers
    )

    return 1 - (total_difference / 200)

In [84]:
# Compare reliable compositions for known SAP-PLM mappings
known_composition_pairs = (
    fabric_materials[
        fabric_materials["plm_code"].notna()
    ][["material_id", "plm_code"]]
    .drop_duplicates()
    .merge(
        sap_composition_profiles,
        on="material_id",
        how="inner"
    )
    .merge(
        plm_composition_profiles,
        on="plm_code",
        how="inner"
    )
)

known_composition_pairs["fiber_composition_similarity"] = (
    known_composition_pairs.apply(
        lambda row: calculate_composition_similarity(
            row["sap_composition"],
            row["plm_composition"]
        ),
        axis=1
    )
)

print(
    "Comparable reliable composition pairs:",
    len(known_composition_pairs)
)

display(
    known_composition_pairs[
        "fiber_composition_similarity"
    ].describe()
)

print(
    "Exact composition match:",
    f"{known_composition_pairs['fiber_composition_similarity'].eq(1).mean():.2%}"
)

print(
    "Similarity >= 0.95:",
    f"{known_composition_pairs['fiber_composition_similarity'].ge(0.95).mean():.2%}"
)

print(
    "Similarity >= 0.90:",
    f"{known_composition_pairs['fiber_composition_similarity'].ge(0.90).mean():.2%}"
)

Comparable reliable composition pairs: 3465


count    3465.000000
mean        0.997562
std         0.033717
min         0.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000000
Name: fiber_composition_similarity, dtype: float64

Exact composition match: 98.30%
Similarity >= 0.95: 99.19%
Similarity >= 0.90: 99.42%


In [85]:
# Inspect composition mismatches
composition_outliers = (
    known_composition_pairs[
        known_composition_pairs["fiber_composition_similarity"] < 0.95
    ]
    .sort_values("fiber_composition_similarity")
    .copy()
)

print("Composition outliers:", len(composition_outliers))

display(
    composition_outliers[
        [
            "material_id",
            "plm_code",
            "sap_composition",
            "plm_composition",
            "fiber_composition_similarity"
        ]
    ].head(50)
)

Composition outliers: 28


,material_id,plm_code,sap_composition,plm_composition,fiber_composition_similarity
250,1020000417,190539,"{'COTTON': 50.0, 'POLYESTER': 35.0, 'VISCOSE':...","{'ELASTANE': 9.0, 'POLYAMIDE': 27.0, 'RAYON': ...",0.00
2332,1020017811,356268,"{'POLYAMIDE': 14.000000000000002, 'VISCOSE': 8...","{'MODAL': 85.0, 'POLYESTER': 15.0}",0.00
349,1020000641,214665,"{'ACRYLIC': 80.0, 'COTTON': 5.0, 'POLYAMIDE': ...","{'ACRYLIC': 5.0, 'COTTON': 22.0, 'POLYAMIDE': ...",0.25
782,1020010396,148745,"{'ELASTANE': 5.0, 'POLYESTER': 79.0, 'VISCOSE'...","{'ELASTANE': 4.0, 'POLYESTER': 27.0, 'VISCOSE'...",0.47
6,1020000026,102617,"{'COTTON': 55.00000000000001, 'POLYESTER': 45.0}",{'COTTON': 100.0},0.55
939,1020011221,358967,"{'ACRYLIC': 4.0, 'COTTON': 37.0, 'POLYAMIDE': ...","{'ACRYLIC': 9.0, 'OTHER': 5.0, 'POLYAMIDE': 6....",0.62
3144,1020021161,370680,"{'COTTON': 93.0, 'ELASTANE': 7.000000000000001}","{'COTTON': 62.0, 'ELASTANE': 1.0, 'POLYESTER':...",0.63
1533,1020014656,198620,"{'ELASTANE': 2.0, 'POLYESTER': 98.0}","{'COTTON': 34.0, 'ELASTANE': 1.0, 'POLYESTER':...",0.66
1417,1020014197,353369,{'COTTON': 100.0},"{'COTTON': 70.0, 'POLYESTER': 30.0}",0.70
3341,1020021706,367162,"{'COTTON': 75.0, 'POLYESTER': 25.0}","{'COTTON': 51.0, 'POLYESTER': 49.0}",0.76


In [86]:
# Add SAP material descriptions for error analysis
composition_outliers = (
    composition_outliers
    .merge(
        fabric_materials[
            [
                "material_id",
                "Türkçe malzeme açıklaması",
                "Türkçe malzeme Uzun açıklaması"
            ]
        ].drop_duplicates("material_id"),
        on="material_id",
        how="left"
    )
)

display(
    composition_outliers[
        [
            "material_id",
            "plm_code",
            "Türkçe malzeme açıklaması",
            "sap_composition",
            "plm_composition",
            "fiber_composition_similarity"
        ]
    ].head(50)
)

,material_id,plm_code,Türkçe malzeme açıklaması,sap_composition,plm_composition,fiber_composition_similarity
0,1020000417,190539,30/1 PENYE KOMPAKT REPORTED IN,"{'COTTON': 50.0, 'POLYESTER': 35.0, 'VISCOSE':...","{'ELASTANE': 9.0, 'POLYAMIDE': 27.0, 'RAYON': ...",0.00
1,1020017811,356268,"Dokuma-Diğer,90000.0,30/1,14.0 Poliamid,","{'POLYAMIDE': 14.000000000000002, 'VISCOSE': 8...","{'MODAL': 85.0, 'POLYESTER': 15.0}",0.00
2,1020000641,214665,FLANEL 280 / 250,"{'ACRYLIC': 80.0, 'COTTON': 5.0, 'POLYAMIDE': ...","{'ACRYLIC': 5.0, 'COTTON': 22.0, 'POLYAMIDE': ...",0.25
3,1020010396,148745,36/1 70D POLY DOUBLEFACE İNTERLOK,"{'ELASTANE': 5.0, 'POLYESTER': 79.0, 'VISCOSE'...","{'ELASTANE': 4.0, 'POLYESTER': 27.0, 'VISCOSE'...",0.47
4,1020000026,102617,30/1 INT IPLK BOYA IP,"{'COTTON': 55.00000000000001, 'POLYESTER': 45.0}",{'COTTON': 100.0},0.55
5,1020011221,358967,275.0,"{'ACRYLIC': 4.0, 'COTTON': 37.0, 'POLYAMIDE': ...","{'ACRYLIC': 9.0, 'OTHER': 5.0, 'POLYAMIDE': 6....",0.62
6,1020021161,370680,"30/1,7.0 Elastan,93.0 Pamuk,SUP,DUZ BYA","{'COTTON': 93.0, 'ELASTANE': 7.000000000000001}","{'COTTON': 62.0, 'ELASTANE': 1.0, 'POLYESTER':...",0.63
7,1020014656,198620,RAŞEL KRINKIL,"{'ELASTANE': 2.0, 'POLYESTER': 98.0}","{'COTTON': 34.0, 'ELASTANE': 1.0, 'POLYESTER':...",0.66
8,1020014197,353369,"100.0 Pamuk,WAF",{'COTTON': 100.0},"{'COTTON': 70.0, 'POLYESTER': 30.0}",0.70
9,1020021706,367162,"Bez ayağı,125000.0,40,25.0 POLYESTER,75.","{'COTTON': 75.0, 'POLYESTER': 25.0}","{'COTTON': 51.0, 'POLYESTER': 49.0}",0.76


Yüksek fiber similarity, aynı malzeme olduğuna dair çok güçlü pozitif kanıttır. Düşük similarity ise tek başına farklı malzeme anlamına gelmez; temporal attribute drift, yanlış mapping veya master-data problemi olabilir.

## 6. Mal grubu bazlı zorunlu alan politikası


In [87]:
# Define the material-group characteristic rules file
mandatory_fields_path = (
    data_folder / "Fabric_Mandatory_Fields.xlsx"
)

# Load material-group characteristic rules
mandatory_fields = pd.read_excel(
    mandatory_fields_path,
    sheet_name="Data"
)

print("Shape:", mandatory_fields.shape)
display(mandatory_fields.head(20))

Shape: (82, 5)


,Mal grubu,Tanım Tipi (Kısa/Uzun),Özellik,Field name,Sıra
0,1010001,K,Kısa Tanım,WEAVETYPE,1
1,1010001,K,Kısa Tanım,FABRICSTRUCTURETNAME,2
2,1010001,K,Kısa Tanım,YARNCOUNT1KNITSID,3
3,1010001,K,Kısa Tanım,WEIGHT,4
4,1010001,K,Kısa Tanım,WEIGHTUOMNAME,5
5,1010001,K,Kısa Tanım,FIBERCONTENTLISTID,6
6,1010001,K,Kısa Tanım,COLORINGID,7
7,1010001,Z,Zorunluluk,COLORINGID,1
8,1010001,Z,Zorunluluk,FABRICSTRUCTURETNAME,1
9,1010001,Z,Zorunluluk,FIBERCONTENTLISTID,1


In [88]:
# Rename columns for easier analysis
mandatory_fields = mandatory_fields.rename(
    columns={
        "Mal grubu": "material_group",
        "Tanım Tipi (Kısa/Uzun)": "rule_type",
        "Özellik": "rule_description",
        "Field name": "field_name",
        "Sıra": "sequence"
    }
)

mandatory_fields["material_group"] = (
    mandatory_fields["material_group"]
    .astype(str)
    .str.strip()
)

mandatory_fields["field_name"] = (
    mandatory_fields["field_name"]
    .astype(str)
    .str.strip()
    .str.upper()
)

mandatory_fields["sequence"] = pd.to_numeric(
    mandatory_fields["sequence"],
    errors="coerce"
)

In [89]:
# Mandatory characteristics
mandatory_policy = (
    mandatory_fields[
        mandatory_fields["rule_type"] == "Z"
    ][["material_group", "field_name"]]
    .drop_duplicates()
    .assign(is_mandatory=True)
)

# Characteristics used in short descriptions
short_description_policy = (
    mandatory_fields[
        mandatory_fields["rule_type"] == "K"
    ][
        [
            "material_group",
            "field_name",
            "sequence"
        ]
    ]
    .rename(
        columns={
            "sequence": "short_description_order"
        }
    )
    .drop_duplicates()
    .assign(in_short_description=True)
)

# Characteristics used in long descriptions
long_description_policy = (
    mandatory_fields[
        mandatory_fields["rule_type"] == "U"
    ][
        [
            "material_group",
            "field_name",
            "sequence"
        ]
    ]
    .rename(
        columns={
            "sequence": "long_description_order"
        }
    )
    .drop_duplicates()
    .assign(in_long_description=True)
)

In [90]:
# Create one row per material group and characteristic
feature_policy = (
    mandatory_fields[
        ["material_group", "field_name"]
    ]
    .drop_duplicates()
    .merge(
        mandatory_policy,
        on=["material_group", "field_name"],
        how="left"
    )
    .merge(
        short_description_policy,
        on=["material_group", "field_name"],
        how="left"
    )
    .merge(
        long_description_policy,
        on=["material_group", "field_name"],
        how="left"
    )
)

boolean_columns = [
    "is_mandatory",
    "in_short_description",
    "in_long_description"
]

feature_policy[boolean_columns] = (
    feature_policy[boolean_columns]
    .fillna(False)
    .astype(bool)
)

feature_policy = feature_policy.sort_values(
    [
        "material_group",
        "is_mandatory",
        "short_description_order",
        "long_description_order"
    ],
    ascending=[True, False, True, True]
)

display(feature_policy)

C:\Users\EMRE.YILMAZ\AppData\Local\Temp\ipykernel_31404\3326567858.py:32: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


,material_group,field_name,is_mandatory,short_description_order,in_short_description,long_description_order,in_long_description
0,1010001,WEAVETYPE,True,1.0,True,NaN,False
1,1010001,FABRICSTRUCTURETNAME,True,2.0,True,NaN,False
2,1010001,YARNCOUNT1KNITSID,True,3.0,True,NaN,False
3,1010001,WEIGHT,True,4.0,True,NaN,False
4,1010001,WEIGHTUOMNAME,True,5.0,True,NaN,False
5,1010001,FIBERCONTENTLISTID,True,6.0,True,NaN,False
6,1010001,COLORINGID,True,7.0,True,NaN,False
7,1020001,YARNCOUNT1KNITSID,True,1.0,True,1.0,True
8,1020001,FIBERCONTENTLISTID,True,3.0,True,NaN,False
10,1020001,FABRICSTRUCTURETNAME,True,4.0,True,5.0,True


In [91]:
# Normalize material group in the SAP material dataset
fabric_materials["material_group"] = (
    fabric_materials["Mal grubu"]
    .astype(str)
    .str.strip()
)

policy_material_groups = set(
    feature_policy["material_group"]
)

fabric_materials["has_feature_policy"] = (
    fabric_materials["material_group"]
    .isin(policy_material_groups)
)

print(
    "Materials covered by feature policy:",
    fabric_materials["has_feature_policy"].sum(),
    "/",
    len(fabric_materials)
)

display(
    fabric_materials[
        "material_group"
    ].value_counts()
)

Materials covered by feature policy: 0 / 21083


material_group
ORME      9547
DOKUMA    8747
DENIM     2789
Name: count, dtype: int64

### 6.1 Mal grubu kodu eşlemesi

> **Not:** 92–105 arası hücreler aynı eşlemenin birkaç denemesidir.
> Geçerli olan son hâldir (`material_group_code_mapping`, ORME / DOKUMA /
> DENIM). `CEPLIK/ASTAR` (1030003) bilinçli olarak kapsam dışıdır.


In [92]:
# Map PLM material-group codes to readable names
plm_material_group_mapping = {
    "1020001": "ORME KUMAS",
    "1020002": "DOKUMA KUMAS",
    "1030003": "CEPLIK/ASTAR",
    "1030004": "DENIM"
}

In [93]:
# Normalize material-group codes
feature_policy["material_group"] = (
    pd.to_numeric(
        feature_policy["material_group"],
        errors="coerce"
    )
    .astype("Int64")
    .astype("string")
)

# Add readable material-group names
feature_policy["material_group_name"] = (
    feature_policy["material_group"]
    .map(plm_material_group_mapping)
)

In [94]:
# Normalize PLM material-group codes
plm_codes["material_group_code"] = (
    pd.to_numeric(
        plm_codes["Mal grubu"],
        errors="coerce"
    )
    .astype("Int64")
    .astype("string")
)

plm_codes["material_group_name"] = (
    plm_codes["material_group_code"]
    .map(plm_material_group_mapping)
)

In [95]:
# Create broader material families for matching
material_family_mapping = {
    "ORME KUMAS": "ORME",
    "DOKUMA KUMAS": "DOKUMA",
    "CEPLIK/ASTAR": "DOKUMA",
    "DENIM": "DENIM"
}

feature_policy["material_family"] = (
    feature_policy["material_group_name"]
    .map(material_family_mapping)
)

plm_codes["material_family"] = (
    plm_codes["material_group_name"]
    .map(material_family_mapping)
)

In [96]:
# Normalize SAP material families
fabric_materials["material_family"] = (
    fabric_materials["Mal grubu"]
    .astype(str)
    .str.strip()
    .str.upper()
)

In [97]:
# Add the actual PLM material group to known SAP-PLM mappings
known_material_groups = (
    fabric_materials[
        fabric_materials["plm_code"].notna()
    ][
        [
            "material_id",
            "plm_code",
            "material_family"
        ]
    ]
    .drop_duplicates()
    .merge(
        plm_codes[
            [
                "plm_code",
                "material_group_code",
                "material_group_name"
            ]
        ].drop_duplicates("plm_code"),
        on="plm_code",
        how="left"
    )
)

display(
    known_material_groups[
        "material_group_code"
    ].value_counts(dropna=False)
)

material_group_code
<NA>    5645
Name: count, dtype: Int64

In [98]:
# Normalize SAP characteristic names
fabric_attributes["field_name"] = (
    fabric_attributes["Dahili krkt.no."]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Keep characteristics that actually have a value
sap_attribute_presence = (
    fabric_attributes[
        fabric_attributes["Karakteristik değeri"].notna()
        & (
            fabric_attributes["Karakteristik değeri"]
            .astype(str)
            .str.strip()
            .ne("")
        )
    ][
        ["material_id", "field_name"]
    ]
    .drop_duplicates()
    .assign(is_present=True)
)

In [99]:
# Extract mandatory characteristics by PLM material group
mandatory_features = (
    feature_policy[
        feature_policy["is_mandatory"]
    ][
        [
            "material_group",
            "field_name"
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "material_group": "material_group_code"
        }
    )
)

In [100]:
# Generate expected mandatory characteristics for each known SAP material
mandatory_coverage_detail = (
    known_material_groups
    .merge(
        mandatory_features,
        on="material_group_code",
        how="inner"
    )
    .merge(
        sap_attribute_presence,
        on=["material_id", "field_name"],
        how="left"
    )
)

mandatory_coverage_detail["is_present"] = (
    mandatory_coverage_detail["is_present"]
    .fillna(False)
    .astype(bool)
)

In [101]:
# Calculate mandatory-field coverage by PLM material group
mandatory_coverage_summary = (
    mandatory_coverage_detail
    .groupby(
        [
            "material_group_code",
            "material_group_name",
            "field_name"
        ],
        dropna=False
    )
    .agg(
        material_count=("material_id", "nunique"),
        present_count=("is_present", "sum")
    )
    .reset_index()
)

mandatory_coverage_summary["coverage_rate"] = (
    mandatory_coverage_summary["present_count"]
    / mandatory_coverage_summary["material_count"]
)

display(
    mandatory_coverage_summary
    .sort_values(
        ["material_group_code", "coverage_rate"],
        ascending=[True, True]
    )
)

,material_group_code,material_group_name,field_name,material_count,present_count,coverage_rate


In [102]:
# Calculate overall mandatory completeness per material group
material_level_completeness = (
    mandatory_coverage_detail
    .groupby(
        [
            "material_id",
            "plm_code",
            "material_group_code",
            "material_group_name"
        ],
        dropna=False
    )
    .agg(
        mandatory_field_count=("field_name", "nunique"),
        mandatory_field_present=("is_present", "sum")
    )
    .reset_index()
)

material_level_completeness["mandatory_completeness"] = (
    material_level_completeness["mandatory_field_present"]
    / material_level_completeness["mandatory_field_count"]
)

display(
    material_level_completeness
    .groupby(
        ["material_group_code", "material_group_name"]
    )["mandatory_completeness"]
    .agg(
        ["count", "mean", "median", "min", "max"]
    )
)

,,count,mean,median,min,max
material_group_code,material_group_name,,,,,


In [103]:
# Inspect raw PLM material-group values
display(
    plm_codes["Mal grubu"]
    .value_counts(dropna=False)
    .head(20)
)

Mal grubu
ORME        12812
DOKUMA      12087
DENIM        3558
TRIKO        2193
LEATHER1     1059
MATERIAL      399
YARN          138
LEATHER2       56
LEATHER        50
AYAKKABI        4
TEST            4
NaN             1
Name: count, dtype: int64

In [104]:
# Map PLM material-group names to policy codes
material_group_code_mapping = {
    "Örme Kumaş": "1020001",
    "Dokuma Kumaş": "1020002",
    "Ceplik/Astar": "1030003",
    "Denim": "1030004"
}

plm_codes["material_group_name"] = (
    plm_codes["Mal grubu"]
    .astype("string")
    .str.strip()
)

plm_codes["material_group_code"] = (
    plm_codes["material_group_name"]
    .map(material_group_code_mapping)
    .astype("string")
)

In [105]:
display(
    plm_codes[
        [
            "material_group_name",
            "material_group_code"
        ]
    ]
    .value_counts()
)

Series([], Name: count, dtype: int64)

In [106]:
# Map current fabric material groups to policy codes
material_group_code_mapping = {
    "ORME": "1020001",
    "DOKUMA": "1020002",
    "DENIM": "1030004"
}

plm_codes["material_group_name"] = (
    plm_codes["Mal grubu"]
    .astype("string")
    .str.strip()
    .str.upper()
)

plm_codes["material_group_code"] = (
    plm_codes["material_group_name"]
    .map(material_group_code_mapping)
    .astype("string")
)

In [107]:
display(
    plm_codes[
        ["material_group_name", "material_group_code"]
    ]
    .value_counts(dropna=False)
)

material_group_name  material_group_code
ORME                 1020001                12812
DOKUMA               1020002                12087
DENIM                1030004                 3558
TRIKO                <NA>                    2193
LEATHER1             <NA>                    1059
MATERIAL             <NA>                     399
YARN                 <NA>                     138
LEATHER2             <NA>                      56
LEATHER              <NA>                      50
AYAKKABI             <NA>                       4
TEST                 <NA>                       4
<NA>                 <NA>                       1
Name: count, dtype: int64

In [108]:
# Add actual PLM material group to known SAP-PLM mappings
known_material_groups = (
    fabric_materials[
        fabric_materials["plm_code"].notna()
    ][
        [
            "material_id",
            "plm_code",
            "material_family"
        ]
    ]
    .drop_duplicates()
    .merge(
        plm_codes[
            [
                "plm_code",
                "material_group_code",
                "material_group_name"
            ]
        ].drop_duplicates("plm_code"),
        on="plm_code",
        how="left"
    )
)

display(
    known_material_groups[
        "material_group_code"
    ].value_counts(dropna=False)
)

material_group_code
1020001    2956
1020002    2399
1030004     280
<NA>         10
Name: count, dtype: Int64

In [109]:
# Inspect known mappings without a material-group policy
unmapped_known_materials = (
    known_material_groups[
        known_material_groups["material_group_code"].isna()
    ]
    .copy()
)

display(unmapped_known_materials)

,material_id,plm_code,material_family,material_group_code,material_group_name
636,1020000649,215007,ORME,<NA>,LEATHER1
763,1020000777,30103,DOKUMA,<NA>,LEATHER1
829,1020000843,352006,ORME,<NA>,TRIKO
1084,1020001098,43466,DOKUMA,<NA>,LEATHER1
1968,1020008269,43466,DOKUMA,<NA>,LEATHER1
2118,1020008482,352006,ORME,<NA>,TRIKO
4050,1020015962,364528,ORME,<NA>,LEATHER1
4666,1020018335,353101,ORME,<NA>,LEATHER1
5328,1020021166,128297,ORME,<NA>,LEATHER1
5330,1020021168,125884,ORME,<NA>,LEATHER1


In [110]:
# Standardize policy material-group codes
feature_policy["material_group"] = (
    feature_policy["material_group"]
    .astype("string")
    .str.strip()
)

# Keep mandatory characteristics only
mandatory_features = (
    feature_policy[
        feature_policy["is_mandatory"]
    ][
        ["material_group", "field_name"]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "material_group": "material_group_code"
        }
    )
)

display(
    mandatory_features[
        "material_group_code"
    ].value_counts()
)

material_group_code
1020001    8
1030004    8
1020002    8
1010001    7
1030003    7
Name: count, dtype: Int64

In [111]:
# Standardize SAP characteristic names
fabric_attributes["field_name"] = (
    fabric_attributes["Dahili krkt.no."]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Identify characteristics with an actual value
sap_attribute_presence = (
    fabric_attributes[
        fabric_attributes["Karakteristik değeri"].notna()
        & fabric_attributes["Karakteristik değeri"]
        .astype("string")
        .str.strip()
        .ne("")
    ][
        ["material_id", "field_name"]
    ]
    .drop_duplicates()
    .assign(is_present=True)
)

In [112]:
# Generate expected mandatory characteristics for known mappings
mandatory_coverage_detail = (
    known_material_groups[
        known_material_groups["material_group_code"].notna()
    ]
    .merge(
        mandatory_features,
        on="material_group_code",
        how="inner"
    )
    .merge(
        sap_attribute_presence,
        on=["material_id", "field_name"],
        how="left"
    )
)

mandatory_coverage_detail["is_present"] = (
    mandatory_coverage_detail["is_present"]
    .fillna(False)
    .astype(bool)
)

print(
    "Materials included:",
    mandatory_coverage_detail["material_id"].nunique()
)

print(
    "Mandatory fields evaluated:",
    mandatory_coverage_detail["field_name"].nunique()
)

Materials included: 5635
Mandatory fields evaluated: 8


C:\Users\EMRE.YILMAZ\AppData\Local\Temp\ipykernel_31404\3361982815.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


In [113]:
# Calculate mandatory-field coverage
mandatory_coverage_summary = (
    mandatory_coverage_detail
    .groupby(
        [
            "material_group_code",
            "material_group_name",
            "field_name"
        ]
    )
    .agg(
        material_count=("material_id", "nunique"),
        present_count=("is_present", "sum")
    )
    .reset_index()
)

mandatory_coverage_summary["coverage_rate"] = (
    mandatory_coverage_summary["present_count"]
    / mandatory_coverage_summary["material_count"]
)

display(
    mandatory_coverage_summary
    .sort_values(
        ["material_group_code", "coverage_rate"],
        ascending=[True, False]
    )
)

,material_group_code,material_group_name,field_name,material_count,present_count,coverage_rate
1,1020001,ORME,FABRICSTRUCTURETNAME,2956,1829,0.618742
2,1020001,ORME,FIBERCONTENTLISTID,2956,1829,0.618742
4,1020001,ORME,WEIGHT,2956,1829,0.618742
6,1020001,ORME,WIDTH,2956,1817,0.614682
5,1020001,ORME,WEIGHTUOMNAME,2956,1768,0.598106
0,1020001,ORME,COLORINGID,2956,1605,0.542963
7,1020001,ORME,YARNCOUNT1KNITSID,2956,1009,0.341340
3,1020001,ORME,WEAVETYPE,2956,2,0.000677
12,1020002,DOKUMA,WEIGHT,2399,1419,0.591496
9,1020002,DOKUMA,FABRICSTRUCTURETNAME,2399,1418,0.591080


In [114]:
# Calculate mandatory completeness per material
material_level_completeness = (
    mandatory_coverage_detail
    .groupby(
        [
            "material_id",
            "plm_code",
            "material_group_code",
            "material_group_name"
        ]
    )
    .agg(
        mandatory_field_count=("field_name", "nunique"),
        mandatory_field_present=("is_present", "sum")
    )
    .reset_index()
)

material_level_completeness["mandatory_completeness"] = (
    material_level_completeness["mandatory_field_present"]
    / material_level_completeness["mandatory_field_count"]
)

display(
    material_level_completeness
    .groupby(
        ["material_group_code", "material_group_name"]
    )["mandatory_completeness"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        min="min",
        max="max"
    )
)

,,count,mean,median,min,max
material_group_code,material_group_name,,,,,
1020001,ORME,2956,0.494249,0.75,0.0,0.875
1020002,DOKUMA,2399,0.432889,0.75,0.0,0.875
1030004,DENIM,280,0.730357,0.75,0.0,0.875


In [115]:
# Identify materials that have at least one SAP attribute
materials_with_attributes = set(
    fabric_attributes["material_id"].dropna()
)

mandatory_coverage_detail["has_any_attribute"] = (
    mandatory_coverage_detail["material_id"]
    .isin(materials_with_attributes)
)

# Avoid the fillna downcasting warning
mandatory_coverage_detail["is_present"] = (
    mandatory_coverage_detail["is_present"].eq(True)
)
# Calculate both absolute and conditional mandatory-field coverage
mandatory_coverage_audit = (
    mandatory_coverage_detail
    .groupby(
        [
            "material_group_code",
            "material_group_name",
            "field_name"
        ]
    )
    .agg(
        total_materials=("material_id", "nunique"),
        materials_with_attributes=(
            "has_any_attribute",
            "sum"
        ),
        present_count=("is_present", "sum")
    )
    .reset_index()
)

# Coverage among all known SAP-PLM mappings
mandatory_coverage_audit["absolute_coverage"] = (
    mandatory_coverage_audit["present_count"]
    / mandatory_coverage_audit["total_materials"]
)

# Coverage only where SAP attribute data exists
mandatory_coverage_audit["conditional_coverage"] = (
    mandatory_coverage_audit["present_count"]
    / mandatory_coverage_audit["materials_with_attributes"]
)

display(
    mandatory_coverage_audit.sort_values(
        [
            "material_group_code",
            "conditional_coverage"
        ],
        ascending=[True, False]
    )
)

,material_group_code,material_group_name,field_name,total_materials,materials_with_attributes,present_count,absolute_coverage,conditional_coverage
1,1020001,ORME,FABRICSTRUCTURETNAME,2956,1831,1829,0.618742,0.998908
2,1020001,ORME,FIBERCONTENTLISTID,2956,1831,1829,0.618742,0.998908
4,1020001,ORME,WEIGHT,2956,1831,1829,0.618742,0.998908
6,1020001,ORME,WIDTH,2956,1831,1817,0.614682,0.992354
5,1020001,ORME,WEIGHTUOMNAME,2956,1831,1768,0.598106,0.965593
0,1020001,ORME,COLORINGID,2956,1831,1605,0.542963,0.876570
7,1020001,ORME,YARNCOUNT1KNITSID,2956,1831,1009,0.341340,0.551065
3,1020001,ORME,WEAVETYPE,2956,1831,2,0.000677,0.001092
12,1020002,DOKUMA,WEIGHT,2399,1420,1419,0.591496,0.999296
9,1020002,DOKUMA,FABRICSTRUCTURETNAME,2399,1420,1418,0.591080,0.998592


### 6.2 İş birimi onaylı politika düzeltmeleri

> **Not:** 113–116 arası zorunlu alan kapsamı hesabı, politika
> düzeltmelerinden sonra 121–123'te yeniden üretilir. Geçerli sonuçlar
> 121–123'tekilerdir.


In [116]:
# Define business-validated policy corrections
policy_overrides = pd.DataFrame(
    [
        {
            "material_group": "1020001",
            "field_name": "WEAVETYPE",
            "is_mandatory_override": False,
            "reason": "WEAVETYPE is not applicable to knitted fabrics."
        },
        {
            "material_group": "1020002",
            "field_name": "YARNCOUNT1KNITSID",
            "is_mandatory_override": False,
            "reason": "Knitting yarn count is not applicable to woven fabrics."
        },
        {
            "material_group": "1030004",
            "field_name": "YARNCOUNT1KNITSID",
            "is_mandatory_override": False,
            "reason": "Knitting yarn count is not applicable to denim fabrics."
        }
    ]
)

display(policy_overrides)

,material_group,field_name,is_mandatory_override,reason
0,1020001,WEAVETYPE,False,WEAVETYPE is not applicable to knitted fabrics.
1,1020002,YARNCOUNT1KNITSID,False,Knitting yarn count is not applicable to woven...
2,1030004,YARNCOUNT1KNITSID,False,Knitting yarn count is not applicable to denim...


In [117]:
# Apply business-validated mandatory-field overrides
feature_policy = feature_policy.merge(
    policy_overrides,
    on=["material_group", "field_name"],
    how="left"
)

feature_policy["is_mandatory_original"] = (
    feature_policy["is_mandatory"]
)

feature_policy["is_mandatory"] = np.where(
    feature_policy["is_mandatory_override"].notna(),
    feature_policy["is_mandatory_override"],
    feature_policy["is_mandatory"]
)

feature_policy["is_mandatory"] = (
    feature_policy["is_mandatory"]
    .astype(bool)
)

In [118]:
# Verify corrected mandatory-field policies
display(
    feature_policy[
        feature_policy["material_group"].isin(
            ["1020001", "1020002", "1030004"]
        )
    ][
        [
            "material_group",
            "field_name",
            "is_mandatory_original",
            "is_mandatory",
            "reason"
        ]
    ]
    .sort_values(
        ["material_group", "field_name"]
    )
)

,material_group,field_name,is_mandatory_original,is_mandatory,reason
10,1020001,COLORINGID,True,True,NaN
9,1020001,FABRICSTRUCTURETNAME,True,True,NaN
8,1020001,FIBERCONTENTLISTID,True,True,NaN
15,1020001,KNITTYPEID,False,False,NaN
11,1020001,WEAVETYPE,True,False,WEAVETYPE is not applicable to knitted fabrics.
12,1020001,WEIGHT,True,True,NaN
13,1020001,WEIGHTUOMNAME,True,True,NaN
14,1020001,WIDTH,True,True,NaN
17,1020001,YARN1TYPEKNITSID,False,False,NaN
16,1020001,YARNCOLORING1KNITSID,False,False,NaN


In [119]:
# Rebuild mandatory characteristics after policy corrections
mandatory_features = (
    feature_policy[
        feature_policy["is_mandatory"]
    ][
        ["material_group", "field_name"]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "material_group": "material_group_code"
        }
    )
)

display(
    mandatory_features[
        "material_group_code"
    ].value_counts()
)

material_group_code
1010001    7
1020001    7
1020002    7
1030003    7
1030004    7
Name: count, dtype: int64

In [120]:
# Rebuild mandatory coverage using the corrected policy
mandatory_coverage_detail = (
    known_material_groups[
        known_material_groups["material_group_code"].notna()
    ]
    .merge(
        mandatory_features,
        on="material_group_code",
        how="inner"
    )
    .merge(
        sap_attribute_presence,
        on=["material_id", "field_name"],
        how="left"
    )
)

mandatory_coverage_detail["is_present"] = (
    mandatory_coverage_detail["is_present"]
    .eq(True)
)

mandatory_coverage_detail["has_any_attribute"] = (
    mandatory_coverage_detail["material_id"]
    .isin(materials_with_attributes)
)

In [121]:
# Recalculate field-level coverage
mandatory_coverage_audit = (
    mandatory_coverage_detail
    .groupby(
        [
            "material_group_code",
            "material_group_name",
            "field_name"
        ]
    )
    .agg(
        total_materials=("material_id", "nunique"),
        materials_with_attributes=("has_any_attribute", "sum"),
        present_count=("is_present", "sum")
    )
    .reset_index()
)

mandatory_coverage_audit["absolute_coverage"] = (
    mandatory_coverage_audit["present_count"]
    / mandatory_coverage_audit["total_materials"]
)

mandatory_coverage_audit["conditional_coverage"] = (
    mandatory_coverage_audit["present_count"]
    / mandatory_coverage_audit["materials_with_attributes"]
)

display(
    mandatory_coverage_audit
    .sort_values(
        ["material_group_code", "conditional_coverage"],
        ascending=[True, False]
    )
)

,material_group_code,material_group_name,field_name,total_materials,materials_with_attributes,present_count,absolute_coverage,conditional_coverage
1,1020001,ORME,FABRICSTRUCTURETNAME,2956,1831,1829,0.618742,0.998908
2,1020001,ORME,FIBERCONTENTLISTID,2956,1831,1829,0.618742,0.998908
3,1020001,ORME,WEIGHT,2956,1831,1829,0.618742,0.998908
5,1020001,ORME,WIDTH,2956,1831,1817,0.614682,0.992354
4,1020001,ORME,WEIGHTUOMNAME,2956,1831,1768,0.598106,0.965593
0,1020001,ORME,COLORINGID,2956,1831,1605,0.542963,0.876570
6,1020001,ORME,YARNCOUNT1KNITSID,2956,1831,1009,0.341340,0.551065
11,1020002,DOKUMA,WEIGHT,2399,1420,1419,0.591496,0.999296
8,1020002,DOKUMA,FABRICSTRUCTURETNAME,2399,1420,1418,0.591080,0.998592
9,1020002,DOKUMA,FIBERCONTENTLISTID,2399,1420,1416,0.590246,0.997183


In [122]:
# Recalculate material-level mandatory completeness
material_level_completeness = (
    mandatory_coverage_detail
    .groupby(
        [
            "material_id",
            "plm_code",
            "material_group_code",
            "material_group_name"
        ]
    )
    .agg(
        mandatory_field_count=("field_name", "nunique"),
        mandatory_field_present=("is_present", "sum")
    )
    .reset_index()
)

material_level_completeness["mandatory_completeness"] = (
    material_level_completeness["mandatory_field_present"]
    / material_level_completeness["mandatory_field_count"]
)

display(
    material_level_completeness
    .groupby(
        ["material_group_code", "material_group_name"]
    )["mandatory_completeness"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        min="min",
        max="max"
    )
)

,,count,mean,median,min,max
material_group_code,material_group_name,,,,,
1020001,ORME,2956,0.564759,0.857143,0.0,1.0
1020002,DOKUMA,2399,0.494730,0.857143,0.0,1.0
1030004,DENIM,280,0.834184,0.857143,0.0,1.0


In [123]:
# Get all policy characteristics for the material groups in scope
all_policy_features = (
    feature_policy[
        feature_policy["material_group"].isin(
            ["1020001", "1020002", "1030004"]
        )
    ][
        [
            "material_group",
            "field_name",
            "is_mandatory",
            "in_short_description",
            "short_description_order",
            "in_long_description",
            "long_description_order"
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "material_group": "material_group_code"
        }
    )
)
# Generate expected policy characteristics for each known material
policy_coverage_detail = (
    known_material_groups[
        known_material_groups["material_group_code"].notna()
    ]
    .merge(
        all_policy_features,
        on="material_group_code",
        how="inner"
    )
    .merge(
        sap_attribute_presence,
        on=["material_id", "field_name"],
        how="left"
    )
)

policy_coverage_detail["is_present"] = (
    policy_coverage_detail["is_present"].eq(True)
)

policy_coverage_detail["has_any_attribute"] = (
    policy_coverage_detail["material_id"]
    .isin(materials_with_attributes)
)
# Calculate observed coverage for every policy characteristic
policy_coverage_summary = (
    policy_coverage_detail
    .groupby(
        [
            "material_group_code",
            "material_group_name",
            "field_name"
        ]
    )
    .agg(
        total_materials=("material_id", "nunique"),
        materials_with_attributes=("has_any_attribute", "sum"),
        present_count=("is_present", "sum")
    )
    .reset_index()
)

policy_coverage_summary["absolute_coverage"] = (
    policy_coverage_summary["present_count"]
    / policy_coverage_summary["total_materials"]
)

policy_coverage_summary["conditional_coverage"] = (
    policy_coverage_summary["present_count"]
    / policy_coverage_summary["materials_with_attributes"]
)

In [124]:
# Combine business policy with observed SAP coverage
feature_policy_audit = (
    all_policy_features
    .merge(
        policy_coverage_summary[
            [
                "material_group_code",
                "field_name",
                "conditional_coverage"
            ]
        ],
        on=["material_group_code", "field_name"],
        how="left"
    )
)

In [125]:
# Assign a high-level role to each characteristic
def assign_feature_role(row):
    if row["is_mandatory"] and (
        row["in_short_description"]
        or row["in_long_description"]
    ):
        return "CORE"

    if row["is_mandatory"]:
        return "MANDATORY"

    if (
        row["in_short_description"]
        or row["in_long_description"]
    ):
        return "DESCRIPTION_SUPPORT"

    return "SUPPORTING"


feature_policy_audit["feature_role"] = (
    feature_policy_audit.apply(
        assign_feature_role,
        axis=1
    )
)

In [126]:
display(
    feature_policy_audit[
        [
            "material_group_code",
            "field_name",
            "feature_role",
            "is_mandatory",
            "in_short_description",
            "short_description_order",
            "in_long_description",
            "long_description_order",
            "conditional_coverage"
        ]
    ]
    .sort_values(
        [
            "material_group_code",
            "feature_role",
            "conditional_coverage"
        ],
        ascending=[True, True, False]
    )
)

,material_group_code,field_name,feature_role,is_mandatory,in_short_description,short_description_order,in_long_description,long_description_order,conditional_coverage
1,1020001,FIBERCONTENTLISTID,CORE,True,True,3.0,False,NaN,0.998908
2,1020001,FABRICSTRUCTURETNAME,CORE,True,True,4.0,True,5.0,0.998908
3,1020001,COLORINGID,CORE,True,True,5.0,True,6.0,0.876570
0,1020001,YARNCOUNT1KNITSID,CORE,True,True,1.0,True,1.0,0.551065
10,1020001,YARN1TYPEKNITSID,DESCRIPTION_SUPPORT,False,False,NaN,True,2.0,0.838340
9,1020001,YARNCOLORING1KNITSID,DESCRIPTION_SUPPORT,False,True,6.0,True,7.0,0.203714
8,1020001,KNITTYPEID,DESCRIPTION_SUPPORT,False,True,3.0,True,4.0,0.013654
5,1020001,WEIGHT,MANDATORY,True,False,NaN,False,NaN,0.998908
7,1020001,WIDTH,MANDATORY,True,False,NaN,False,NaN,0.992354
6,1020001,WEIGHTUOMNAME,MANDATORY,True,False,NaN,False,NaN,0.965593


In [127]:
# Assign modeling tiers based on business role and observed data coverage
def assign_modeling_tier(row):
    coverage = row["conditional_coverage"]

    if pd.isna(coverage):
        return "EXCLUDE"

    # Strong business relevance and high observed coverage
    if row["feature_role"] == "CORE" and coverage >= 0.80:
        return "PRIMARY"

    # Mandatory fields with high coverage
    if row["feature_role"] == "MANDATORY" and coverage >= 0.80:
        return "PRIMARY"

    # Relevant fields with moderate coverage
    if (
        row["feature_role"] in [
            "CORE",
            "MANDATORY",
            "DESCRIPTION_SUPPORT"
        ]
        and coverage >= 0.40
    ):
        return "SECONDARY"

    # Low-coverage fields may still provide useful evidence when available
    if coverage >= 0.10:
        return "OPTIONAL"

    return "EXCLUDE"


feature_policy_audit["modeling_tier"] = (
    feature_policy_audit.apply(
        assign_modeling_tier,
        axis=1
    )
)

In [128]:
display(
    feature_policy_audit[
        [
            "material_group_code",
            "field_name",
            "feature_role",
            "conditional_coverage",
            "modeling_tier"
        ]
    ]
    .sort_values(
        [
            "material_group_code",
            "modeling_tier",
            "conditional_coverage"
        ],
        ascending=[True, True, False]
    )
)

,material_group_code,field_name,feature_role,conditional_coverage,modeling_tier
8,1020001,KNITTYPEID,DESCRIPTION_SUPPORT,0.013654,EXCLUDE
4,1020001,WEAVETYPE,SUPPORTING,0.001092,EXCLUDE
9,1020001,YARNCOLORING1KNITSID,DESCRIPTION_SUPPORT,0.203714,OPTIONAL
1,1020001,FIBERCONTENTLISTID,CORE,0.998908,PRIMARY
2,1020001,FABRICSTRUCTURETNAME,CORE,0.998908,PRIMARY
5,1020001,WEIGHT,MANDATORY,0.998908,PRIMARY
7,1020001,WIDTH,MANDATORY,0.992354,PRIMARY
6,1020001,WEIGHTUOMNAME,MANDATORY,0.965593,PRIMARY
3,1020001,COLORINGID,CORE,0.876570,PRIMARY
10,1020001,YARN1TYPEKNITSID,DESCRIPTION_SUPPORT,0.838340,SECONDARY


In [129]:
# Build material-group-specific feature sets
matching_feature_sets = (
    feature_policy_audit[
        feature_policy_audit["modeling_tier"]
        .isin(["PRIMARY", "SECONDARY"])
    ]
    .groupby(
        [
            "material_group_code",
            "modeling_tier"
        ]
    )["field_name"]
    .apply(list)
)

display(matching_feature_sets)

material_group_code  modeling_tier
1020001              PRIMARY          [FIBERCONTENTLISTID, FABRICSTRUCTURETNAME, COL...
                     SECONDARY                    [YARNCOUNT1KNITSID, YARN1TYPEKNITSID]
1020002              PRIMARY          [FABRICSTRUCTURETNAME, WEIGHT, FIBERCONTENTLIS...
                     SECONDARY                                       [WEFTYARNCOUNT1ID]
1030004              PRIMARY          [FIBERCONTENTLISTID, WEIGHT, COLORINGID, FABRI...
                     SECONDARY                                       [WIDTH, WEAVETYPE]
Name: field_name, dtype: object

In [130]:
# Check usable coverage for numeric characteristics
numeric_features = [
    "WEIGHT",
    "WIDTH"
]

numeric_quality_records = []

for material_group_code in ["1020001", "1020002", "1030004"]:
    
    group_materials = set(
        known_material_groups.loc[
            known_material_groups["material_group_code"]
            == material_group_code,
            "material_id"
        ]
    )

    group_attributes = fabric_attributes[
        fabric_attributes["material_id"].isin(group_materials)
    ]

    for field_name in numeric_features:

        field_values = group_attributes[
            group_attributes["field_name"] == field_name
        ].copy()

        field_values["numeric_value"] = pd.to_numeric(
            field_values["Karakteristik değeri"],
            errors="coerce"
        )

        materials_with_any_attributes = len(
            group_materials
            & materials_with_attributes
        )

        materials_with_valid_value = (
            field_values.loc[
                field_values["numeric_value"] > 0,
                "material_id"
            ]
            .nunique()
        )

        numeric_quality_records.append(
            {
                "material_group_code": material_group_code,
                "field_name": field_name,
                "materials_with_attributes":
                    materials_with_any_attributes,
                "materials_with_valid_value":
                    materials_with_valid_value,
                "usable_coverage":
                    materials_with_valid_value
                    / materials_with_any_attributes
                    if materials_with_any_attributes > 0
                    else np.nan
            }
        )


numeric_feature_quality = pd.DataFrame(
    numeric_quality_records
)

display(numeric_feature_quality)

,material_group_code,field_name,materials_with_attributes,materials_with_valid_value,usable_coverage
0,1020001,WEIGHT,1831,1807,0.986892
1,1020001,WIDTH,1831,38,0.020754
2,1020002,WEIGHT,1420,1415,0.996479
3,1020002,WIDTH,1420,67,0.047183
4,1030004,WEIGHT,262,262,1.000000
5,1030004,WIDTH,262,19,0.072519


In [131]:
# Add usable coverage for numeric characteristics
feature_policy_audit = feature_policy_audit.merge(
    numeric_feature_quality[
        [
            "material_group_code",
            "field_name",
            "usable_coverage"
        ]
    ],
    on=["material_group_code", "field_name"],
    how="left"
)

# Use usable coverage when available, otherwise regular conditional coverage
feature_policy_audit["effective_coverage"] = (
    feature_policy_audit["usable_coverage"]
    .fillna(feature_policy_audit["conditional_coverage"])
)

In [132]:
# Reassign modeling tiers using effective coverage
def assign_modeling_tier(row):
    coverage = row["effective_coverage"]

    if pd.isna(coverage):
        return "EXCLUDE"

    if (
        row["feature_role"] in ["CORE", "MANDATORY"]
        and coverage >= 0.80
    ):
        return "PRIMARY"

    if (
        row["feature_role"] in [
            "CORE",
            "MANDATORY",
            "DESCRIPTION_SUPPORT"
        ]
        and coverage >= 0.40
    ):
        return "SECONDARY"

    if coverage >= 0.10:
        return "OPTIONAL"

    return "EXCLUDE"


feature_policy_audit["modeling_tier"] = (
    feature_policy_audit.apply(
        assign_modeling_tier,
        axis=1
    )
)

In [133]:
display(
    feature_policy_audit[
        [
            "material_group_code",
            "field_name",
            "feature_role",
            "conditional_coverage",
            "usable_coverage",
            "effective_coverage",
            "modeling_tier"
        ]
    ]
    .sort_values(
        [
            "material_group_code",
            "modeling_tier",
            "effective_coverage"
        ],
        ascending=[True, True, False]
    )
)

,material_group_code,field_name,feature_role,conditional_coverage,usable_coverage,effective_coverage,modeling_tier
7,1020001,WIDTH,MANDATORY,0.992354,0.020754,0.020754,EXCLUDE
8,1020001,KNITTYPEID,DESCRIPTION_SUPPORT,0.013654,NaN,0.013654,EXCLUDE
4,1020001,WEAVETYPE,SUPPORTING,0.001092,NaN,0.001092,EXCLUDE
9,1020001,YARNCOLORING1KNITSID,DESCRIPTION_SUPPORT,0.203714,NaN,0.203714,OPTIONAL
1,1020001,FIBERCONTENTLISTID,CORE,0.998908,NaN,0.998908,PRIMARY
2,1020001,FABRICSTRUCTURETNAME,CORE,0.998908,NaN,0.998908,PRIMARY
5,1020001,WEIGHT,MANDATORY,0.998908,0.986892,0.986892,PRIMARY
6,1020001,WEIGHTUOMNAME,MANDATORY,0.965593,NaN,0.965593,PRIMARY
3,1020001,COLORINGID,CORE,0.876570,NaN,0.876570,PRIMARY
10,1020001,YARN1TYPEKNITSID,DESCRIPTION_SUPPORT,0.838340,NaN,0.838340,SECONDARY


In [134]:
# Select categorical characteristics that may be used for matching
categorical_features = [
    "FABRICSTRUCTURETNAME",
    "COLORINGID",
    "WEIGHTUOMNAME",
    "WEAVETYPE",
    "YARNCOUNT1KNITSID",
    "YARN1TYPEKNITSID",
    "WEFTYARNCOUNT1ID"
]

categorical_value_summary = (
    fabric_attributes[
        fabric_attributes["field_name"].isin(categorical_features)
    ]
    .merge(
        known_material_groups[
            [
                "material_id",
                "material_group_code",
                "material_group_name"
            ]
        ].drop_duplicates("material_id"),
        on="material_id",
        how="inner"
    )
    .groupby(
        [
            "material_group_code",
            "material_group_name",
            "field_name",
            "Karakteristik değeri"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="count")
)

categorical_value_summary["rank"] = (
    categorical_value_summary
    .groupby(
        ["material_group_code", "field_name"]
    )["count"]
    .rank(
        method="first",
        ascending=False
    )
)

display(
    categorical_value_summary[
        categorical_value_summary["rank"] <= 20
    ]
    .sort_values(
        [
            "material_group_code",
            "field_name",
            "count"
        ],
        ascending=[True, True, False]
    )
)

,material_group_code,material_group_name,field_name,Karakteristik değeri,count,rank
17,1020001,ORME,COLORINGID,SOLIDDYED,856,1.0
19,1020001,ORME,COLORINGID,YARNDYED,213,2.0
1,1020001,ORME,COLORINGID,ALLOVERPRINT,152,3.0
14,1020001,ORME,COLORINGID,REACTIVEDYED,116,4.0
16,1020001,ORME,COLORINGID,REAKTIVEDISPERSE,105,5.0
...,...,...,...,...,...,...
430,1030004,DENIM,WEAVETYPE,LCWWEAVETYPELIST6,3,5.0
434,1030004,DENIM,WEIGHTUOMNAME,VRDOZPERSQYARD,234,1.0
433,1030004,DENIM,WEIGHTUOMNAME,GSM,26,2.0
435,1030004,DENIM,YARN1TYPEKNITSID,COMBED,1,1.0


In [135]:
# Measure categorical feature diversity
categorical_feature_profile = (
    fabric_attributes[
        fabric_attributes["field_name"].isin(categorical_features)
    ]
    .merge(
        known_material_groups[
            [
                "material_id",
                "material_group_code",
                "material_group_name"
            ]
        ].drop_duplicates("material_id"),
        on="material_id",
        how="inner"
    )
    .groupby(
        [
            "material_group_code",
            "material_group_name",
            "field_name"
        ]
    )
    .agg(
        material_count=("material_id", "nunique"),
        unique_values=("Karakteristik değeri", "nunique")
    )
    .reset_index()
)

display(categorical_feature_profile)

,material_group_code,material_group_name,field_name,material_count,unique_values
0,1020001,ORME,COLORINGID,1605,20
1,1020001,ORME,FABRICSTRUCTURETNAME,1829,49
2,1020001,ORME,WEAVETYPE,2,2
3,1020001,ORME,WEFTYARNCOUNT1ID,1,1
4,1020001,ORME,WEIGHTUOMNAME,1768,11
5,1020001,ORME,YARN1TYPEKNITSID,1535,24
6,1020001,ORME,YARNCOUNT1KNITSID,1009,79
7,1020002,DOKUMA,COLORINGID,1358,22
8,1020002,DOKUMA,FABRICSTRUCTURETNAME,1418,46
9,1020002,DOKUMA,WEAVETYPE,1288,46


In [136]:
# Inspect weight-unit values by material group
weight_unit_distribution = (
    categorical_value_summary[
        categorical_value_summary["field_name"] == "WEIGHTUOMNAME"
    ][
        [
            "material_group_code",
            "material_group_name",
            "Karakteristik değeri",
            "count"
        ]
    ]
    .sort_values(
        ["material_group_code", "count"],
        ascending=[True, False]
    )
)

display(weight_unit_distribution)

,material_group_code,material_group_name,Karakteristik değeri,count
81,1020001,ORME,GSM,1745
82,1020001,ORME,KG,7
79,1020001,ORME,GR,6
77,1020001,ORME,DENYE,2
80,1020001,ORME,GRAM,2
72,1020001,ORME,180,1
73,1020001,ORME,210,1
74,1020001,ORME,220,1
75,1020001,ORME,250,1
76,1020001,ORME,280,1


In [137]:
# Inspect PLM weight-unit values by material group
plm_weight_unit_distribution = (
    plm_codes[
        plm_codes["material_group_code"].isin(
            ["1020001", "1020002", "1030004"]
        )
    ]
    .groupby(
        [
            "material_group_code",
            "material_group_name",
            "Kumaş ağırlığı birimi"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        ["material_group_code", "count"],
        ascending=[True, False]
    )
)

display(plm_weight_unit_distribution)

,material_group_code,material_group_name,Kumaş ağırlığı birimi,count
2,1020001,ORME,GSM,12086
5,1020001,ORME,NaN,701
0,1020001,ORME,CMS,9
1,1020001,ORME,DENYE,8
3,1020001,ORME,NE,7
4,1020001,ORME,VRDOZPERSQYARD,1
8,1020002,DOKUMA,GSM,11635
12,1020002,DOKUMA,NaN,362
11,1020002,DOKUMA,VRDOZPERSQYARD,66
6,1020002,DOKUMA,CMS,20


In [138]:
# Prepare SAP weight information
sap_weight_audit = weight_values[
    [
        "material_id",
        "sap_weight_raw",
        "WEIGHTUOMNAME"
    ]
].copy()

sap_weight_audit = sap_weight_audit.rename(
    columns={
        "WEIGHTUOMNAME": "sap_weight_unit_raw"
    }
)

In [139]:
# Normalize only clearly equivalent weight-unit labels
def normalize_weight_unit(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().upper()

    unit_mapping = {
        "GSM": "GSM",
        "G/M2": "GSM",
        "GR/M2": "GSM",
        "VRDOZPERSQYARD": "VRDOZPERSQYARD"
    }

    return unit_mapping.get(value, value)


sap_weight_audit["sap_weight_unit"] = (
    sap_weight_audit["sap_weight_unit_raw"]
    .apply(normalize_weight_unit)
)

In [140]:
# Prepare PLM weight information
plm_weight_audit = plm_codes[
    [
        "plm_code",
        "Kumaş ağırlığı",
        "Kumaş ağırlığı birimi"
    ]
].copy()

plm_weight_audit["plm_weight"] = pd.to_numeric(
    plm_weight_audit["Kumaş ağırlığı"],
    errors="coerce"
)

plm_weight_audit["plm_weight_unit"] = (
    plm_weight_audit["Kumaş ağırlığı birimi"]
    .apply(normalize_weight_unit)
)

In [141]:
# Build known SAP-PLM weight pairs
known_weight_audit = (
    known_material_groups[
        [
            "material_id",
            "plm_code",
            "material_group_code",
            "material_group_name"
        ]
    ]
    .merge(
        sap_weight_audit,
        on="material_id",
        how="left"
    )
    .merge(
        plm_weight_audit,
        on="plm_code",
        how="left"
    )
)

In [142]:
# Inspect SAP vs PLM weight-unit combinations
weight_unit_pair_summary = (
    known_weight_audit
    .groupby(
        [
            "material_group_code",
            "material_group_name",
            "sap_weight_unit",
            "plm_weight_unit"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        ["material_group_code", "count"],
        ascending=[True, False]
    )
)

display(weight_unit_pair_summary)

,material_group_code,material_group_name,sap_weight_unit,plm_weight_unit,count
8,1020001,ORME,GSM,GSM,1746
10,1020001,ORME,NaN,GSM,1150
12,1020001,ORME,NaN,NaN,37
9,1020001,ORME,KG,GSM,7
6,1020001,ORME,GR,GSM,6
5,1020001,ORME,DENYE,DENYE,2
7,1020001,ORME,GRAM,GSM,2
0,1020001,ORME,180,GSM,1
1,1020001,ORME,210,GSM,1
2,1020001,ORME,220,GSM,1


In [143]:
# Analyze scaling patterns where SAP and PLM use the same unit
same_unit_weight_pairs = known_weight_audit[
    known_weight_audit["sap_weight_raw"].notna()
    & known_weight_audit["plm_weight"].notna()
    & (known_weight_audit["sap_weight_raw"] > 0)
    & (known_weight_audit["plm_weight"] > 0)
    & known_weight_audit["sap_weight_unit"].notna()
    & (
        known_weight_audit["sap_weight_unit"]
        == known_weight_audit["plm_weight_unit"]
    )
].copy()

same_unit_weight_pairs["weight_ratio"] = (
    same_unit_weight_pairs["sap_weight_raw"]
    / same_unit_weight_pairs["plm_weight"]
)

display(
    same_unit_weight_pairs.groupby(
        [
            "material_group_name",
            "sap_weight_unit"
        ]
    )["weight_ratio"].describe()
)

count         mean         std  \
material_group_name sap_weight_unit                                    
DENIM               GSM                25.0   320.680000  475.619133   
                    VRDOZPERSQYARD    234.0   202.683150  400.733487   
DOKUMA              GSM                 6.0   500.500000  547.174835   
                    VRDOZPERSQYARD      1.0  1000.000000         NaN   
LEATHER1            GSM                 5.0   200.800000  446.766382   
ORME                DENYE               2.0  1000.000000    0.000000   
                    GSM              1724.0   340.059958  473.054103   
TRIKO               GSM                 1.0  1000.000000         NaN   

                                            min     25%     50%     75%  \
material_group_name sap_weight_unit                                       
DENIM               GSM                 1.00000     1.0     1.0  1000.0   
                    VRDOZPERSQYARD      1.00000     1.0     1.0     1.0   
DOKUMA              GSM                 1.00000     1.0   500.5  1000.0   
                    VRDOZPERSQYARD   1000.00000  1000.0  1000.0  1000.0   
LEATHER1            GSM                 1.00000     1.0     1.0     1.0   
ORME                DENYE            1000.00000  1000.0  1000.0  1000.0   
                    GSM                 0.73913     1.0     1.0  1000.0   
TRIKO               GSM              1000.00000  1000.0  1000.0  1000.0   

                                             max  
material_group_name sap_weight_unit               
DENIM               GSM              1000.000000  
                    VRDOZPERSQYARD   1000.000000  
DOKUMA              GSM              1000.000000  
                    VRDOZPERSQYARD   1000.000000  
LEATHER1            GSM              1000.000000  
ORME                DENYE            1000.000000  
                    GSM              1181.818182  
TRIKO               GSM              1000.000000

In [144]:
# Classify common SAP-to-PLM weight scaling patterns
def classify_weight_scale(ratio):
    if pd.isna(ratio):
        return "unknown"

    if np.isclose(ratio, 1, rtol=0.05):
        return "same_scale"

    if np.isclose(ratio, 1000, rtol=0.05):
        return "sap_x1000"

    if np.isclose(ratio, 100, rtol=0.05):
        return "sap_x100"

    if np.isclose(ratio, 0.001, rtol=0.05):
        return "plm_x1000"

    return "other"


same_unit_weight_pairs["scale_pattern"] = (
    same_unit_weight_pairs["weight_ratio"]
    .apply(classify_weight_scale)
)

display(
    same_unit_weight_pairs.groupby(
        [
            "material_group_name",
            "sap_weight_unit",
            "scale_pattern"
        ]
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        ["material_group_name", "sap_weight_unit", "count"],
        ascending=[True, True, False]
    )
)

,material_group_name,sap_weight_unit,scale_pattern,count
0,DENIM,GSM,same_scale,17
1,DENIM,GSM,sap_x1000,8
3,DENIM,VRDOZPERSQYARD,same_scale,185
4,DENIM,VRDOZPERSQYARD,sap_x1000,47
2,DENIM,VRDOZPERSQYARD,other,2
5,DOKUMA,GSM,same_scale,3
6,DOKUMA,GSM,sap_x1000,3
7,DOKUMA,VRDOZPERSQYARD,sap_x1000,1
8,LEATHER1,GSM,same_scale,4
9,LEATHER1,GSM,sap_x1000,1


In [145]:
# Compare SAP and PLM weights while accounting for SAP x1000 scaling
def calculate_weight_similarity(
    sap_weight,
    plm_weight,
    sap_unit=None,
    plm_unit=None
):
    if (
        pd.isna(sap_weight)
        or pd.isna(plm_weight)
        or sap_weight <= 0
        or plm_weight <= 0
    ):
        return pd.Series(
            [np.nan, np.nan, pd.NA]
        )

    # If both units exist and disagree, do not compare them directly
    if (
        pd.notna(sap_unit)
        and pd.notna(plm_unit)
        and sap_unit != plm_unit
    ):
        return pd.Series(
            [np.nan, np.nan, "unit_mismatch"]
        )

    candidates = {
        "same_scale": sap_weight,
        "sap_x1000": sap_weight / 1000
    }

    relative_errors = {
        scale: abs(value - plm_weight) / plm_weight
        for scale, value in candidates.items()
    }

    best_scale = min(
        relative_errors,
        key=relative_errors.get
    )

    best_error = relative_errors[best_scale]

    # 1 = perfect match, approaches 0 as the error increases
    similarity = 1 / (1 + best_error)

    return pd.Series(
        [
            similarity,
            best_error,
            best_scale
        ]
    )

In [146]:
# Calculate scale-aware weight similarity for known mappings
known_weight_audit[
    [
        "weight_similarity",
        "weight_relative_error",
        "weight_scale"
    ]
] = known_weight_audit.apply(
    lambda row: calculate_weight_similarity(
        row["sap_weight_raw"],
        row["plm_weight"],
        row["sap_weight_unit"],
        row["plm_weight_unit"]
    ),
    axis=1
)

In [147]:
# Evaluate weight agreement
comparable_weight_pairs = known_weight_audit[
    known_weight_audit["weight_similarity"].notna()
].copy()

print(
    "Comparable weight pairs:",
    len(comparable_weight_pairs)
)

print(
    "Exact / nearly exact (<= 1% error):",
    f"{comparable_weight_pairs['weight_relative_error'].le(0.01).mean():.2%}"
)

print(
    "Within 5%:",
    f"{comparable_weight_pairs['weight_relative_error'].le(0.05).mean():.2%}"
)

print(
    "Within 10%:",
    f"{comparable_weight_pairs['weight_relative_error'].le(0.10).mean():.2%}"
)

display(
    comparable_weight_pairs[
        "weight_scale"
    ].value_counts()
)

Comparable weight pairs: 3470
Exact / nearly exact (<= 1% error): 98.01%
Within 5%: 98.59%
Within 10%: 99.14%


weight_scale
same_scale    2205
sap_x1000     1265
Name: count, dtype: int64

In [148]:
display(
    comparable_weight_pairs
    .groupby("material_group_name")
    .agg(
        comparable_pairs=("material_id", "size"),
        within_5_percent=(
            "weight_relative_error",
            lambda x: (x <= 0.05).mean()
        ),
        within_10_percent=(
            "weight_relative_error",
            lambda x: (x <= 0.10).mean()
        )
    )
)

,comparable_pairs,within_5_percent,within_10_percent
material_group_name,,,
DENIM,261,0.992337,0.992337
DOKUMA,1415,0.983039,0.990813
LEATHER1,6,1.000000,1.000000
ORME,1787,0.987129,0.991606
TRIKO,1,1.000000,1.000000


## 7. Özellik bazlı uyum (agreement) ölçümü


In [149]:
# Keep only material groups currently in modeling scope
model_material_groups = [
    "1020001",  # ORME
    "1020002",  # DOKUMA
    "1030004"   # DENIM
]

known_material_groups_model = (
    known_material_groups[
        known_material_groups["material_group_code"]
        .isin(model_material_groups)
    ]
    .copy()
)

In [150]:
# Prepare SAP fabric structure values
sap_fabric_structure = (
    fabric_attributes[
        fabric_attributes["field_name"]
        == "FABRICSTRUCTURETNAME"
    ][
        ["material_id", "Karakteristik değeri"]
    ]
    .drop_duplicates("material_id")
    .rename(
        columns={
            "Karakteristik değeri":
            "sap_fabric_structure"
        }
    )
)

sap_fabric_structure["sap_fabric_structure"] = (
    sap_fabric_structure["sap_fabric_structure"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [151]:
# Prepare PLM fabric structure values
plm_fabric_structure = (
    plm_codes[
        [
            "plm_code",
            "Kumaş tipi"
        ]
    ]
    .copy()
    .rename(
        columns={
            "Kumaş tipi":
            "plm_fabric_structure"
        }
    )
)

plm_fabric_structure["plm_fabric_structure"] = (
    plm_fabric_structure["plm_fabric_structure"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [152]:
# Compare SAP and PLM fabric structure values
fabric_structure_pairs = (
    known_material_groups_model[
        [
            "material_id",
            "plm_code",
            "material_group_code",
            "material_group_name"
        ]
    ]
    .merge(
        sap_fabric_structure,
        on="material_id",
        how="inner"
    )
    .merge(
        plm_fabric_structure,
        on="plm_code",
        how="inner"
    )
)

fabric_structure_pairs = fabric_structure_pairs[
    fabric_structure_pairs["sap_fabric_structure"].notna()
    & fabric_structure_pairs["plm_fabric_structure"].notna()
].copy()

fabric_structure_pairs["fabric_structure_match"] = (
    fabric_structure_pairs["sap_fabric_structure"]
    == fabric_structure_pairs["plm_fabric_structure"]
)

print(
    "Comparable fabric structure pairs:",
    len(fabric_structure_pairs)
)

print(
    "Exact match rate:",
    f"{fabric_structure_pairs['fabric_structure_match'].mean():.2%}"
)

display(
    fabric_structure_pairs
    .groupby("material_group_name")
    ["fabric_structure_match"]
    .agg(
        comparable_pairs="count",
        exact_match_rate="mean"
    )
)

Comparable fabric structure pairs: 3508
Exact match rate: 93.36%


,comparable_pairs,exact_match_rate
material_group_name,,
DENIM,262,0.996183
DOKUMA,1418,0.907616
ORME,1828,0.944748


In [153]:
# Inspect common fabric-structure mismatches
display(
    fabric_structure_pairs[
        ~fabric_structure_pairs["fabric_structure_match"]
    ]
    .groupby(
        [
            "sap_fabric_structure",
            "plm_fabric_structure"
        ]
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        "count",
        ascending=False
    )
    .head(30)
)

,sap_fabric_structure,plm_fabric_structure,count
44,POPLIN,PLAINWEAVE,38
56,THREETHREADFLEECE,THREETHREADLOOPBACK,34
18,INTERLOCK,SOFTTOUCHSCUBA,21
41,POLYVISCON,POLYVISCOSE,18
33,OTHERWOVEN,PLAINWEAVE,13
24,LINEN,PLAINWEAVE,7
47,RIB,RIB2X2,7
58,TWOTHREAD,TWOTHREADLOOPBACK,7
54,TAFFETA,PLAINWEAVE,6
40,POLYVISCON,PLAINWEAVE,4


In [154]:
# Normalize only clearly equivalent fabric-structure labels
fabric_structure_mapping = {
    "POLYVISCON": "POLYVISCOSE",
    "BEZAYAĞI": "PLAINWEAVE"
}

fabric_structure_pairs["sap_fabric_structure_normalized"] = (
    fabric_structure_pairs["sap_fabric_structure"]
    .replace(fabric_structure_mapping)
)

fabric_structure_pairs["plm_fabric_structure_normalized"] = (
    fabric_structure_pairs["plm_fabric_structure"]
    .replace(fabric_structure_mapping)
)

fabric_structure_pairs["fabric_structure_match_normalized"] = (
    fabric_structure_pairs["sap_fabric_structure_normalized"]
    == fabric_structure_pairs["plm_fabric_structure_normalized"]
)

print(
    "Normalized exact match rate:",
    f"{fabric_structure_pairs['fabric_structure_match_normalized'].mean():.2%}"
)

Normalized exact match rate: 93.87%


In [155]:
# Prepare SAP coloring values
sap_coloring = (
    fabric_attributes[
        fabric_attributes["field_name"] == "COLORINGID"
    ][
        ["material_id", "Karakteristik değeri"]
    ]
    .dropna(subset=["Karakteristik değeri"])
    .copy()
)

sap_coloring["coloring_value"] = (
    sap_coloring["Karakteristik değeri"]
    .astype("string")
    .str.strip()
    .str.upper()
)

sap_coloring_sets = (
    sap_coloring
    .groupby("material_id")["coloring_value"]
    .apply(set)
    .rename("sap_coloring_set")
    .reset_index()
)

In [156]:
# Prepare PLM coloring values from the multi-value characteristic table
plm_coloring = (
    plm_multi_value_valid[
        plm_multi_value_valid["PLM Karakteristik Tanımı"]
        == "COLORINGID"
    ][
        ["plm_code", "PLM Karakteristik Değeri"]
    ]
    .dropna(subset=["PLM Karakteristik Değeri"])
    .copy()
)

plm_coloring["coloring_value"] = (
    plm_coloring["PLM Karakteristik Değeri"]
    .astype("string")
    .str.strip()
    .str.upper()
)

plm_coloring_sets = (
    plm_coloring
    .groupby("plm_code")["coloring_value"]
    .apply(set)
    .rename("plm_coloring_set")
    .reset_index()
)

In [157]:
# Compare coloring sets for known SAP-PLM mappings
coloring_pairs = (
    known_material_groups_model[
        [
            "material_id",
            "plm_code",
            "material_group_name"
        ]
    ]
    .merge(
        sap_coloring_sets,
        on="material_id",
        how="inner"
    )
    .merge(
        plm_coloring_sets,
        on="plm_code",
        how="inner"
    )
)

coloring_pairs["coloring_similarity"] = (
    coloring_pairs.apply(
        lambda row: calculate_jaccard_similarity(
            row["sap_coloring_set"],
            row["plm_coloring_set"]
        ),
        axis=1
    )
)

print(
    "Comparable coloring pairs:",
    len(coloring_pairs)
)

print(
    "Exact coloring-set match:",
    f"{coloring_pairs['coloring_similarity'].eq(1).mean():.2%}"
)

display(
    coloring_pairs
    .groupby("material_group_name")
    ["coloring_similarity"]
    .agg(
        comparable_pairs="count",
        mean_similarity="mean",
        exact_match_rate=lambda x: x.eq(1).mean()
    )
)

Comparable coloring pairs: 3003
Exact coloring-set match: 79.95%


,comparable_pairs,mean_similarity,exact_match_rate
material_group_name,,,
DENIM,226,0.980088,0.977876
DOKUMA,1336,0.836702,0.835329
ORME,1441,0.738839,0.738376


In [158]:
# Inspect common coloring mismatches
coloring_mismatches = (
    coloring_pairs[
        coloring_pairs["coloring_similarity"] < 1
    ].copy()
)

# Convert sets to hashable tuples for counting
coloring_mismatches["sap_coloring_tuple"] = (
    coloring_mismatches["sap_coloring_set"]
    .apply(lambda x: tuple(sorted(x)))
)

coloring_mismatches["plm_coloring_tuple"] = (
    coloring_mismatches["plm_coloring_set"]
    .apply(lambda x: tuple(sorted(x)))
)

display(
    coloring_mismatches[
        [
            "sap_coloring_tuple",
            "plm_coloring_tuple"
        ]
    ]
    .value_counts()
    .head(30)
)

sap_coloring_tuple                                                             plm_coloring_tuple          
(SOLIDDYED,)                                                                   (REACTIVEDYED,)                 200
                                                                               (REAKTIVEDISPERSE,)             113
(YARNDYED,)                                                                    (WASHING,)                       48
(SOLIDDYED,)                                                                   (DISPERSEDYED,)                  38
                                                                               (PADBATCH,)                      35
(ALLOVERPRINT,)                                                                (BLEACHING,)                     24
                                                                               (REACTIVEDYED,)                  22
(YARNDYED,)                                                                    (REAKTIV

In [159]:
display(
    coloring_mismatches[
        [
            "material_group_name",
            "sap_coloring_tuple",
            "plm_coloring_tuple"
        ]
    ]
    .value_counts()
    .head(40)
)

material_group_name  sap_coloring_tuple         plm_coloring_tuple          
ORME                 (SOLIDDYED,)               (REACTIVEDYED,)                 143
                                                (REAKTIVEDISPERSE,)              95
DOKUMA               (SOLIDDYED,)               (REACTIVEDYED,)                  57
                                                (PADBATCH,)                      35
                     (YARNDYED,)                (WASHING,)                       35
ORME                 (ALLOVERPRINT,)            (REACTIVEDYED,)                  21
                     (SOLIDDYED,)               (DISPERSEDYED,)                  20
DOKUMA               (SOLIDDYED,)               (REAKTIVEDISPERSE,)              18
                                                (DISPERSEDYED,)                  18
ORME                 (ALLOVERPRINT,)            (BLEACHING,)                     14
                     (YARNDYED,)                (WASHING,)                       13

In [160]:
# Prepare SAP COLORINGID and DYETYPEID values
sap_coloring_dye = (
    fabric_attributes[
        fabric_attributes["field_name"].isin(
            ["COLORINGID", "DYETYPEID"]
        )
    ][
        [
            "material_id",
            "field_name",
            "Karakteristik değeri"
        ]
    ]
    .dropna(subset=["Karakteristik değeri"])
    .copy()
)

sap_coloring_dye["value"] = (
    sap_coloring_dye["Karakteristik değeri"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [161]:
# Create SAP value sets by characteristic
sap_coloring_dye_sets = (
    sap_coloring_dye
    .groupby(
        ["material_id", "field_name"]
    )["value"]
    .apply(set)
    .unstack()
    .reset_index()
    .rename(
        columns={
            "COLORINGID": "sap_coloring_set",
            "DYETYPEID": "sap_dye_type_set"
        }
    )
)

In [162]:
# Prepare PLM COLORINGID and DYETYPEID values
plm_coloring_dye = (
    plm_multi_value_valid[
        plm_multi_value_valid[
            "PLM Karakteristik Tanımı"
        ].isin(
            ["COLORINGID", "DYETYPEID"]
        )
    ][
        [
            "plm_code",
            "PLM Karakteristik Tanımı",
            "PLM Karakteristik Değeri"
        ]
    ]
    .dropna(subset=["PLM Karakteristik Değeri"])
    .copy()
)

plm_coloring_dye["value"] = (
    plm_coloring_dye["PLM Karakteristik Değeri"]
    .astype("string")
    .str.strip()
    .str.upper()
)

plm_coloring_dye_sets = (
    plm_coloring_dye
    .groupby(
        [
            "plm_code",
            "PLM Karakteristik Tanımı"
        ]
    )["value"]
    .apply(set)
    .unstack()
    .reset_index()
    .rename(
        columns={
            "COLORINGID": "plm_coloring_set",
            "DYETYPEID": "plm_dye_type_set"
        }
    )
)

In [163]:
# Build cross-field comparison table
coloring_semantic_audit = (
    known_material_groups_model[
        [
            "material_id",
            "plm_code",
            "material_group_name"
        ]
    ]
    .merge(
        sap_coloring_dye_sets,
        on="material_id",
        how="left"
    )
    .merge(
        plm_coloring_dye_sets,
        on="plm_code",
        how="left"
    )
)

In [164]:
# Compare same-field and cross-field similarities
comparison_pairs = {
    "coloring_to_coloring": (
        "sap_coloring_set",
        "plm_coloring_set"
    ),
    "dye_to_dye": (
        "sap_dye_type_set",
        "plm_dye_type_set"
    ),
    "dye_to_coloring": (
        "sap_dye_type_set",
        "plm_coloring_set"
    ),
    "coloring_to_dye": (
        "sap_coloring_set",
        "plm_dye_type_set"
    )
}

for score_name, (sap_col, plm_col) in comparison_pairs.items():
    coloring_semantic_audit[score_name] = (
        coloring_semantic_audit.apply(
            lambda row: calculate_jaccard_similarity(
                row[sap_col],
                row[plm_col]
            ),
            axis=1
        )
    )

In [165]:
# Summarize semantic agreement by material group
semantic_summary = []

for material_group, group in (
    coloring_semantic_audit.groupby(
        "material_group_name"
    )
):
    for score_name in comparison_pairs:

        comparable = group[
            score_name
        ].dropna()

        semantic_summary.append(
            {
                "material_group_name": material_group,
                "comparison": score_name,
                "comparable_pairs": len(comparable),
                "mean_similarity": comparable.mean(),
                "exact_match_rate":
                    comparable.eq(1).mean()
            }
        )

semantic_summary = pd.DataFrame(
    semantic_summary
)

display(semantic_summary)

,material_group_name,comparison,comparable_pairs,mean_similarity,exact_match_rate
0,DENIM,coloring_to_coloring,226,0.980088,0.977876
1,DENIM,dye_to_dye,2,1.000000,1.000000
2,DENIM,dye_to_coloring,45,0.044444,0.044444
3,DENIM,coloring_to_dye,2,0.000000,0.000000
4,DOKUMA,coloring_to_coloring,1336,0.836702,0.835329
5,DOKUMA,dye_to_dye,508,0.890748,0.801181
6,DOKUMA,dye_to_coloring,840,0.105357,0.102381
7,DOKUMA,coloring_to_dye,503,0.000000,0.000000
8,ORME,coloring_to_coloring,1441,0.738839,0.738376
9,ORME,dye_to_dye,476,0.861695,0.760504


In [166]:
# Prepare SAP knitting yarn count
sap_yarn_count_knit = (
    fabric_attributes[
        fabric_attributes["field_name"] == "YARNCOUNT1KNITSID"
    ][
        ["material_id", "Karakteristik değeri"]
    ]
    .dropna(subset=["Karakteristik değeri"])
    .drop_duplicates("material_id")
    .rename(
        columns={
            "Karakteristik değeri": "sap_yarn_count"
        }
    )
)

sap_yarn_count_knit["sap_yarn_count"] = (
    sap_yarn_count_knit["sap_yarn_count"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [167]:
# Prepare PLM knitting yarn count
plm_yarn_count_knit = (
    plm_codes[
        [
            "plm_code",
            "1.İplik numarası örme"
        ]
    ]
    .copy()
    .rename(
        columns={
            "1.İplik numarası örme": "plm_yarn_count"
        }
    )
)

plm_yarn_count_knit["plm_yarn_count"] = (
    plm_yarn_count_knit["plm_yarn_count"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [168]:
# Compare knitting yarn counts for known knitted-fabric mappings
yarn_count_pairs = (
    known_material_groups_model[
        known_material_groups_model["material_group_code"]
        == "1020001"
    ][
        ["material_id", "plm_code"]
    ]
    .merge(
        sap_yarn_count_knit,
        on="material_id",
        how="inner"
    )
    .merge(
        plm_yarn_count_knit,
        on="plm_code",
        how="inner"
    )
)

yarn_count_pairs = yarn_count_pairs[
    yarn_count_pairs["sap_yarn_count"].notna()
    & yarn_count_pairs["plm_yarn_count"].notna()
].copy()

yarn_count_pairs["yarn_count_match"] = (
    yarn_count_pairs["sap_yarn_count"]
    == yarn_count_pairs["plm_yarn_count"]
)

print(
    "Comparable yarn-count pairs:",
    len(yarn_count_pairs)
)

print(
    "Exact match rate:",
    f"{yarn_count_pairs['yarn_count_match'].mean():.2%}"
)

Comparable yarn-count pairs: 1007
Exact match rate: 87.98%


In [169]:
# Inspect common yarn-count mismatches
display(
    yarn_count_pairs[
        ~yarn_count_pairs["yarn_count_match"]
    ]
    .groupby(
        [
            "sap_yarn_count",
            "plm_yarn_count"
        ]
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        "count",
        ascending=False
    )
    .head(30)
)

,sap_yarn_count,plm_yarn_count,count
17,30/1,30,73
22,36/1,36,6
12,26/1,26,6
9,20/1,20,3
19,30/2,30,3
13,28/1,28,3
5,16,16/1,2
14,30/1,150 D,2
24,40/1,40,2
4,150 D,150,2


In [170]:
import re

# Parse yarn count into structured components
def parse_yarn_count(value):
    if pd.isna(value):
        return {
            "count": np.nan,
            "ply": np.nan,
            "unit": pd.NA
        }

    value = str(value).strip().upper()

    # Remove extra spaces
    value = re.sub(r"\s+", " ", value)

    # Detect count system
    unit = pd.NA

    if re.search(r"\bNE\b", value):
        unit = "NE"

    elif re.search(r"\bD\b", value):
        unit = "DENIER"

    # Extract count and optional ply
    match = re.search(
        r"(\d+(?:\.\d+)?)"
        r"(?:\s*/\s*(\d+))?",
        value
    )

    if not match:
        return {
            "count": np.nan,
            "ply": np.nan,
            "unit": unit
        }

    count = float(match.group(1))

    ply = (
        float(match.group(2))
        if match.group(2)
        else np.nan
    )

    return {
        "count": count,
        "ply": ply,
        "unit": unit
    }

In [171]:
# Compare parsed yarn counts conservatively
def calculate_yarn_count_match(sap_value, plm_value):

    sap = parse_yarn_count(sap_value)
    plm = parse_yarn_count(plm_value)

    if pd.isna(sap["count"]) or pd.isna(plm["count"]):
        return np.nan

    # Main yarn count must agree
    if sap["count"] != plm["count"]:
        return False

    # If both units are known, they must agree
    if (
        pd.notna(sap["unit"])
        and pd.notna(plm["unit"])
        and sap["unit"] != plm["unit"]
    ):
        return False

    # If both ply values are known, they must agree
    if (
        pd.notna(sap["ply"])
        and pd.notna(plm["ply"])
        and sap["ply"] != plm["ply"]
    ):
        return False

    # /1 and missing ply are treated as equivalent
    explicit_ply = (
        sap["ply"]
        if pd.notna(sap["ply"])
        else plm["ply"]
    )

    if pd.notna(explicit_ply) and explicit_ply != 1:
        # Do not assume that missing ply equals /2, /3, etc.
        if pd.isna(sap["ply"]) or pd.isna(plm["ply"]):
            return False

    return True

In [172]:
# Recalculate yarn-count agreement after semantic normalization
yarn_count_pairs["yarn_count_semantic_match"] = (
    yarn_count_pairs.apply(
        lambda row: calculate_yarn_count_match(
            row["sap_yarn_count"],
            row["plm_yarn_count"]
        ),
        axis=1
    )
)

comparable_semantic_yarn_pairs = (
    yarn_count_pairs[
        yarn_count_pairs[
            "yarn_count_semantic_match"
        ].notna()
    ]
)

print(
    "Semantic match rate:",
    f"{comparable_semantic_yarn_pairs['yarn_count_semantic_match'].mean():.2%}"
)

print(
    "Raw exact match rate:",
    f"{yarn_count_pairs['yarn_count_match'].mean():.2%}"
)

Semantic match rate: 98.61%
Raw exact match rate: 87.98%


In [173]:
# Prepare SAP knitting yarn type values
sap_yarn_type_knit = (
    fabric_attributes[
        fabric_attributes["field_name"] == "YARN1TYPEKNITSID"
    ][
        ["material_id", "Karakteristik değeri"]
    ]
    .dropna(subset=["Karakteristik değeri"])
    .copy()
)

sap_yarn_type_knit["yarn_type"] = (
    sap_yarn_type_knit["Karakteristik değeri"]
    .astype("string")
    .str.strip()
    .str.upper()
)

sap_yarn_type_sets = (
    sap_yarn_type_knit
    .groupby("material_id")["yarn_type"]
    .apply(set)
    .rename("sap_yarn_type_set")
    .reset_index()
)

In [174]:
# Prepare PLM knitting yarn type values
plm_yarn_type_knit = (
    plm_multi_value_valid[
        plm_multi_value_valid["PLM Karakteristik Tanımı"]
        == "YARN1TYPEKNITSID"
    ][
        ["plm_code", "PLM Karakteristik Değeri"]
    ]
    .dropna(subset=["PLM Karakteristik Değeri"])
    .copy()
)

plm_yarn_type_knit["yarn_type"] = (
    plm_yarn_type_knit["PLM Karakteristik Değeri"]
    .astype("string")
    .str.strip()
    .str.upper()
)

plm_yarn_type_sets = (
    plm_yarn_type_knit
    .groupby("plm_code")["yarn_type"]
    .apply(set)
    .rename("plm_yarn_type_set")
    .reset_index()
)

In [175]:
# Compare knitting yarn types for known knitted-fabric mappings
yarn_type_pairs = (
    known_material_groups_model[
        known_material_groups_model["material_group_code"]
        == "1020001"
    ][
        ["material_id", "plm_code"]
    ]
    .merge(
        sap_yarn_type_sets,
        on="material_id",
        how="inner"
    )
    .merge(
        plm_yarn_type_sets,
        on="plm_code",
        how="inner"
    )
)

yarn_type_pairs["yarn_type_similarity"] = (
    yarn_type_pairs.apply(
        lambda row: calculate_jaccard_similarity(
            row["sap_yarn_type_set"],
            row["plm_yarn_type_set"]
        ),
        axis=1
    )
)

print(
    "Comparable yarn-type pairs:",
    len(yarn_type_pairs)
)

print(
    "Exact yarn-type match:",
    f"{yarn_type_pairs['yarn_type_similarity'].eq(1).mean():.2%}"
)

print(
    "Mean similarity:",
    f"{yarn_type_pairs['yarn_type_similarity'].mean():.3f}"
)

Comparable yarn-type pairs: 724
Exact yarn-type match: 46.41%
Mean similarity: 0.464


In [176]:
# Inspect common yarn-type mismatches
yarn_type_mismatches = (
    yarn_type_pairs[
        yarn_type_pairs["yarn_type_similarity"] < 1
    ].copy()
)

yarn_type_mismatches["sap_yarn_type_tuple"] = (
    yarn_type_mismatches["sap_yarn_type_set"]
    .apply(lambda x: tuple(sorted(x)))
)

yarn_type_mismatches["plm_yarn_type_tuple"] = (
    yarn_type_mismatches["plm_yarn_type_set"]
    .apply(lambda x: tuple(sorted(x)))
)

display(
    yarn_type_mismatches[
        [
            "sap_yarn_type_tuple",
            "plm_yarn_type_tuple"
        ]
    ]
    .value_counts()
    .head(30)
)

sap_yarn_type_tuple      plm_yarn_type_tuple
(VORTEX,)                (VORTEKS,)             186
(COMBED,)                (COMPACTCOMBED,)       112
(COMBEDCOMPACT,)         (COMPACTCOMBED,)        45
(TEXTURED,)              (CONTINUEFLAMENT,)       9
(COMPACT,)               (COMPACTCOMBED,)         5
(RING,)                  (COMPACTCOMBED,)         4
(VORTEX,)                (VORTEKSCOMBED,)         3
(COMBED,)                (RING,)                  2
                         (RINGCOMBED,)            2
                         (RINGCARDED,)            1
                         (COMPACTCARDED,)         1
                         (COMPACT,)               1
(COMBED, RING)           (COMPACTCOMBED,)         1
(COMBING,)               (COMPACTCOMBED,)         1
(COMBEDCOMPACT,)         (RING,)                  1
                         (COMPACT,)               1
                         (COMPACTCARDED,)         1
(MICRO,)                 (CONTINUEFLAMENT,)       1
(ELASTHANEYARN,)   

In [177]:
# Normalize only clearly equivalent yarn-type labels
yarn_type_mapping = {
    "VORTEX": "VORTEKS",
    "COMBEDCOMPACT": "COMPACTCOMBED"
}


def normalize_yarn_type(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().upper()

    return yarn_type_mapping.get(
        value,
        value
    )

In [178]:
# Normalize SAP yarn-type sets
sap_yarn_type_sets_normalized = (
    sap_yarn_type_knit
    .assign(
        yarn_type_normalized=lambda df:
            df["yarn_type"].apply(normalize_yarn_type)
    )
    .groupby("material_id")["yarn_type_normalized"]
    .apply(set)
    .rename("sap_yarn_type_set")
    .reset_index()
)

# Normalize PLM yarn-type sets
plm_yarn_type_sets_normalized = (
    plm_yarn_type_knit
    .assign(
        yarn_type_normalized=lambda df:
            df["yarn_type"].apply(normalize_yarn_type)
    )
    .groupby("plm_code")["yarn_type_normalized"]
    .apply(set)
    .rename("plm_yarn_type_set")
    .reset_index()
)

In [179]:
# Recalculate yarn-type similarity after safe normalization
yarn_type_pairs_normalized = (
    known_material_groups_model[
        known_material_groups_model["material_group_code"]
        == "1020001"
    ][
        ["material_id", "plm_code"]
    ]
    .merge(
        sap_yarn_type_sets_normalized,
        on="material_id",
        how="inner"
    )
    .merge(
        plm_yarn_type_sets_normalized,
        on="plm_code",
        how="inner"
    )
)

yarn_type_pairs_normalized["yarn_type_similarity"] = (
    yarn_type_pairs_normalized.apply(
        lambda row: calculate_jaccard_similarity(
            row["sap_yarn_type_set"],
            row["plm_yarn_type_set"]
        ),
        axis=1
    )
)

print(
    "Comparable yarn-type pairs:",
    len(yarn_type_pairs_normalized)
)

print(
    "Normalized exact match:",
    f"{yarn_type_pairs_normalized['yarn_type_similarity'].eq(1).mean():.2%}"
)

print(
    "Mean similarity:",
    f"{yarn_type_pairs_normalized['yarn_type_similarity'].mean():.3f}"
)

Comparable yarn-type pairs: 724
Normalized exact match: 78.31%
Mean similarity: 0.783


In [180]:
# Prepare SAP weft yarn count values
sap_weft_yarn_count = (
    fabric_attributes[
        fabric_attributes["field_name"] == "WEFTYARNCOUNT1ID"
    ][
        ["material_id", "Karakteristik değeri"]
    ]
    .dropna(subset=["Karakteristik değeri"])
    .drop_duplicates("material_id")
    .rename(
        columns={
            "Karakteristik değeri": "sap_weft_yarn_count"
        }
    )
)

sap_weft_yarn_count["sap_weft_yarn_count"] = (
    sap_weft_yarn_count["sap_weft_yarn_count"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [181]:
# Prepare PLM weft yarn count values
plm_weft_yarn_count = (
    plm_codes[
        [
            "plm_code",
            "1.Atkı iplik numarası"
        ]
    ]
    .copy()
    .rename(
        columns={
            "1.Atkı iplik numarası":
            "plm_weft_yarn_count"
        }
    )
)

plm_weft_yarn_count["plm_weft_yarn_count"] = (
    plm_weft_yarn_count["plm_weft_yarn_count"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [182]:
# Compare weft yarn counts for known woven-fabric mappings
weft_yarn_count_pairs = (
    known_material_groups_model[
        known_material_groups_model["material_group_code"]
        == "1020002"
    ][
        ["material_id", "plm_code"]
    ]
    .merge(
        sap_weft_yarn_count,
        on="material_id",
        how="inner"
    )
    .merge(
        plm_weft_yarn_count,
        on="plm_code",
        how="inner"
    )
)

weft_yarn_count_pairs = (
    weft_yarn_count_pairs[
        weft_yarn_count_pairs["sap_weft_yarn_count"].notna()
        & weft_yarn_count_pairs["plm_weft_yarn_count"].notna()
    ]
    .copy()
)

In [183]:
# Calculate raw and semantic yarn-count agreement
weft_yarn_count_pairs["raw_match"] = (
    weft_yarn_count_pairs["sap_weft_yarn_count"]
    == weft_yarn_count_pairs["plm_weft_yarn_count"]
)

weft_yarn_count_pairs["semantic_match"] = (
    weft_yarn_count_pairs.apply(
        lambda row: calculate_yarn_count_match(
            row["sap_weft_yarn_count"],
            row["plm_weft_yarn_count"]
        ),
        axis=1
    )
)

comparable_weft_pairs = (
    weft_yarn_count_pairs[
        weft_yarn_count_pairs["semantic_match"].notna()
    ]
)

print(
    "Comparable weft yarn-count pairs:",
    len(comparable_weft_pairs)
)

print(
    "Raw exact match rate:",
    f"{comparable_weft_pairs['raw_match'].mean():.2%}"
)

print(
    "Semantic match rate:",
    f"{comparable_weft_pairs['semantic_match'].mean():.2%}"
)

Comparable weft yarn-count pairs: 887
Raw exact match rate: 90.19%
Semantic match rate: 98.76%


In [184]:
# Inspect semantic mismatches
display(
    comparable_weft_pairs[
        ~comparable_weft_pairs["semantic_match"]
    ]
    .groupby(
        [
            "sap_weft_yarn_count",
            "plm_weft_yarn_count"
        ]
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        "count",
        ascending=False
    )
    .head(30)
)

,sap_weft_yarn_count,plm_weft_yarn_count,count
5,28/2,28,4
0,100/96,100,1
2,20,16,1
1,12/2,"4,2",1
3,20/1,21,1
4,20/1,30,1
6,40/2,40,1
7,50 D,70,1


In [185]:
def parse_yarn_count(value):
    if pd.isna(value):
        return {
            "count": np.nan,
            "ply": np.nan,
            "unit": pd.NA
        }

    value = str(value).strip().upper()

    # Normalize decimal separator
    value = value.replace(",", ".")

    # Remove extra spaces
    value = re.sub(r"\s+", " ", value)

    # Detect yarn count system
    unit = pd.NA

    if re.search(r"\bNE\b", value):
        unit = "NE"

    elif re.search(r"\bD\b", value):
        unit = "DENIER"

    # Extract count and optional ply
    match = re.search(
        r"(\d+(?:\.\d+)?)"
        r"(?:\s*/\s*(\d+(?:\.\d+)?))?",
        value
    )

    if not match:
        return {
            "count": np.nan,
            "ply": np.nan,
            "unit": unit
        }

    count = float(match.group(1))

    ply = (
        float(match.group(2))
        if match.group(2)
        else np.nan
    )

    return {
        "count": count,
        "ply": ply,
        "unit": unit
    }

In [186]:
# Build empirical feature-agreement results
agreement_records = []

# WEIGHT
weight_group_map = {
    "ORME": "1020001",
    "DOKUMA": "1020002",
    "DENIM": "1030004"
}

for group_name, group in comparable_weight_pairs.groupby("material_group_name"):
    if group_name not in weight_group_map:
        continue

    agreement_records.append({
        "material_group": weight_group_map[group_name],
        "field_name": "WEIGHT",
        "agreement_rate": (group["weight_relative_error"] <= 0.05).mean(),
        "agreement_metric": "within_5_percent"
    })

# FABRICSTRUCTURETNAME
for group_name, group in fabric_structure_pairs.groupby("material_group_name"):
    if group_name not in weight_group_map:
        continue

    agreement_records.append({
        "material_group": weight_group_map[group_name],
        "field_name": "FABRICSTRUCTURETNAME",
        "agreement_rate": group["fabric_structure_match_normalized"].mean(),
        "agreement_metric": "normalized_exact_match"
    })

# COLORINGID
for group_name, group in coloring_pairs.groupby("material_group_name"):
    if group_name not in weight_group_map:
        continue

    agreement_records.append({
        "material_group": weight_group_map[group_name],
        "field_name": "COLORINGID",
        "agreement_rate": group["coloring_similarity"].eq(1).mean(),
        "agreement_metric": "exact_set_match"
    })

# ORME - YARNCOUNT1KNITSID
agreement_records.append({
    "material_group": "1020001",
    "field_name": "YARNCOUNT1KNITSID",
    "agreement_rate": yarn_count_pairs[
        "yarn_count_semantic_match"
    ].mean(),
    "agreement_metric": "semantic_match"
})

# ORME - YARN1TYPEKNITSID
agreement_records.append({
    "material_group": "1020001",
    "field_name": "YARN1TYPEKNITSID",
    "agreement_rate": yarn_type_pairs_normalized[
        "yarn_type_similarity"
    ].eq(1).mean(),
    "agreement_metric": "normalized_exact_set_match"
})

# DOKUMA - WEFTYARNCOUNT1ID
agreement_records.append({
    "material_group": "1020002",
    "field_name": "WEFTYARNCOUNT1ID",
    "agreement_rate": comparable_weft_pairs[
        "semantic_match"
    ].mean(),
    "agreement_metric": "semantic_match"
})

feature_agreement = pd.DataFrame(agreement_records)

In [187]:
# Add previously measured agreement results
previous_agreement = pd.DataFrame([
    {
        "material_group": "1020001",
        "field_name": "FIBERCONTENTLISTID",
        "agreement_rate": 0.9804,
        "agreement_metric": "exact_fiber_set_match_global"
    },
    {
        "material_group": "1020002",
        "field_name": "FIBERCONTENTLISTID",
        "agreement_rate": 0.9804,
        "agreement_metric": "exact_fiber_set_match_global"
    },
    {
        "material_group": "1030004",
        "field_name": "FIBERCONTENTLISTID",
        "agreement_rate": 0.9804,
        "agreement_metric": "exact_fiber_set_match_global"
    },
    {
        "material_group": "1020002",
        "field_name": "WEAVETYPE",
        "agreement_rate": 0.9715,
        "agreement_metric": "exact_match_previous_audit"
    },
    {
        "material_group": "1030004",
        "field_name": "WEAVETYPE",
        "agreement_rate": 0.9715,
        "agreement_metric": "exact_match_previous_audit"
    }
])

feature_agreement = pd.concat(
    [feature_agreement, previous_agreement],
    ignore_index=True
)

In [188]:
print(feature_policy_audit.columns.tolist())

['material_group_code', 'field_name', 'is_mandatory', 'in_short_description', 'short_description_order', 'in_long_description', 'long_description_order', 'conditional_coverage', 'feature_role', 'modeling_tier', 'usable_coverage', 'effective_coverage']


In [189]:
# Prepare feature policy audit for merging
final_feature_base = feature_policy_audit.copy()

# Standardize material group column name
if "material_group" not in final_feature_base.columns:
    if "material_group_code" in final_feature_base.columns:
        final_feature_base = final_feature_base.rename(
            columns={
                "material_group_code": "material_group"
            }
        )
    else:
        raise KeyError(
            "Neither 'material_group' nor 'material_group_code' "
            "exists in feature_policy_audit."
        )

# Standardize data types before merge
final_feature_base["material_group"] = (
    final_feature_base["material_group"]
    .astype("string")
)

feature_agreement["material_group"] = (
    feature_agreement["material_group"]
    .astype("string")
)

# Remove old agreement columns if this cell is rerun
columns_to_remove = [
    "agreement_rate",
    "agreement_metric",
    "evidence_score"
]

final_feature_base = final_feature_base.drop(
    columns=[
        col
        for col in columns_to_remove
        if col in final_feature_base.columns
    ],
    errors="ignore"
)

# Combine business policy, coverage and empirical agreement
final_feature_audit = (
    final_feature_base
    .merge(
        feature_agreement,
        on=["material_group", "field_name"],
        how="left"
    )
)

final_feature_audit["evidence_score"] = (
    final_feature_audit["effective_coverage"]
    * final_feature_audit["agreement_rate"]
)

In [190]:
# Keep only audited features in modeling scope
model_feature_audit = (
    final_feature_audit[
        final_feature_audit["material_group"].isin(
            ["1020001", "1020002", "1030004"]
        )
        & final_feature_audit["agreement_rate"].notna()
    ]
    .copy()
)

display(
    model_feature_audit[
        [
            "material_group",
            "field_name",
            "feature_role",
            "effective_coverage",
            "agreement_rate",
            "agreement_metric",
            "evidence_score"
        ]
    ]
    .sort_values(
        ["material_group", "evidence_score"],
        ascending=[True, False]
    )
)

,material_group,field_name,feature_role,effective_coverage,agreement_rate,agreement_metric,evidence_score
1,1020001,FIBERCONTENTLISTID,CORE,0.998908,0.980400,exact_fiber_set_match_global,0.979329
5,1020001,WEIGHT,MANDATORY,0.986892,0.987129,within_5_percent,0.974190
2,1020001,FABRICSTRUCTURETNAME,CORE,0.998908,0.944748,normalized_exact_match,0.943716
10,1020001,YARN1TYPEKNITSID,DESCRIPTION_SUPPORT,0.838340,0.783149,normalized_exact_set_match,0.656545
3,1020001,COLORINGID,CORE,0.876570,0.738376,exact_set_match,0.647238
0,1020001,YARNCOUNT1KNITSID,CORE,0.551065,0.986083,semantic_match,0.543396
12,1020002,WEIGHT,CORE,0.996479,0.983039,within_5_percent,0.979577
13,1020002,FIBERCONTENTLISTID,CORE,0.997183,0.980400,exact_fiber_set_match_global,0.977638
11,1020002,FABRICSTRUCTURETNAME,CORE,0.998592,0.920310,normalized_exact_match,0.919014
14,1020002,WEAVETYPE,CORE,0.907042,0.971500,exact_match_previous_audit,0.881192


In [191]:
# Assign final feature tiers using coverage, agreement and business role
def assign_final_feature_tier(row):
    coverage = row["effective_coverage"]
    agreement = row["agreement_rate"]
    role = row["feature_role"]

    if pd.isna(agreement):
        return "NOT_AUDITED"

    # Strong and broadly available evidence
    if coverage >= 0.80 and agreement >= 0.90:
        return "PRIMARY"

    # Reliable when available
    if coverage >= 0.40 and agreement >= 0.75:
        return "SECONDARY"

    # Business-important fields with moderate agreement
    if (
        role in ["CORE", "MANDATORY"]
        and coverage >= 0.40
        and agreement >= 0.70
    ):
        return "SECONDARY"

    if coverage >= 0.10 and agreement >= 0.70:
        return "OPTIONAL"

    return "EXCLUDE"


model_feature_audit["final_feature_tier"] = (
    model_feature_audit.apply(
        assign_final_feature_tier,
        axis=1
    )
)

display(
    model_feature_audit[
        [
            "material_group",
            "field_name",
            "feature_role",
            "effective_coverage",
            "agreement_rate",
            "evidence_score",
            "final_feature_tier"
        ]
    ]
    .sort_values(
        ["material_group", "final_feature_tier", "evidence_score"],
        ascending=[True, True, False]
    )
)

,material_group,field_name,feature_role,effective_coverage,agreement_rate,evidence_score,final_feature_tier
1,1020001,FIBERCONTENTLISTID,CORE,0.998908,0.980400,0.979329,PRIMARY
5,1020001,WEIGHT,MANDATORY,0.986892,0.987129,0.974190,PRIMARY
2,1020001,FABRICSTRUCTURETNAME,CORE,0.998908,0.944748,0.943716,PRIMARY
10,1020001,YARN1TYPEKNITSID,DESCRIPTION_SUPPORT,0.838340,0.783149,0.656545,SECONDARY
3,1020001,COLORINGID,CORE,0.876570,0.738376,0.647238,SECONDARY
0,1020001,YARNCOUNT1KNITSID,CORE,0.551065,0.986083,0.543396,SECONDARY
12,1020002,WEIGHT,CORE,0.996479,0.983039,0.979577,PRIMARY
13,1020002,FIBERCONTENTLISTID,CORE,0.997183,0.980400,0.977638,PRIMARY
11,1020002,FABRICSTRUCTURETNAME,CORE,0.998592,0.920310,0.919014,PRIMARY
14,1020002,WEAVETYPE,CORE,0.907042,0.971500,0.881192,PRIMARY


## 8. Aday üretimi (candidate generation)


In [192]:
import re
import unicodedata

# Normalize text for retrieval
def normalize_retrieval_text(value):
    if pd.isna(value):
        return ""

    value = str(value).upper().strip()

    # Normalize whitespace and punctuation
    value = re.sub(r"[^A-ZÇĞİÖŞÜ0-9]+", " ", value)
    value = re.sub(r"\s+", " ", value)

    return value.strip()

In [193]:
# Build SAP retrieval text
sap_candidates_source = (
    fabric_materials[
        [
            "material_id",
            "Mal grubu",
            "Türkçe malzeme açıklaması",
            "Türkçe malzeme Uzun açıklaması"
        ]
    ]
    .copy()
)

sap_candidates_source["material_group_code"] = (
    sap_candidates_source["Mal grubu"]
    .map({
        "ORME": "1020001",
        "DOKUMA": "1020002",
        "DENIM": "1030004"
    })
)

sap_candidates_source["retrieval_text"] = (
    sap_candidates_source[
        "Türkçe malzeme açıklaması"
    ].fillna("")
    + " "
    + sap_candidates_source[
        "Türkçe malzeme Uzun açıklaması"
    ].fillna("")
)

sap_candidates_source["retrieval_text"] = (
    sap_candidates_source["retrieval_text"]
    .apply(normalize_retrieval_text)
)

In [194]:
# Build PLM retrieval text
plm_candidates_source = (
    plm_codes[
        [
            "plm_code",
            "material_group_code",
            "Türkçe malzeme açıklaması",
            "Malzeme Türkçe Adı",
            "Malzeme ingilizce adı"
        ]
    ]
    .copy()
)

plm_candidates_source["retrieval_text"] = (
    plm_candidates_source[
        "Türkçe malzeme açıklaması"
    ].fillna("")
    + " "
    + plm_candidates_source[
        "Malzeme Türkçe Adı"
    ].fillna("")
    + " "
    + plm_candidates_source[
        "Malzeme ingilizce adı"
    ].fillna("")
)

plm_candidates_source["retrieval_text"] = (
    plm_candidates_source["retrieval_text"]
    .apply(normalize_retrieval_text)
)

In [195]:
# Check retrieval-text availability
print(
    "SAP materials:",
    len(sap_candidates_source)
)

print(
    "SAP with retrieval text:",
    (
        sap_candidates_source["retrieval_text"]
        .str.len()
        .gt(0)
        .mean()
    )
)

print(
    "PLM codes:",
    len(plm_candidates_source)
)

print(
    "PLM with retrieval text:",
    (
        plm_candidates_source["retrieval_text"]
        .str.len()
        .gt(0)
        .mean()
    )
)

SAP materials: 21083
SAP with retrieval text: 0.9999525684200541
PLM codes: 32361
PLM with retrieval text: 0.35564413955069374


In [196]:
display(
    sap_candidates_source[
        [
            "material_id",
            "material_group_code",
            "retrieval_text"
        ]
    ]
    .sample(10, random_state=42)
)

display(
    plm_candidates_source[
        [
            "plm_code",
            "material_group_code",
            "retrieval_text"
        ]
    ]
    .sample(10, random_state=42)
)

,material_id,material_group_code,retrieval_text
3921,1020004063,1020001,30 1 100 0 PAMUK SUP BYA 30 1 PENYE SÜPREM DÜZ...
13183,1020013720,1030004,KİPAŞ BCI KINSEY ULT BLACK 70BCI30CLY
6753,1020006940,1020002,MIA 70D 40D 150D 40D ARM 100 PES 8YJ AQ MIA 56...
12403,1020012835,1020002,GAB 16 1 63 PES 33 VIS 3EA GABARDIN 312
2862,1020002978,1020001,30 1 1 1 LYC RIB 30 1 RIBANA REAKTIF
16720,1020017516,1020001,100 0 PAMUK SUP IPLK BOYA FLAM PENYE SÜPREM İP...
11005,1020011318,1020002,24 1X20DEN VİS PA EL DOKUMA 115
4935,1020005081,1020001,30 1 LYC SUP G00 BEYAZ 30 1 LYC SUP G00 BEYAZ
15175,1020015832,1020001,4 0 ELASTAN 96 0 PAMUK KAŞKORSE 5 2 KAŞKORSE D...
4400,1020004545,1030004,ATLAS ANTONY GREY 168 CM 98 2 ATLAS ANTONY GRE...


,plm_code,material_group_code,retrieval_text
11513,351843,1020001,
31782,83947,1020001,DOUBLE FACE RIBANA
31115,53548,1030004,
11472,351796,1020002,
5191,196881,1020002,
14299,354817,1020002,
8477,214452,1020002,
1837,124118,1020002,
19777,360875,1020001,
30777,4726,1020001,


In [197]:
# PLM fields useful for candidate retrieval
plm_retrieval_fields = [
    "Kumaş tipi",
    "Dokuma tipi",
    "Örme alt tipi",
    "1.İplik numarası örme",
    "1.Atkı iplik numarası",
    "1.Çözgü iplik numarası",
    "Kumaş ağırlığı"
]


def combine_retrieval_fields(row, fields):
    values = []

    for field in fields:
        value = row.get(field)

        if pd.notna(value):
            value = normalize_retrieval_text(value)

            if value:
                values.append(value)

    return " ".join(values)


plm_structured_text = plm_codes[
    ["plm_code"] + plm_retrieval_fields
].copy()

plm_structured_text["structured_text"] = (
    plm_structured_text.apply(
        lambda row: combine_retrieval_fields(
            row,
            plm_retrieval_fields
        ),
        axis=1
    )
)

In [198]:
# Characteristics to include in retrieval representation
retrieval_multi_fields = [
    "FIBERCONTENTLISTID",
    "COLORINGID",
    "YARN1TYPEKNITSID"
]

plm_multi_retrieval = (
    plm_multi_value_valid[
        plm_multi_value_valid[
            "PLM Karakteristik Tanımı"
        ].isin(retrieval_multi_fields)
    ]
    .copy()
)

plm_multi_retrieval["retrieval_value"] = (
    plm_multi_retrieval[
        "PLM Karakteristik Değeri"
    ]
    .apply(normalize_retrieval_text)
)

plm_multi_text = (
    plm_multi_retrieval
    .groupby("plm_code")["retrieval_value"]
    .apply(
        lambda values:
            " ".join(sorted(set(values)))
    )
    .rename("multi_value_text")
    .reset_index()
)

In [199]:
# Enrich PLM retrieval text with structured characteristics
plm_candidates_enriched = (
    plm_candidates_source
    .merge(
        plm_structured_text[
            ["plm_code", "structured_text"]
        ],
        on="plm_code",
        how="left"
    )
    .merge(
        plm_multi_text,
        on="plm_code",
        how="left"
    )
)

plm_candidates_enriched["structured_text"] = (
    plm_candidates_enriched[
        "structured_text"
    ].fillna("")
)

plm_candidates_enriched["multi_value_text"] = (
    plm_candidates_enriched[
        "multi_value_text"
    ].fillna("")
)

plm_candidates_enriched["retrieval_text_enriched"] = (
    plm_candidates_enriched["retrieval_text"]
    + " "
    + plm_candidates_enriched["structured_text"]
    + " "
    + plm_candidates_enriched["multi_value_text"]
).str.strip()

In [200]:
# Compare original and enriched PLM retrieval coverage
original_coverage = (
    plm_candidates_enriched["retrieval_text"]
    .str.len()
    .gt(0)
    .mean()
)

enriched_coverage = (
    plm_candidates_enriched[
        "retrieval_text_enriched"
    ]
    .str.len()
    .gt(0)
    .mean()
)

print(
    "Original PLM text coverage:",
    f"{original_coverage:.2%}"
)

print(
    "Enriched PLM text coverage:",
    f"{enriched_coverage:.2%}"
)

Original PLM text coverage: 35.57%
Enriched PLM text coverage: 100.00%


In [201]:
display(
    plm_candidates_enriched
    .assign(
        has_retrieval_text=lambda df:
            df["retrieval_text_enriched"]
            .str.len()
            .gt(0)
    )
    .groupby("material_group_code")
    .agg(
        plm_codes=("plm_code", "nunique"),
        retrieval_coverage=(
            "has_retrieval_text",
            "mean"
        )
    )
)

,plm_codes,retrieval_coverage
material_group_code,,
1020001,12812,1.0
1020002,12087,1.0
1030004,3558,1.0


In [202]:
display(
    plm_candidates_enriched[
        [
            "plm_code",
            "material_group_code",
            "retrieval_text",
            "structured_text",
            "multi_value_text",
            "retrieval_text_enriched"
        ]
    ]
    .sample(10, random_state=42)
)

,plm_code,material_group_code,retrieval_text,structured_text,multi_value_text,retrieval_text_enriched
21528,362762,1020001,C0828,RIB2X2 210 0,5 00 ELASTHANE 95 00 POLIESTER,C0828 RIB2X2 210 0 5 00 ELASTHANE 95 00 POLIESTER
30697,45832,1030004,,DENIM LCWWEAVETYPELIST7 9 9,1 00 ELASTHANE 99 00 COTTON INDIGOINDIGO YARNDYED,DENIM LCWWEAVETYPELIST7 9 9 1 00 ELASTHANE 99 ...
11393,351708,1020002,AKBEY CEPLİK 1306 ECRU,PLAINWEAVE LCWWEAVETYPELIST1 125 0,23 00 COTTON 77 00 POLIESTER,AKBEY CEPLİK 1306 ECRU PLAINWEAVE LCWWEAVETYPE...
32260,ARGE108252,1020001,,INTERLOCK,30 00 POLIESTER 70 00 COTTON,INTERLOCK 30 00 POLIESTER 70 00 COTTON
4201,177409,1020001,KAPPA,INTERLOCK CIRCULAR 70 D 340 0,10 00 ELASTHANE 90 00 POLIESTER CONTINUEFLAMEN...,KAPPA INTERLOCK CIRCULAR 70 D 340 0 10 00 ELAS...
12376,352743,1020001,N,JERSEY 30 1 330 0,21 00 VISCOSE 3 00 ELASTHANE 76 00 POLIESTER S...,N JERSEY 30 1 330 0 21 00 VISCOSE 3 00 ELASTHA...
6390,207551,1020001,,INTERLOCK CIRCULAR 30 2 300 0,2 00 ELASTHANE 33 00 VISCOSE 65 00 POLIESTER Y...,INTERLOCK CIRCULAR 30 2 300 0 2 00 ELASTHANE 3...
13953,354440,<NA>,,IMITATIONLEATHER 275 0,100 00 POLYURETHANE SOLIDDYED,IMITATIONLEATHER 275 0 100 00 POLYURETHANE SOL...
14682,355254,1020001,,PUNTODIROMA CIRCULAR 28 1 350 0,22 00 VISCOSE 3 00 ELASTHANE 75 00 POLIESTER Y...,PUNTODIROMA CIRCULAR 28 1 350 0 22 00 VISCOSE ...
9125,2529,1020002,REFERENCENO PP12385 0H,PLAINWEAVE LCWWEAVETYPELIST1 40 125 0,100 00 COTTON REACTIVEDYED,REFERENCENO PP12385 0H PLAINWEAVE LCWWEAVETYPE...


In [203]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

In [204]:
# Generate top-k PLM candidates using character n-gram TF-IDF
def generate_tfidf_candidates(
    sap_df,
    plm_df,
    material_group_code,
    top_k=50
):
    sap_group = (
        sap_df[
            sap_df["material_group_code"] == material_group_code
        ]
        .copy()
        .reset_index(drop=True)
    )

    plm_group = (
        plm_df[
            plm_df["material_group_code"] == material_group_code
        ]
        .copy()
        .reset_index(drop=True)
    )

    # Keep only rows with usable retrieval text
    sap_group = sap_group[
        sap_group["retrieval_text"].str.len() > 0
    ].reset_index(drop=True)

    plm_group = plm_group[
        plm_group["retrieval_text_enriched"].str.len() > 0
    ].reset_index(drop=True)

    # Character n-grams are robust to spelling and formatting differences
    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=1,
        sublinear_tf=True,
        norm="l2"
    )

    plm_matrix = vectorizer.fit_transform(
        plm_group["retrieval_text_enriched"]
    )

    sap_matrix = vectorizer.transform(
        sap_group["retrieval_text"]
    )

    n_neighbors = min(
        top_k,
        len(plm_group)
    )

    nn_model = NearestNeighbors(
        n_neighbors=n_neighbors,
        metric="cosine",
        algorithm="brute"
    )

    nn_model.fit(plm_matrix)

    distances, indices = nn_model.kneighbors(
        sap_matrix
    )

    candidate_records = []

    for sap_idx in range(len(sap_group)):
        for rank, (plm_idx, distance) in enumerate(
            zip(indices[sap_idx], distances[sap_idx]),
            start=1
        ):
            candidate_records.append({
                "material_id":
                    sap_group.loc[sap_idx, "material_id"],
                "material_group_code":
                    material_group_code,
                "candidate_plm_code":
                    plm_group.loc[plm_idx, "plm_code"],
                "candidate_rank":
                    rank,
                "retrieval_similarity":
                    1 - distance
            })

    return pd.DataFrame(candidate_records)

In [205]:
# Generate candidates for each material group
candidate_tables = []

for material_group_code in [
    "1020001",  # ORME
    "1020002",  # DOKUMA
    "1030004"   # DENIM
]:
    group_candidates = generate_tfidf_candidates(
        sap_candidates_source,
        plm_candidates_enriched,
        material_group_code=material_group_code,
        top_k=50
    )

    candidate_tables.append(
        group_candidates
    )

tfidf_candidates = pd.concat(
    candidate_tables,
    ignore_index=True
)

print(
    "Candidate pairs:",
    len(tfidf_candidates)
)

display(
    tfidf_candidates.head(10)
)

Candidate pairs: 1054100


,material_id,material_group_code,candidate_plm_code,candidate_rank,retrieval_similarity
0,1020000002,1020001,361215,1,0.379164
1,1020000002,1020001,358349,2,0.375990
2,1020000002,1020001,354454,3,0.367678
3,1020000002,1020001,359991,4,0.362639
4,1020000002,1020001,366601,5,0.205123
5,1020000002,1020001,363436,6,0.198500
6,1020000002,1020001,353396,7,0.197695
7,1020000002,1020001,366056,8,0.197640
8,1020000002,1020001,366023,9,0.194396
9,1020000002,1020001,356138,10,0.183935


In [206]:
# Prepare known SAP-PLM mappings for retrieval evaluation
known_retrieval_pairs = (
    known_material_groups_model[
        [
            "material_id",
            "plm_code",
            "material_group_code"
        ]
    ]
    .dropna(subset=["plm_code"])
    .copy()
)

retrieval_evaluation = (
    known_retrieval_pairs
    .merge(
        tfidf_candidates,
        left_on=[
            "material_id",
            "material_group_code",
            "plm_code"
        ],
        right_on=[
            "material_id",
            "material_group_code",
            "candidate_plm_code"
        ],
        how="left"
    )
)

In [207]:
# Calculate candidate-generation recall
def calculate_recall_at_k(df, k):
    return (
        df["candidate_rank"]
        .le(k)
        .fillna(False)
        .mean()
    )


for k in [1, 3, 5, 10, 20, 50]:
    print(
        f"Recall@{k}:",
        f"{calculate_recall_at_k(retrieval_evaluation, k):.2%}"
    )

Recall@1: 3.69%
Recall@3: 6.78%
Recall@5: 8.64%
Recall@10: 13.17%
Recall@20: 19.50%
Recall@50: 30.83%


In [208]:
# Evaluate retrieval recall by material group
retrieval_group_summary = []

for group_name, group in (
    retrieval_evaluation
    .merge(
        known_material_groups_model[
            [
                "material_id",
                "material_group_name"
            ]
        ].drop_duplicates("material_id"),
        on="material_id",
        how="left"
    )
    .groupby("material_group_name")
):
    retrieval_group_summary.append({
        "material_group_name": group_name,
        "known_pairs": len(group),
        "recall_at_1":
            calculate_recall_at_k(group, 1),
        "recall_at_5":
            calculate_recall_at_k(group, 5),
        "recall_at_10":
            calculate_recall_at_k(group, 10),
        "recall_at_20":
            calculate_recall_at_k(group, 20),
        "recall_at_50":
            calculate_recall_at_k(group, 50)
    })

retrieval_group_summary = pd.DataFrame(
    retrieval_group_summary
)

display(retrieval_group_summary)

,material_group_name,known_pairs,recall_at_1,recall_at_5,recall_at_10,recall_at_20,recall_at_50
0,DENIM,280,0.110714,0.185714,0.239286,0.321429,0.432143
1,DOKUMA,2399,0.055857,0.143393,0.224677,0.334306,0.508128
2,ORME,2956,0.014547,0.030785,0.046008,0.070027,0.134303


In [209]:
# Add diagnostic flags to retrieval evaluation
materials_with_attributes = set(
    fabric_attributes["material_id"]
    .dropna()
    .astype("string")
)

plm_original_text_lookup = (
    plm_candidates_source[
        ["plm_code", "retrieval_text"]
    ]
    .drop_duplicates("plm_code")
    .set_index("plm_code")["retrieval_text"]
)

retrieval_diagnostic = (
    retrieval_evaluation.copy()
)

retrieval_diagnostic["has_sap_attributes"] = (
    retrieval_diagnostic["material_id"]
    .astype("string")
    .isin(materials_with_attributes)
)

retrieval_diagnostic["target_plm_has_original_text"] = (
    retrieval_diagnostic["plm_code"]
    .map(plm_original_text_lookup)
    .fillna("")
    .str.len()
    .gt(0)
)

In [210]:
# Recall by SAP attribute availability
attribute_recall_summary = []

for has_attributes, group in (
    retrieval_diagnostic
    .groupby("has_sap_attributes")
):
    attribute_recall_summary.append({
        "has_sap_attributes": has_attributes,
        "known_pairs": len(group),
        "recall_at_10": calculate_recall_at_k(group, 10),
        "recall_at_20": calculate_recall_at_k(group, 20),
        "recall_at_50": calculate_recall_at_k(group, 50)
    })

display(
    pd.DataFrame(attribute_recall_summary)
)

,has_sap_attributes,known_pairs,recall_at_10,recall_at_20,recall_at_50
0,False,2122,0.154100,0.212535,0.305372
1,True,3513,0.118133,0.184458,0.309991


In [211]:
# Recall by target PLM description availability
target_text_recall_summary = []

for has_text, group in (
    retrieval_diagnostic
    .groupby("target_plm_has_original_text")
):
    target_text_recall_summary.append({
        "target_plm_has_original_text": has_text,
        "known_pairs": len(group),
        "recall_at_10": calculate_recall_at_k(group, 10),
        "recall_at_20": calculate_recall_at_k(group, 20),
        "recall_at_50": calculate_recall_at_k(group, 50)
    })

display(
    pd.DataFrame(target_text_recall_summary)
)

,target_plm_has_original_text,known_pairs,recall_at_10,recall_at_20,recall_at_50
0,False,3723,0.149073,0.224281,0.368251
1,True,1912,0.097803,0.138075,0.191423


In [212]:
# Normalize fabric structure for candidate generation
def normalize_fabric_structure(value):
    if pd.isna(value):
        return pd.NA

    value = (
        str(value)
        .strip()
        .upper()
    )

    mapping = {
        "POLYVISCON": "POLYVISCOSE",
        "BEZAYAĞI": "PLAINWEAVE"
    }

    return mapping.get(value, value)

In [213]:
# Prepare SAP fabric structures
sap_structure_candidates = (
    fabric_attributes[
        fabric_attributes["field_name"]
        == "FABRICSTRUCTURETNAME"
    ][
        ["material_id", "Karakteristik değeri"]
    ]
    .dropna(subset=["Karakteristik değeri"])
    .drop_duplicates("material_id")
    .copy()
)

sap_structure_candidates["structure"] = (
    sap_structure_candidates[
        "Karakteristik değeri"
    ]
    .apply(normalize_fabric_structure)
)

In [214]:
# Prepare PLM fabric structures
plm_structure_candidates = (
    plm_codes[
        [
            "plm_code",
            "material_group_code",
            "Kumaş tipi"
        ]
    ]
    .dropna(subset=["Kumaş tipi"])
    .copy()
)

plm_structure_candidates["structure"] = (
    plm_structure_candidates["Kumaş tipi"]
    .apply(normalize_fabric_structure)
)

In [215]:
# Generate candidates from exact normalized fabric structure
structure_candidates = (
    sap_candidates_source[
        [
            "material_id",
            "material_group_code"
        ]
    ]
    .merge(
        sap_structure_candidates[
            ["material_id", "structure"]
        ],
        on="material_id",
        how="inner"
    )
    .merge(
        plm_structure_candidates[
            [
                "plm_code",
                "material_group_code",
                "structure"
            ]
        ],
        on=[
            "material_group_code",
            "structure"
        ],
        how="inner"
    )
    .rename(
        columns={
            "plm_code": "candidate_plm_code"
        }
    )
)

structure_candidates["candidate_source"] = (
    "fabric_structure"
)

print(
    "Structure candidate pairs:",
    len(structure_candidates)
)

print(
    "SAP materials with structure candidates:",
    structure_candidates[
        "material_id"
    ].nunique()
)

Structure candidate pairs: 10618940
SAP materials with structure candidates: 7466


In [216]:
# Evaluate structure-channel candidate recall
structure_recall = (
    known_retrieval_pairs
    .merge(
        structure_candidates[
            [
                "material_id",
                "candidate_plm_code"
            ]
        ].drop_duplicates(),
        left_on=[
            "material_id",
            "plm_code"
        ],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left",
        indicator=True
    )
)

structure_recall["retrieved"] = (
    structure_recall["_merge"]
    == "both"
)

print(
    "Structure channel recall:",
    f"{structure_recall['retrieved'].mean():.2%}"
)

Structure channel recall: 58.24%


In [217]:
structure_available = (
    structure_recall["material_id"]
    .isin(
        set(
            sap_structure_candidates[
                "material_id"
            ]
        )
    )
)

print(
    "Structure recall when SAP structure exists:",
    f"{structure_recall.loc[structure_available, 'retrieved'].mean():.2%}"
)

Structure recall when SAP structure exists: 93.53%


In [218]:
# Prepare SAP structure + weight representation
sap_structure_weight = (
    sap_candidates_source[
        [
            "material_id",
            "material_group_code"
        ]
    ]
    .merge(
        sap_structure_candidates[
            ["material_id", "structure"]
        ],
        on="material_id",
        how="inner"
    )
    .merge(
        sap_weight_audit[
            [
                "material_id",
                "sap_weight_raw",
                "sap_weight_unit"
            ]
        ],
        on="material_id",
        how="left"
    )
)

sap_structure_weight = sap_structure_weight[
    sap_structure_weight["sap_weight_raw"].notna()
    & (sap_structure_weight["sap_weight_raw"] > 0)
].copy()

In [219]:
# Prepare PLM structure + weight representation
plm_structure_weight = (
    plm_structure_candidates[
        [
            "plm_code",
            "material_group_code",
            "structure"
        ]
    ]
    .merge(
        plm_weight_audit[
            [
                "plm_code",
                "plm_weight",
                "plm_weight_unit"
            ]
        ],
        on="plm_code",
        how="left"
    )
)

plm_structure_weight = plm_structure_weight[
    plm_structure_weight["plm_weight"].notna()
    & (plm_structure_weight["plm_weight"] > 0)
].copy()

In [220]:
# Build a fast lookup index
plm_structure_index = {
    key: group.reset_index(drop=True)
    for key, group in (
        plm_structure_weight
        .groupby(
            [
                "material_group_code",
                "structure"
            ],
            dropna=False
        )
    )
}

In [221]:
# Generate candidates that agree on structure
# and are within a weight tolerance
def generate_structure_weight_candidates(
    sap_df,
    plm_index,
    weight_tolerance=0.10
):
    records = []

    for row in sap_df.itertuples(index=False):

        key = (
            row.material_group_code,
            row.structure
        )

        plm_pool = plm_index.get(key)

        if plm_pool is None or plm_pool.empty:
            continue

        sap_weight = row.sap_weight_raw
        sap_unit = row.sap_weight_unit

        plm_weights = (
            plm_pool["plm_weight"]
            .astype(float)
            .to_numpy()
        )

        # Test both observed SAP scale patterns
        same_scale_error = (
            np.abs(sap_weight - plm_weights)
            / plm_weights
        )

        x1000_error = (
            np.abs((sap_weight / 1000) - plm_weights)
            / plm_weights
        )

        best_error = np.minimum(
            same_scale_error,
            x1000_error
        )

        valid = best_error <= weight_tolerance

        # If both units exist, require agreement
        if pd.notna(sap_unit):

            plm_units = (
                plm_pool["plm_weight_unit"]
                .astype("string")
            )

            unit_valid = (
                plm_units.isna()
                | (plm_units == str(sap_unit))
            ).to_numpy()

            valid = valid & unit_valid

        candidate_indices = np.where(valid)[0]

        for idx in candidate_indices:

            records.append(
                (
                    row.material_id,
                    row.material_group_code,
                    plm_pool.iloc[idx]["plm_code"],
                    best_error[idx]
                )
            )

    return pd.DataFrame(
        records,
        columns=[
            "material_id",
            "material_group_code",
            "candidate_plm_code",
            "weight_relative_error"
        ]
    )

In [222]:
structure_weight_candidates = (
    generate_structure_weight_candidates(
        sap_structure_weight,
        plm_structure_index,
        weight_tolerance=0.10
    )
)

structure_weight_candidates[
    "candidate_source"
] = "structure_weight"

print(
    "Structure + weight candidate pairs:",
    len(structure_weight_candidates)
)

print(
    "SAP materials covered:",
    structure_weight_candidates[
        "material_id"
    ].nunique()
)

print(
    "Average candidates per covered SAP:",
    len(structure_weight_candidates)
    / structure_weight_candidates["material_id"].nunique()
)

Structure + weight candidate pairs: 1757470
SAP materials covered: 6079
Average candidates per covered SAP: 289.1051159730219


In [223]:
# Evaluate composite structured channel
structure_weight_recall = (
    known_retrieval_pairs
    .merge(
        structure_weight_candidates[
            [
                "material_id",
                "candidate_plm_code"
            ]
        ].drop_duplicates(),
        left_on=[
            "material_id",
            "plm_code"
        ],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left",
        indicator=True
    )
)

structure_weight_recall["retrieved"] = (
    structure_weight_recall["_merge"]
    == "both"
)

print(
    "Overall structure + weight recall:",
    f"{structure_weight_recall['retrieved'].mean():.2%}"
)

Overall structure + weight recall: 57.32%


In [224]:
# Evaluate only materials where both SAP features exist
available_structure_weight_materials = set(
    sap_structure_weight["material_id"]
)

available_mask = (
    structure_weight_recall["material_id"]
    .isin(available_structure_weight_materials)
)

print(
    "Conditional structure + weight recall:",
    f"{structure_weight_recall.loc[available_mask, 'retrieved'].mean():.2%}"
)

Conditional structure + weight recall: 92.74%


In [225]:
# Combine text and structured candidate channels
text_candidate_pairs = (
    tfidf_candidates[
        [
            "material_id",
            "candidate_plm_code"
        ]
    ]
    .drop_duplicates()
)

structured_candidate_pairs = (
    structure_weight_candidates[
        [
            "material_id",
            "candidate_plm_code"
        ]
    ]
    .drop_duplicates()
)

combined_candidates = (
    pd.concat(
        [
            text_candidate_pairs,
            structured_candidate_pairs
        ],
        ignore_index=True
    )
    .drop_duplicates()
)

combined_recall = (
    known_retrieval_pairs
    .merge(
        combined_candidates,
        left_on=[
            "material_id",
            "plm_code"
        ],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left",
        indicator=True
    )
)

print(
    "Text + structure/weight union recall:",
    f"{(combined_recall['_merge'] == 'both').mean():.2%}"
)

Text + structure/weight union recall: 69.60%


In [226]:
# Evaluate combined candidate recall by SAP attribute availability
combined_recall_diagnostic = (
    known_retrieval_pairs
    .merge(
        combined_candidates,
        left_on=["material_id", "plm_code"],
        right_on=["material_id", "candidate_plm_code"],
        how="left",
        indicator=True
    )
)

combined_recall_diagnostic["retrieved"] = (
    combined_recall_diagnostic["_merge"] == "both"
)

combined_recall_diagnostic["has_sap_attributes"] = (
    combined_recall_diagnostic["material_id"]
    .astype("string")
    .isin(materials_with_attributes)
)

display(
    combined_recall_diagnostic
    .groupby("has_sap_attributes")
    .agg(
        known_pairs=("material_id", "size"),
        candidate_recall=("retrieved", "mean")
    )
)

,known_pairs,candidate_recall
has_sap_attributes,,
False,2122,0.305372
True,3513,0.931967


In [227]:
# Normalize fiber names for retrieval
def normalize_fiber_for_retrieval(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().upper()

    # Remove leading percentage / numeric composition
    value = re.sub(
        r"^\s*\d+(?:[.,]\d+)?\s*%?\s*",
        "",
        value
    ).strip()

    if not value or re.fullmatch(r"[\d.,]+", value):
        return pd.NA

    mapping = {
        "POLIESTER": "POLYESTER",
        "ELASTHANE": "ELASTANE",
        "SPANDEX": "ELASTANE",
        "POLYAMIDE6": "POLYAMIDE",
        "POLIAMID6": "POLYAMIDE",
        "TENCEL": "LYOCELL",
        "ACETAT": "ACETATE"
    }

    return mapping.get(value, value)

In [228]:
# Prepare SAP fiber-name sets
sap_fiber_candidates = (
    fabric_attributes[
        fabric_attributes["field_name"]
        == "FIBERCONTENTLISTID"
    ][
        ["material_id", "Karakteristik değeri"]
    ]
    .copy()
)

sap_fiber_candidates["fiber_name"] = (
    sap_fiber_candidates["Karakteristik değeri"]
    .apply(normalize_fiber_for_retrieval)
)

sap_fiber_sets = (
    sap_fiber_candidates
    .dropna(subset=["fiber_name"])
    .groupby("material_id")["fiber_name"]
    .apply(lambda x: tuple(sorted(set(x))))
    .rename("fiber_key")
    .reset_index()
)

In [229]:
# Prepare PLM fiber-name sets
plm_fiber_candidates = (
    plm_multi_value_valid[
        plm_multi_value_valid[
            "PLM Karakteristik Tanımı"
        ] == "FIBERCONTENTLISTID"
    ][
        ["plm_code", "PLM Karakteristik Değeri"]
    ]
    .copy()
)

plm_fiber_candidates["fiber_name"] = (
    plm_fiber_candidates[
        "PLM Karakteristik Değeri"
    ]
    .apply(normalize_fiber_for_retrieval)
)

plm_fiber_sets = (
    plm_fiber_candidates
    .dropna(subset=["fiber_name"])
    .groupby("plm_code")["fiber_name"]
    .apply(lambda x: tuple(sorted(set(x))))
    .rename("fiber_key")
    .reset_index()
)

In [230]:
# SAP fiber + weight representation
sap_fiber_weight = (
    sap_candidates_source[
        ["material_id", "material_group_code"]
    ]
    .merge(
        sap_fiber_sets,
        on="material_id",
        how="inner"
    )
    .merge(
        sap_weight_audit[
            [
                "material_id",
                "sap_weight_raw",
                "sap_weight_unit"
            ]
        ],
        on="material_id",
        how="inner"
    )
)

sap_fiber_weight = sap_fiber_weight[
    sap_fiber_weight["sap_weight_raw"].notna()
    & (sap_fiber_weight["sap_weight_raw"] > 0)
].copy()

In [231]:
# PLM fiber + weight representation
plm_fiber_weight = (
    plm_codes[
        ["plm_code", "material_group_code"]
    ]
    .merge(
        plm_fiber_sets,
        on="plm_code",
        how="inner"
    )
    .merge(
        plm_weight_audit[
            [
                "plm_code",
                "plm_weight",
                "plm_weight_unit"
            ]
        ],
        on="plm_code",
        how="inner"
    )
)

plm_fiber_weight = plm_fiber_weight[
    plm_fiber_weight["plm_weight"].notna()
    & (plm_fiber_weight["plm_weight"] > 0)
].copy()

In [232]:
# Build PLM lookup by material group and fiber set
plm_fiber_index = {
    key: group.reset_index(drop=True)
    for key, group in (
        plm_fiber_weight
        .groupby(
            ["material_group_code", "fiber_key"],
            dropna=False
        )
    )
}

In [233]:
# Generate exact-fiber + weight candidates
def generate_fiber_weight_candidates(
    sap_df,
    plm_index,
    weight_tolerance=0.10
):
    records = []

    for row in sap_df.itertuples(index=False):

        key = (
            row.material_group_code,
            row.fiber_key
        )

        plm_pool = plm_index.get(key)

        if plm_pool is None or plm_pool.empty:
            continue

        plm_weights = (
            plm_pool["plm_weight"]
            .astype(float)
            .to_numpy()
        )

        same_scale_error = (
            np.abs(row.sap_weight_raw - plm_weights)
            / plm_weights
        )

        x1000_error = (
            np.abs(
                (row.sap_weight_raw / 1000)
                - plm_weights
            )
            / plm_weights
        )

        best_error = np.minimum(
            same_scale_error,
            x1000_error
        )

        valid = best_error <= weight_tolerance

        # Only enforce unit when SAP unit is known
        if pd.notna(row.sap_weight_unit):

            plm_units = (
                plm_pool["plm_weight_unit"]
                .astype("string")
            )

            unit_valid = (
                plm_units.isna()
                | (
                    plm_units
                    == str(row.sap_weight_unit)
                )
            ).to_numpy()

            valid = valid & unit_valid

        for idx in np.where(valid)[0]:

            records.append({
                "material_id": row.material_id,
                "material_group_code":
                    row.material_group_code,
                "candidate_plm_code":
                    plm_pool.iloc[idx]["plm_code"],
                "weight_relative_error":
                    best_error[idx],
                "candidate_source":
                    "fiber_weight"
            })

    return pd.DataFrame(records)

In [234]:
fiber_weight_candidates = (
    generate_fiber_weight_candidates(
        sap_fiber_weight,
        plm_fiber_index,
        weight_tolerance=0.10
    )
)

print(
    "Fiber + weight candidate pairs:",
    len(fiber_weight_candidates)
)

print(
    "SAP materials covered:",
    fiber_weight_candidates[
        "material_id"
    ].nunique()
)

print(
    "Average candidates per covered SAP:",
    len(fiber_weight_candidates)
    / fiber_weight_candidates["material_id"].nunique()
)

Fiber + weight candidate pairs: 1468405
SAP materials covered: 6377
Average candidates per covered SAP: 230.26579896503057


In [235]:
fiber_weight_recall = (
    known_retrieval_pairs
    .merge(
        fiber_weight_candidates[
            ["material_id", "candidate_plm_code"]
        ].drop_duplicates(),
        left_on=["material_id", "plm_code"],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left",
        indicator=True
    )
)

fiber_weight_recall["retrieved"] = (
    fiber_weight_recall["_merge"] == "both"
)

available_fiber_weight_materials = set(
    sap_fiber_weight["material_id"]
)

available_mask = (
    fiber_weight_recall["material_id"]
    .isin(available_fiber_weight_materials)
)

print(
    "Overall fiber + weight recall:",
    f"{fiber_weight_recall['retrieved'].mean():.2%}"
)

print(
    "Conditional fiber + weight recall:",
    f"{fiber_weight_recall.loc[available_mask, 'retrieved'].mean():.2%}"
)

Overall fiber + weight recall: 60.37%
Conditional fiber + weight recall: 97.70%


In [236]:
# Union of text, structure-weight and fiber-weight channels
all_candidate_pairs = (
    pd.concat(
        [
            tfidf_candidates[
                ["material_id", "candidate_plm_code"]
            ],
            structure_weight_candidates[
                ["material_id", "candidate_plm_code"]
            ],
            fiber_weight_candidates[
                ["material_id", "candidate_plm_code"]
            ]
        ],
        ignore_index=True
    )
    .drop_duplicates()
)

all_candidate_recall = (
    known_retrieval_pairs
    .merge(
        all_candidate_pairs,
        left_on=["material_id", "plm_code"],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left",
        indicator=True
    )
)

all_candidate_recall["retrieved"] = (
    all_candidate_recall["_merge"] == "both"
)

print(
    "Combined candidate recall:",
    f"{all_candidate_recall['retrieved'].mean():.2%}"
)

Combined candidate recall: 72.42%


Bu sonuç mimariyi bayağı netleştirdi:

Attribute'u olan SAP'lerde: text + structure/weight ile bile recall %93,20.
fiber + weight tek başına, kullanılabildiği materyallerde %97,70 conditional recall veriyor.
Attribute'u olmayan SAP'lerde: şu an text-only recall sadece %30,54.
Dolayısıyla overall %72,42 düşük görünse de asıl problem structured tarafta değil; attribute'suz grup.

Şu aşamada iki ayrı pipeline düşünmek daha doğru:

A) SAP attributes available
   → structured candidate generation
   → text supplementary channel
   → pairwise reranking

B) SAP attributes unavailable
   → text-only / description-derived retrieval
   → lower-confidence recommendation

In [237]:
# Evaluate final candidate union by SAP attribute availability
final_candidate_recall = (
    known_retrieval_pairs
    .merge(
        all_candidate_pairs,
        left_on=["material_id", "plm_code"],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left",
        indicator=True
    )
)

final_candidate_recall["retrieved"] = (
    final_candidate_recall["_merge"] == "both"
)

final_candidate_recall["has_sap_attributes"] = (
    final_candidate_recall["material_id"]
    .astype("string")
    .isin(materials_with_attributes)
)

display(
    final_candidate_recall
    .groupby("has_sap_attributes")
    .agg(
        known_pairs=("material_id", "size"),
        candidate_recall=("retrieved", "mean")
    )
)

,known_pairs,candidate_recall
has_sap_attributes,,
False,2122,0.305372
True,3513,0.977227


In [238]:
# Evaluate attribute-rich candidate recall by material group
final_candidate_recall_grouped = (
    final_candidate_recall
    .merge(
        known_material_groups_model[
            [
                "material_id",
                "material_group_name"
            ]
        ].drop_duplicates("material_id"),
        on="material_id",
        how="left"
    )
)

display(
    final_candidate_recall_grouped[
        final_candidate_recall_grouped[
            "has_sap_attributes"
        ]
    ]
    .groupby("material_group_name")
    .agg(
        known_pairs=("material_id", "size"),
        candidate_recall=("retrieved", "mean")
    )
)

,known_pairs,candidate_recall
material_group_name,,
DENIM,262,0.996183
DOKUMA,1420,0.983099
ORME,1831,0.969962


In [239]:
# Candidate pool size after all retrieval channels
candidate_pool_size = (
    all_candidate_pairs
    .groupby("material_id")
    .size()
    .rename("candidate_count")
)

print(
    candidate_pool_size.describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

count    21082.000000
mean       183.266958
std        303.360004
min         49.000000
50%         50.000000
75%         66.000000
90%        642.000000
95%        942.000000
99%       1392.000000
max       1895.000000
Name: candidate_count, dtype: float64


In [240]:
attribute_candidate_pool_size = (
    candidate_pool_size[
        candidate_pool_size.index.astype("string")
        .isin(materials_with_attributes)
    ]
)

print(
    attribute_candidate_pool_size.describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

count    9465.000000
mean      346.845747
std       395.507062
min        49.000000
50%       116.000000
75%       591.000000
90%       982.600000
95%      1184.600000
99%      1474.000000
max      1895.000000
Name: candidate_count, dtype: float64


## 9. Pairwise reranker — ilk sürüm

> ⚠️ **SUPERSEDED.** 243–295 arası `candidate_master` / `pairwise_training`
> sürümü, her SAP–PLM çifti için birden fazla satır üretiyordu (bkz. 293–295
> teşhis hücreleri). Tüm blok 296–332'de `*_clean` tabloları ile yeniden
> kurulmuştur ve geçerli sonuçlar oradadır.
>
> Blok silinmedi çünkü 264 (`sap_coloring_model` / `plm_coloring_model`) ve
> 265 (`sap_weave_type` / `plm_weave_type`) hücreleri temiz sürümde de
> kullanılıyor.


In [241]:
# Preserve retrieval-source information
text_channel = (
    tfidf_candidates[
        [
            "material_id",
            "candidate_plm_code",
            "retrieval_similarity"
        ]
    ]
    .copy()
)

text_channel["from_text"] = 1

structure_channel = (
    structure_weight_candidates[
        [
            "material_id",
            "candidate_plm_code",
            "weight_relative_error"
        ]
    ]
    .copy()
)

structure_channel["from_structure_weight"] = 1

fiber_channel = (
    fiber_weight_candidates[
        [
            "material_id",
            "candidate_plm_code",
            "weight_relative_error"
        ]
    ]
    .copy()
)

fiber_channel["from_fiber_weight"] = 1

In [242]:
# Create candidate master table with retrieval-channel flags
candidate_master = (
    all_candidate_pairs
    .merge(
        text_channel,
        on=["material_id", "candidate_plm_code"],
        how="left"
    )
    .merge(
        structure_channel,
        on=["material_id", "candidate_plm_code"],
        how="left",
        suffixes=("", "_structure")
    )
    .merge(
        fiber_channel,
        on=["material_id", "candidate_plm_code"],
        how="left",
        suffixes=("", "_fiber")
    )
)

for column in [
    "from_text",
    "from_structure_weight",
    "from_fiber_weight"
]:
    candidate_master[column] = (
        candidate_master[column]
        .fillna(0)
        .astype(int)
    )

In [243]:
# Count how many independent retrieval channels selected each pair
candidate_master["retrieval_channel_count"] = (
    candidate_master[
        [
            "from_text",
            "from_structure_weight",
            "from_fiber_weight"
        ]
    ].sum(axis=1)
)

display(
    candidate_master[
        "retrieval_channel_count"
    ].value_counts()
)

retrieval_channel_count
1    3463515
2     385221
3      15363
Name: count, dtype: int64

In [244]:
print(candidate_master.columns.tolist())

['material_id', 'candidate_plm_code', 'retrieval_similarity', 'from_text', 'weight_relative_error', 'from_structure_weight', 'weight_relative_error_fiber', 'from_fiber_weight', 'retrieval_channel_count']


In [245]:
# Add material group back to candidate master
material_group_lookup = (
    sap_candidates_source[
        [
            "material_id",
            "material_group_code"
        ]
    ]
    .drop_duplicates("material_id")
)

candidate_master = (
    candidate_master
    .drop(
        columns=["material_group_code"],
        errors="ignore"
    )
    .merge(
        material_group_lookup,
        on="material_id",
        how="left"
    )
)

In [246]:
print(
    candidate_master[
        "material_group_code"
    ].value_counts(dropna=False)
)

print(
    candidate_master.columns.tolist()
)

material_group_code
1020001    2191890
1020002    1255243
1030004     416966
Name: count, dtype: int64
['material_id', 'candidate_plm_code', 'retrieval_similarity', 'from_text', 'weight_relative_error', 'from_structure_weight', 'weight_relative_error_fiber', 'from_fiber_weight', 'retrieval_channel_count', 'material_group_code']


In [247]:
# Prepare known mappings with SAP attributes
known_attribute_mappings = (
    known_retrieval_pairs[
        known_retrieval_pairs["material_id"]
        .astype("string")
        .isin(materials_with_attributes)
    ]
    [
        [
            "material_id",
            "plm_code",
            "material_group_code"
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "plm_code": "true_plm_code"
        }
    )
    .copy()
)

# Standardize merge-key data types
candidate_master["material_id"] = (
    candidate_master["material_id"]
    .astype("string")
)

candidate_master["material_group_code"] = (
    candidate_master["material_group_code"]
    .astype("string")
)

known_attribute_mappings["material_id"] = (
    known_attribute_mappings["material_id"]
    .astype("string")
)

known_attribute_mappings["material_group_code"] = (
    known_attribute_mappings["material_group_code"]
    .astype("string")
)

In [248]:
# Build labeled candidate-pair dataset
pairwise_dataset = (
    candidate_master
    .merge(
        known_attribute_mappings,
        on=[
            "material_id",
            "material_group_code"
        ],
        how="inner"
    )
)

pairwise_dataset["is_match"] = (
    pairwise_dataset["candidate_plm_code"]
    == pairwise_dataset["true_plm_code"]
).astype(int)

print(
    "Pairwise rows:",
    len(pairwise_dataset)
)

print(
    "SAP materials:",
    pairwise_dataset["material_id"].nunique()
)

print(
    "Positive pairs:",
    pairwise_dataset["is_match"].sum()
)

print(
    "Positive rate:",
    f"{pairwise_dataset['is_match'].mean():.4%}"
)

Pairwise rows: 2220089
SAP materials: 3499
Positive pairs: 3433
Positive rate: 0.1546%


In [249]:
# Compare the material group used by known mappings
# with the material group used during candidate generation

candidate_group_lookup = (
    candidate_master[
        [
            "material_id",
            "material_group_code"
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "material_group_code":
            "candidate_material_group"
        }
    )
)

group_alignment_audit = (
    known_attribute_mappings
    .merge(
        candidate_group_lookup,
        on="material_id",
        how="left"
    )
)

group_alignment_audit["group_match"] = (
    group_alignment_audit["material_group_code"]
    == group_alignment_audit["candidate_material_group"]
)

print(
    "Known attribute mappings:",
    len(group_alignment_audit)
)

print(
    "Unique known materials:",
    group_alignment_audit[
        "material_id"
    ].nunique()
)

print(
    "Missing from candidate master:",
    group_alignment_audit[
        "candidate_material_group"
    ].isna().sum()
)

print(
    "Material-group mismatches:",
    (~group_alignment_audit["group_match"])
    .sum()
)

Known attribute mappings: 3513
Unique known materials: 3513
Missing from candidate master: 0
Material-group mismatches: 14


In [250]:
# Inspect material-group inconsistencies
display(
    group_alignment_audit[
        ~group_alignment_audit["group_match"]
    ][
        [
            "material_id",
            "true_plm_code",
            "material_group_code",
            "candidate_material_group"
        ]
    ]
)

,material_id,true_plm_code,material_group_code,candidate_material_group
23,1020000050,10507,1020001,1020002
171,1020000293,16840,1020001,1020002
257,1020000428,19254,1020001,1020002
312,1020000557,211678,1020001,1020002
558,1020001005,355934,1020001,1020002
655,1020001192,57720,1020002,1020001
997,1020011294,101336,1020002,1030004
1138,1020011901,201237,1020002,1030004
1376,1020013592,114296,1020002,1030004
1698,1020015126,118072,1020002,1030004


In [251]:
# Check which known mappings were not retrieved
positive_candidate_check = (
    known_attribute_mappings
    .merge(
        candidate_master[
            [
                "material_id",
                "candidate_plm_code"
            ]
        ].drop_duplicates(),
        left_on=[
            "material_id",
            "true_plm_code"
        ],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left"
    )
)

positive_candidate_check["true_plm_retrieved"] = (
    positive_candidate_check[
        "candidate_plm_code"
    ].notna()
)

print(
    "Known mappings:",
    len(positive_candidate_check)
)

print(
    "True PLM retrieved:",
    positive_candidate_check[
        "true_plm_retrieved"
    ].sum()
)

print(
    "True PLM missed:",
    (
        ~positive_candidate_check[
            "true_plm_retrieved"
        ]
    ).sum()
)

Known mappings: 3513
True PLM retrieved: 3433
True PLM missed: 80


In [252]:
# Add SAP fiber representation
pairwise_dataset = (
    pairwise_dataset
    .merge(
        sap_fiber_sets.rename(
            columns={
                "fiber_key":
                "sap_fiber_key"
            }
        ),
        on="material_id",
        how="left"
    )
)

# Add PLM fiber representation
pairwise_dataset = (
    pairwise_dataset
    .merge(
        plm_fiber_sets.rename(
            columns={
                "plm_code":
                "candidate_plm_code",
                "fiber_key":
                "plm_fiber_key"
            }
        ),
        on="candidate_plm_code",
        how="left"
    )
)

In [253]:
# Calculate fiber-set similarity
def tuple_jaccard(left, right):
    if not isinstance(left, tuple) or not isinstance(right, tuple):
        return np.nan

    left_set = set(left)
    right_set = set(right)

    if not left_set or not right_set:
        return np.nan

    return (
        len(left_set & right_set)
        / len(left_set | right_set)
    )


pairwise_dataset["fiber_similarity"] = [
    tuple_jaccard(left, right)
    for left, right in zip(
        pairwise_dataset["sap_fiber_key"],
        pairwise_dataset["plm_fiber_key"]
    )
]

In [254]:
# Add normalized SAP fabric structure
pairwise_dataset = (
    pairwise_dataset
    .merge(
        sap_structure_candidates[
            ["material_id", "structure"]
        ].rename(
            columns={
                "structure":
                "sap_structure"
            }
        ),
        on="material_id",
        how="left"
    )
)

# Add normalized PLM fabric structure
pairwise_dataset = (
    pairwise_dataset
    .merge(
        plm_structure_candidates[
            ["plm_code", "structure"]
        ].rename(
            columns={
                "plm_code":
                "candidate_plm_code",
                "structure":
                "plm_structure"
            }
        ),
        on="candidate_plm_code",
        how="left"
    )
)

pairwise_dataset["fabric_structure_match"] = np.where(
    pairwise_dataset["sap_structure"].notna()
    & pairwise_dataset["plm_structure"].notna(),
    (
        pairwise_dataset["sap_structure"]
        == pairwise_dataset["plm_structure"]
    ).astype(float),
    np.nan
)

In [255]:
feature_columns = [
    "retrieval_similarity",
    "retrieval_channel_count",
    "fiber_similarity",
    "fabric_structure_match"
]

display(
    pairwise_dataset
    .groupby("is_match")[
        feature_columns
    ]
    .mean()
    .T
    .rename(
        columns={
            0: "non_match_mean",
            1: "match_mean"
        }
    )
)

is_match,non_match_mean,match_mean
retrieval_similarity,0.268871,0.291782
retrieval_channel_count,1.140854,2.249053
fiber_similarity,0.685094,0.997348
fabric_structure_match,0.597499,0.944364


In [256]:
display(
    pairwise_dataset[
        pairwise_dataset["is_match"] == 1
    ][feature_columns]
    .notna()
    .mean()
    .rename(
        "positive_feature_coverage"
    )
    .to_frame()
)

,positive_feature_coverage
retrieval_similarity,0.317215
retrieval_channel_count,1.000000
fiber_similarity,0.999126
fabric_structure_match,1.000000


In [257]:
# Audit candidate-generation misses
missed_mappings = (
    positive_candidate_check[
        ~positive_candidate_check["true_plm_retrieved"]
    ][
        [
            "material_id",
            "true_plm_code",
            "material_group_code"
        ]
    ]
    .copy()
)

missed_mappings = (
    missed_mappings
    .merge(
        candidate_group_lookup,
        on="material_id",
        how="left"
    )
)

missed_mappings["miss_reason"] = np.where(
    missed_mappings["material_group_code"]
    != missed_mappings["candidate_material_group"],
    "material_group_mismatch",
    "retrieval_miss_same_group"
)

display(
    missed_mappings[
        "miss_reason"
    ].value_counts()
)

miss_reason
retrieval_miss_same_group    66
material_group_mismatch      14
Name: count, dtype: int64

In [258]:
# Keep only materials whose true PLM reached the candidate pool
retrieved_material_ids = set(
    pairwise_dataset.loc[
        pairwise_dataset["is_match"] == 1,
        "material_id"
    ]
)

pairwise_training = (
    pairwise_dataset[
        pairwise_dataset["material_id"]
        .isin(retrieved_material_ids)
    ]
    .copy()
)

print(
    "Training materials:",
    pairwise_training["material_id"].nunique()
)

print(
    "Positive pairs:",
    pairwise_training["is_match"].sum()
)

print(
    "Materials without a positive:",
    (
        pairwise_training
        .groupby("material_id")["is_match"]
        .sum()
        .eq(0)
        .sum()
    )
)

Training materials: 3433
Positive pairs: 3433
Materials without a positive: 0


In [259]:
# Add SAP weight data to every candidate pair
pairwise_training = (
    pairwise_training
    .merge(
        sap_weight_audit[
            [
                "material_id",
                "sap_weight_raw",
                "sap_weight_unit"
            ]
        ],
        on="material_id",
        how="left"
    )
)

# Add PLM weight data to every candidate pair
pairwise_training = (
    pairwise_training
    .merge(
        plm_weight_audit[
            [
                "plm_code",
                "plm_weight",
                "plm_weight_unit"
            ]
        ].rename(
            columns={
                "plm_code": "candidate_plm_code"
            }
        ),
        on="candidate_plm_code",
        how="left"
    )
)

In [260]:
# Vectorized scale-aware weight similarity
valid_weight = (
    pairwise_training["sap_weight_raw"].notna()
    & pairwise_training["plm_weight"].notna()
    & (pairwise_training["sap_weight_raw"] > 0)
    & (pairwise_training["plm_weight"] > 0)
)

same_scale_error = (
    abs(
        pairwise_training["sap_weight_raw"]
        - pairwise_training["plm_weight"]
    )
    / pairwise_training["plm_weight"]
)

x1000_error = (
    abs(
        pairwise_training["sap_weight_raw"] / 1000
        - pairwise_training["plm_weight"]
    )
    / pairwise_training["plm_weight"]
)

pairwise_training["weight_relative_error_all"] = (
    np.minimum(
        same_scale_error,
        x1000_error
    )
)

# If both units exist and disagree, comparison is invalid
unit_mismatch = (
    pairwise_training["sap_weight_unit"].notna()
    & pairwise_training["plm_weight_unit"].notna()
    & (
        pairwise_training["sap_weight_unit"]
        != pairwise_training["plm_weight_unit"]
    )
)

pairwise_training.loc[
    ~valid_weight | unit_mismatch,
    "weight_relative_error_all"
] = np.nan

pairwise_training["weight_similarity"] = (
    1 / (
        1
        + pairwise_training[
            "weight_relative_error_all"
        ]
    )
)

In [261]:
feature_columns = [
    "retrieval_similarity",
    "retrieval_channel_count",
    "fiber_similarity",
    "weight_similarity",
    "fabric_structure_match"
]

display(
    pairwise_training
    .groupby("is_match")[feature_columns]
    .mean()
    .T
    .rename(
        columns={
            0: "non_match_mean",
            1: "match_mean"
        }
    )
)

display(
    pairwise_training[
        pairwise_training["is_match"] == 1
    ][feature_columns]
    .notna()
    .mean()
    .rename("positive_feature_coverage")
    .to_frame()
)

is_match,non_match_mean,match_mean
retrieval_similarity,0.269066,0.291782
retrieval_channel_count,1.141070,2.249053
fiber_similarity,0.685238,0.997348
weight_similarity,0.944690,0.998952
fabric_structure_match,0.597962,0.944364


,positive_feature_coverage
retrieval_similarity,0.317215
retrieval_channel_count,1.000000
fiber_similarity,0.999126
weight_similarity,0.997670
fabric_structure_match,1.000000


In [262]:
# Convert coloring sets to canonical tuples
sap_coloring_model = sap_coloring_sets.copy()
plm_coloring_model = plm_coloring_sets.copy()

sap_coloring_model["sap_coloring_key"] = (
    sap_coloring_model["sap_coloring_set"]
    .apply(lambda x: tuple(sorted(x)))
)

plm_coloring_model["plm_coloring_key"] = (
    plm_coloring_model["plm_coloring_set"]
    .apply(lambda x: tuple(sorted(x)))
)

pairwise_training = (
    pairwise_training
    .merge(
        sap_coloring_model[
            ["material_id", "sap_coloring_key"]
        ],
        on="material_id",
        how="left"
    )
    .merge(
        plm_coloring_model[
            ["plm_code", "plm_coloring_key"]
        ].rename(
            columns={
                "plm_code": "candidate_plm_code"
            }
        ),
        on="candidate_plm_code",
        how="left"
    )
)

pairwise_training["coloring_match"] = np.where(
    pairwise_training["sap_coloring_key"].notna()
    & pairwise_training["plm_coloring_key"].notna(),
    (
        pairwise_training["sap_coloring_key"]
        == pairwise_training["plm_coloring_key"]
    ).astype(float),
    np.nan
)

In [263]:
# Prepare SAP weave type
sap_weave_type = (
    fabric_attributes[
        fabric_attributes["field_name"] == "WEAVETYPE"
    ][
        ["material_id", "Karakteristik değeri"]
    ]
    .dropna()
    .drop_duplicates("material_id")
    .rename(
        columns={
            "Karakteristik değeri": "sap_weave_type"
        }
    )
)

sap_weave_type["sap_weave_type"] = (
    sap_weave_type["sap_weave_type"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Prepare PLM weave type
plm_weave_type = (
    plm_codes[
        ["plm_code", "Dokuma tipi"]
    ]
    .copy()
    .rename(
        columns={
            "Dokuma tipi": "plm_weave_type"
        }
    )
)

plm_weave_type["plm_weave_type"] = (
    plm_weave_type["plm_weave_type"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [264]:
pairwise_training = (
    pairwise_training
    .merge(
        sap_weave_type,
        on="material_id",
        how="left"
    )
    .merge(
        plm_weave_type.rename(
            columns={
                "plm_code": "candidate_plm_code"
            }
        ),
        on="candidate_plm_code",
        how="left"
    )
)

pairwise_training["weave_type_match"] = np.where(
    pairwise_training["sap_weave_type"].notna()
    & pairwise_training["plm_weave_type"].notna(),
    (
        pairwise_training["sap_weave_type"]
        == pairwise_training["plm_weave_type"]
    ).astype(float),
    np.nan
)

In [265]:
model_features = [
    "retrieval_similarity",
    "retrieval_channel_count",
    "fiber_similarity",
    "weight_similarity",
    "fabric_structure_match",
    "coloring_match",
    "weave_type_match"
]

display(
    pairwise_training
    .groupby("is_match")[model_features]
    .mean()
    .T
    .rename(
        columns={
            0: "non_match_mean",
            1: "match_mean"
        }
    )
)

is_match,non_match_mean,match_mean
retrieval_similarity,0.268782,0.291782
retrieval_channel_count,1.141037,2.249053
fiber_similarity,0.684743,0.997348
weight_similarity,0.944640,0.998952
fabric_structure_match,0.597706,0.944364
coloring_match,0.227986,0.803875
weave_type_match,0.514691,0.975387


In [266]:
# Build a simple heuristic only for selecting difficult negatives
pairwise_training["hardness_score"] = (
    pairwise_training["fiber_similarity"].fillna(0)
    + pairwise_training["weight_similarity"].fillna(0)
    + pairwise_training["fabric_structure_match"].fillna(0)
    + 0.5 * pairwise_training["coloring_match"].fillna(0)
    + 0.25 * pairwise_training["retrieval_channel_count"]
    + 0.25 * pairwise_training["retrieval_similarity"].fillna(0)
)

In [267]:
positive_pairs = (
    pairwise_training[
        pairwise_training["is_match"] == 1
    ]
    .copy()
)

negative_pairs = (
    pairwise_training[
        pairwise_training["is_match"] == 0
    ]
    .copy()
)

In [268]:
# Keep the most confusing negative candidates per SAP material
hard_negatives = (
    negative_pairs
    .sort_values(
        ["material_id", "hardness_score"],
        ascending=[True, False]
    )
    .groupby("material_id")
    .head(20)
)

In [269]:
# Add random negatives for broader coverage
negative_pairs["random_score"] = np.random.default_rng(
    42
).random(len(negative_pairs))

random_negatives = (
    negative_pairs
    .sort_values(
        ["material_id", "random_score"]
    )
    .groupby("material_id")
    .head(20)
)

In [270]:
pairwise_sample = (
    pd.concat(
        [
            positive_pairs,
            hard_negatives,
            random_negatives
        ],
        ignore_index=True
    )
    .drop_duplicates(
        subset=[
            "material_id",
            "candidate_plm_code"
        ]
    )
)

print(
    "Sample rows:",
    len(pairwise_sample)
)

print(
    "Materials:",
    pairwise_sample["material_id"].nunique()
)

print(
    "Positive pairs:",
    pairwise_sample["is_match"].sum()
)

print(
    "Positive rate:",
    f"{pairwise_sample['is_match'].mean():.2%}"
)

Sample rows: 136730
Materials: 3433
Positive pairs: 3433
Positive rate: 2.51%


In [271]:
from sklearn.model_selection import train_test_split

material_split = (
    pairwise_sample[
        ["material_id", "material_group_code"]
    ]
    .drop_duplicates("material_id")
)

train_materials, test_materials = train_test_split(
    material_split,
    test_size=0.20,
    random_state=42,
    stratify=material_split["material_group_code"]
)

train_ids = set(train_materials["material_id"])
test_ids = set(test_materials["material_id"])

train_df = pairwise_sample[
    pairwise_sample["material_id"].isin(train_ids)
].copy()

test_df = pairwise_sample[
    pairwise_sample["material_id"].isin(test_ids)
].copy()

print("Train materials:", len(train_ids))
print("Test materials:", len(test_ids))

Train materials: 2746
Test materials: 687


In [272]:
model_features = [
    "retrieval_similarity",
    "retrieval_channel_count",
    "from_text",
    "from_structure_weight",
    "from_fiber_weight",
    "fiber_similarity",
    "weight_similarity",
    "fabric_structure_match",
    "coloring_match",
    "weave_type_match"
]

In [273]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Build an interpretable baseline reranker
logistic_model = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value=-1,
            add_indicator=True
        )
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    )
])

In [274]:
X_train = train_df[model_features]
y_train = train_df["is_match"]

logistic_model.fit(
    X_train,
    y_train
)

print("Model trained.")

Model trained.


In [275]:
# Evaluate on all candidates, not only sampled negatives
test_full = (
    pairwise_training[
        pairwise_training["material_id"]
        .isin(test_ids)
    ]
    .copy()
)

print(
    "Test materials:",
    test_full["material_id"].nunique()
)

print(
    "Full test candidate pairs:",
    len(test_full)
)

print(
    "Test positives:",
    test_full["is_match"].sum()
)

Test materials: 687
Full test candidate pairs: 433930
Test positives: 687


In [276]:
# Predict match probabilities for every candidate
test_full["match_probability"] = (
    logistic_model.predict_proba(
        test_full[model_features]
    )[:, 1]
)

In [277]:
# Rank PLM candidates within each SAP material
test_full["predicted_rank"] = (
    test_full
    .groupby("material_id")[
        "match_probability"
    ]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)

In [278]:
# Keep the true PLM row for each test material
true_match_ranks = (
    test_full[
        test_full["is_match"] == 1
    ][
        [
            "material_id",
            "material_group_code",
            "true_plm_code",
            "predicted_rank",
            "match_probability"
        ]
    ]
    .copy()
)

def top_k_accuracy(df, k):
    return (
        df["predicted_rank"] <= k
    ).mean()

print(
    "Top-1:",
    f"{top_k_accuracy(true_match_ranks, 1):.2%}"
)

print(
    "Top-3:",
    f"{top_k_accuracy(true_match_ranks, 3):.2%}"
)

print(
    "Top-5:",
    f"{top_k_accuracy(true_match_ranks, 5):.2%}"
)

print(
    "Top-10:",
    f"{top_k_accuracy(true_match_ranks, 10):.2%}"
)

mrr = (
    1 / true_match_ranks["predicted_rank"]
).mean()

print(
    "MRR:",
    f"{mrr:.4f}"
)

Top-1: 46.43%
Top-3: 69.14%
Top-5: 75.55%
Top-10: 82.97%
MRR: 0.5979


In [279]:
# Ranking performance by material group
group_ranking_summary = []

group_name_mapping = {
    "1020001": "ORME",
    "1020002": "DOKUMA",
    "1030004": "DENIM"
}

for group_code, group in (
    true_match_ranks
    .groupby("material_group_code")
):
    group_ranking_summary.append({
        "material_group":
            group_name_mapping.get(
                group_code,
                group_code
            ),
        "test_materials":
            len(group),
        "top_1":
            top_k_accuracy(group, 1),
        "top_3":
            top_k_accuracy(group, 3),
        "top_5":
            top_k_accuracy(group, 5),
        "top_10":
            top_k_accuracy(group, 10),
        "mrr":
            (1 / group["predicted_rank"]).mean()
    })

display(
    pd.DataFrame(
        group_ranking_summary
    )
)

,material_group,test_materials,top_1,top_3,top_5,top_10,mrr
0,ORME,356,0.317416,0.581461,0.668539,0.758427,0.478138
1,DOKUMA,279,0.634409,0.813620,0.853047,0.910394,0.733556
2,DENIM,52,0.557692,0.788462,0.826923,0.884615,0.690119


In [280]:
print(
    true_match_ranks[
        "predicted_rank"
    ].describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

count    687.000000
mean      11.518195
std       36.735115
min        1.000000
50%        2.000000
75%        5.000000
90%       20.000000
95%       42.100000
99%      232.280000
max      379.000000
Name: predicted_rank, dtype: float64


In [281]:
# Inspect Logistic Regression feature coefficients
imputer = logistic_model.named_steps["imputer"]
classifier = logistic_model.named_steps["model"]

feature_names = imputer.get_feature_names_out(
    model_features
)

coefficient_table = (
    pd.DataFrame({
        "feature": feature_names,
        "coefficient": classifier.coef_[0]
    })
    .sort_values(
        "coefficient",
        ascending=False
    )
)

display(coefficient_table)

,feature,coefficient
6,weight_similarity,20.869349
12,missingindicator_weight_similarity,19.186162
7,fabric_structure_match,4.276502
15,missingindicator_weave_type_match,2.231400
9,weave_type_match,2.088882
4,from_fiber_weight,2.065629
0,retrieval_similarity,0.960079
5,fiber_similarity,0.719237
8,coloring_match,0.393125
14,missingindicator_coloring_match,0.251134


In [282]:
# Initialize group-specific yarn features
pairwise_training["knit_yarn_count_match"] = np.nan
pairwise_training["knit_yarn_type_similarity"] = np.nan
pairwise_training["weft_yarn_count_match"] = np.nan

In [283]:
# Build knitting yarn-count lookups
sap_knit_yarn_count_lookup = (
    sap_yarn_count_knit
    .set_index("material_id")["sap_yarn_count"]
    .to_dict()
)

plm_knit_yarn_count_lookup = (
    plm_yarn_count_knit
    .set_index("plm_code")["plm_yarn_count"]
    .to_dict()
)

orme_mask = (
    pairwise_training["material_group_code"]
    == "1020001"
)

pairwise_training.loc[
    orme_mask,
    "knit_yarn_count_match"
] = [
    calculate_yarn_count_match(
        sap_knit_yarn_count_lookup.get(material_id),
        plm_knit_yarn_count_lookup.get(plm_code)
    )
    for material_id, plm_code in zip(
        pairwise_training.loc[orme_mask, "material_id"],
        pairwise_training.loc[orme_mask, "candidate_plm_code"]
    )
]

In [284]:
# Build normalized knitting yarn-type lookups
sap_knit_yarn_type_lookup = (
    sap_yarn_type_sets_normalized
    .set_index("material_id")["sap_yarn_type_set"]
    .to_dict()
)

plm_knit_yarn_type_lookup = (
    plm_yarn_type_sets_normalized
    .set_index("plm_code")["plm_yarn_type_set"]
    .to_dict()
)


def safe_set_jaccard(left, right):
    if not isinstance(left, set) or not isinstance(right, set):
        return np.nan

    if not left or not right:
        return np.nan

    return len(left & right) / len(left | right)


pairwise_training.loc[
    orme_mask,
    "knit_yarn_type_similarity"
] = [
    safe_set_jaccard(
        sap_knit_yarn_type_lookup.get(material_id),
        plm_knit_yarn_type_lookup.get(plm_code)
    )
    for material_id, plm_code in zip(
        pairwise_training.loc[orme_mask, "material_id"],
        pairwise_training.loc[orme_mask, "candidate_plm_code"]
    )
]

In [285]:
# Build woven weft yarn-count lookups
sap_weft_yarn_count_lookup = (
    sap_weft_yarn_count
    .set_index("material_id")["sap_weft_yarn_count"]
    .to_dict()
)

plm_weft_yarn_count_lookup = (
    plm_weft_yarn_count
    .set_index("plm_code")["plm_weft_yarn_count"]
    .to_dict()
)

dokuma_mask = (
    pairwise_training["material_group_code"]
    == "1020002"
)

pairwise_training.loc[
    dokuma_mask,
    "weft_yarn_count_match"
] = [
    calculate_yarn_count_match(
        sap_weft_yarn_count_lookup.get(material_id),
        plm_weft_yarn_count_lookup.get(plm_code)
    )
    for material_id, plm_code in zip(
        pairwise_training.loc[dokuma_mask, "material_id"],
        pairwise_training.loc[dokuma_mask, "candidate_plm_code"]
    )
]

In [286]:
# Rebuild sampled dataset with newly added features
sample_keys = (
    pairwise_sample[
        ["material_id", "candidate_plm_code"]
    ]
    .drop_duplicates()
)

pairwise_sample_v2 = (
    sample_keys
    .merge(
        pairwise_training,
        on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="inner"
    )
)

print("Sample rows:", len(pairwise_sample_v2))
print(
    "Materials:",
    pairwise_sample_v2["material_id"].nunique()
)

Sample rows: 138292
Materials: 3433


In [287]:
# Common non-redundant features
common_features = [
    "retrieval_similarity",
    "from_structure_weight",
    "from_fiber_weight",
    "fiber_similarity",
    "weight_similarity",
    "fabric_structure_match",
    "coloring_match"
]

group_features = {
    "1020001": common_features + [
        "knit_yarn_count_match",
        "knit_yarn_type_similarity"
    ],

    "1020002": common_features + [
        "weave_type_match",
        "weft_yarn_count_match"
    ],

    "1030004": common_features + [
        "weave_type_match"
    ]
}

In [288]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

group_models = {}
group_results = []

group_name_mapping = {
    "1020001": "ORME",
    "1020002": "DOKUMA",
    "1030004": "DENIM"
}

for group_code, features in group_features.items():

    train_group = pairwise_sample_v2[
        pairwise_sample_v2["material_id"].isin(train_ids)
        & (
            pairwise_sample_v2["material_group_code"]
            == group_code
        )
    ].copy()

    # Full candidate pool for unbiased ranking evaluation
    test_group = pairwise_training[
        pairwise_training["material_id"].isin(test_ids)
        & (
            pairwise_training["material_group_code"]
            == group_code
        )
    ].copy()

    model = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=-1,
                add_indicator=True
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    ])

    model.fit(
        train_group[features],
        train_group["is_match"]
    )

    test_group["match_probability"] = (
        model.predict_proba(
            test_group[features]
        )[:, 1]
    )

    test_group["predicted_rank"] = (
        test_group
        .groupby("material_id")[
            "match_probability"
        ]
        .rank(
            method="first",
            ascending=False
        )
        .astype(int)
    )

    true_ranks = test_group[
        test_group["is_match"] == 1
    ].copy()

    group_models[group_code] = model

    group_results.append({
        "material_group":
            group_name_mapping[group_code],
        "test_materials":
            len(true_ranks),
        "top_1":
            (true_ranks["predicted_rank"] <= 1).mean(),
        "top_3":
            (true_ranks["predicted_rank"] <= 3).mean(),
        "top_5":
            (true_ranks["predicted_rank"] <= 5).mean(),
        "top_10":
            (true_ranks["predicted_rank"] <= 10).mean(),
        "mrr":
            (
                1 / true_ranks["predicted_rank"]
            ).mean()
    })

group_results = pd.DataFrame(group_results)

display(group_results)

,material_group,test_materials,top_1,top_3,top_5,top_10,mrr
0,ORME,356,0.435393,0.668539,0.738764,0.811798,0.572140
1,DOKUMA,279,0.584229,0.774194,0.824373,0.896057,0.696218
2,DENIM,52,0.250000,0.692308,0.788462,0.865385,0.491029


In [289]:
# Weighted overall ranking performance
total_test_materials = (
    group_results["test_materials"].sum()
)

for metric in [
    "top_1",
    "top_3",
    "top_5",
    "top_10",
    "mrr"
]:
    weighted_metric = (
        (
            group_results[metric]
            * group_results["test_materials"]
        ).sum()
        / total_test_materials
    )

    print(
        f"{metric}:",
        f"{weighted_metric:.4f}"
    )

top_1: 0.4818
top_3: 0.7132
top_5: 0.7773
top_10: 0.8501
mrr: 0.6164


### 9.1 Teşhis: yinelenen aday çiftleri

Aşağıdaki üç hücre yukarıdaki sürümdeki satır çoğalması hatasını ortaya
çıkarır; devamındaki `_clean` blokları düzeltilmiş kurulumdur.


In [290]:
# Check duplicate SAP-PLM candidate pairs
duplicate_pair_summary = (
    pairwise_training
    .groupby(
        ["material_id", "candidate_plm_code"]
    )
    .size()
    .reset_index(name="row_count")
)

print(
    "Unique candidate pairs:",
    len(duplicate_pair_summary)
)

print(
    "Duplicated candidate pairs:",
    (duplicate_pair_summary["row_count"] > 1).sum()
)

print(
    "Maximum duplicate count:",
    duplicate_pair_summary["row_count"].max()
)

display(
    duplicate_pair_summary[
        duplicate_pair_summary["row_count"] > 1
    ]
    .sort_values(
        "row_count",
        ascending=False
    )
    .head(20)
)

Unique candidate pairs: 2206292
Duplicated candidate pairs: 283
Maximum duplicate count: 32


,material_id,candidate_plm_code,row_count
1327977,1020016850,"DOBBY, WOVEN-OTHERS, 75 D X 75",32
936826,1020014318,"DOBBY, WOVEN-OTHERS, 75 D X 75",32
56389,1020000156,"DOBBY, WOVEN-OTHERS, 75 D X 75",32
1225019,1020016031,"DOBBY, WOVEN-OTHERS, 75 D X 75",32
1365293,1020017076,"DOBBY, WOVEN-OTHERS, 75 D X 75",32
1056766,1020015051,"DOBBY, WOVEN-OTHERS, 75 D X 75",32
1050881,1020015029,"DOBBY, WOVEN-OTHERS, 75 D X 75",32
398161,1020001190,"DOBBY, WOVEN-OTHERS, 75 D X 75",32
413831,1020001238,"DOBBY, WOVEN-OTHERS, 75 D X 75",32
246321,1020000725,"DOBBY, WOVEN-OTHERS, 75 D X 75",32


In [291]:
print(
    "Pairwise training rows:",
    len(pairwise_training)
)

print(
    "Unique SAP-PLM pairs:",
    pairwise_training[
        ["material_id", "candidate_plm_code"]
    ].drop_duplicates().shape[0]
)

Pairwise training rows: 2210673
Unique SAP-PLM pairs: 2206292


In [292]:
# Check whether duplication already exists in candidate master
candidate_master_duplicates = (
    candidate_master
    .groupby(
        ["material_id", "candidate_plm_code"]
    )
    .size()
)

print(
    "Candidate master duplicated pairs:",
    (candidate_master_duplicates > 1).sum()
)

print(
    "Candidate master maximum duplicate count:",
    candidate_master_duplicates.max()
)

Candidate master duplicated pairs: 431
Candidate master maximum duplicate count: 4


## 10. Pairwise reranker — temiz sürüm (geçerli)


In [293]:
# Aggregate text channel to one row per SAP-PLM pair
text_channel_clean = (
    tfidf_candidates
    .groupby(
        ["material_id", "candidate_plm_code"],
        as_index=False
    )
    .agg(
        retrieval_similarity=(
            "retrieval_similarity",
            "max"
        )
    )
)

text_channel_clean["from_text"] = 1

In [294]:
# Aggregate structure-weight channel
structure_channel_clean = (
    structure_weight_candidates
    .groupby(
        ["material_id", "candidate_plm_code"],
        as_index=False
    )
    .agg(
        structure_weight_error=(
            "weight_relative_error",
            "min"
        )
    )
)

structure_channel_clean[
    "from_structure_weight"
] = 1

In [295]:
# Aggregate fiber-weight channel
fiber_channel_clean = (
    fiber_weight_candidates
    .groupby(
        ["material_id", "candidate_plm_code"],
        as_index=False
    )
    .agg(
        fiber_weight_error=(
            "weight_relative_error",
            "min"
        )
    )
)

fiber_channel_clean[
    "from_fiber_weight"
] = 1

In [296]:
# Rebuild a strictly one-row-per-pair candidate master
candidate_master_clean = (
    all_candidate_pairs[
        ["material_id", "candidate_plm_code"]
    ]
    .drop_duplicates()
    .merge(
        text_channel_clean,
        on=["material_id", "candidate_plm_code"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        structure_channel_clean,
        on=["material_id", "candidate_plm_code"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        fiber_channel_clean,
        on=["material_id", "candidate_plm_code"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        material_group_lookup,
        on="material_id",
        how="left",
        validate="many_to_one"
    )
)

for column in [
    "from_text",
    "from_structure_weight",
    "from_fiber_weight"
]:
    candidate_master_clean[column] = (
        candidate_master_clean[column]
        .fillna(0)
        .astype(int)
    )

candidate_master_clean["retrieval_channel_count"] = (
    candidate_master_clean[
        [
            "from_text",
            "from_structure_weight",
            "from_fiber_weight"
        ]
    ].sum(axis=1)
)

In [297]:
# Candidate pair must be unique
assert not candidate_master_clean.duplicated(
    ["material_id", "candidate_plm_code"]
).any()

print(
    "Clean candidate pairs:",
    len(candidate_master_clean)
)

Clean candidate pairs: 3863634


In [298]:
# Define the unique pair key
pair_key = [
    "material_id",
    "candidate_plm_code"
]

assert not candidate_master_clean.duplicated(pair_key).any()

# Rebuild labeled candidate pairs from the clean candidate master
pairwise_dataset_clean = (
    candidate_master_clean
    .merge(
        known_attribute_mappings,
        on=[
            "material_id",
            "material_group_code"
        ],
        how="inner",
        validate="many_to_one"
    )
)

pairwise_dataset_clean["is_match"] = (
    pairwise_dataset_clean["candidate_plm_code"]
    == pairwise_dataset_clean["true_plm_code"]
).astype(int)

print(
    "Pairwise rows:",
    len(pairwise_dataset_clean)
)

print(
    "Unique pairs:",
    pairwise_dataset_clean[pair_key]
    .drop_duplicates()
    .shape[0]
)

assert not pairwise_dataset_clean.duplicated(pair_key).any()

Pairwise rows: 2219789
Unique pairs: 2219789


In [299]:
# Keep only materials with a positive candidate
retrieved_material_ids = set(
    pairwise_dataset_clean.loc[
        pairwise_dataset_clean["is_match"] == 1,
        "material_id"
    ]
)

pairwise_training_clean = (
    pairwise_dataset_clean[
        pairwise_dataset_clean["material_id"]
        .isin(retrieved_material_ids)
    ]
    .copy()
)

print(
    "Training materials:",
    pairwise_training_clean[
        "material_id"
    ].nunique()
)

print(
    "Positive pairs:",
    pairwise_training_clean[
        "is_match"
    ].sum()
)

print(
    "Duplicate pairs:",
    pairwise_training_clean
    .duplicated(pair_key)
    .sum()
)

Training materials: 3433
Positive pairs: 3433
Duplicate pairs: 0


In [300]:
# Build scalar lookup dictionaries
sap_structure_lookup = (
    sap_structure_candidates
    .drop_duplicates("material_id")
    .set_index("material_id")["structure"]
    .to_dict()
)

plm_structure_lookup = (
    plm_structure_candidates
    .drop_duplicates("plm_code")
    .set_index("plm_code")["structure"]
    .to_dict()
)

sap_weight_raw_lookup = (
    sap_weight_audit
    .drop_duplicates("material_id")
    .set_index("material_id")["sap_weight_raw"]
    .to_dict()
)

sap_weight_unit_lookup = (
    sap_weight_audit
    .drop_duplicates("material_id")
    .set_index("material_id")["sap_weight_unit"]
    .to_dict()
)

plm_weight_value_lookup = (
    plm_weight_audit
    .drop_duplicates("plm_code")
    .set_index("plm_code")["plm_weight"]
    .to_dict()
)

plm_weight_unit_lookup = (
    plm_weight_audit
    .drop_duplicates("plm_code")
    .set_index("plm_code")["plm_weight_unit"]
    .to_dict()
)

In [301]:
sap_fiber_lookup = (
    sap_fiber_sets
    .set_index("material_id")["fiber_key"]
    .to_dict()
)

plm_fiber_lookup = (
    plm_fiber_sets
    .set_index("plm_code")["fiber_key"]
    .to_dict()
)

sap_coloring_lookup = (
    sap_coloring_model
    .set_index("material_id")["sap_coloring_key"]
    .to_dict()
)

plm_coloring_lookup = (
    plm_coloring_model
    .set_index("plm_code")["plm_coloring_key"]
    .to_dict()
)

sap_weave_lookup = (
    sap_weave_type
    .drop_duplicates("material_id")
    .set_index("material_id")["sap_weave_type"]
    .to_dict()
)

plm_weave_lookup = (
    plm_weave_type
    .drop_duplicates("plm_code")
    .set_index("plm_code")["plm_weave_type"]
    .to_dict()
)

In [302]:
# Fiber similarity
pairwise_training_clean["fiber_similarity"] = [
    tuple_jaccard(
        sap_fiber_lookup.get(material_id),
        plm_fiber_lookup.get(plm_code)
    )
    for material_id, plm_code in zip(
        pairwise_training_clean["material_id"],
        pairwise_training_clean["candidate_plm_code"]
    )
]

# Fabric structure match
pairwise_training_clean["fabric_structure_match"] = [
    (
        float(
            sap_structure_lookup.get(material_id)
            == plm_structure_lookup.get(plm_code)
        )
        if (
            sap_structure_lookup.get(material_id) is not None
            and plm_structure_lookup.get(plm_code) is not None
        )
        else np.nan
    )
    for material_id, plm_code in zip(
        pairwise_training_clean["material_id"],
        pairwise_training_clean["candidate_plm_code"]
    )
]

# Coloring match
pairwise_training_clean["coloring_match"] = [
    (
        float(
            sap_coloring_lookup.get(material_id)
            == plm_coloring_lookup.get(plm_code)
        )
        if (
            sap_coloring_lookup.get(material_id) is not None
            and plm_coloring_lookup.get(plm_code) is not None
        )
        else np.nan
    )
    for material_id, plm_code in zip(
        pairwise_training_clean["material_id"],
        pairwise_training_clean["candidate_plm_code"]
    )
]

# Weave type match
pairwise_training_clean["weave_type_match"] = [
    (
        float(
            sap_weave_lookup.get(material_id)
            == plm_weave_lookup.get(plm_code)
        )
        if (
            sap_weave_lookup.get(material_id) is not None
            and plm_weave_lookup.get(plm_code) is not None
        )
        else np.nan
    )
    for material_id, plm_code in zip(
        pairwise_training_clean["material_id"],
        pairwise_training_clean["candidate_plm_code"]
    )
]

In [303]:
# Map weight values without changing row count
pairwise_training_clean["sap_weight_raw"] = (
    pairwise_training_clean["material_id"]
    .map(sap_weight_raw_lookup)
)

pairwise_training_clean["sap_weight_unit"] = (
    pairwise_training_clean["material_id"]
    .map(sap_weight_unit_lookup)
)

pairwise_training_clean["plm_weight"] = (
    pairwise_training_clean["candidate_plm_code"]
    .map(plm_weight_value_lookup)
)

pairwise_training_clean["plm_weight_unit"] = (
    pairwise_training_clean["candidate_plm_code"]
    .map(plm_weight_unit_lookup)
)

valid_weight = (
    pairwise_training_clean["sap_weight_raw"].notna()
    & pairwise_training_clean["plm_weight"].notna()
    & (pairwise_training_clean["sap_weight_raw"] > 0)
    & (pairwise_training_clean["plm_weight"] > 0)
)

same_scale_error = (
    abs(
        pairwise_training_clean["sap_weight_raw"]
        - pairwise_training_clean["plm_weight"]
    )
    / pairwise_training_clean["plm_weight"]
)

x1000_error = (
    abs(
        pairwise_training_clean["sap_weight_raw"] / 1000
        - pairwise_training_clean["plm_weight"]
    )
    / pairwise_training_clean["plm_weight"]
)

weight_error = np.minimum(
    same_scale_error,
    x1000_error
)

unit_mismatch = (
    pairwise_training_clean["sap_weight_unit"].notna()
    & pairwise_training_clean["plm_weight_unit"].notna()
    & (
        pairwise_training_clean["sap_weight_unit"]
        != pairwise_training_clean["plm_weight_unit"]
    )
)

weight_error[
    ~valid_weight | unit_mismatch
] = np.nan

pairwise_training_clean["weight_similarity"] = (
    1 / (1 + weight_error)
)

In [304]:
assert not pairwise_training_clean.duplicated(
    pair_key
).any()

print(
    "Clean pairwise rows:",
    len(pairwise_training_clean)
)

print(
    "Clean unique pairs:",
    pairwise_training_clean[
        pair_key
    ].drop_duplicates().shape[0]
)

Clean pairwise rows: 2206292
Clean unique pairs: 2206292


In [305]:
# Final integrity checks
print(
    "Training materials:",
    pairwise_training_clean["material_id"].nunique()
)

print(
    "Positive pairs:",
    pairwise_training_clean["is_match"].sum()
)

print(
    "Materials without a positive:",
    (
        pairwise_training_clean
        .groupby("material_id")["is_match"]
        .sum()
        .eq(0)
        .sum()
    )
)

assert not pairwise_training_clean.duplicated(
    ["material_id", "candidate_plm_code"]
).any()

Training materials: 3433
Positive pairs: 3433
Materials without a positive: 0


In [306]:
# Build yarn-count lookup dictionaries
sap_knit_yarn_count_lookup = (
    sap_yarn_count_knit
    .drop_duplicates("material_id")
    .set_index("material_id")["sap_yarn_count"]
    .to_dict()
)

plm_knit_yarn_count_lookup = (
    plm_yarn_count_knit
    .drop_duplicates("plm_code")
    .set_index("plm_code")["plm_yarn_count"]
    .to_dict()
)

sap_weft_yarn_count_lookup = (
    sap_weft_yarn_count
    .drop_duplicates("material_id")
    .set_index("material_id")["sap_weft_yarn_count"]
    .to_dict()
)

plm_weft_yarn_count_lookup = (
    plm_weft_yarn_count
    .drop_duplicates("plm_code")
    .set_index("plm_code")["plm_weft_yarn_count"]
    .to_dict()
)

In [307]:
# Build normalized knitting yarn-type lookups
sap_knit_yarn_type_lookup = (
    sap_yarn_type_sets_normalized
    .set_index("material_id")["sap_yarn_type_set"]
    .to_dict()
)

plm_knit_yarn_type_lookup = (
    plm_yarn_type_sets_normalized
    .set_index("plm_code")["plm_yarn_type_set"]
    .to_dict()
)


def safe_set_jaccard(left, right):
    if not isinstance(left, set) or not isinstance(right, set):
        return np.nan

    if not left or not right:
        return np.nan

    return len(left & right) / len(left | right)

In [308]:
pairwise_training_clean[
    "knit_yarn_count_match"
] = np.nan

pairwise_training_clean[
    "knit_yarn_type_similarity"
] = np.nan

pairwise_training_clean[
    "weft_yarn_count_match"
] = np.nan

In [309]:
orme_mask = (
    pairwise_training_clean["material_group_code"]
    == "1020001"
)

pairwise_training_clean.loc[
    orme_mask,
    "knit_yarn_count_match"
] = [
    calculate_yarn_count_match(
        sap_knit_yarn_count_lookup.get(material_id),
        plm_knit_yarn_count_lookup.get(plm_code)
    )
    for material_id, plm_code in zip(
        pairwise_training_clean.loc[
            orme_mask,
            "material_id"
        ],
        pairwise_training_clean.loc[
            orme_mask,
            "candidate_plm_code"
        ]
    )
]

pairwise_training_clean.loc[
    orme_mask,
    "knit_yarn_type_similarity"
] = [
    safe_set_jaccard(
        sap_knit_yarn_type_lookup.get(material_id),
        plm_knit_yarn_type_lookup.get(plm_code)
    )
    for material_id, plm_code in zip(
        pairwise_training_clean.loc[
            orme_mask,
            "material_id"
        ],
        pairwise_training_clean.loc[
            orme_mask,
            "candidate_plm_code"
        ]
    )
]

In [310]:
dokuma_mask = (
    pairwise_training_clean["material_group_code"]
    == "1020002"
)

pairwise_training_clean.loc[
    dokuma_mask,
    "weft_yarn_count_match"
] = [
    calculate_yarn_count_match(
        sap_weft_yarn_count_lookup.get(material_id),
        plm_weft_yarn_count_lookup.get(plm_code)
    )
    for material_id, plm_code in zip(
        pairwise_training_clean.loc[
            dokuma_mask,
            "material_id"
        ],
        pairwise_training_clean.loc[
            dokuma_mask,
            "candidate_plm_code"
        ]
    )
]

In [311]:
assert not pairwise_training_clean.duplicated(
    ["material_id", "candidate_plm_code"]
).any()

In [312]:
# Heuristic score used only for negative sampling
pairwise_training_clean["hardness_score"] = (
    pairwise_training_clean[
        "fiber_similarity"
    ].fillna(0)
    + pairwise_training_clean[
        "weight_similarity"
    ].fillna(0)
    + pairwise_training_clean[
        "fabric_structure_match"
    ].fillna(0)
    + 0.5
    * pairwise_training_clean[
        "coloring_match"
    ].fillna(0)
    + 0.25
    * pairwise_training_clean[
        "retrieval_similarity"
    ].fillna(0)
)

positive_pairs_clean = (
    pairwise_training_clean[
        pairwise_training_clean["is_match"] == 1
    ]
    .copy()
)

negative_pairs_clean = (
    pairwise_training_clean[
        pairwise_training_clean["is_match"] == 0
    ]
    .copy()
)

In [313]:
hard_negatives_clean = (
    negative_pairs_clean
    .sort_values(
        ["material_id", "hardness_score"],
        ascending=[True, False]
    )
    .groupby("material_id")
    .head(20)
)

In [314]:
rng = np.random.default_rng(42)

negative_pairs_clean["random_score"] = (
    rng.random(len(negative_pairs_clean))
)

random_negatives_clean = (
    negative_pairs_clean
    .sort_values(
        ["material_id", "random_score"]
    )
    .groupby("material_id")
    .head(20)
)

In [315]:
pairwise_sample_clean = (
    pd.concat(
        [
            positive_pairs_clean,
            hard_negatives_clean,
            random_negatives_clean
        ],
        ignore_index=True
    )
    .drop_duplicates(
        [
            "material_id",
            "candidate_plm_code"
        ]
    )
)

print(
    "Sample rows:",
    len(pairwise_sample_clean)
)

print(
    "Materials:",
    pairwise_sample_clean[
        "material_id"
    ].nunique()
)

print(
    "Positive pairs:",
    pairwise_sample_clean[
        "is_match"
    ].sum()
)

Sample rows: 136647
Materials: 3433
Positive pairs: 3433


In [316]:
common_features = [
    "retrieval_similarity",
    "from_structure_weight",
    "from_fiber_weight",
    "fiber_similarity",
    "weight_similarity",
    "fabric_structure_match",
    "coloring_match"
]

baseline_group_features = {
    "1020001": common_features,
    "1020002": common_features + [
        "weave_type_match"
    ],
    "1030004": common_features + [
        "weave_type_match"
    ]
}

yarn_group_features = {
    "1020001": common_features + [
        "knit_yarn_count_match",
        "knit_yarn_type_similarity"
    ],
    "1020002": common_features + [
        "weave_type_match",
        "weft_yarn_count_match"
    ],
    "1030004": common_features + [
        "weave_type_match"
    ]
}

In [317]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


def evaluate_group_models(
    feature_config,
    sample_df,
    full_df,
    train_ids,
    test_ids
):
    results = []

    group_name_mapping = {
        "1020001": "ORME",
        "1020002": "DOKUMA",
        "1030004": "DENIM"
    }

    for group_code, features in feature_config.items():

        train_group = sample_df[
            sample_df["material_id"].isin(train_ids)
            & (
                sample_df["material_group_code"]
                == group_code
            )
        ].copy()

        test_group = full_df[
            full_df["material_id"].isin(test_ids)
            & (
                full_df["material_group_code"]
                == group_code
            )
        ].copy()

        model = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value=-1,
                    add_indicator=True
                )
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "model",
                LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    random_state=42
                )
            )
        ])

        model.fit(
            train_group[features],
            train_group["is_match"]
        )

        test_group["score"] = (
            model.predict_proba(
                test_group[features]
            )[:, 1]
        )

        test_group["rank"] = (
            test_group
            .groupby("material_id")["score"]
            .rank(
                ascending=False,
                method="first"
            )
        )

        true_rows = (
            test_group[
                test_group["is_match"] == 1
            ]
        )

        results.append({
            "material_group":
                group_name_mapping[group_code],
            "test_materials":
                len(true_rows),
            "top_1":
                (true_rows["rank"] <= 1).mean(),
            "top_3":
                (true_rows["rank"] <= 3).mean(),
            "top_5":
                (true_rows["rank"] <= 5).mean(),
            "top_10":
                (true_rows["rank"] <= 10).mean(),
            "mrr":
                (1 / true_rows["rank"]).mean()
        })

    return pd.DataFrame(results)

In [318]:
baseline_results = evaluate_group_models(
    baseline_group_features,
    pairwise_sample_clean,
    pairwise_training_clean,
    train_ids,
    test_ids
)

display(baseline_results)

,material_group,test_materials,top_1,top_3,top_5,top_10,mrr
0,ORME,356,0.328652,0.578652,0.662921,0.755618,0.481022
1,DOKUMA,279,0.562724,0.759857,0.820789,0.899642,0.681414
2,DENIM,52,0.250000,0.673077,0.769231,0.865385,0.480153


In [319]:
yarn_results = evaluate_group_models(
    yarn_group_features,
    pairwise_sample_clean,
    pairwise_training_clean,
    train_ids,
    test_ids
)

display(yarn_results)

,material_group,test_materials,top_1,top_3,top_5,top_10,mrr
0,ORME,356,0.306180,0.511236,0.589888,0.702247,0.443131
1,DOKUMA,279,0.620072,0.799283,0.849462,0.903226,0.725650
2,DENIM,52,0.250000,0.673077,0.769231,0.865385,0.480153


In [320]:
comparison = (
    baseline_results
    .merge(
        yarn_results,
        on=[
            "material_group",
            "test_materials"
        ],
        suffixes=(
            "_baseline",
            "_with_yarn"
        )
    )
)

for metric in [
    "top_1",
    "top_3",
    "top_5",
    "top_10",
    "mrr"
]:
    comparison[
        f"{metric}_delta"
    ] = (
        comparison[
            f"{metric}_with_yarn"
        ]
        - comparison[
            f"{metric}_baseline"
        ]
    )

display(comparison)

,material_group,test_materials,top_1_baseline,top_3_baseline,top_5_baseline,top_10_baseline,mrr_baseline,top_1_with_yarn,top_3_with_yarn,top_5_with_yarn,top_10_with_yarn,mrr_with_yarn,top_1_delta,top_3_delta,top_5_delta,top_10_delta,mrr_delta
0,ORME,356,0.328652,0.578652,0.662921,0.755618,0.481022,0.306180,0.511236,0.589888,0.702247,0.443131,-0.022472,-0.067416,-0.073034,-0.053371,-0.037891
1,DOKUMA,279,0.562724,0.759857,0.820789,0.899642,0.681414,0.620072,0.799283,0.849462,0.903226,0.725650,0.057348,0.039427,0.028674,0.003584,0.044236
2,DENIM,52,0.250000,0.673077,0.769231,0.865385,0.480153,0.250000,0.673077,0.769231,0.865385,0.480153,0.000000,0.000000,0.000000,0.000000,0.000000


In [321]:
# ORME feature ablation
orme_feature_configs = {
    "baseline": common_features,

    "yarn_count_only": common_features + [
        "knit_yarn_count_match"
    ],

    "yarn_type_only": common_features + [
        "knit_yarn_type_similarity"
    ],

    "both_yarn_features": common_features + [
        "knit_yarn_count_match",
        "knit_yarn_type_similarity"
    ]
}

In [322]:
# Evaluate each ORME feature configuration
orme_ablation_results = []

for experiment_name, features in orme_feature_configs.items():

    result = evaluate_group_models(
        {"1020001": features},
        pairwise_sample_clean,
        pairwise_training_clean,
        train_ids,
        test_ids
    )

    result["experiment"] = experiment_name

    orme_ablation_results.append(result)

orme_ablation_results = pd.concat(
    orme_ablation_results,
    ignore_index=True
)

display(
    orme_ablation_results[
        [
            "experiment",
            "test_materials",
            "top_1",
            "top_3",
            "top_5",
            "top_10",
            "mrr"
        ]
    ]
)

,experiment,test_materials,top_1,top_3,top_5,top_10,mrr
0,baseline,356,0.328652,0.578652,0.662921,0.755618,0.481022
1,yarn_count_only,356,0.396067,0.632022,0.702247,0.786517,0.539068
2,yarn_type_only,356,0.258427,0.441011,0.561798,0.685393,0.392837
3,both_yarn_features,356,0.306180,0.511236,0.589888,0.702247,0.443131


In [323]:
# Compare every experiment with the ORME baseline
baseline_row = (
    orme_ablation_results[
        orme_ablation_results["experiment"] == "baseline"
    ]
    .iloc[0]
)

for metric in [
    "top_1",
    "top_3",
    "top_5",
    "top_10",
    "mrr"
]:
    orme_ablation_results[
        f"{metric}_delta"
    ] = (
        orme_ablation_results[metric]
        - baseline_row[metric]
    )

display(
    orme_ablation_results[
        [
            "experiment",
            "top_1_delta",
            "top_3_delta",
            "top_5_delta",
            "top_10_delta",
            "mrr_delta"
        ]
    ]
)

,experiment,top_1_delta,top_3_delta,top_5_delta,top_10_delta,mrr_delta
0,baseline,0.000000,0.000000,0.000000,0.000000,0.000000
1,yarn_count_only,0.067416,0.053371,0.039326,0.030899,0.058046
2,yarn_type_only,-0.070225,-0.137640,-0.101124,-0.070225,-0.088184
3,both_yarn_features,-0.022472,-0.067416,-0.073034,-0.053371,-0.037891


In [324]:
# Final group-specific feature configuration
final_group_features = {
    "1020001": common_features + [
        "knit_yarn_count_match"
    ],

    "1020002": common_features + [
        "weave_type_match",
        "weft_yarn_count_match"
    ],

    "1030004": common_features + [
        "weave_type_match"
    ]
}

In [325]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [326]:
def evaluate_random_forest_models(
    feature_config,
    sample_df,
    full_df,
    train_ids,
    test_ids
):
    results = []
    models = {}

    group_name_mapping = {
        "1020001": "ORME",
        "1020002": "DOKUMA",
        "1030004": "DENIM"
    }

    for group_code, features in feature_config.items():

        train_group = sample_df[
            sample_df["material_id"].isin(train_ids)
            & (
                sample_df["material_group_code"]
                == group_code
            )
        ].copy()

        test_group = full_df[
            full_df["material_id"].isin(test_ids)
            & (
                full_df["material_group_code"]
                == group_code
            )
        ].copy()

        model = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value=-1,
                    add_indicator=True
                )
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=300,
                    max_depth=12,
                    min_samples_leaf=2,
                    class_weight="balanced_subsample",
                    n_jobs=-1,
                    random_state=42
                )
            )
        ])

        model.fit(
            train_group[features],
            train_group["is_match"]
        )

        test_group["score"] = (
            model.predict_proba(
                test_group[features]
            )[:, 1]
        )

        test_group["rank"] = (
            test_group
            .groupby("material_id")["score"]
            .rank(
                ascending=False,
                method="first"
            )
        )

        true_rows = (
            test_group[
                test_group["is_match"] == 1
            ]
            .copy()
        )

        results.append({
            "material_group":
                group_name_mapping[group_code],
            "test_materials":
                len(true_rows),
            "top_1":
                (true_rows["rank"] <= 1).mean(),
            "top_3":
                (true_rows["rank"] <= 3).mean(),
            "top_5":
                (true_rows["rank"] <= 5).mean(),
            "top_10":
                (true_rows["rank"] <= 10).mean(),
            "mrr":
                (1 / true_rows["rank"]).mean()
        })

        models[group_code] = model

    return pd.DataFrame(results), models

In [327]:
rf_results, rf_models = (
    evaluate_random_forest_models(
        final_group_features,
        pairwise_sample_clean,
        pairwise_training_clean,
        train_ids,
        test_ids
    )
)

display(rf_results)

,material_group,test_materials,top_1,top_3,top_5,top_10,mrr
0,ORME,356,0.370787,0.547753,0.609551,0.699438,0.486701
1,DOKUMA,279,0.612903,0.763441,0.863799,0.939068,0.714702
2,DENIM,52,0.557692,0.730769,0.788462,0.826923,0.659534


In [328]:
logistic_final_results = evaluate_group_models(
    final_group_features,
    pairwise_sample_clean,
    pairwise_training_clean,
    train_ids,
    test_ids
)

display(logistic_final_results)

,material_group,test_materials,top_1,top_3,top_5,top_10,mrr
0,ORME,356,0.396067,0.632022,0.702247,0.786517,0.539068
1,DOKUMA,279,0.620072,0.799283,0.849462,0.903226,0.725650
2,DENIM,52,0.250000,0.673077,0.769231,0.865385,0.480153


In [329]:
model_comparison = (
    logistic_final_results
    .merge(
        rf_results,
        on=[
            "material_group",
            "test_materials"
        ],
        suffixes=(
            "_logistic",
            "_random_forest"
        )
    )
)

for metric in [
    "top_1",
    "top_3",
    "top_5",
    "top_10",
    "mrr"
]:
    model_comparison[
        f"{metric}_delta"
    ] = (
        model_comparison[
            f"{metric}_random_forest"
        ]
        - model_comparison[
            f"{metric}_logistic"
        ]
    )

display(model_comparison)

,material_group,test_materials,top_1_logistic,top_3_logistic,top_5_logistic,top_10_logistic,mrr_logistic,top_1_random_forest,top_3_random_forest,top_5_random_forest,top_10_random_forest,mrr_random_forest,top_1_delta,top_3_delta,top_5_delta,top_10_delta,mrr_delta
0,ORME,356,0.396067,0.632022,0.702247,0.786517,0.539068,0.370787,0.547753,0.609551,0.699438,0.486701,-0.025281,-0.084270,-0.092697,-0.087079,-0.052367
1,DOKUMA,279,0.620072,0.799283,0.849462,0.903226,0.725650,0.612903,0.763441,0.863799,0.939068,0.714702,-0.007168,-0.035842,0.014337,0.035842,-0.010948
2,DENIM,52,0.250000,0.673077,0.769231,0.865385,0.480153,0.557692,0.730769,0.788462,0.826923,0.659534,0.307692,0.057692,0.019231,-0.038462,0.179382


## 11. Train / validation / holdout ayrımı ve model seçimi


In [330]:
# Current split has been used for model/feature selection,
# so it should be treated as validation data
validation_ids = test_ids
development_train_ids = train_ids

In [331]:
from sklearn.model_selection import train_test_split

material_table = (
    pairwise_training_clean[
        ["material_id", "material_group_code"]
    ]
    .drop_duplicates("material_id")
)

# First reserve 20% as untouched holdout test
development_materials, holdout_materials = train_test_split(
    material_table,
    test_size=0.20,
    random_state=123,
    stratify=material_table["material_group_code"]
)

# Split development portion into train and validation
train_materials, validation_materials = train_test_split(
    development_materials,
    test_size=0.20,
    random_state=123,
    stratify=development_materials["material_group_code"]
)

train_ids_final = set(train_materials["material_id"])
validation_ids_final = set(validation_materials["material_id"])
holdout_ids = set(holdout_materials["material_id"])

print("Train materials:", len(train_ids_final))
print("Validation materials:", len(validation_ids_final))
print("Holdout materials:", len(holdout_ids))

Train materials: 2196
Validation materials: 550
Holdout materials: 687


In [332]:
for name, df in {
    "Train": train_materials,
    "Validation": validation_materials,
    "Holdout": holdout_materials
}.items():
    print(f"\n{name}")
    print(
        df["material_group_code"]
        .value_counts()
        .sort_index()
    )


Train
material_group_code
1020001    1136
1020002     893
1030004     167
Name: count, dtype: int64

Validation
material_group_code
1020001    284
1020002    224
1030004     42
Name: count, dtype: int64

Holdout
material_group_code
1020001    356
1020002    279
1030004     52
Name: count, dtype: int64


In [333]:
def build_training_sample(
    full_df,
    train_ids,
    hard_negatives_per_material=20,
    random_negatives_per_material=20,
    random_state=42
):
    train_pool = full_df[
        full_df["material_id"].isin(train_ids)
    ].copy()

    positive_pairs = train_pool[
        train_pool["is_match"] == 1
    ].copy()

    negative_pairs = train_pool[
        train_pool["is_match"] == 0
    ].copy()

    # Heuristic used only to identify difficult negatives
    negative_pairs["hardness_score"] = (
        negative_pairs["fiber_similarity"].fillna(0)
        + negative_pairs["weight_similarity"].fillna(0)
        + negative_pairs["fabric_structure_match"].fillna(0)
        + 0.5 * negative_pairs["coloring_match"].fillna(0)
        + 0.25 * negative_pairs["retrieval_similarity"].fillna(0)
    )

    hard_negatives = (
        negative_pairs
        .sort_values(
            ["material_id", "hardness_score"],
            ascending=[True, False]
        )
        .groupby("material_id")
        .head(hard_negatives_per_material)
    )

    rng = np.random.default_rng(random_state)

    negative_pairs["random_score"] = (
        rng.random(len(negative_pairs))
    )

    random_negatives = (
        negative_pairs
        .sort_values(
            ["material_id", "random_score"]
        )
        .groupby("material_id")
        .head(random_negatives_per_material)
    )

    training_sample = (
        pd.concat(
            [
                positive_pairs,
                hard_negatives,
                random_negatives
            ],
            ignore_index=True
        )
        .drop_duplicates(
            ["material_id", "candidate_plm_code"]
        )
    )

    return training_sample

In [334]:
training_sample_final = build_training_sample(
    pairwise_training_clean,
    train_ids_final
)

print(
    "Training sample rows:",
    len(training_sample_final)
)

print(
    "Training materials:",
    training_sample_final["material_id"].nunique()
)

print(
    "Positive pairs:",
    training_sample_final["is_match"].sum()
)

print(
    "Positive rate:",
    f"{training_sample_final['is_match'].mean():.2%}"
)

Training sample rows: 87544
Training materials: 2196
Positive pairs: 2196
Positive rate: 2.51%


In [335]:
common_features = [
    "retrieval_similarity",
    "from_structure_weight",
    "from_fiber_weight",
    "fiber_similarity",
    "weight_similarity",
    "fabric_structure_match",
    "coloring_match"
]

final_group_features = {
    # ORME
    "1020001": common_features + [
        "knit_yarn_count_match"
    ],

    # DOKUMA
    "1020002": common_features + [
        "weave_type_match",
        "weft_yarn_count_match"
    ],

    # DENIM
    "1030004": common_features + [
        "weave_type_match"
    ]
}

In [336]:
logistic_validation_results = evaluate_group_models(
    final_group_features,
    training_sample_final,
    pairwise_training_clean,
    train_ids_final,
    validation_ids_final
)

display(logistic_validation_results)

,material_group,test_materials,top_1,top_3,top_5,top_10,mrr
0,ORME,284,0.302817,0.503521,0.588028,0.676056,0.432422
1,DOKUMA,224,0.580357,0.785714,0.875000,0.919643,0.706336
2,DENIM,42,0.547619,0.952381,0.976190,0.976190,0.741911


In [337]:
rf_validation_results, rf_validation_models = (
    evaluate_random_forest_models(
        final_group_features,
        training_sample_final,
        pairwise_training_clean,
        train_ids_final,
        validation_ids_final
    )
)

display(rf_validation_results)

,material_group,test_materials,top_1,top_3,top_5,top_10,mrr
0,ORME,284,0.366197,0.517606,0.563380,0.633803,0.462374
1,DOKUMA,224,0.642857,0.808036,0.866071,0.919643,0.737909
2,DENIM,42,0.595238,0.857143,0.952381,0.976190,0.740260


In [338]:
validation_model_comparison = (
    logistic_validation_results
    .merge(
        rf_validation_results,
        on=[
            "material_group",
            "test_materials"
        ],
        suffixes=(
            "_logistic",
            "_random_forest"
        )
    )
)

for metric in [
    "top_1",
    "top_3",
    "top_5",
    "top_10",
    "mrr"
]:
    validation_model_comparison[
        f"{metric}_delta"
    ] = (
        validation_model_comparison[
            f"{metric}_random_forest"
        ]
        - validation_model_comparison[
            f"{metric}_logistic"
        ]
    )

display(validation_model_comparison)

,material_group,test_materials,top_1_logistic,top_3_logistic,top_5_logistic,top_10_logistic,mrr_logistic,top_1_random_forest,top_3_random_forest,top_5_random_forest,top_10_random_forest,mrr_random_forest,top_1_delta,top_3_delta,top_5_delta,top_10_delta,mrr_delta
0,ORME,284,0.302817,0.503521,0.588028,0.676056,0.432422,0.366197,0.517606,0.563380,0.633803,0.462374,0.063380,0.014085,-0.024648,-0.042254,0.029953
1,DOKUMA,224,0.580357,0.785714,0.875000,0.919643,0.706336,0.642857,0.808036,0.866071,0.919643,0.737909,0.062500,0.022321,-0.008929,0.000000,0.031573
2,DENIM,42,0.547619,0.952381,0.976190,0.976190,0.741911,0.595238,0.857143,0.952381,0.976190,0.740260,0.047619,-0.095238,-0.023810,0.000000,-0.001651


In [339]:
from sklearn.model_selection import KFold

development_ids = set(
    development_materials["material_id"]
)

development_pairwise = (
    pairwise_training_clean[
        pairwise_training_clean["material_id"]
        .isin(development_ids)
    ]
    .copy()
)

In [340]:
def cross_validate_rerankers(
    full_df,
    feature_config,
    n_splits=5,
    random_state=42
):
    cv_results = []

    group_name_mapping = {
        "1020001": "ORME",
        "1020002": "DOKUMA",
        "1030004": "DENIM"
    }

    for group_code, features in feature_config.items():

        material_ids = (
            full_df.loc[
                full_df["material_group_code"] == group_code,
                "material_id"
            ]
            .drop_duplicates()
            .to_numpy()
        )

        kfold = KFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=random_state
        )

        for fold, (train_idx, val_idx) in enumerate(
            kfold.split(material_ids),
            start=1
        ):
            fold_train_ids = set(
                material_ids[train_idx]
            )

            fold_val_ids = set(
                material_ids[val_idx]
            )

            # Negative sampling only from fold training materials
            training_sample = build_training_sample(
                full_df,
                fold_train_ids,
                random_state=42 + fold
            )

            validation_full = (
                full_df[
                    full_df["material_id"]
                    .isin(fold_val_ids)
                ]
                .copy()
            )

            models = {
                "logistic": Pipeline([
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="constant",
                            fill_value=-1,
                            add_indicator=True
                        )
                    ),
                    (
                        "scaler",
                        StandardScaler()
                    ),
                    (
                        "model",
                        LogisticRegression(
                            max_iter=2000,
                            class_weight="balanced",
                            random_state=42
                        )
                    )
                ]),

                "random_forest":
                    Pipeline([
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="constant",
                                fill_value=-1,
                                add_indicator=True
                            )
                        ),
                        (
                            "model",
                            RandomForestClassifier(
                                n_estimators=300,
                                max_depth=12,
                                min_samples_leaf=2,
                                class_weight=
                                    "balanced_subsample",
                                n_jobs=-1,
                                random_state=42
                            )
                        )
                    ])
            }

            for model_name, model in models.items():

                model.fit(
                    training_sample[features],
                    training_sample["is_match"]
                )

                scored = validation_full.copy()

                scored["score"] = (
                    model.predict_proba(
                        scored[features]
                    )[:, 1]
                )

                scored["rank"] = (
                    scored
                    .groupby("material_id")["score"]
                    .rank(
                        ascending=False,
                        method="first"
                    )
                )

                true_rows = (
                    scored[
                        scored["is_match"] == 1
                    ]
                )

                cv_results.append({
                    "material_group":
                        group_name_mapping[group_code],
                    "fold": fold,
                    "model": model_name,
                    "test_materials":
                        len(true_rows),

                    "top_1":
                        (
                            true_rows["rank"] <= 1
                        ).mean(),

                    "top_3":
                        (
                            true_rows["rank"] <= 3
                        ).mean(),

                    "top_5":
                        (
                            true_rows["rank"] <= 5
                        ).mean(),

                    "top_10":
                        (
                            true_rows["rank"] <= 10
                        ).mean(),

                    "mrr":
                        (
                            1 / true_rows["rank"]
                        ).mean()
                })

    return pd.DataFrame(cv_results)

In [341]:
cv_results = cross_validate_rerankers(
    development_pairwise,
    final_group_features,
    n_splits=5,
    random_state=42
)

In [342]:
cv_summary = (
    cv_results
    .groupby(
        ["material_group", "model"]
    )
    .agg(
        folds=("fold", "count"),
        top_1_mean=("top_1", "mean"),
        top_1_std=("top_1", "std"),
        top_3_mean=("top_3", "mean"),
        top_5_mean=("top_5", "mean"),
        top_10_mean=("top_10", "mean"),
        mrr_mean=("mrr", "mean"),
        mrr_std=("mrr", "std")
    )
    .reset_index()
)

display(cv_summary)

,material_group,model,folds,top_1_mean,top_1_std,top_3_mean,top_5_mean,top_10_mean,mrr_mean,mrr_std
0,DENIM,logistic,5,0.321138,0.154024,0.651336,0.794890,0.885366,0.513038,0.148220
1,DENIM,random_forest,5,0.597793,0.081624,0.779675,0.837398,0.880488,0.701727,0.050760
2,DOKUMA,logistic,5,0.572137,0.044983,0.773551,0.836187,0.902462,0.691884,0.035342
3,DOKUMA,random_forest,5,0.636575,0.036208,0.805782,0.854108,0.906042,0.736020,0.029526
4,ORME,logistic,5,0.297887,0.044162,0.493662,0.580986,0.701408,0.432059,0.032373
5,ORME,random_forest,5,0.352817,0.029607,0.523239,0.605634,0.699296,0.468950,0.026933


In [343]:
cv_comparison = (
    cv_summary
    .pivot(
        index="material_group",
        columns="model",
        values=[
            "top_1_mean",
            "top_5_mean",
            "mrr_mean"
        ]
    )
)

display(cv_comparison)

top_1_mean               top_5_mean                mrr_mean  \
model            logistic random_forest   logistic random_forest  logistic   
material_group                                                               
DENIM            0.321138      0.597793   0.794890      0.837398  0.513038   
DOKUMA           0.572137      0.636575   0.836187      0.854108  0.691884   
ORME             0.297887      0.352817   0.580986      0.605634  0.432059   

                              
model          random_forest  
material_group                
DENIM               0.701727  
DOKUMA              0.736020  
ORME                0.468950

In [344]:
# Combine train and validation for final development training
development_ids_final = (
    train_ids_final
    | validation_ids_final
)

final_training_sample = build_training_sample(
    pairwise_training_clean,
    development_ids_final,
    hard_negatives_per_material=20,
    random_negatives_per_material=20,
    random_state=42
)

print(
    "Development training materials:",
    final_training_sample["material_id"].nunique()
)

print(
    "Training sample rows:",
    len(final_training_sample)
)

print(
    "Positive pairs:",
    final_training_sample["is_match"].sum()
)

Development training materials: 2746
Training sample rows: 109398
Positive pairs: 2746


In [345]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

final_rf_models = {}
holdout_results = []
holdout_scored_parts = []

group_name_mapping = {
    "1020001": "ORME",
    "1020002": "DOKUMA",
    "1030004": "DENIM"
}

for group_code, features in final_group_features.items():

    train_group = final_training_sample[
        final_training_sample["material_group_code"]
        == group_code
    ].copy()

    holdout_group = pairwise_training_clean[
        pairwise_training_clean["material_id"].isin(holdout_ids)
        & (
            pairwise_training_clean["material_group_code"]
            == group_code
        )
    ].copy()

    model = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=-1,
                add_indicator=True
            )
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=12,
                min_samples_leaf=2,
                class_weight="balanced_subsample",
                n_jobs=-1,
                random_state=42
            )
        )
    ])

    model.fit(
        train_group[features],
        train_group["is_match"]
    )

    holdout_group["match_probability"] = (
        model.predict_proba(
            holdout_group[features]
        )[:, 1]
    )

    holdout_group["predicted_rank"] = (
        holdout_group
        .groupby("material_id")["match_probability"]
        .rank(
            ascending=False,
            method="first"
        )
    )

    true_rows = holdout_group[
        holdout_group["is_match"] == 1
    ].copy()

    holdout_results.append({
        "material_group":
            group_name_mapping[group_code],

        "holdout_materials":
            len(true_rows),

        "top_1":
            (true_rows["predicted_rank"] <= 1).mean(),

        "top_3":
            (true_rows["predicted_rank"] <= 3).mean(),

        "top_5":
            (true_rows["predicted_rank"] <= 5).mean(),

        "top_10":
            (true_rows["predicted_rank"] <= 10).mean(),

        "mrr":
            (
                1 / true_rows["predicted_rank"]
            ).mean()
    })

    final_rf_models[group_code] = model
    holdout_scored_parts.append(holdout_group)

holdout_results = pd.DataFrame(
    holdout_results
)

holdout_scored = pd.concat(
    holdout_scored_parts,
    ignore_index=True
)

display(holdout_results)

,material_group,holdout_materials,top_1,top_3,top_5,top_10,mrr
0,ORME,356,0.345506,0.528090,0.603933,0.696629,0.470249
1,DOKUMA,279,0.620072,0.749104,0.842294,0.917563,0.713574
2,DENIM,52,0.615385,0.807692,0.903846,0.942308,0.728622


In [346]:
total_holdout = (
    holdout_results["holdout_materials"].sum()
)

overall_holdout = {}

for metric in [
    "top_1",
    "top_3",
    "top_5",
    "top_10",
    "mrr"
]:
    overall_holdout[metric] = (
        (
            holdout_results[metric]
            * holdout_results["holdout_materials"]
        ).sum()
        / total_holdout
    )

print("Reranker holdout performance")

for metric, value in overall_holdout.items():
    print(
        f"{metric}: {value:.4f}"
    )

Reranker holdout performance
top_1: 0.4774
top_3: 0.6390
top_5: 0.7234
top_10: 0.8049
mrr: 0.5886


In [347]:
holdout_true_ranks = (
    holdout_scored[
        holdout_scored["is_match"] == 1
    ][
        [
            "material_id",
            "material_group_code",
            "true_plm_code",
            "predicted_rank",
            "match_probability"
        ]
    ]
    .copy()
)

display(
    holdout_true_ranks[
        "predicted_rank"
    ].describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

count    687.000000
mean      12.196507
std       37.029270
min        1.000000
50%        2.000000
75%        7.000000
90%       28.400000
95%       54.000000
99%      163.360000
max      528.000000
Name: predicted_rank, dtype: float64

In [348]:
# Score validation candidates using the selected Random Forest models
validation_scored_parts = []

for group_code, features in final_group_features.items():

    train_group = training_sample_final[
        training_sample_final["material_group_code"]
        == group_code
    ].copy()

    validation_group = pairwise_training_clean[
        pairwise_training_clean["material_id"]
        .isin(validation_ids_final)
        & (
            pairwise_training_clean["material_group_code"]
            == group_code
        )
    ].copy()

    model = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=-1,
                add_indicator=True
            )
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=12,
                min_samples_leaf=2,
                class_weight="balanced_subsample",
                n_jobs=-1,
                random_state=42
            )
        )
    ])

    model.fit(
        train_group[features],
        train_group["is_match"]
    )

    validation_group["score"] = (
        model.predict_proba(
            validation_group[features]
        )[:, 1]
    )

    validation_scored_parts.append(
        validation_group
    )

validation_scored = pd.concat(
    validation_scored_parts,
    ignore_index=True
)

In [349]:
# Rank validation candidates
validation_scored["rank"] = (
    validation_scored
    .groupby("material_id")["score"]
    .rank(
        ascending=False,
        method="first"
    )
)

top_two = (
    validation_scored[
        validation_scored["rank"] <= 2
    ]
    .sort_values(
        ["material_id", "rank"]
    )
)

confidence_table = (
    top_two
    .pivot(
        index="material_id",
        columns="rank",
        values="score"
    )
    .rename(
        columns={
            1.0: "top1_score",
            2.0: "top2_score"
        }
    )
    .reset_index()
)

confidence_table["score_margin"] = (
    confidence_table["top1_score"]
    - confidence_table["top2_score"]
)

In [350]:
top1_predictions = (
    validation_scored[
        validation_scored["rank"] == 1
    ][
        [
            "material_id",
            "material_group_code",
            "candidate_plm_code",
            "true_plm_code",
            "score"
        ]
    ]
    .copy()
)

top1_predictions["is_top1_correct"] = (
    top1_predictions["candidate_plm_code"]
    == top1_predictions["true_plm_code"]
)

confidence_audit = (
    top1_predictions
    .merge(
        confidence_table,
        on="material_id",
        how="left"
    )
)

In [351]:
confidence_audit["margin_band"] = pd.cut(
    confidence_audit["score_margin"],
    bins=[
        -np.inf,
        0.05,
        0.10,
        0.20,
        0.30,
        np.inf
    ],
    labels=[
        "<=0.05",
        "0.05-0.10",
        "0.10-0.20",
        "0.20-0.30",
        ">0.30"
    ]
)

display(
    confidence_audit
    .groupby(
        "margin_band",
        observed=True
    )
    .agg(
        materials=("material_id", "size"),
        top1_accuracy=("is_top1_correct", "mean"),
        mean_top1_score=("top1_score", "mean"),
        mean_margin=("score_margin", "mean")
    )
)

,materials,top1_accuracy,mean_top1_score,mean_margin
margin_band,,,,
<=0.05,319,0.228840,0.863717,0.004680
0.05-0.10,54,0.722222,0.906277,0.073826
0.10-0.20,31,0.774194,0.906730,0.128830
0.20-0.30,39,0.897436,0.885950,0.236523
>0.30,107,0.953271,0.906672,0.600656


In [352]:
display(
    confidence_audit
    .groupby("material_group_code")
    .agg(
        materials=("material_id", "size"),
        top1_accuracy=("is_top1_correct", "mean"),
        mean_top1_score=("top1_score", "mean"),
        mean_margin=("score_margin", "mean")
    )
)

,materials,top1_accuracy,mean_top1_score,mean_margin
material_group_code,,,,
1020001,284,0.366197,0.841860,0.068221
1020002,224,0.642857,0.926459,0.227312
1030004,42,0.595238,0.893436,0.301785


In [353]:
# Evaluate cumulative confidence thresholds
margin_thresholds = [
    0.00,
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.40,
    0.50
]

threshold_results = []

total_materials = len(confidence_audit)

for threshold in margin_thresholds:

    selected = confidence_audit[
        confidence_audit["score_margin"] >= threshold
    ]

    if len(selected) == 0:
        continue

    threshold_results.append({
        "margin_threshold": threshold,
        "selected_materials": len(selected),
        "coverage": len(selected) / total_materials,
        "precision": selected["is_top1_correct"].mean()
    })

threshold_results = pd.DataFrame(
    threshold_results
)

display(threshold_results)

,margin_threshold,selected_materials,coverage,precision
0,0.00,550,1.000000,0.496364
1,0.05,231,0.420000,0.865801
2,0.10,177,0.321818,0.909605
3,0.15,155,0.281818,0.935484
4,0.20,146,0.265455,0.938356
5,0.25,119,0.216364,0.941176
6,0.30,107,0.194545,0.953271
7,0.40,85,0.154545,0.976471
8,0.50,66,0.120000,1.000000


In [354]:
# Evaluate confidence thresholds by material group
group_threshold_results = []

for group_code, group in confidence_audit.groupby(
    "material_group_code"
):

    group_total = len(group)

    for threshold in margin_thresholds:

        selected = group[
            group["score_margin"] >= threshold
        ]

        if len(selected) == 0:
            continue

        group_threshold_results.append({
            "material_group_code": group_code,
            "margin_threshold": threshold,
            "selected_materials": len(selected),
            "coverage": len(selected) / group_total,
            "precision": selected[
                "is_top1_correct"
            ].mean()
        })

group_threshold_results = pd.DataFrame(
    group_threshold_results
)

display(group_threshold_results)

,material_group_code,margin_threshold,selected_materials,coverage,precision
0,1020001,0.00,284,1.000000,0.366197
1,1020001,0.05,86,0.302817,0.848837
2,1020001,0.10,58,0.204225,0.879310
3,1020001,0.15,40,0.140845,0.950000
4,1020001,0.20,35,0.123239,0.942857
5,1020001,0.25,20,0.070423,0.950000
6,1020001,0.30,17,0.059859,0.941176
7,1020001,0.40,12,0.042254,1.000000
8,1020001,0.50,10,0.035211,1.000000
9,1020002,0.00,224,1.000000,0.642857


In [355]:
# Find the highest-coverage threshold
# satisfying a target precision
def find_best_threshold(
    results,
    target_precision
):
    eligible = results[
        results["precision"] >= target_precision
    ].copy()

    if eligible.empty:
        return None

    return (
        eligible
        .sort_values(
            [
                "coverage",
                "margin_threshold"
            ],
            ascending=[False, True]
        )
        .iloc[0]
    )


for target in [0.90, 0.95]:
    result = find_best_threshold(
        threshold_results,
        target
    )

    print(
        f"\nTarget precision: {target:.0%}"
    )

    if result is None:
        print("No threshold reached the target.")
    else:
        print(
            f"Margin threshold: "
            f"{result['margin_threshold']:.2f}"
        )
        print(
            f"Coverage: "
            f"{result['coverage']:.2%}"
        )
        print(
            f"Observed precision: "
            f"{result['precision']:.2%}"
        )


Target precision: 90%
Margin threshold: 0.10
Coverage: 32.18%
Observed precision: 90.96%

Target precision: 95%
Margin threshold: 0.30
Coverage: 19.45%
Observed precision: 95.33%


## 12. Güven eşikleri ve AUTO / REVIEW karar katmanı


In [356]:
# Confidence thresholds selected using validation data only
auto_margin_thresholds = {
    "1020001": 0.15,  # ORME
    "1020002": 0.30,  # DOKUMA
    "1030004": 0.40   # DENIM
}

review_margin_threshold = 0.05

In [357]:
# Extract top two candidates from holdout results
holdout_top_two = (
    holdout_scored[
        holdout_scored["predicted_rank"] <= 2
    ]
    .sort_values(
        ["material_id", "predicted_rank"]
    )
)

holdout_confidence = (
    holdout_top_two
    .pivot(
        index="material_id",
        columns="predicted_rank",
        values="match_probability"
    )
    .rename(
        columns={
            1.0: "top1_score",
            2.0: "top2_score"
        }
    )
    .reset_index()
)

holdout_confidence["score_margin"] = (
    holdout_confidence["top1_score"]
    - holdout_confidence["top2_score"]
)

In [358]:
# Add top-1 prediction and correctness
holdout_top1 = (
    holdout_scored[
        holdout_scored["predicted_rank"] == 1
    ][
        [
            "material_id",
            "material_group_code",
            "candidate_plm_code",
            "true_plm_code",
            "match_probability"
        ]
    ]
    .copy()
)

holdout_top1["is_top1_correct"] = (
    holdout_top1["candidate_plm_code"]
    == holdout_top1["true_plm_code"]
)

holdout_decisions = (
    holdout_top1
    .merge(
        holdout_confidence,
        on="material_id",
        how="left",
        validate="one_to_one"
    )
)

In [359]:
# Assign operational decision based on fixed validation thresholds
def assign_confidence_decision(row):

    auto_threshold = auto_margin_thresholds[
        row["material_group_code"]
    ]

    if row["score_margin"] >= auto_threshold:
        return "AUTO"

    if row["score_margin"] >= review_margin_threshold:
        return "REVIEW"

    return "LOW_CONFIDENCE"


holdout_decisions["decision"] = (
    holdout_decisions.apply(
        assign_confidence_decision,
        axis=1
    )
)

In [360]:
# Evaluate decision-layer performance
decision_summary = (
    holdout_decisions
    .groupby("decision")
    .agg(
        materials=("material_id", "size"),
        top1_accuracy=("is_top1_correct", "mean"),
        mean_margin=("score_margin", "mean")
    )
    .reset_index()
)

decision_summary["coverage"] = (
    decision_summary["materials"]
    / len(holdout_decisions)
)

display(decision_summary)

,decision,materials,top1_accuracy,mean_margin,coverage
0,AUTO,156,0.955128,0.528796,0.227074
1,LOW_CONFIDENCE,418,0.244019,0.005825,0.608443
2,REVIEW,113,0.681416,0.122491,0.164483


In [361]:
# Audit automatic recommendations by material group
auto_holdout_summary = (
    holdout_decisions[
        holdout_decisions["decision"] == "AUTO"
    ]
    .groupby("material_group_code")
    .agg(
        auto_materials=("material_id", "size"),
        auto_precision=("is_top1_correct", "mean"),
        mean_margin=("score_margin", "mean")
    )
    .reset_index()
)

group_holdout_counts = (
    holdout_decisions
    .groupby("material_group_code")
    ["material_id"]
    .size()
    .rename("total_materials")
    .reset_index()
)

auto_holdout_summary = (
    auto_holdout_summary
    .merge(
        group_holdout_counts,
        on="material_group_code",
        how="left"
    )
)

auto_holdout_summary["auto_coverage"] = (
    auto_holdout_summary["auto_materials"]
    / auto_holdout_summary["total_materials"]
)

display(auto_holdout_summary)

,material_group_code,auto_materials,auto_precision,mean_margin,total_materials,auto_coverage
0,1020001,54,0.925926,0.336858,356,0.151685
1,1020002,81,0.975309,0.607317,279,0.290323
2,1030004,21,0.952381,0.719483,52,0.403846


In [362]:
# Overall automatic-decision performance
auto_predictions = (
    holdout_decisions[
        holdout_decisions["decision"] == "AUTO"
    ]
)

print(
    "AUTO coverage:",
    f"{len(auto_predictions) / len(holdout_decisions):.2%}"
)

print(
    "AUTO precision:",
    f"{auto_predictions['is_top1_correct'].mean():.2%}"
)

AUTO coverage: 22.71%
AUTO precision: 95.51%


In [363]:
# Add true-PLM presence within Top-K to the holdout decision table
holdout_rank_lookup = (
    holdout_scored[
        holdout_scored["is_match"] == 1
    ][
        [
            "material_id",
            "predicted_rank"
        ]
    ]
    .rename(
        columns={
            "predicted_rank": "true_plm_rank"
        }
    )
)

holdout_decisions = (
    holdout_decisions
    .merge(
        holdout_rank_lookup,
        on="material_id",
        how="left",
        validate="one_to_one"
    )
)

holdout_decisions["true_in_top3"] = (
    holdout_decisions["true_plm_rank"] <= 3
)

holdout_decisions["true_in_top5"] = (
    holdout_decisions["true_plm_rank"] <= 5
)

In [364]:
review_holdout = (
    holdout_decisions[
        holdout_decisions["decision"] == "REVIEW"
    ]
)

print(
    "REVIEW coverage:",
    f"{len(review_holdout) / len(holdout_decisions):.2%}"
)

print(
    "REVIEW Top-1 accuracy:",
    f"{review_holdout['is_top1_correct'].mean():.2%}"
)

print(
    "REVIEW Top-3 success:",
    f"{review_holdout['true_in_top3'].mean():.2%}"
)

print(
    "REVIEW Top-5 success:",
    f"{review_holdout['true_in_top5'].mean():.2%}"
)

REVIEW coverage: 16.45%
REVIEW Top-1 accuracy: 68.14%
REVIEW Top-3 success: 77.88%
REVIEW Top-5 success: 84.07%


In [365]:
review_group_summary = (
    review_holdout
    .groupby("material_group_code")
    .agg(
        review_materials=("material_id", "size"),
        top1_accuracy=("is_top1_correct", "mean"),
        top3_success=("true_in_top3", "mean"),
        top5_success=("true_in_top5", "mean"),
        mean_margin=("score_margin", "mean")
    )
    .reset_index()
)

display(review_group_summary)

,material_group_code,review_materials,top1_accuracy,top3_success,top5_success,mean_margin
0,1020001,56,0.660714,0.696429,0.732143,0.093911
1,1020002,52,0.692308,0.865385,0.942308,0.147452
2,1030004,5,0.800000,0.800000,1.000000,0.182986


In [366]:
# Operational success under the fixed decision policy
holdout_decisions["workflow_success"] = np.select(
    [
        holdout_decisions["decision"] == "AUTO",
        holdout_decisions["decision"] == "REVIEW"
    ],
    [
        holdout_decisions["is_top1_correct"],
        holdout_decisions["true_in_top3"]
    ],
    default=False
)

actionable = (
    holdout_decisions["decision"]
    .isin(["AUTO", "REVIEW"])
)

print(
    "AUTO + REVIEW coverage:",
    f"{actionable.mean():.2%}"
)

print(
    "Success among actionable materials:",
    f"{holdout_decisions.loc[actionable, 'workflow_success'].mean():.2%}"
)

AUTO + REVIEW coverage: 39.16%
Success among actionable materials: 88.10%


## 13. Attribute'u olmayan SAP malzemeleri için ayrı hat


In [367]:
# Prepare known mappings without SAP attributes
known_no_attribute_mappings = (
    known_retrieval_pairs[
        ~known_retrieval_pairs["material_id"]
        .astype("string")
        .isin(materials_with_attributes)
    ]
    .copy()
)

print(
    "Known mappings without SAP attributes:",
    len(known_no_attribute_mappings)
)

Known mappings without SAP attributes: 2122


In [368]:
# Add the SAP-side material group used for blocking
known_no_attribute_audit = (
    known_no_attribute_mappings
    .merge(
        material_group_lookup.rename(
            columns={
                "material_group_code":
                "sap_material_group_code"
            }
        ),
        on="material_id",
        how="left"
    )
)

known_no_attribute_audit["group_match"] = (
    known_no_attribute_audit["material_group_code"]
    == known_no_attribute_audit["sap_material_group_code"]
)

print(
    "Material-group mismatches:",
    (~known_no_attribute_audit["group_match"]).sum()
)

display(
    known_no_attribute_audit[
        ~known_no_attribute_audit["group_match"]
    ].head(20)
)

Material-group mismatches: 37


,material_id,plm_code,material_group_code,sap_material_group_code,group_match
34,1020000086,119700,1030004,1020002,False
75,1020000225,132775,1030004,1020002,False
103,1020000275,166671,1030004,1020002,False
287,1020000693,24653,1030004,1020002,False
306,1020000743,28305,1020001,1020002,False
326,1020000802,32356,1030004,1020002,False
328,1020000805,32708,1030004,1020002,False
394,1020000942,354117,1020001,1020002,False
405,1020000961,354677,1020001,1020002,False
443,1020001022,356274,1020001,1020002,False


In [369]:
from sklearn.model_selection import train_test_split

no_attribute_materials = (
    known_no_attribute_audit[
        [
            "material_id",
            "sap_material_group_code"
        ]
    ]
    .drop_duplicates("material_id")
    .rename(
        columns={
            "sap_material_group_code":
            "material_group_code"
        }
    )
)

no_attr_development, no_attr_holdout = train_test_split(
    no_attribute_materials,
    test_size=0.20,
    random_state=321,
    stratify=no_attribute_materials[
        "material_group_code"
    ]
)

no_attr_dev_ids = set(
    no_attr_development["material_id"]
)

no_attr_holdout_ids = set(
    no_attr_holdout["material_id"]
)

print(
    "Development materials:",
    len(no_attr_dev_ids)
)

print(
    "Untouched holdout materials:",
    len(no_attr_holdout_ids)
)

Development materials: 1697
Untouched holdout materials: 425


In [370]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors


def generate_text_channel_candidates(
    sap_df,
    plm_df,
    material_group_code,
    plm_text_column,
    channel_name,
    top_k=50,
    analyzer="char_wb",
    ngram_range=(3, 5)
):
    sap_group = (
        sap_df[
            sap_df["material_group_code"]
            == material_group_code
        ]
        .copy()
        .reset_index(drop=True)
    )

    plm_group = (
        plm_df[
            plm_df["material_group_code"]
            == material_group_code
        ]
        .copy()
        .reset_index(drop=True)
    )

    sap_group = sap_group[
        sap_group["retrieval_text"].str.len() > 0
    ].reset_index(drop=True)

    plm_group = plm_group[
        plm_group[plm_text_column]
        .fillna("")
        .str.len() > 0
    ].reset_index(drop=True)

    if sap_group.empty or plm_group.empty:
        return pd.DataFrame()

    vectorizer = TfidfVectorizer(
        analyzer=analyzer,
        ngram_range=ngram_range,
        sublinear_tf=True,
        norm="l2"
    )

    plm_matrix = vectorizer.fit_transform(
        plm_group[plm_text_column]
    )

    sap_matrix = vectorizer.transform(
        sap_group["retrieval_text"]
    )

    n_neighbors = min(
        top_k,
        len(plm_group)
    )

    model = NearestNeighbors(
        n_neighbors=n_neighbors,
        metric="cosine",
        algorithm="brute"
    )

    model.fit(plm_matrix)

    distances, indices = model.kneighbors(
        sap_matrix
    )

    records = []

    for sap_idx in range(len(sap_group)):
        for rank, (plm_idx, distance) in enumerate(
            zip(
                indices[sap_idx],
                distances[sap_idx]
            ),
            start=1
        ):
            records.append({
                "material_id":
                    sap_group.loc[
                        sap_idx,
                        "material_id"
                    ],
                "candidate_plm_code":
                    plm_group.loc[
                        plm_idx,
                        "plm_code"
                    ],
                "channel":
                    channel_name,
                "channel_rank":
                    rank,
                "channel_similarity":
                    1 - distance
            })

    return pd.DataFrame(records)

In [371]:
sap_no_attr_dev = (
    sap_candidates_source[
        sap_candidates_source["material_id"]
        .isin(no_attr_dev_ids)
    ]
    .copy()
)

In [372]:
text_channel_tables = []

channel_config = {
    "original_text": "retrieval_text",
    "structured_text": "structured_text",
    "multi_value_text": "multi_value_text",
    "enriched_text": "retrieval_text_enriched"
}

for group_code in [
    "1020001",
    "1020002",
    "1030004"
]:
    for channel_name, text_column in (
        channel_config.items()
    ):
        result = generate_text_channel_candidates(
            sap_no_attr_dev,
            plm_candidates_enriched,
            material_group_code=group_code,
            plm_text_column=text_column,
            channel_name=channel_name,
            top_k=50
        )

        if not result.empty:
            text_channel_tables.append(result)

multi_channel_candidates = pd.concat(
    text_channel_tables,
    ignore_index=True
)

In [373]:
multi_channel_union = (
    multi_channel_candidates[
        [
            "material_id",
            "candidate_plm_code"
        ]
    ]
    .drop_duplicates()
)

print(
    "Candidate pairs:",
    len(multi_channel_union)
)

print(
    "Average candidates per SAP:",
    len(multi_channel_union)
    / multi_channel_union["material_id"].nunique()
)

Candidate pairs: 274353
Average candidates per SAP: 161.6694166175604


In [374]:
no_attr_dev_truth = (
    known_no_attribute_audit[
        known_no_attribute_audit["material_id"]
        .isin(no_attr_dev_ids)
    ][
        [
            "material_id",
            "plm_code"
        ]
    ]
    .copy()
)

no_attr_dev_recall = (
    no_attr_dev_truth
    .merge(
        multi_channel_union,
        left_on=[
            "material_id",
            "plm_code"
        ],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left"
    )
)

no_attr_dev_recall["retrieved"] = (
    no_attr_dev_recall[
        "candidate_plm_code"
    ].notna()
)

print(
    "Multi-channel development recall:",
    f"{no_attr_dev_recall['retrieved'].mean():.2%}"
)

Multi-channel development recall: 43.72%


In [375]:
channel_recall_summary = []

for channel_name in (
    multi_channel_candidates["channel"]
    .unique()
):
    channel_candidates = (
        multi_channel_candidates[
            multi_channel_candidates["channel"]
            == channel_name
        ][
            [
                "material_id",
                "candidate_plm_code"
            ]
        ]
        .drop_duplicates()
    )

    evaluation = (
        no_attr_dev_truth
        .merge(
            channel_candidates,
            left_on=[
                "material_id",
                "plm_code"
            ],
            right_on=[
                "material_id",
                "candidate_plm_code"
            ],
            how="left"
        )
    )

    channel_recall_summary.append({
        "channel": channel_name,
        "recall": (
            evaluation[
                "candidate_plm_code"
            ].notna().mean()
        )
    })

display(
    pd.DataFrame(
        channel_recall_summary
    ).sort_values(
        "recall",
        ascending=False
    )
)

,channel,recall
1,structured_text,0.324101
3,enriched_text,0.304066
2,multi_value_text,0.131408
0,original_text,0.025928


In [376]:
# Evaluate multi-channel recall by material group
no_attr_dev_recall_grouped = (
    no_attr_dev_recall
    .merge(
        no_attribute_materials[
            [
                "material_id",
                "material_group_code"
            ]
        ],
        on="material_id",
        how="left"
    )
)

display(
    no_attr_dev_recall_grouped
    .groupby("material_group_code")
    .agg(
        known_materials=("material_id", "size"),
        candidate_recall=("retrieved", "mean")
    )
)

,known_materials,candidate_recall
material_group_code,,
1020001,880,0.190909
1020002,808,0.702970
1030004,9,0.666667


In [377]:
# Build training data for description-to-structure prediction
structure_prediction_data = (
    sap_candidates_source[
        [
            "material_id",
            "material_group_code",
            "retrieval_text"
        ]
    ]
    .merge(
        sap_structure_candidates[
            [
                "material_id",
                "structure"
            ]
        ],
        on="material_id",
        how="inner"
    )
)

structure_prediction_data = (
    structure_prediction_data[
        structure_prediction_data["retrieval_text"].str.len() > 0
    ]
    .dropna(
        subset=[
            "material_group_code",
            "structure"
        ]
    )
    .copy()
)

display(
    structure_prediction_data
    .groupby("material_group_code")
    .agg(
        materials=("material_id", "nunique"),
        structure_classes=("structure", "nunique")
    )
)

,materials,structure_classes
material_group_code,,
1020001,4663,98
1020002,3235,101
1030004,1178,16


In [378]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, top_k_accuracy_score

In [379]:
# Evaluate description-based fabric-structure prediction
structure_model_results = []
structure_models = {}

for group_code in [
    "1020001",
    "1020002",
    "1030004"
]:
    group_data = (
        structure_prediction_data[
            structure_prediction_data["material_group_code"]
            == group_code
        ]
        .copy()
    )

    # Very rare labels are not suitable for stratified classification
    label_counts = group_data["structure"].value_counts()

    valid_labels = label_counts[
        label_counts >= 2
    ].index

    group_data = group_data[
        group_data["structure"].isin(valid_labels)
    ].copy()

    train_data, test_data = train_test_split(
        group_data,
        test_size=0.20,
        random_state=42,
        stratify=group_data["structure"]
    )

    model = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(3, 5),
                sublinear_tf=True,
                min_df=2
            )
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    ])

    model.fit(
        train_data["retrieval_text"],
        train_data["structure"]
    )

    predicted = model.predict(
        test_data["retrieval_text"]
    )

    probabilities = model.predict_proba(
        test_data["retrieval_text"]
    )

    classes = model.named_steps[
        "model"
    ].classes_

    structure_model_results.append({
        "material_group_code": group_code,
        "test_materials": len(test_data),
        "classes": len(classes),
        "top_1_accuracy":
            accuracy_score(
                test_data["structure"],
                predicted
            ),
        "top_3_accuracy":
            top_k_accuracy_score(
                test_data["structure"],
                probabilities,
                k=min(3, len(classes)),
                labels=classes
            )
    })

    structure_models[group_code] = model

display(
    pd.DataFrame(structure_model_results)
)

,material_group_code,test_materials,classes,top_1_accuracy,top_3_accuracy
0,1020001,928,71,0.872845,0.959052
1,1020002,640,64,0.912500,0.968750
2,1030004,235,11,0.902128,0.995745


In [380]:
# Retrain structure classifiers using all available labeled SAP materials
final_structure_models = {}

for group_code in [
    "1020001",
    "1020002",
    "1030004"
]:
    group_data = (
        structure_prediction_data[
            structure_prediction_data["material_group_code"]
            == group_code
        ]
        .copy()
    )

    # Keep labels with at least two observations
    label_counts = group_data["structure"].value_counts()

    valid_labels = label_counts[
        label_counts >= 2
    ].index

    group_data = group_data[
        group_data["structure"].isin(valid_labels)
    ].copy()

    model = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(3, 5),
                sublinear_tf=True,
                min_df=2
            )
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    ])

    model.fit(
        group_data["retrieval_text"],
        group_data["structure"]
    )

    final_structure_models[group_code] = model

In [381]:
# Predict Top-3 fabric structures for SAP materials without attributes
predicted_structure_records = []

for group_code, model in final_structure_models.items():

    sap_group = (
        sap_no_attr_dev[
            sap_no_attr_dev["material_group_code"]
            == group_code
        ]
        .copy()
    )

    if sap_group.empty:
        continue

    probabilities = model.predict_proba(
        sap_group["retrieval_text"]
    )

    classes = model.named_steps[
        "model"
    ].classes_

    top_k = min(3, len(classes))

    top_indices = np.argsort(
        probabilities,
        axis=1
    )[:, -top_k:][:, ::-1]

    for row_idx, material_id in enumerate(
        sap_group["material_id"]
    ):
        for rank, class_idx in enumerate(
            top_indices[row_idx],
            start=1
        ):
            predicted_structure_records.append({
                "material_id": material_id,
                "material_group_code": group_code,
                "predicted_structure":
                    classes[class_idx],
                "structure_rank": rank,
                "structure_probability":
                    probabilities[row_idx, class_idx]
            })

predicted_structures = pd.DataFrame(
    predicted_structure_records
)

display(predicted_structures.head(10))

,material_id,material_group_code,predicted_structure,structure_rank,structure_probability
0,1020000013,1020001,JERSEY,1,0.103946
1,1020000013,1020001,FLAT,2,0.051972
2,1020000013,1020001,QUILTED,3,0.043342
3,1020000015,1020001,WRAPKNITTED,1,0.348364
4,1020000015,1020001,VELOUR,2,0.044243
5,1020000015,1020001,POLARFLEECE,3,0.032921
6,1020000018,1020001,JERSEY,1,0.106730
7,1020000018,1020001,MESHRIB,2,0.064524
8,1020000018,1020001,FLAT,3,0.040869
9,1020000024,1020001,THREETHREADFLEECE,1,0.217749


In [382]:
# Prepare PLM structure-aware retrieval table
plm_structure_retrieval = (
    plm_candidates_enriched[
        [
            "plm_code",
            "material_group_code",
            "structured_text"
        ]
    ]
    .merge(
        plm_structure_candidates[
            [
                "plm_code",
                "structure"
            ]
        ],
        on="plm_code",
        how="inner"
    )
)

plm_structure_retrieval = (
    plm_structure_retrieval[
        plm_structure_retrieval[
            "structured_text"
        ].str.len() > 0
    ]
    .copy()
)

In [383]:
def generate_predicted_structure_candidates(
    sap_df,
    predicted_structures,
    plm_df,
    top_k_per_structure=30
):
    candidate_records = []

    for group_code in predicted_structures[
        "material_group_code"
    ].unique():

        sap_group = (
            sap_df[
                sap_df["material_group_code"]
                == group_code
            ]
            .set_index("material_id")
        )

        predictions_group = (
            predicted_structures[
                predicted_structures[
                    "material_group_code"
                ] == group_code
            ]
        )

        for structure_value, structure_predictions in (
            predictions_group.groupby(
                "predicted_structure"
            )
        ):

            plm_pool = (
                plm_df[
                    (plm_df["material_group_code"]
                     == group_code)
                    & (
                        plm_df["structure"]
                        == structure_value
                    )
                ]
                .reset_index(drop=True)
            )

            if plm_pool.empty:
                continue

            vectorizer = TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(3, 5),
                sublinear_tf=True,
                norm="l2"
            )

            plm_matrix = vectorizer.fit_transform(
                plm_pool["structured_text"]
            )

            n_neighbors = min(
                top_k_per_structure,
                len(plm_pool)
            )

            nn_model = NearestNeighbors(
                n_neighbors=n_neighbors,
                metric="cosine",
                algorithm="brute"
            )

            nn_model.fit(plm_matrix)

            for pred in structure_predictions.itertuples(
                index=False
            ):

                if pred.material_id not in sap_group.index:
                    continue

                sap_text = sap_group.loc[
                    pred.material_id,
                    "retrieval_text"
                ]

                sap_vector = vectorizer.transform(
                    [sap_text]
                )

                distances, indices = (
                    nn_model.kneighbors(
                        sap_vector
                    )
                )

                for retrieval_rank, (
                    plm_idx,
                    distance
                ) in enumerate(
                    zip(
                        indices[0],
                        distances[0]
                    ),
                    start=1
                ):
                    candidate_records.append({
                        "material_id":
                            pred.material_id,
                        "candidate_plm_code":
                            plm_pool.loc[
                                plm_idx,
                                "plm_code"
                            ],
                        "predicted_structure":
                            structure_value,
                        "structure_rank":
                            pred.structure_rank,
                        "structure_probability":
                            pred.structure_probability,
                        "structure_retrieval_rank":
                            retrieval_rank,
                        "structure_retrieval_similarity":
                            1 - distance
                    })

    return pd.DataFrame(candidate_records)

In [384]:
pseudo_structure_candidates = (
    generate_predicted_structure_candidates(
        sap_no_attr_dev,
        predicted_structures,
        plm_structure_retrieval,
        top_k_per_structure=30
    )
)

pseudo_structure_pairs = (
    pseudo_structure_candidates[
        [
            "material_id",
            "candidate_plm_code"
        ]
    ]
    .drop_duplicates()
)

print(
    "Pseudo-structure candidate pairs:",
    len(pseudo_structure_pairs)
)

print(
    "Average candidates per SAP:",
    len(pseudo_structure_pairs)
    / pseudo_structure_pairs[
        "material_id"
    ].nunique()
)

Pseudo-structure candidate pairs: 108435
Average candidates per SAP: 64.4295900178253


In [385]:
pseudo_structure_recall = (
    no_attr_dev_truth
    .merge(
        pseudo_structure_pairs,
        left_on=[
            "material_id",
            "plm_code"
        ],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left"
    )
)

pseudo_structure_recall["retrieved"] = (
    pseudo_structure_recall[
        "candidate_plm_code"
    ].notna()
)

print(
    "Pseudo-structure recall:",
    f"{pseudo_structure_recall['retrieved'].mean():.2%}"
)

Pseudo-structure recall: 38.72%


In [386]:
# Union text channels with predicted-structure retrieval
no_attr_combined_candidates = (
    pd.concat(
        [
            multi_channel_union,
            pseudo_structure_pairs
        ],
        ignore_index=True
    )
    .drop_duplicates()
)

no_attr_combined_recall = (
    no_attr_dev_truth
    .merge(
        no_attr_combined_candidates,
        left_on=[
            "material_id",
            "plm_code"
        ],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left"
    )
)

no_attr_combined_recall["retrieved"] = (
    no_attr_combined_recall[
        "candidate_plm_code"
    ].notna()
)

print(
    "Combined no-attribute recall:",
    f"{no_attr_combined_recall['retrieved'].mean():.2%}"
)

Combined no-attribute recall: 51.74%


In [387]:
no_attr_combined_grouped = (
    no_attr_combined_recall
    .merge(
        no_attribute_materials[
            [
                "material_id",
                "material_group_code"
            ]
        ],
        on="material_id",
        how="left"
    )
)

display(
    no_attr_combined_grouped
    .groupby("material_group_code")
    .agg(
        known_materials=("material_id", "size"),
        candidate_recall=("retrieved", "mean")
    )
)

,known_materials,candidate_recall
material_group_code,,
1020001,880,0.285227
1020002,808,0.768564
1030004,9,0.666667


In [388]:
# Convert yarn count into a retrieval-friendly canonical label
def canonicalize_yarn_count(value):
    parsed = parse_yarn_count(value)

    if pd.isna(parsed["count"]):
        return pd.NA

    count = parsed["count"]

    if float(count).is_integer():
        count_text = str(int(count))
    else:
        count_text = str(count)

    ply = parsed["ply"]

    # Missing ply and /1 are treated as equivalent for candidate retrieval
    if pd.isna(ply) or ply == 1:
        return count_text

    if float(ply).is_integer():
        ply_text = str(int(ply))
    else:
        ply_text = str(ply)

    return f"{count_text}/{ply_text}"

In [389]:
# Build description-to-yarn-count training data for knitted fabrics
yarn_count_prediction_data = (
    sap_candidates_source[
        [
            "material_id",
            "material_group_code",
            "retrieval_text"
        ]
    ]
    .merge(
        sap_yarn_count_knit[
            [
                "material_id",
                "sap_yarn_count"
            ]
        ],
        on="material_id",
        how="inner"
    )
)

yarn_count_prediction_data = (
    yarn_count_prediction_data[
        yarn_count_prediction_data["material_group_code"]
        == "1020001"
    ]
    .copy()
)

yarn_count_prediction_data["yarn_count_class"] = (
    yarn_count_prediction_data["sap_yarn_count"]
    .apply(canonicalize_yarn_count)
)

yarn_count_prediction_data = (
    yarn_count_prediction_data
    .dropna(subset=["yarn_count_class"])
)

print(
    "ORME materials:",
    len(yarn_count_prediction_data)
)

print(
    "Yarn-count classes:",
    yarn_count_prediction_data[
        "yarn_count_class"
    ].nunique()
)

display(
    yarn_count_prediction_data[
        "yarn_count_class"
    ].value_counts().head(20)
)

ORME materials: 2908
Yarn-count classes: 79


yarn_count_class
30       1472
1         411
20        201
40        111
28         88
36         86
70         79
26         54
24         41
150        38
30/2       34
100        34
50         25
75         23
3          15
10         14
30/3       13
30/20      12
30/70      12
16         11
Name: count, dtype: int64

In [390]:
# Keep classes that have enough observations for stratified evaluation
class_counts = (
    yarn_count_prediction_data[
        "yarn_count_class"
    ].value_counts()
)

valid_classes = class_counts[
    class_counts >= 2
].index

yarn_count_model_data = (
    yarn_count_prediction_data[
        yarn_count_prediction_data[
            "yarn_count_class"
        ].isin(valid_classes)
    ]
    .copy()
)

train_yarn, test_yarn = train_test_split(
    yarn_count_model_data,
    test_size=0.20,
    random_state=42,
    stratify=yarn_count_model_data[
        "yarn_count_class"
    ]
)

yarn_count_text_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            sublinear_tf=True,
            min_df=2
        )
    ),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    )
])

yarn_count_text_model.fit(
    train_yarn["retrieval_text"],
    train_yarn["yarn_count_class"]
)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(analyzer='char_wb', min_df=2,
                                 ngram_range=(3, 5), sublinear_tf=True)),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=2000,
                                    random_state=42))])

In [391]:
yarn_predictions = (
    yarn_count_text_model.predict(
        test_yarn["retrieval_text"]
    )
)

yarn_probabilities = (
    yarn_count_text_model.predict_proba(
        test_yarn["retrieval_text"]
    )
)

yarn_classes = (
    yarn_count_text_model
    .named_steps["model"]
    .classes_
)

print(
    "Top-1 yarn-count accuracy:",
    f"{accuracy_score(test_yarn['yarn_count_class'], yarn_predictions):.2%}"
)

top3_yarn_accuracy = top_k_accuracy_score(
    test_yarn["yarn_count_class"],
    yarn_probabilities,
    k=min(3, len(yarn_classes)),
    labels=yarn_classes
)

print(
    "Top-3 yarn-count accuracy:",
    f"{top3_yarn_accuracy:.2%}"
)

Top-1 yarn-count accuracy: 54.69%
Top-3 yarn-count accuracy: 89.93%


In [392]:
print(
    "Materials used for evaluation:",
    len(yarn_count_model_data)
)

print(
    "Materials excluded due to rare class:",
    len(yarn_count_prediction_data)
    - len(yarn_count_model_data)
)

Materials used for evaluation: 2879
Materials excluded due to rare class: 29


In [393]:
# Retrain the yarn-count classifier using all available labeled data
final_yarn_count_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            sublinear_tf=True,
            min_df=2
        )
    ),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    )
])

final_yarn_count_model.fit(
    yarn_count_model_data["retrieval_text"],
    yarn_count_model_data["yarn_count_class"]
)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(analyzer='char_wb', min_df=2,
                                 ngram_range=(3, 5), sublinear_tf=True)),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=2000,
                                    random_state=42))])

In [394]:
# Predict Top-3 yarn counts for no-attribute knitted fabrics
sap_no_attr_orme = (
    sap_no_attr_dev[
        sap_no_attr_dev["material_group_code"] == "1020001"
    ]
    .copy()
)

yarn_probabilities = (
    final_yarn_count_model.predict_proba(
        sap_no_attr_orme["retrieval_text"]
    )
)

yarn_classes = (
    final_yarn_count_model
    .named_steps["model"]
    .classes_
)

top_indices = np.argsort(
    yarn_probabilities,
    axis=1
)[:, -3:][:, ::-1]

predicted_yarn_records = []

for row_idx, material_id in enumerate(
    sap_no_attr_orme["material_id"]
):
    for rank, class_idx in enumerate(
        top_indices[row_idx],
        start=1
    ):
        predicted_yarn_records.append({
            "material_id": material_id,
            "predicted_yarn_count":
                yarn_classes[class_idx],
            "yarn_rank": rank,
            "yarn_probability":
                yarn_probabilities[row_idx, class_idx]
        })

predicted_yarn_counts = pd.DataFrame(
    predicted_yarn_records
)

display(predicted_yarn_counts.head(10))

,material_id,predicted_yarn_count,yarn_rank,yarn_probability
0,1020000013,30,1,0.095914
1,1020000013,70,2,0.064229
2,1020000013,26,3,0.057424
3,1020000015,150,1,0.199862
4,1020000015,100,2,0.177146
5,1020000015,70,3,0.094513
6,1020000018,24,1,0.474012
7,1020000018,30,2,0.046541
8,1020000018,26,3,0.034842
9,1020000024,30,1,0.132137


In [395]:
# Prepare canonical PLM knitting yarn counts
plm_yarn_count_retrieval = (
    plm_yarn_count_knit[
        ["plm_code", "plm_yarn_count"]
    ]
    .copy()
)

plm_yarn_count_retrieval["yarn_count_class"] = (
    plm_yarn_count_retrieval["plm_yarn_count"]
    .apply(canonicalize_yarn_count)
)

plm_yarn_count_retrieval = (
    plm_yarn_count_retrieval
    .dropna(subset=["yarn_count_class"])
    .drop_duplicates("plm_code")
)

In [396]:
# Build PLM pool indexed by structure and yarn count
plm_orme_structure_yarn = (
    plm_candidates_enriched[
        [
            "plm_code",
            "material_group_code",
            "structured_text"
        ]
    ]
    .merge(
        plm_structure_candidates[
            ["plm_code", "structure"]
        ],
        on="plm_code",
        how="inner"
    )
    .merge(
        plm_yarn_count_retrieval[
            ["plm_code", "yarn_count_class"]
        ],
        on="plm_code",
        how="inner"
    )
)

plm_orme_structure_yarn = (
    plm_orme_structure_yarn[
        plm_orme_structure_yarn[
            "material_group_code"
        ] == "1020001"
    ]
    .copy()
)

In [397]:
# Combine predicted structures and yarn counts
orme_structure_predictions = (
    predicted_structures[
        predicted_structures["material_group_code"]
        == "1020001"
    ][
        [
            "material_id",
            "predicted_structure",
            "structure_rank",
            "structure_probability"
        ]
    ]
)

prediction_combinations = (
    orme_structure_predictions
    .merge(
        predicted_yarn_counts,
        on="material_id",
        how="inner"
    )
)

print(
    "Prediction combinations:",
    len(prediction_combinations)
)

Prediction combinations: 7920


In [398]:
def generate_structure_yarn_candidates(
    sap_df,
    prediction_df,
    plm_df,
    top_k_per_combination=20
):
    records = []

    sap_lookup = (
        sap_df
        .set_index("material_id")["retrieval_text"]
        .to_dict()
    )

    for (
        structure_value,
        yarn_value
    ), predictions in prediction_df.groupby(
        [
            "predicted_structure",
            "predicted_yarn_count"
        ]
    ):

        plm_pool = (
            plm_df[
                (plm_df["structure"] == structure_value)
                & (
                    plm_df["yarn_count_class"]
                    == yarn_value
                )
            ]
            .reset_index(drop=True)
        )

        if plm_pool.empty:
            continue

        vectorizer = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            sublinear_tf=True,
            norm="l2"
        )

        plm_matrix = vectorizer.fit_transform(
            plm_pool["structured_text"]
        )

        n_neighbors = min(
            top_k_per_combination,
            len(plm_pool)
        )

        nn_model = NearestNeighbors(
            n_neighbors=n_neighbors,
            metric="cosine",
            algorithm="brute"
        )

        nn_model.fit(plm_matrix)

        material_ids = [
            material_id
            for material_id in predictions["material_id"]
            if material_id in sap_lookup
        ]

        if not material_ids:
            continue

        sap_texts = [
            sap_lookup[material_id]
            for material_id in material_ids
        ]

        sap_matrix = vectorizer.transform(
            sap_texts
        )

        distances, indices = nn_model.kneighbors(
            sap_matrix
        )

        for row_idx, material_id in enumerate(
            material_ids
        ):
            prediction_row = (
                predictions[
                    predictions["material_id"]
                    == material_id
                ]
                .iloc[0]
            )

            for retrieval_rank, (
                plm_idx,
                distance
            ) in enumerate(
                zip(
                    indices[row_idx],
                    distances[row_idx]
                ),
                start=1
            ):
                records.append({
                    "material_id": material_id,
                    "candidate_plm_code":
                        plm_pool.loc[
                            plm_idx,
                            "plm_code"
                        ],
                    "predicted_structure":
                        structure_value,
                    "predicted_yarn_count":
                        yarn_value,
                    "structure_rank":
                        prediction_row[
                            "structure_rank"
                        ],
                    "yarn_rank":
                        prediction_row[
                            "yarn_rank"
                        ],
                    "retrieval_rank":
                        retrieval_rank,
                    "retrieval_similarity":
                        1 - distance
                })

    return pd.DataFrame(records)

In [399]:
pseudo_structure_yarn_candidates = (
    generate_structure_yarn_candidates(
        sap_no_attr_orme,
        prediction_combinations,
        plm_orme_structure_yarn,
        top_k_per_combination=20
    )
)

pseudo_structure_yarn_pairs = (
    pseudo_structure_yarn_candidates[
        [
            "material_id",
            "candidate_plm_code"
        ]
    ]
    .drop_duplicates()
)

print(
    "Structure + yarn candidate pairs:",
    len(pseudo_structure_yarn_pairs)
)

print(
    "Average candidates per covered SAP:",
    len(pseudo_structure_yarn_pairs)
    / pseudo_structure_yarn_pairs[
        "material_id"
    ].nunique()
)

Structure + yarn candidate pairs: 80057
Average candidates per covered SAP: 91.0773606370876


In [400]:
orme_dev_truth = (
    no_attr_dev_truth[
        no_attr_dev_truth["material_id"]
        .isin(set(sap_no_attr_orme["material_id"]))
    ]
)

orme_structure_yarn_recall = (
    orme_dev_truth
    .merge(
        pseudo_structure_yarn_pairs,
        left_on=[
            "material_id",
            "plm_code"
        ],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left"
    )
)

print(
    "Structure + yarn ORME recall:",
    f"{orme_structure_yarn_recall['candidate_plm_code'].notna().mean():.2%}"
)

Structure + yarn ORME recall: 16.02%


In [401]:
# Add structure+yarn channel to the existing no-attribute candidate pool
no_attr_candidates_v2 = (
    pd.concat(
        [
            no_attr_combined_candidates,
            pseudo_structure_yarn_pairs
        ],
        ignore_index=True
    )
    .drop_duplicates()
)

no_attr_recall_v2 = (
    no_attr_dev_truth
    .merge(
        no_attr_candidates_v2,
        left_on=[
            "material_id",
            "plm_code"
        ],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left"
    )
)

no_attr_recall_v2["retrieved"] = (
    no_attr_recall_v2[
        "candidate_plm_code"
    ].notna()
)

print(
    "Overall no-attribute recall v2:",
    f"{no_attr_recall_v2['retrieved'].mean():.2%}"
)

Overall no-attribute recall v2: 52.45%


In [402]:
display(
    no_attr_recall_v2
    .merge(
        no_attribute_materials[
            [
                "material_id",
                "material_group_code"
            ]
        ],
        on="material_id",
        how="left"
    )
    .groupby("material_group_code")
    .agg(
        known_materials=("material_id", "size"),
        candidate_recall=("retrieved", "mean")
    )
)

,known_materials,candidate_recall
material_group_code,,
1020001,880,0.298864
1020002,808,0.768564
1030004,9,0.666667


In [403]:
# Audit yarn-count availability on true PLM codes
orme_truth_yarn_audit = (
    orme_dev_truth
    .merge(
        plm_yarn_count_retrieval[
            [
                "plm_code",
                "yarn_count_class"
            ]
        ],
        on="plm_code",
        how="left"
    )
)

print(
    "ORME development materials:",
    len(orme_truth_yarn_audit)
)

print(
    "True PLM with yarn count:",
    f"{orme_truth_yarn_audit['yarn_count_class'].notna().mean():.2%}"
)

ORME development materials: 880
True PLM with yarn count: 96.48%


In [404]:
# Check whether predicted Top-3 yarn counts contain the true PLM yarn count
predicted_yarn_sets = (
    predicted_yarn_counts
    .groupby("material_id")[
        "predicted_yarn_count"
    ]
    .apply(set)
    .rename("predicted_yarn_set")
    .reset_index()
)

orme_yarn_prediction_audit = (
    orme_truth_yarn_audit
    .merge(
        predicted_yarn_sets,
        on="material_id",
        how="left"
    )
)

orme_yarn_prediction_audit[
    "true_yarn_in_top3"
] = (
    orme_yarn_prediction_audit.apply(
        lambda row:
            (
                row["yarn_count_class"]
                in row["predicted_yarn_set"]
            )
            if (
                pd.notna(row["yarn_count_class"])
                and isinstance(
                    row["predicted_yarn_set"],
                    set
                )
            )
            else np.nan,
        axis=1
    )
)

valid_yarn_audit = (
    orme_yarn_prediction_audit[
        orme_yarn_prediction_audit[
            "true_yarn_in_top3"
        ].notna()
    ]
)

print(
    "Comparable materials:",
    len(valid_yarn_audit)
)

print(
    "Predicted yarn Top-3 agreement with true PLM:",
    f"{valid_yarn_audit['true_yarn_in_top3'].mean():.2%}"
)

Comparable materials: 849
Predicted yarn Top-3 agreement with true PLM: 92.46%


In [405]:
# Get true PLM fabric structure
orme_truth_structure_audit = (
    orme_dev_truth
    .merge(
        plm_structure_candidates[
            [
                "plm_code",
                "structure"
            ]
        ].rename(
            columns={
                "structure":
                "true_plm_structure"
            }
        ),
        on="plm_code",
        how="left"
    )
)

predicted_structure_sets = (
    predicted_structures[
        predicted_structures[
            "material_group_code"
        ] == "1020001"
    ]
    .groupby("material_id")[
        "predicted_structure"
    ]
    .apply(set)
    .rename("predicted_structure_set")
    .reset_index()
)

orme_structure_prediction_audit = (
    orme_truth_structure_audit
    .merge(
        predicted_structure_sets,
        on="material_id",
        how="left"
    )
)

orme_structure_prediction_audit[
    "true_structure_in_top3"
] = (
    orme_structure_prediction_audit.apply(
        lambda row:
            (
                row["true_plm_structure"]
                in row["predicted_structure_set"]
            )
            if (
                pd.notna(row["true_plm_structure"])
                and isinstance(
                    row["predicted_structure_set"],
                    set
                )
            )
            else np.nan,
        axis=1
    )
)

valid_structure_audit = (
    orme_structure_prediction_audit[
        orme_structure_prediction_audit[
            "true_structure_in_top3"
        ].notna()
    ]
)

print(
    "Predicted structure Top-3 agreement with true PLM:",
    f"{valid_structure_audit['true_structure_in_top3'].mean():.2%}"
)

Predicted structure Top-3 agreement with true PLM: 95.68%


In [406]:
# Audit whether both true PLM structure and yarn count
# are contained in the predicted Top-3 sets
orme_joint_audit = (
    orme_truth_yarn_audit[
        [
            "material_id",
            "plm_code",
            "yarn_count_class"
        ]
    ]
    .merge(
        orme_truth_structure_audit[
            [
                "material_id",
                "true_plm_structure"
            ]
        ],
        on="material_id",
        how="left"
    )
    .merge(
        predicted_yarn_sets,
        on="material_id",
        how="left"
    )
    .merge(
        predicted_structure_sets,
        on="material_id",
        how="left"
    )
)

orme_joint_audit["true_yarn_in_top3"] = (
    orme_joint_audit.apply(
        lambda row:
            row["yarn_count_class"] in row["predicted_yarn_set"]
            if (
                pd.notna(row["yarn_count_class"])
                and isinstance(row["predicted_yarn_set"], set)
            )
            else np.nan,
        axis=1
    )
)

orme_joint_audit["true_structure_in_top3"] = (
    orme_joint_audit.apply(
        lambda row:
            row["true_plm_structure"] in row["predicted_structure_set"]
            if (
                pd.notna(row["true_plm_structure"])
                and isinstance(row["predicted_structure_set"], set)
            )
            else np.nan,
        axis=1
    )
)

comparable_joint = orme_joint_audit[
    orme_joint_audit["true_yarn_in_top3"].notna()
    & orme_joint_audit["true_structure_in_top3"].notna()
].copy()

comparable_joint["both_in_top3"] = (
    comparable_joint["true_yarn_in_top3"]
    & comparable_joint["true_structure_in_top3"]
)

print(
    "Comparable ORME materials:",
    len(comparable_joint)
)

print(
    "Both true structure and yarn in predicted Top-3:",
    f"{comparable_joint['both_in_top3'].mean():.2%}"
)

Comparable ORME materials: 848
Both true structure and yarn in predicted Top-3: 88.33%


In [407]:
# Test candidate-pool recall before text ranking
true_plm_attributes = (
    orme_dev_truth
    .merge(
        plm_structure_candidates[
            ["plm_code", "structure"]
        ],
        on="plm_code",
        how="left"
    )
    .merge(
        plm_yarn_count_retrieval[
            ["plm_code", "yarn_count_class"]
        ],
        on="plm_code",
        how="left"
    )
    .merge(
        predicted_structure_sets,
        on="material_id",
        how="left"
    )
    .merge(
        predicted_yarn_sets,
        on="material_id",
        how="left"
    )
)

true_plm_attributes["eligible_by_pseudo_attributes"] = (
    true_plm_attributes.apply(
        lambda row:
            (
                row["structure"]
                in row["predicted_structure_set"]
            )
            and (
                row["yarn_count_class"]
                in row["predicted_yarn_set"]
            )
            if (
                pd.notna(row["structure"])
                and pd.notna(row["yarn_count_class"])
                and isinstance(
                    row["predicted_structure_set"],
                    set
                )
                and isinstance(
                    row["predicted_yarn_set"],
                    set
                )
            )
            else False,
        axis=1
    )
)

print(
    "Pseudo-attribute pool recall:",
    f"{true_plm_attributes['eligible_by_pseudo_attributes'].mean():.2%}"
)

Pseudo-attribute pool recall: 85.11%


In [408]:
# Build the full pseudo-attribute candidate pool
pseudo_attribute_candidates = (
    prediction_combinations[
        [
            "material_id",
            "predicted_structure",
            "structure_rank",
            "structure_probability",
            "predicted_yarn_count",
            "yarn_rank",
            "yarn_probability"
        ]
    ]
    .merge(
        plm_orme_structure_yarn[
            [
                "plm_code",
                "structure",
                "yarn_count_class"
            ]
        ],
        left_on=[
            "predicted_structure",
            "predicted_yarn_count"
        ],
        right_on=[
            "structure",
            "yarn_count_class"
        ],
        how="inner"
    )
    .rename(
        columns={
            "plm_code": "candidate_plm_code"
        }
    )
)

In [409]:
# Keep one row per SAP-PLM pair
pseudo_attribute_candidates_clean = (
    pseudo_attribute_candidates
    .groupby(
        [
            "material_id",
            "candidate_plm_code"
        ],
        as_index=False
    )
    .agg(
        structure_probability=(
            "structure_probability",
            "max"
        ),
        yarn_probability=(
            "yarn_probability",
            "max"
        ),
        best_structure_rank=(
            "structure_rank",
            "min"
        ),
        best_yarn_rank=(
            "yarn_rank",
            "min"
        )
    )
)

pseudo_attribute_candidates_clean[
    "pseudo_attribute_score"
] = (
    pseudo_attribute_candidates_clean[
        "structure_probability"
    ]
    * pseudo_attribute_candidates_clean[
        "yarn_probability"
    ]
)

In [410]:
print(
    "Pseudo-attribute candidate pairs:",
    len(pseudo_attribute_candidates_clean)
)

print(
    "Covered SAP materials:",
    pseudo_attribute_candidates_clean[
        "material_id"
    ].nunique()
)

print(
    "Average candidates per covered SAP:",
    len(pseudo_attribute_candidates_clean)
    / pseudo_attribute_candidates_clean[
        "material_id"
    ].nunique()
)

Pseudo-attribute candidate pairs: 1128656
Covered SAP materials: 879
Average candidates per covered SAP: 1284.0227531285552


In [411]:
# Combine text retrieval and pseudo-attribute candidate generation
no_attr_orme_candidate_union = (
    pd.concat(
        [
            multi_channel_union[
                multi_channel_union["material_id"]
                .isin(set(sap_no_attr_orme["material_id"]))
            ],
            pseudo_attribute_candidates_clean[
                [
                    "material_id",
                    "candidate_plm_code"
                ]
            ]
        ],
        ignore_index=True
    )
    .drop_duplicates()
)

In [412]:
orme_union_recall = (
    orme_dev_truth
    .merge(
        no_attr_orme_candidate_union,
        left_on=[
            "material_id",
            "plm_code"
        ],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left"
    )
)

orme_union_recall["retrieved"] = (
    orme_union_recall[
        "candidate_plm_code"
    ].notna()
)

print(
    "ORME candidate union recall:",
    f"{orme_union_recall['retrieved'].mean():.2%}"
)

print(
    "Average union candidates per SAP:",
    len(no_attr_orme_candidate_union)
    / no_attr_orme_candidate_union[
        "material_id"
    ].nunique()
)

ORME candidate union recall: 88.64%
Average union candidates per SAP: 1416.6875


In [413]:
# Final no-attribute candidate union for development
no_attr_candidate_union_v3 = (
    pd.concat(
        [
            no_attr_combined_candidates,
            pseudo_attribute_candidates_clean[
                ["material_id", "candidate_plm_code"]
            ]
        ],
        ignore_index=True
    )
    .drop_duplicates()
)

print(
    "Final no-attribute candidate pairs:",
    len(no_attr_candidate_union_v3)
)

print(
    "Average candidates per SAP:",
    len(no_attr_candidate_union_v3)
    / no_attr_candidate_union_v3["material_id"].nunique()
)

Final no-attribute candidate pairs: 1416078
Average candidates per SAP: 834.4596346493813


In [414]:
# Aggregate text retrieval features
text_pair_features = (
    multi_channel_candidates
    .groupby(
        [
            "material_id",
            "candidate_plm_code",
            "channel"
        ],
        as_index=False
    )
    .agg(
        similarity=("channel_similarity", "max"),
        rank=("channel_rank", "min")
    )
)

text_similarity_wide = (
    text_pair_features
    .pivot(
        index=["material_id", "candidate_plm_code"],
        columns="channel",
        values="similarity"
    )
    .add_suffix("_similarity")
    .reset_index()
)

text_rank_wide = (
    text_pair_features
    .pivot(
        index=["material_id", "candidate_plm_code"],
        columns="channel",
        values="rank"
    )
    .add_suffix("_rank")
    .reset_index()
)

text_channel_count = (
    text_pair_features
    .groupby(
        ["material_id", "candidate_plm_code"]
    )["channel"]
    .nunique()
    .rename("text_channel_count")
    .reset_index()
)

In [415]:
# Aggregate pseudo-structure retrieval features
pseudo_structure_features = (
    pseudo_structure_candidates
    .groupby(
        ["material_id", "candidate_plm_code"],
        as_index=False
    )
    .agg(
        structure_probability=(
            "structure_probability",
            "max"
        ),
        best_structure_rank=(
            "structure_rank",
            "min"
        ),
        structure_retrieval_similarity=(
            "structure_retrieval_similarity",
            "max"
        ),
        structure_retrieval_rank=(
            "structure_retrieval_rank",
            "min"
        )
    )
)

pseudo_structure_features[
    "from_pseudo_structure"
] = 1

In [416]:
pseudo_attribute_features = (
    pseudo_attribute_candidates_clean.copy()
)

pseudo_attribute_features[
    "from_structure_yarn"
] = 1

In [417]:
# Build one-row-per-SAP-PLM candidate master
no_attr_candidate_master = (
    no_attr_candidate_union_v3
    .merge(
        text_similarity_wide,
        on=["material_id", "candidate_plm_code"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        text_rank_wide,
        on=["material_id", "candidate_plm_code"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        text_channel_count,
        on=["material_id", "candidate_plm_code"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        pseudo_structure_features,
        on=["material_id", "candidate_plm_code"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        pseudo_attribute_features,
        on=["material_id", "candidate_plm_code"],
        how="left",
        validate="one_to_one"
    )
)

for column in [
    "text_channel_count",
    "from_pseudo_structure",
    "from_structure_yarn"
]:
    no_attr_candidate_master[column] = (
        no_attr_candidate_master[column]
        .fillna(0)
    )

assert not no_attr_candidate_master.duplicated(
    ["material_id", "candidate_plm_code"]
).any()

In [418]:
# Add material group
no_attr_candidate_master = (
    no_attr_candidate_master
    .merge(
        material_group_lookup,
        on="material_id",
        how="left",
        validate="many_to_one"
    )
)

# Add known target PLM for development materials only
no_attr_dev_truth_labeled = (
    no_attr_dev_truth
    .rename(
        columns={
            "plm_code": "true_plm_code"
        }
    )
)

no_attr_dev_pairs = (
    no_attr_candidate_master[
        no_attr_candidate_master["material_id"]
        .isin(no_attr_dev_ids)
    ]
    .merge(
        no_attr_dev_truth_labeled,
        on="material_id",
        how="inner",
        validate="many_to_one"
    )
)

no_attr_dev_pairs["is_match"] = (
    no_attr_dev_pairs["candidate_plm_code"]
    == no_attr_dev_pairs["true_plm_code"]
).astype(int)

In [419]:
print(
    "Development SAP materials:",
    no_attr_dev_pairs["material_id"].nunique()
)

print(
    "Candidate pairs:",
    len(no_attr_dev_pairs)
)

print(
    "Positive pairs retrieved:",
    no_attr_dev_pairs["is_match"].sum()
)

candidate_recall = (
    no_attr_dev_pairs
    .groupby("material_id")["is_match"]
    .max()
    .mean()
)

print(
    "Candidate recall:",
    f"{candidate_recall:.2%}"
)

Development SAP materials: 1697
Candidate pairs: 1416078
Positive pairs retrieved: 1412
Candidate recall: 83.21%


In [420]:
candidate_feature_columns = [
    column
    for column in no_attr_dev_pairs.columns
    if (
        column.endswith("_similarity")
        or column.endswith("_rank")
        or column in [
            "text_channel_count",
            "structure_probability",
            "yarn_probability",
            "pseudo_attribute_score",
            "from_pseudo_structure",
            "from_structure_yarn"
        ]
    )
]

print(candidate_feature_columns)

['enriched_text_similarity', 'multi_value_text_similarity', 'original_text_similarity', 'structured_text_similarity', 'enriched_text_rank', 'multi_value_text_rank', 'original_text_rank', 'structured_text_rank', 'text_channel_count', 'structure_retrieval_similarity', 'structure_retrieval_rank', 'from_pseudo_structure', 'yarn_probability', 'best_yarn_rank', 'pseudo_attribute_score', 'from_structure_yarn']


In [421]:
# Inspect duplicated pseudo-structure feature names
print([
    column
    for column in no_attr_dev_pairs.columns
    if "structure_probability" in column
    or "best_structure_rank" in column
])

['structure_probability_x', 'best_structure_rank_x', 'structure_probability_y', 'best_structure_rank_y']


In [422]:
# Consolidate structure probability from multiple candidate channels
structure_probability_columns = [
    column
    for column in no_attr_dev_pairs.columns
    if column.startswith("structure_probability")
]

structure_rank_columns = [
    column
    for column in no_attr_dev_pairs.columns
    if column.startswith("best_structure_rank")
]

no_attr_dev_pairs["structure_probability"] = (
    no_attr_dev_pairs[
        structure_probability_columns
    ]
    .max(axis=1)
)

no_attr_dev_pairs["best_structure_rank"] = (
    no_attr_dev_pairs[
        structure_rank_columns
    ]
    .min(axis=1)
)

In [423]:
# Add material-group indicators
for group_code in [
    "1020001",
    "1020002",
    "1030004"
]:
    no_attr_dev_pairs[
        f"group_{group_code}"
    ] = (
        no_attr_dev_pairs[
            "material_group_code"
        ] == group_code
    ).astype(int)

In [424]:
no_attr_features = [
    "structured_text_similarity",
    "enriched_text_similarity",
    "multi_value_text_similarity",
    "original_text_similarity",

    "structure_retrieval_similarity",
    "structure_probability",
    "best_structure_rank",

    "yarn_probability",
    "best_yarn_rank",
    "pseudo_attribute_score",

    "text_channel_count",
    "from_pseudo_structure",
    "from_structure_yarn",

    "group_1020001",
    "group_1020002",
    "group_1030004"
]

In [425]:
# Create material-level development split
no_attr_dev_material_table = (
    no_attr_development[
        [
            "material_id",
            "material_group_code"
        ]
    ]
    .copy()
)

no_attr_train_materials, no_attr_validation_materials = (
    train_test_split(
        no_attr_dev_material_table,
        test_size=0.20,
        random_state=777,
        stratify=no_attr_dev_material_table[
            "material_group_code"
        ]
    )
)

no_attr_train_ids = set(
    no_attr_train_materials["material_id"]
)

no_attr_validation_ids = set(
    no_attr_validation_materials["material_id"]
)

print(
    "Train materials:",
    len(no_attr_train_ids)
)

print(
    "Validation materials:",
    len(no_attr_validation_ids)
)

Train materials: 1357
Validation materials: 340


In [426]:
# Identify train materials with a retrieved positive candidate
train_pair_pool = (
    no_attr_dev_pairs[
        no_attr_dev_pairs["material_id"]
        .isin(no_attr_train_ids)
    ]
    .copy()
)

retrieved_train_ids = set(
    train_pair_pool
    .groupby("material_id")["is_match"]
    .max()
    .loc[lambda x: x == 1]
    .index
)

train_pair_pool = (
    train_pair_pool[
        train_pair_pool["material_id"]
        .isin(retrieved_train_ids)
    ]
    .copy()
)

print(
    "Train materials with true PLM retrieved:",
    len(retrieved_train_ids)
)

Train materials with true PLM retrieved: 1126


In [427]:
# Score used only for hard-negative selection
train_pair_pool["hardness_score"] = (
    train_pair_pool[
        "structured_text_similarity"
    ].fillna(0)

    + train_pair_pool[
        "enriched_text_similarity"
    ].fillna(0)

    + train_pair_pool[
        "structure_retrieval_similarity"
    ].fillna(0)

    + train_pair_pool[
        "pseudo_attribute_score"
    ].fillna(0)

    + 0.25 * train_pair_pool[
        "multi_value_text_similarity"
    ].fillna(0)
)

positive_pairs = (
    train_pair_pool[
        train_pair_pool["is_match"] == 1
    ]
    .copy()
)

negative_pairs = (
    train_pair_pool[
        train_pair_pool["is_match"] == 0
    ]
    .copy()
)

hard_negatives = (
    negative_pairs
    .sort_values(
        ["material_id", "hardness_score"],
        ascending=[True, False]
    )
    .groupby("material_id")
    .head(30)
)

rng = np.random.default_rng(42)

negative_pairs["random_score"] = (
    rng.random(len(negative_pairs))
)

random_negatives = (
    negative_pairs
    .sort_values(
        ["material_id", "random_score"]
    )
    .groupby("material_id")
    .head(20)
)

no_attr_training_sample = (
    pd.concat(
        [
            positive_pairs,
            hard_negatives,
            random_negatives
        ],
        ignore_index=True
    )
    .drop_duplicates(
        ["material_id", "candidate_plm_code"]
    )
)

print(
    "Training rows:",
    len(no_attr_training_sample)
)

print(
    "Training positives:",
    no_attr_training_sample[
        "is_match"
    ].sum()
)

Training rows: 55509
Training positives: 1126


In [428]:
# Train first no-attribute reranker
no_attr_rf_model = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value=-1,
            add_indicator=True
        )
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=300,
            max_depth=12,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=42
        )
    )
])

no_attr_rf_model.fit(
    no_attr_training_sample[
        no_attr_features
    ],
    no_attr_training_sample[
        "is_match"
    ]
)

Pipeline(steps=[('imputer',
                 SimpleImputer(add_indicator=True, fill_value=-1,
                               strategy='constant')),
                ('model',
                 RandomForestClassifier(class_weight='balanced_subsample',
                                        max_depth=12, min_samples_leaf=2,
                                        n_estimators=300, n_jobs=-1,
                                        random_state=42))])

In [429]:
# Score the full validation candidate pool
no_attr_validation_pairs = (
    no_attr_dev_pairs[
        no_attr_dev_pairs["material_id"]
        .isin(no_attr_validation_ids)
    ]
    .copy()
)

no_attr_validation_pairs["score"] = (
    no_attr_rf_model.predict_proba(
        no_attr_validation_pairs[
            no_attr_features
        ]
    )[:, 1]
)

no_attr_validation_pairs["rank"] = (
    no_attr_validation_pairs
    .groupby("material_id")["score"]
    .rank(
        ascending=False,
        method="first"
    )
)

In [430]:
validation_true_ranks = (
    no_attr_validation_pairs[
        no_attr_validation_pairs["is_match"] == 1
    ][
        [
            "material_id",
            "material_group_code",
            "rank"
        ]
    ]
    .copy()
)

print(
    "Conditional Top-1:",
    f"{(validation_true_ranks['rank'] <= 1).mean():.2%}"
)

print(
    "Conditional Top-3:",
    f"{(validation_true_ranks['rank'] <= 3).mean():.2%}"
)

print(
    "Conditional Top-5:",
    f"{(validation_true_ranks['rank'] <= 5).mean():.2%}"
)

print(
    "Conditional Top-10:",
    f"{(validation_true_ranks['rank'] <= 10).mean():.2%}"
)

print(
    "Conditional MRR:",
    f"{(1 / validation_true_ranks['rank']).mean():.4f}"
)

Conditional Top-1: 17.13%
Conditional Top-3: 25.87%
Conditional Top-5: 28.32%
Conditional Top-10: 38.11%
Conditional MRR: 0.2417


In [431]:
# End-to-end validation evaluation
validation_rank_table = (
    no_attr_validation_materials[
        [
            "material_id",
            "material_group_code"
        ]
    ]
    .merge(
        validation_true_ranks[
            [
                "material_id",
                "rank"
            ]
        ],
        on="material_id",
        how="left"
    )
)

print(
    "Validation candidate recall:",
    f"{validation_rank_table['rank'].notna().mean():.2%}"
)

for k in [1, 3, 5, 10]:
    success = (
        validation_rank_table["rank"]
        .le(k)
        .fillna(False)
        .mean()
    )

    print(
        f"End-to-end Top-{k}:",
        f"{success:.2%}"
    )

Validation candidate recall: 84.12%
End-to-end Top-1: 14.41%
End-to-end Top-3: 21.76%
End-to-end Top-5: 23.82%
End-to-end Top-10: 32.06%


In [432]:
# Evaluate no-attribute reranker by material group
group_results = []

for group_code, group in validation_rank_table.groupby(
    "material_group_code"
):
    retrieved = group[group["rank"].notna()]

    group_results.append({
        "material_group_code": group_code,
        "materials": len(group),

        "candidate_recall":
            group["rank"].notna().mean(),

        "conditional_top1":
            (retrieved["rank"] <= 1).mean(),

        "conditional_top3":
            (retrieved["rank"] <= 3).mean(),

        "conditional_top5":
            (retrieved["rank"] <= 5).mean(),

        "end_to_end_top5":
            group["rank"]
            .le(5)
            .fillna(False)
            .mean()
    })

display(pd.DataFrame(group_results))

,material_group_code,materials,candidate_recall,conditional_top1,conditional_top3,conditional_top5,end_to_end_top5
0,1020001,176,0.880682,0.032258,0.070968,0.070968,0.062500
1,1020002,162,0.796296,0.325581,0.472868,0.527132,0.419753
2,1030004,2,1.000000,1.000000,1.000000,1.000000,1.000000


In [433]:
# Inspect feature importance
imputer = no_attr_rf_model.named_steps["imputer"]
rf_model = no_attr_rf_model.named_steps["model"]

feature_names = imputer.get_feature_names_out(
    no_attr_features
)

feature_importance = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": rf_model.feature_importances_
    })
    .sort_values(
        "importance",
        ascending=False
    )
)

display(feature_importance.head(25))

,feature,importance
5,structure_probability,0.174051
4,structure_retrieval_similarity,0.113364
0,structured_text_similarity,0.085491
1,enriched_text_similarity,0.084633
9,pseudo_attribute_score,0.078991
6,best_structure_rank,0.070818
7,yarn_probability,0.060206
2,multi_value_text_similarity,0.058643
10,text_channel_count,0.038985
17,missingindicator_enriched_text_similarity,0.027684


In [434]:
# Build a canonical fiber lexicon
fiber_alias_mapping = {
    "PAMUK": "COTTON",
    "COTTON": "COTTON",

    "PES": "POLYESTER",
    "POLYESTER": "POLYESTER",
    "POLIESTER": "POLYESTER",

    "VIS": "VISCOSE",
    "VISCOSE": "VISCOSE",

    "EA": "ELASTANE",
    "ELASTAN": "ELASTANE",
    "ELASTANE": "ELASTANE",
    "ELASTHANE": "ELASTANE",
    "SPANDEX": "ELASTANE",

    "PA": "POLYAMIDE",
    "POLYAMIDE": "POLYAMIDE",
    "POLYAMIDE6": "POLYAMIDE",

    "LYOCELL": "LYOCELL",
    "TENCEL": "LYOCELL",

    "ACRYLIC": "ACRYLIC",
    "AKRILIK": "ACRYLIC",

    "LINEN": "LINEN",
    "KETEN": "LINEN",

    "WOOL": "WOOL",
    "YUN": "WOOL",
    "YÜN": "WOOL"
}

In [435]:
import re

def extract_fiber_names_from_text(text):
    if pd.isna(text):
        return set()

    text = str(text).upper()

    found = set()

    for alias, canonical in fiber_alias_mapping.items():

        # Short abbreviations need token boundaries
        pattern = rf"(?<![A-ZÇĞİÖŞÜ]){re.escape(alias)}(?![A-ZÇĞİÖŞÜ])"

        if re.search(pattern, text):
            found.add(canonical)

    return found

In [436]:
# Compare description-derived fibers with actual SAP fiber attributes
fiber_extraction_audit = (
    sap_candidates_source[
        [
            "material_id",
            "material_group_code",
            "retrieval_text"
        ]
    ]
    .merge(
        sap_fiber_sets,
        on="material_id",
        how="inner"
    )
)

fiber_extraction_audit[
    "predicted_fiber_set"
] = (
    fiber_extraction_audit[
        "retrieval_text"
    ]
    .apply(extract_fiber_names_from_text)
)

fiber_extraction_audit[
    "actual_fiber_set"
] = (
    fiber_extraction_audit[
        "fiber_key"
    ]
    .apply(set)
)

fiber_extraction_audit[
    "fiber_set_match"
] = (
    fiber_extraction_audit[
        "predicted_fiber_set"
    ]
    == fiber_extraction_audit[
        "actual_fiber_set"
    ]
)

fiber_extraction_audit[
    "fiber_jaccard"
] = [
    (
        len(pred & actual)
        / len(pred | actual)
        if pred and actual
        else np.nan
    )
    for pred, actual in zip(
        fiber_extraction_audit[
            "predicted_fiber_set"
        ],
        fiber_extraction_audit[
            "actual_fiber_set"
        ]
    )
]

In [437]:
print(
    "Materials:",
    len(fiber_extraction_audit)
)

print(
    "Description with at least one extracted fiber:",
    f"{fiber_extraction_audit['predicted_fiber_set'].apply(bool).mean():.2%}"
)

print(
    "Exact fiber-set match:",
    f"{fiber_extraction_audit['fiber_set_match'].mean():.2%}"
)

print(
    "Mean fiber Jaccard:",
    f"{fiber_extraction_audit['fiber_jaccard'].mean():.3f}"
)

Materials: 8634
Description with at least one extracted fiber: 63.99%
Exact fiber-set match: 41.13%
Mean fiber Jaccard: 0.779


In [438]:
display(
    fiber_extraction_audit
    .groupby("material_group_code")
    .agg(
        materials=("material_id", "size"),
        extraction_coverage=(
            "predicted_fiber_set",
            lambda x: x.apply(bool).mean()
        ),
        exact_match=(
            "fiber_set_match",
            "mean"
        ),
        mean_jaccard=(
            "fiber_jaccard",
            "mean"
        )
    )
)

,materials,extraction_coverage,exact_match,mean_jaccard
material_group_code,,,,
1020001,4568,0.617119,0.329466,0.692769
1020002,2993,0.561978,0.386235,0.826387
1030004,1073,0.954334,0.829450,0.937598


In [439]:
# Inspect validation candidate-pool size by material group
validation_candidate_stats = (
    no_attr_dev_pairs[
        no_attr_dev_pairs["material_id"]
        .isin(no_attr_validation_ids)
    ]
    .groupby(
        ["material_id", "material_group_code"],
        as_index=False
    )
    .agg(
        candidate_count=("candidate_plm_code", "size"),
        true_retrieved=("is_match", "max")
    )
)

candidate_pool_summary = (
    validation_candidate_stats
    .groupby("material_group_code")
    .agg(
        materials=("material_id", "size"),
        candidate_recall=("true_retrieved", "mean"),
        mean_candidates=("candidate_count", "mean"),
        median_candidates=("candidate_count", "median"),
        p90_candidates=(
            "candidate_count",
            lambda x: x.quantile(0.90)
        )
    )
)

display(candidate_pool_summary)

,materials,candidate_recall,mean_candidates,median_candidates,p90_candidates
material_group_code,,,,,
1020001,176,0.880682,1481.784091,1712.0,2375.5
1020002,162,0.796296,186.796296,183.0,210.0
1030004,2,1.000000,196.000000,196.0,207.2


In [440]:
group_specific_features = [
    feature
    for feature in no_attr_features
    if not feature.startswith("group_")
]

print(group_specific_features)

['structured_text_similarity', 'enriched_text_similarity', 'multi_value_text_similarity', 'original_text_similarity', 'structure_retrieval_similarity', 'structure_probability', 'best_structure_rank', 'yarn_probability', 'best_yarn_rank', 'pseudo_attribute_score', 'text_channel_count', 'from_pseudo_structure', 'from_structure_yarn']


In [441]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


def build_no_attr_lr():
    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=-1,
                add_indicator=True
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    ])


def build_no_attr_rf():
    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=-1,
                add_indicator=True
            )
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=12,
                min_samples_leaf=2,
                class_weight="balanced_subsample",
                n_jobs=-1,
                random_state=42
            )
        )
    ])

In [442]:
def evaluate_group_reranker(
    group_code,
    model,
    model_name,
    features
):
    train_group = (
        no_attr_training_sample[
            (
                no_attr_training_sample["material_id"]
                .isin(no_attr_train_ids)
            )
            & (
                no_attr_training_sample[
                    "material_group_code"
                ] == group_code
            )
        ]
        .copy()
    )

    validation_group = (
        no_attr_dev_pairs[
            (
                no_attr_dev_pairs["material_id"]
                .isin(no_attr_validation_ids)
            )
            & (
                no_attr_dev_pairs[
                    "material_group_code"
                ] == group_code
            )
        ]
        .copy()
    )

    validation_materials_group = (
        no_attr_validation_materials[
            no_attr_validation_materials[
                "material_group_code"
            ] == group_code
        ][["material_id"]]
        .copy()
    )

    model.fit(
        train_group[features],
        train_group["is_match"]
    )

    validation_group["score"] = (
        model.predict_proba(
            validation_group[features]
        )[:, 1]
    )

    validation_group["rank"] = (
        validation_group
        .groupby("material_id")["score"]
        .rank(
            ascending=False,
            method="first"
        )
    )

    true_ranks = (
        validation_group[
            validation_group["is_match"] == 1
        ][
            ["material_id", "rank"]
        ]
        .copy()
    )

    evaluation_table = (
        validation_materials_group
        .merge(
            true_ranks,
            on="material_id",
            how="left",
            validate="one_to_one"
        )
    )

    retrieved = (
        evaluation_table[
            evaluation_table["rank"].notna()
        ]
    )

    result = {
        "material_group_code": group_code,
        "model": model_name,

        "validation_materials":
            len(evaluation_table),

        "candidate_recall":
            evaluation_table["rank"]
            .notna()
            .mean(),

        "conditional_top1":
            (retrieved["rank"] <= 1).mean(),

        "conditional_top3":
            (retrieved["rank"] <= 3).mean(),

        "conditional_top5":
            (retrieved["rank"] <= 5).mean(),

        "conditional_top10":
            (retrieved["rank"] <= 10).mean(),

        "conditional_mrr":
            (1 / retrieved["rank"]).mean(),

        "end_to_end_top1":
            evaluation_table["rank"]
            .le(1)
            .fillna(False)
            .mean(),

        "end_to_end_top3":
            evaluation_table["rank"]
            .le(3)
            .fillna(False)
            .mean(),

        "end_to_end_top5":
            evaluation_table["rank"]
            .le(5)
            .fillna(False)
            .mean(),

        "end_to_end_top10":
            evaluation_table["rank"]
            .le(10)
            .fillna(False)
            .mean()
    }

    return result, model, validation_group

In [443]:
group_model_results = []

trained_group_models = {}
scored_group_validation = {}

group_codes = [
    "1020001",  # ORME
    "1020002"   # DOKUMA
]

for group_code in group_codes:

    lr_result, lr_model, lr_scored = (
        evaluate_group_reranker(
            group_code=group_code,
            model=build_no_attr_lr(),
            model_name="Logistic Regression",
            features=group_specific_features
        )
    )

    group_model_results.append(lr_result)

    trained_group_models[
        (group_code, "LR")
    ] = lr_model

    scored_group_validation[
        (group_code, "LR")
    ] = lr_scored


    rf_result, rf_model, rf_scored = (
        evaluate_group_reranker(
            group_code=group_code,
            model=build_no_attr_rf(),
            model_name="Random Forest",
            features=group_specific_features
        )
    )

    group_model_results.append(rf_result)

    trained_group_models[
        (group_code, "RF")
    ] = rf_model

    scored_group_validation[
        (group_code, "RF")
    ] = rf_scored

c:\Users\EMRE.YILMAZ\Downloads\Codes\.venv\Lib\site-packages\sklearn\impute\_base.py:572: FutureWarning: Currently, when `keep_empty_feature=False` and `strategy="constant"`, empty features are not dropped. This behaviour will change in version 1.8. Set `keep_empty_feature=True` to preserve this behaviour.
  warnings.warn(
c:\Users\EMRE.YILMAZ\Downloads\Codes\.venv\Lib\site-packages\sklearn\impute\_base.py:572: FutureWarning: Currently, when `keep_empty_feature=False` and `strategy="constant"`, empty features are not dropped. This behaviour will change in version 1.8. Set `keep_empty_feature=True` to preserve this behaviour.
  warnings.warn(


In [444]:
group_model_results_df = (
    pd.DataFrame(group_model_results)
    .sort_values(
        [
            "material_group_code",
            "conditional_mrr"
        ],
        ascending=[True, False]
    )
)

display(group_model_results_df)

,material_group_code,model,validation_materials,candidate_recall,conditional_top1,conditional_top3,conditional_top5,conditional_top10,conditional_mrr,end_to_end_top1,end_to_end_top3,end_to_end_top5,end_to_end_top10
1,1020001,Random Forest,176,0.880682,0.032258,0.070968,0.070968,0.161290,0.070409,0.028409,0.062500,0.062500,0.142045
0,1020001,Logistic Regression,176,0.880682,0.025806,0.058065,0.064516,0.154839,0.064145,0.022727,0.051136,0.056818,0.136364
3,1020002,Random Forest,162,0.796296,0.348837,0.480620,0.542636,0.666667,0.454674,0.277778,0.382716,0.432099,0.530864
2,1020002,Logistic Regression,162,0.796296,0.248062,0.387597,0.449612,0.589147,0.350450,0.197531,0.308642,0.358025,0.469136


In [445]:
rank_distribution_results = []

for (
    group_code,
    model_name
), scored_data in scored_group_validation.items():

    true_ranks = (
        scored_data[
            scored_data["is_match"] == 1
        ]["rank"]
    )

    rank_distribution_results.append({
        "material_group_code":
            group_code,

        "model":
            model_name,

        "median_rank":
            true_ranks.median(),

        "p75_rank":
            true_ranks.quantile(0.75),

        "p90_rank":
            true_ranks.quantile(0.90),

        "mean_rank":
            true_ranks.mean(),

        "max_rank":
            true_ranks.max()
    })

display(
    pd.DataFrame(
        rank_distribution_results
    )
)

,material_group_code,model,median_rank,p75_rank,p90_rank,mean_rank,max_rank
0,1020001,LR,86.0,307.0,1143.4,289.967742,2029.0
1,1020001,RF,107.0,328.0,1156.4,320.922581,1998.0
2,1020002,LR,6.0,28.0,73.2,22.325581,129.0
3,1020002,RF,4.0,19.0,32.2,11.558140,94.0


In [446]:
# Build true PLM structure + yarn attributes for ORME development materials
orme_oracle_truth = (
    orme_dev_truth
    .merge(
        plm_structure_candidates[
            ["plm_code", "structure"]
        ],
        on="plm_code",
        how="left"
    )
    .merge(
        plm_yarn_count_retrieval[
            ["plm_code", "yarn_count_class"]
        ],
        on="plm_code",
        how="left"
    )
)

# Count how many PLM codes share the exact same structure + yarn combination
orme_pool_sizes = (
    plm_orme_structure_yarn
    .groupby(
        ["structure", "yarn_count_class"]
    )
    .size()
    .rename("oracle_pool_size")
    .reset_index()
)

orme_oracle_truth = (
    orme_oracle_truth
    .merge(
        orme_pool_sizes,
        on=["structure", "yarn_count_class"],
        how="left"
    )
)

display(
    orme_oracle_truth["oracle_pool_size"]
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
)

count     848.000000
mean      624.284198
std       522.302658
min         1.000000
25%       124.000000
50%       349.000000
75%      1265.000000
90%      1265.000000
95%      1265.000000
max      1265.000000
Name: oracle_pool_size, dtype: float64

In [447]:
print(
    "Mean oracle pool:",
    orme_oracle_truth["oracle_pool_size"].mean()
)

print(
    "Median oracle pool:",
    orme_oracle_truth["oracle_pool_size"].median()
)

print(
    "P90 oracle pool:",
    orme_oracle_truth["oracle_pool_size"].quantile(0.90)
)

Mean oracle pool: 624.2841981132076
Median oracle pool: 349.0
P90 oracle pool: 1265.0


In [448]:
# Audit exact description ambiguity in no-attribute ORME
orme_description_truth = (
    sap_no_attr_dev[
        sap_no_attr_dev["material_group_code"] == "1020001"
    ][
        ["material_id", "retrieval_text"]
    ]
    .merge(
        orme_dev_truth[
            ["material_id", "plm_code"]
        ],
        on="material_id",
        how="inner"
    )
)

description_ambiguity = (
    orme_description_truth
    .groupby("retrieval_text")
    .agg(
        sap_materials=("material_id", "nunique"),
        mapped_plms=("plm_code", "nunique")
    )
    .reset_index()
)

print(
    "Unique descriptions:",
    len(description_ambiguity)
)

print(
    "Descriptions mapping to >1 PLM:",
    (
        description_ambiguity["mapped_plms"] > 1
    ).sum()
)

ambiguous_descriptions = set(
    description_ambiguity.loc[
        description_ambiguity["mapped_plms"] > 1,
        "retrieval_text"
    ]
)

materials_inside_ambiguous_descriptions = (
    orme_description_truth[
        orme_description_truth[
            "retrieval_text"
        ].isin(ambiguous_descriptions)
    ]["material_id"]
    .nunique()
)

print(
    "Materials inside ambiguous descriptions:",
    materials_inside_ambiguous_descriptions
)

Unique descriptions: 863
Descriptions mapping to >1 PLM: 6
Materials inside ambiguous descriptions: 16


In [449]:
# How unique is the PLM structured representation?
orme_plm_text_ambiguity = (
    plm_candidates_enriched[
        plm_candidates_enriched[
            "material_group_code"
        ] == "1020001"
    ]
    .groupby("structured_text")
    .agg(
        plm_codes=("plm_code", "nunique")
    )
    .reset_index()
)

print(
    "ORME PLM codes:",
    plm_candidates_enriched[
        plm_candidates_enriched[
            "material_group_code"
        ] == "1020001"
    ]["plm_code"].nunique()
)

print(
    "Unique structured texts:",
    len(orme_plm_text_ambiguity)
)

print(
    "Structured texts shared by multiple PLMs:",
    (
        orme_plm_text_ambiguity["plm_codes"] > 1
    ).sum()
)

display(
    orme_plm_text_ambiguity[
        orme_plm_text_ambiguity["plm_codes"] > 1
    ]
    .sort_values(
        "plm_codes",
        ascending=False
    )
    .head(20)
)

ORME PLM codes: 12812
Unique structured texts: 4740
Structured texts shared by multiple PLMs: 1611


,structured_text,plm_codes
1476,JERSEY CIRCULAR 30 1 150 0,314
3863,THREETHREADLOOPBACK CIRCULAR 30 1 280 0,210
3658,SOCKS CIRCULAR 20 1 0 0,168
4220,TWOTHREADLOOPBACK CIRCULAR 30 1 240 0,142
4218,TWOTHREADLOOPBACK CIRCULAR 30 1 230 0,97
1490,JERSEY CIRCULAR 30 1 210 0,96
3859,THREETHREADLOOPBACK CIRCULAR 30 1 260 0,87
1485,JERSEY CIRCULAR 30 1 190 0,84
1478,JERSEY CIRCULAR 30 1 160 0,82
685,INTERLOCK CIRCULAR 30 1 250 0,68


## 14. PLM master içinde mükerrer kayıt tespiti


In [450]:
# Audit PLM master columns before duplicate detection
plm_column_audit = pd.DataFrame({
    "column": plm_codes.columns,
    "dtype": [
        str(plm_codes[column].dtype)
        for column in plm_codes.columns
    ],
    "non_null": [
        plm_codes[column].notna().sum()
        for column in plm_codes.columns
    ],
    "unique_values": [
        plm_codes[column].nunique(dropna=True)
        for column in plm_codes.columns
    ]
})

display(plm_column_audit)

,column,dtype,non_null,unique_values
0,PLM Kodu,object,32361,32361
1,Örme alt tipi,object,11741,9
2,İlmek uzunluğu/50 iğne,object,8016,1148
3,Malzeme statüs,object,13105,6
4,1.İplik numarası örme,object,11937,399
5,2.İplik numarası örm,object,8393,306
6,3.İplik numarası örme,object,3019,167
7,Kumaş ağırlığı,float64,31265,647
8,Kumaş ağırlığı birimi,object,28772,10
9,Kumaş eni,float64,21375,177


In [451]:
import re
import numpy as np
import pandas as pd


def normalize_duplicate_text(series):
    return (
        series
        .astype("string")
        .str.normalize("NFKC")
        .str.strip()
        .str.upper()
        .str.replace(r"\s+", " ", regex=True)
        .fillna("<MISSING>")
        .replace("", "<MISSING>")
    )


def normalize_duplicate_column(series):
    # Preserve numeric equivalence such as 150 and 150.0
    if pd.api.types.is_numeric_dtype(series):
        numeric = pd.to_numeric(
            series,
            errors="coerce"
        )

        return (
            numeric
            .round(8)
            .astype("string")
            .fillna("<MISSING>")
        )

    return normalize_duplicate_text(series)

### 14.1 Teşhis: normalizasyonun yarattığı PLM kodu çakışmaları

> Büyük harfe çevirme, yalnızca büyük/küçük harf ile ayrışan PLM kodlarını
> birleştiriyordu. Bu yüzden kimlik normalizasyonu (`strip` only) kullanılır.


In [452]:
# Inspect PLM-code collisions introduced by normalization
plm_code_collision_audit = (
    plm_codes[
        ["PLM Kodu"]
    ]
    .assign(
        normalized_plm_code=(
            plm_codes["PLM Kodu"]
            .astype("string")
            .str.strip()
            .str.upper()
            .str.replace(r"\.0$", "", regex=True)
        )
    )
)

collision_codes = (
    plm_code_collision_audit
    .groupby("normalized_plm_code")
    .agg(
        raw_code_count=("PLM Kodu", "nunique"),
        raw_codes=(
            "PLM Kodu",
            lambda x: list(x.astype(str).unique())
        )
    )
    .query("raw_code_count > 1")
)

display(collision_codes)

,raw_code_count,raw_codes
normalized_plm_code,,
"DOBBY, WOVEN-OTHERS, 75 D X 75",2,"[DOBBY, WOVEN-OTHERS, 75 D X 75, Dobby, Woven-..."


In [453]:
# Inspect the raw PLM rows that collide after case normalization
collision_raw_codes = (
    plm_code_collision_audit[
        plm_code_collision_audit["normalized_plm_code"]
        == "DOBBY, WOVEN-OTHERS, 75 D X 75"
    ]["PLM Kodu"]
    .tolist()
)

display(
    plm_codes[
        plm_codes["PLM Kodu"].isin(collision_raw_codes)
    ].T
)

,32357,32358
PLM Kodu,"DOBBY, WOVEN-OTHERS, 75 D X 75","Dobby, Woven-Others, 75 D X 75"
Örme alt tipi,CIRCULAR,NaN
İlmek uzunluğu/50 iğne,NaN,NaN
Malzeme statüs,NaN,INCONCEPT
1.İplik numarası örme,NaN,NaN
2.İplik numarası örm,NaN,NaN
3.İplik numarası örme,NaN,NaN
Kumaş ağırlığı,190.0,190.0
Kumaş ağırlığı birimi,GSM,GSM
Kumaş eni,0.0,0.0


In [454]:
# Check whether the colliding raw identifiers exist in the multi-value file
mv_raw_codes = (
    plm_multi_value_valid["plm_code"]
    .astype("string")
    .str.strip()
)

for raw_code in collision_raw_codes:
    count = (
        mv_raw_codes
        == str(raw_code).strip()
    ).sum()

    print(
        repr(raw_code),
        "multi-value rows:",
        count
    )

'DOBBY, WOVEN-OTHERS, 75 D X 75' multi-value rows: 4
'Dobby, Woven-Others, 75 D X 75' multi-value rows: 0


In [455]:
def normalize_plm_code_identity(series):
    return (
        series
        .astype("string")
        .str.strip()
    )

In [456]:
plm_duplicate_base = plm_codes.copy()

plm_duplicate_base["plm_code"] = (
    normalize_plm_code_identity(
        plm_duplicate_base["PLM Kodu"]
    )
)

print(
    "Rows:",
    len(plm_duplicate_base)
)

print(
    "Unique PLM codes:",
    plm_duplicate_base["plm_code"].nunique()
)

assert plm_duplicate_base["plm_code"].is_unique

Rows: 32361
Unique PLM codes: 32361


In [457]:
mv_duplicate = plm_multi_value_valid.copy()

mv_duplicate["plm_code"] = (
    normalize_plm_code_identity(
        mv_duplicate["plm_code"]
    )
)

mv_duplicate["characteristic"] = (
    normalize_duplicate_text(
        mv_duplicate[
            "PLM Karakteristik Tanımı"
        ]
    )
)

mv_duplicate["value"] = (
    normalize_duplicate_text(
        mv_duplicate[
            "PLM Karakteristik Değeri"
        ]
    )
)

In [458]:
master_codes = set(
    plm_duplicate_base["plm_code"]
)

multi_value_codes = set(
    mv_duplicate["plm_code"]
)

print(
    "Master PLM codes:",
    len(master_codes)
)

print(
    "Multi-value PLM codes:",
    len(multi_value_codes)
)

print(
    "Multi-value codes found in master:",
    len(
        multi_value_codes
        & master_codes
    )
)

print(
    "Multi-value codes NOT found in master:",
    len(
        multi_value_codes
        - master_codes
    )
)

Master PLM codes: 32361
Multi-value PLM codes: 31953
Multi-value codes found in master: 31953
Multi-value codes NOT found in master: 0


In [459]:
duplicate_excluded_columns = {
    # Identity
    "PLM Kodu",
    "plm_code",

    # Descriptive fields
    "Türkçe malzeme açıklaması",
    "Malzeme Türkçe Adı",
    "Malzeme ingilizce adı",

    # Administrative / derived fields
    "Malzeme statüs",
    "material_group_code",
    "material_group_name",
    "material_family"
}

structured_duplicate_features = [
    column
    for column in plm_duplicate_base.columns
    if column not in duplicate_excluded_columns
]

print(
    "Structured duplicate features:",
    len(structured_duplicate_features)
)

print(structured_duplicate_features)

Structured duplicate features: 22
['Örme alt tipi', 'İlmek uzunluğu/50 iğne', '1.İplik numarası örme', '2.İplik numarası örm', '3.İplik numarası örme', 'Kumaş ağırlığı', 'Kumaş ağırlığı birimi', 'Kumaş eni', 'Kumaş eni birimi', 'Pus', 'Fine', 'Mal grubu', 'Kumaş tipi', 'Çözgü sıklığı (tel /in', 'Atkı sıklığı (tel /inç)', 'Dokuma tipi', '1.Çözgü iplik numarası', '1.Atkı iplik numarası', '2.Çözgü iplik numarası', '2.Atkı iplik numarası', '3.Çözgü iplik numarası', '3.Atkı iplik numaras']


In [460]:
structured_signature = (
    plm_duplicate_base[
        ["plm_code"] + structured_duplicate_features
    ]
    .copy()
)

for column in structured_duplicate_features:
    structured_signature[column] = (
        normalize_duplicate_column(
            structured_signature[column]
        )
    )

assert structured_signature["plm_code"].is_unique

print(
    "Structured PLM rows:",
    len(structured_signature)
)

Structured PLM rows: 32361


In [461]:
mv_characteristic_sets = (
    mv_duplicate
    .groupby(
        [
            "plm_code",
            "characteristic"
        ]
    )["value"]
    .agg(
        lambda values:
            " || ".join(
                sorted(set(values))
            )
    )
    .reset_index()
)

mv_signature_wide = (
    mv_characteristic_sets
    .pivot(
        index="plm_code",
        columns="characteristic",
        values="value"
    )
    .fillna("<ABSENT>")
    .reset_index()
)

mv_feature_columns = [
    column
    for column in mv_signature_wide.columns
    if column != "plm_code"
]

mv_signature_wide = (
    mv_signature_wide.rename(
        columns={
            column: f"MV__{column}"
            for column in mv_feature_columns
        }
    )
)

assert mv_signature_wide["plm_code"].is_unique

print(
    "Multi-value PLM profiles:",
    len(mv_signature_wide)
)

print(
    "Multi-value characteristics:",
    len(mv_feature_columns)
)

Multi-value PLM profiles: 31953
Multi-value characteristics: 64


In [462]:
plm_duplicate_master = (
    structured_signature
    .merge(
        mv_signature_wide,
        on="plm_code",
        how="left",
        validate="one_to_one"
    )
)

mv_columns = [
    column
    for column in plm_duplicate_master.columns
    if column.startswith("MV__")
]

plm_duplicate_master[mv_columns] = (
    plm_duplicate_master[mv_columns]
    .fillna("<ABSENT>")
)

assert plm_duplicate_master["plm_code"].is_unique

# The merge must not add or drop rows relative to the structured signature.
assert len(plm_duplicate_master) == len(structured_signature)

print(
    "Merged PLM rows:",
    len(plm_duplicate_master)
)

print(
    "Structured features:",
    len(structured_duplicate_features)
)

print(
    "Multi-value features:",
    len(mv_columns)
)

Merged PLM rows: 32361
Structured features: 22
Multi-value features: 64


In [463]:
duplicate_feature_columns = (
    structured_duplicate_features
    + mv_columns
)

plm_duplicate_master[
    "material_profile_hash"
] = (
    pd.util.hash_pandas_object(
        plm_duplicate_master[
            duplicate_feature_columns
        ],
        index=False
    )
    .astype("uint64")
)

In [464]:
profile_counts = (
    plm_duplicate_master[
        "material_profile_hash"
    ]
    .value_counts()
)

duplicate_hashes = set(
    profile_counts[
        profile_counts > 1
    ].index
)

exact_duplicate_plms = (
    plm_duplicate_master[
        plm_duplicate_master[
            "material_profile_hash"
        ].isin(duplicate_hashes)
    ]
    .copy()
)

print(
    "Total PLM codes:",
    len(plm_duplicate_master)
)

print(
    "Unique technical profiles:",
    plm_duplicate_master[
        "material_profile_hash"
    ].nunique()
)

print(
    "Exact duplicate groups:",
    len(duplicate_hashes)
)

print(
    "PLM codes inside exact duplicate groups:",
    len(exact_duplicate_plms)
)

print(
    "Duplicate PLM rate:",
    f"{len(exact_duplicate_plms) / len(plm_duplicate_master):.2%}"
)

Total PLM codes: 32361
Unique technical profiles: 29778
Exact duplicate groups: 1289
PLM codes inside exact duplicate groups: 3872
Duplicate PLM rate: 11.97%


In [465]:
duplicate_verification = (
    exact_duplicate_plms
    .groupby("material_profile_hash")[duplicate_feature_columns]
    .apply(
        lambda group: group.drop_duplicates().shape[0]
    )
)

print(
    "Groups with more than one distinct feature row:",
    (duplicate_verification > 1).sum()
)

assert (duplicate_verification == 1).all()

Groups with more than one distinct feature row: 0


In [466]:
duplicate_hash_order = (
    exact_duplicate_plms[
        "material_profile_hash"
    ]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

duplicate_group_mapping = {
    profile_hash: f"PLM_DUP_{index:05d}"
    for index, profile_hash in enumerate(
        duplicate_hash_order,
        start=1
    )
}

exact_duplicate_plms[
    "duplicate_group_id"
] = (
    exact_duplicate_plms[
        "material_profile_hash"
    ]
    .map(duplicate_group_mapping)
)

exact_duplicate_plms[
    "duplicate_group_size"
] = (
    exact_duplicate_plms
    .groupby("duplicate_group_id")[
        "plm_code"
    ]
    .transform("size")
)

In [467]:
duplicate_review = (
    exact_duplicate_plms[
        [
            "duplicate_group_id",
            "duplicate_group_size",
            "plm_code",
            "material_profile_hash"
        ]
    ]
    .merge(
        plm_duplicate_base[
            [
                "plm_code",
                "Mal grubu",
                "Türkçe malzeme açıklaması",
                "Malzeme Türkçe Adı",
                "Malzeme ingilizce adı",
                "Malzeme statüs"
            ]
        ],
        on="plm_code",
        how="left",
        validate="one_to_one"
    )
    .sort_values(
        [
            "duplicate_group_size",
            "duplicate_group_id",
            "plm_code"
        ],
        ascending=[
            False,
            True,
            True
        ]
    )
)

display(
    duplicate_review.head(100)
)

,duplicate_group_id,duplicate_group_size,plm_code,material_profile_hash,Mal grubu,Türkçe malzeme açıklaması,Malzeme Türkçe Adı,Malzeme ingilizce adı,Malzeme statüs
29,PLM_DUP_00154,62,100000253,2095193622556995218,MATERIAL,CHİRPY; SÜET TEKSTİL /150 CM/100 GM2 (367780: ...,NaN,NaN,NaN
2324,PLM_DUP_00154,62,360687,2095193622556995218,MATERIAL,NaN,NaN,NaN,INCONCEPT
2325,PLM_DUP_00154,62,360688,2095193622556995218,MATERIAL,NaN,NaN,NaN,INCONCEPT
2667,PLM_DUP_00154,62,363164,2095193622556995218,MATERIAL,NaN,NaN,NaN,INCONCEPT
2690,PLM_DUP_00154,62,363329,2095193622556995218,MATERIAL,NaN,NaN,NaN,INCONCEPT
2705,PLM_DUP_00154,62,363437,2095193622556995218,MATERIAL,NaN,NaN,NaN,INCONCEPT
2708,PLM_DUP_00154,62,363442,2095193622556995218,MATERIAL,NaN,NaN,NaN,INCONCEPT
2712,PLM_DUP_00154,62,363467,2095193622556995218,MATERIAL,NaN,NaN,NaN,INCONCEPT
2904,PLM_DUP_00154,62,365250,2095193622556995218,MATERIAL,NaN,NaN,NaN,INCONCEPT
2906,PLM_DUP_00154,62,365255,2095193622556995218,MATERIAL,NaN,NaN,NaN,INCONCEPT


In [468]:
duplicate_impact_by_group = (
    plm_duplicate_base[
        [
            "plm_code",
            "Mal grubu"
        ]
    ]
    .merge(
        exact_duplicate_plms[
            [
                "plm_code",
                "duplicate_group_id"
            ]
        ],
        on="plm_code",
        how="left",
        validate="one_to_one"
    )
    .groupby("Mal grubu")
    .agg(
        plm_codes=(
            "plm_code",
            "nunique"
        ),
        duplicate_plm_codes=(
            "duplicate_group_id",
            lambda x: x.notna().sum()
        ),
        duplicate_groups=(
            "duplicate_group_id",
            "nunique"
        )
    )
)

duplicate_impact_by_group[
    "duplicate_plm_rate"
] = (
    duplicate_impact_by_group[
        "duplicate_plm_codes"
    ]
    / duplicate_impact_by_group[
        "plm_codes"
    ]
)

display(
    duplicate_impact_by_group
    .sort_values(
        "duplicate_plm_rate",
        ascending=False
    )
)

,plm_codes,duplicate_plm_codes,duplicate_groups,duplicate_plm_rate
Mal grubu,,,,
TEST,4,4,1,1.000000
LEATHER1,1059,807,148,0.762040
LEATHER2,56,42,16,0.750000
MATERIAL,399,239,42,0.598997
YARN,138,44,16,0.318841
DENIM,3558,947,331,0.266161
TRIKO,2193,227,85,0.103511
LEATHER,50,4,2,0.080000
ORME,12812,919,381,0.071730


In [469]:
duplicate_group_summary = (
    exact_duplicate_plms
    .groupby("duplicate_group_id")
    .agg(
        duplicate_group_size=(
            "plm_code",
            "nunique"
        )
    )
    .reset_index()
)

display(
    duplicate_group_summary[
        "duplicate_group_size"
    ]
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

count    1289.000000
mean        3.003879
std         2.750350
min         2.000000
50%         2.000000
75%         3.000000
90%         6.000000
95%         6.000000
99%        13.120000
max        62.000000
Name: duplicate_group_size, dtype: float64

## 15. Statü ve teknik doluluk analizi


In [470]:
# Normalize PLM status for analysis
plm_status_audit = (
    plm_duplicate_base[
        [
            "plm_code",
            "Mal grubu",
            "Malzeme statüs"
        ]
    ]
    .copy()
)

plm_status_audit["status"] = (
    plm_status_audit["Malzeme statüs"]
    .astype("string")
    .str.strip()
    .str.upper()
    .fillna("<MISSING>")
)

status_distribution = (
    plm_status_audit["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="plm_codes")
)

status_distribution["rate"] = (
    status_distribution["plm_codes"]
    / len(plm_status_audit)
)

display(status_distribution)

,status,plm_codes,rate
0,<MISSING>,19256,0.595037
1,INCONCEPT,9975,0.308241
2,LCWPS9,2665,0.082352
3,PRODUCTCARDAPPROVED,424,0.013102
4,WS005,21,0.000649
5,ACTIVE,14,0.000433
6,TECHNICALMEETWAITING,6,0.000185


In [471]:
# Add status to PLMs that belong to repeated exact technical profiles
# Same data, clearer terminology:
# these are repeated exact observed profiles, not yet confirmed duplicates
exact_profile_plms = exact_duplicate_plms.copy()
profile_status_audit = (
    exact_profile_plms[
        [
            "plm_code",
            "duplicate_group_id"
        ]
    ]
    .merge(
        plm_status_audit[
            [
                "plm_code",
                "Mal grubu",
                "status"
            ]
        ],
        on="plm_code",
        how="left",
        validate="many_to_one"
    )
)

profile_status_summary = (
    profile_status_audit["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="plm_codes")
)

profile_status_summary["rate"] = (
    profile_status_summary["plm_codes"]
    / len(profile_status_audit)
)

display(profile_status_summary)

,status,plm_codes,rate
0,<MISSING>,2343,0.605114
1,INCONCEPT,1080,0.278926
2,LCWPS9,432,0.11157
3,PRODUCTCARDAPPROVED,11,0.002841
4,TECHNICALMEETWAITING,5,0.001291
5,WS005,1,0.000258


In [472]:
print(
    "PLMs inside repeated profiles:",
    len(profile_status_audit)
)

print(
    "INCONCEPT PLMs inside repeated profiles:",
    profile_status_audit["status"]
    .eq("INCONCEPT")
    .sum()
)

print(
    "INCONCEPT rate inside repeated profiles:",
    f"{profile_status_audit['status'].eq('INCONCEPT').mean():.2%}"
)

PLMs inside repeated profiles: 3872
INCONCEPT PLMs inside repeated profiles: 1080
INCONCEPT rate inside repeated profiles: 27.89%


In [473]:
profile_status_by_group = (
    profile_status_audit
    .groupby("duplicate_group_id")
    .agg(
        profile_size=("plm_code", "nunique"),

        inconcept_count=(
            "status",
            lambda x: (x == "INCONCEPT").sum()
        ),

        distinct_statuses=(
            "status",
            lambda x: tuple(sorted(set(x)))
        )
    )
    .reset_index()
)

profile_status_by_group["inconcept_rate"] = (
    profile_status_by_group["inconcept_count"]
    / profile_status_by_group["profile_size"]
)

profile_status_by_group["profile_status_type"] = np.select(
    [
        profile_status_by_group["inconcept_rate"].eq(1),
        profile_status_by_group["inconcept_rate"].eq(0)
    ],
    [
        "ALL_INCONCEPT",
        "NO_INCONCEPT"
    ],
    default="MIXED"
)

display(
    profile_status_by_group[
        "profile_status_type"
    ]
    .value_counts()
    .rename_axis("profile_status_type")
    .reset_index(name="profile_groups")
)

,profile_status_type,profile_groups
0,NO_INCONCEPT,832
1,ALL_INCONCEPT,287
2,MIXED,170


In [474]:
# Build PLM completeness table cleanly
plm_completeness = (
    plm_duplicate_master[
        ["plm_code"]
        + structured_duplicate_features
        + mv_columns
    ]
    .copy()
)

plm_completeness["structured_filled_count"] = (
    plm_completeness[
        structured_duplicate_features
    ]
    .apply(
        lambda row: sum(
            value != "<MISSING>"
            for value in row
        ),
        axis=1
    )
)

plm_completeness["mv_filled_count"] = (
    plm_completeness[
        mv_columns
    ]
    .apply(
        lambda row: sum(
            value not in {
                "<ABSENT>",
                "<MISSING>"
            }
            for value in row
        ),
        axis=1
    )
)

plm_completeness["technical_filled_count"] = (
    plm_completeness["structured_filled_count"]
    + plm_completeness["mv_filled_count"]
)

plm_completeness["technical_completeness_rate"] = (
    plm_completeness["technical_filled_count"]
    / (
        len(structured_duplicate_features)
        + len(mv_columns)
    )
)

In [475]:
plm_completeness = (
    plm_completeness
    .merge(
        plm_status_audit[
            [
                "plm_code",
                "status"
            ]
        ],
        on="plm_code",
        how="left",
        validate="one_to_one"
    )
)

assert "Mal grubu" in plm_completeness.columns
assert "status" in plm_completeness.columns
assert plm_completeness["plm_code"].is_unique

In [476]:
plm_completeness["status_bucket"] = np.select(
    [
        plm_completeness["status"] == "INCONCEPT",
        plm_completeness["status"] == "<MISSING>"
    ],
    [
        "INCONCEPT",
        "MISSING"
    ],
    default="OTHER_KNOWN"
)

In [477]:
fabric_completeness = (
    plm_completeness[
        plm_completeness["Mal grubu"]
        .isin(
            [
                "ORME",
                "DOKUMA",
                "DENIM"
            ]
        )
    ]
    .copy()
)

fabric_status_completeness = (
    fabric_completeness
    .groupby(
        [
            "Mal grubu",
            "status_bucket"
        ]
    )
    .agg(
        plm_codes=(
            "plm_code",
            "nunique"
        ),
        mean_structured_fields=(
            "structured_filled_count",
            "mean"
        ),
        mean_mv_fields=(
            "mv_filled_count",
            "mean"
        ),
        mean_total_fields=(
            "technical_filled_count",
            "mean"
        ),
        median_total_fields=(
            "technical_filled_count",
            "median"
        ),
        mean_completeness_rate=(
            "technical_completeness_rate",
            "mean"
        )
    )
    .reset_index()
)

display(fabric_status_completeness)

,Mal grubu,status_bucket,plm_codes,mean_structured_fields,mean_mv_fields,mean_total_fields,median_total_fields,mean_completeness_rate
0,DENIM,INCONCEPT,1143,5.623797,1.993876,7.617673,7.0,0.088578
1,DENIM,MISSING,2318,6.241156,2.343831,8.584987,9.0,0.099825
2,DENIM,OTHER_KNOWN,97,5.206186,0.865979,6.072165,7.0,0.070607
3,DOKUMA,INCONCEPT,3314,9.518709,6.194931,15.713639,16.0,0.182717
4,DOKUMA,MISSING,6859,8.916314,4.587695,13.504009,14.0,0.157023
5,DOKUMA,OTHER_KNOWN,1914,6.196447,1.613898,7.810345,7.0,0.090818
6,ORME,INCONCEPT,4624,9.896626,6.248919,16.145545,16.0,0.187739
7,ORME,MISSING,7296,9.335115,6.115132,15.450247,16.0,0.179654
8,ORME,OTHER_KNOWN,892,6.978700,2.782511,9.761211,7.0,0.113502


In [478]:
display(
    fabric_completeness
    .groupby(
        [
            "Mal grubu",
            "status_bucket"
        ]
    )[
        "technical_filled_count"
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    )
)

count       mean       std  min   10%   25%   50%  \
Mal grubu status_bucket                                                       
DENIM     INCONCEPT      1143.0   7.617673  1.778428  2.0   6.0   7.0   7.0   
          MISSING        2318.0   8.584987  2.186272  2.0   6.0   7.0   9.0   
          OTHER_KNOWN      97.0   6.072165  1.894330  2.0   2.0   7.0   7.0   
DOKUMA    INCONCEPT      3314.0  15.713639  4.339197  2.0   9.0  14.0  16.0   
          MISSING        6859.0  13.504009  3.524456  2.0   8.0  12.0  14.0   
          OTHER_KNOWN    1914.0   7.810345  3.476164  2.0   7.0   7.0   7.0   
ORME      INCONCEPT      4624.0  16.145545  4.130340  3.0  11.0  14.0  16.0   
          MISSING        7296.0  15.450247  4.220871  2.0  10.0  14.0  16.0   
          OTHER_KNOWN     892.0   9.761211  5.606648  2.0   6.0   7.0   7.0   

                          75%   90%   max  
Mal grubu status_bucket                    
DENIM     INCONCEPT       9.0   9.0  19.0  
          MISSING        10.0  11.0  19.0  
          OTHER_KNOWN     7.0   7.0   7.0  
DOKUMA    INCONCEPT      18.0  21.0  47.0  
          MISSING        15.0  17.0  41.0  
          OTHER_KNOWN     7.0   7.0  27.0  
ORME      INCONCEPT      19.0  21.0  30.0  
          MISSING        18.0  20.0  31.0  
          OTHER_KNOWN    13.0  19.0  26.0

In [479]:
sap_used_plm = (
    sap_plm_mapping[
        ["PLM Kodu"]
    ]
    .dropna()
    .copy()
)

sap_used_plm["plm_code"] = (
    normalize_plm_code_identity(
        sap_used_plm["PLM Kodu"]
    )
)

sap_used_unique = (
    sap_used_plm[
        ["plm_code"]
    ]
    .drop_duplicates()
    .merge(
        plm_status_audit[
            [
                "plm_code",
                "Mal grubu",
                "status"
            ]
        ],
        on="plm_code",
        how="left",
        validate="one_to_one"
    )
)

In [480]:
print(
    "Unique SAP-used PLM codes:",
    len(sap_used_unique)
)

print(
    "Found in PLM master:",
    sap_used_unique["Mal grubu"]
    .notna()
    .sum()
)

print(
    "Not found in PLM master:",
    sap_used_unique["Mal grubu"]
    .isna()
    .sum()
)

Unique SAP-used PLM codes: 4012
Found in PLM master: 0
Not found in PLM master: 4012


In [481]:
# Attach technical completeness to repeated exact-profile PLMs
exact_profile_quality = (
    exact_profile_plms[
        [
            "plm_code",
            "duplicate_group_id",
            "duplicate_group_size"
        ]
    ]
    .merge(
        plm_completeness[
            [
                "plm_code",
                "Mal grubu",
                "structured_filled_count",
                "mv_filled_count",
                "technical_filled_count",
                "technical_completeness_rate"
            ]
        ],
        on="plm_code",
        how="left",
        validate="one_to_one"
    )
)

# All PLMs in the same exact profile must have the same technical completeness
profile_quality_summary = (
    exact_profile_quality
    .groupby(
        [
            "duplicate_group_id",
            "Mal grubu"
        ],
        as_index=False
    )
    .agg(
        profile_size=(
            "plm_code",
            "nunique"
        ),
        structured_filled_count=(
            "structured_filled_count",
            "first"
        ),
        mv_filled_count=(
            "mv_filled_count",
            "first"
        ),
        technical_filled_count=(
            "technical_filled_count",
            "first"
        ),
        technical_completeness_rate=(
            "technical_completeness_rate",
            "first"
        )
    )
)

display(
    profile_quality_summary.head()
)

,duplicate_group_id,Mal grubu,profile_size,structured_filled_count,mv_filled_count,technical_filled_count,technical_completeness_rate
0,PLM_DUP_00001,DENIM,2,5,1,6,0.069767
1,PLM_DUP_00002,TRIKO,7,5,1,6,0.069767
2,PLM_DUP_00003,ORME,2,11,7,18,0.209302
3,PLM_DUP_00004,TRIKO,2,4,6,10,0.116279
4,PLM_DUP_00005,DENIM,2,6,4,10,0.116279


In [482]:
fabric_profile_quality = (
    profile_quality_summary[
        profile_quality_summary["Mal grubu"]
        .isin(
            [
                "ORME",
                "DOKUMA",
                "DENIM"
            ]
        )
    ]
    .copy()
)

display(
    fabric_profile_quality
    .groupby("Mal grubu")[
        [
            "profile_size",
            "structured_filled_count",
            "mv_filled_count",
            "technical_filled_count"
        ]
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    )
)

profile_size                                                    \
                 count      mean       std  min  10%  25%  50%  75%  90%   
Mal grubu                                                                  
DENIM            331.0  2.861027  2.433110  2.0  2.0  2.0  2.0  3.0  4.0   
DOKUMA           267.0  2.393258  1.434940  2.0  2.0  2.0  2.0  2.0  3.0   
ORME             381.0  2.412073  1.698532  2.0  2.0  2.0  2.0  2.0  3.0   

                 ... technical_filled_count                                 \
            max  ...                  count       mean       std  min  10%   
Mal grubu        ...                                                         
DENIM      32.0  ...                  331.0   7.604230  1.903071  2.0  6.0   
DOKUMA     20.0  ...                  267.0   9.385768  4.507832  2.0  5.0   
ORME       23.0  ...                  381.0  13.627297  5.172568  2.0  5.0   

                                         
            25%   50%   75%   90%   max  
Mal grubu                                
DENIM       7.0   7.0   9.0  10.0  17.0  
DOKUMA      7.0   7.0  13.0  15.4  27.0  
ORME       10.0  15.0  17.0  19.0  25.0  

[3 rows x 40 columns]

In [483]:
profile_information_flags = (
    fabric_profile_quality.copy()
)

profile_information_flags["no_mv_information"] = (
    profile_information_flags[
        "mv_filled_count"
    ] == 0
)

profile_information_flags["only_few_structured_fields"] = (
    profile_information_flags[
        "structured_filled_count"
    ] <= 3
)

profile_information_flags["very_low_information"] = (
    (
        profile_information_flags[
            "mv_filled_count"
        ] == 0
    )
    &
    (
        profile_information_flags[
            "structured_filled_count"
        ] <= 3
    )
)

display(
    profile_information_flags
    .groupby("Mal grubu")
    .agg(
        profile_groups=(
            "duplicate_group_id",
            "nunique"
        ),
        no_mv_groups=(
            "no_mv_information",
            "sum"
        ),
        very_low_information_groups=(
            "very_low_information",
            "sum"
        )
    )
)

,profile_groups,no_mv_groups,very_low_information_groups
Mal grubu,,,
DENIM,331,9,5
DOKUMA,267,8,5
ORME,381,11,8


In [484]:
fabric_all_plm_completeness = (
    plm_completeness[
        plm_completeness["Mal grubu"]
        .isin(
            [
                "ORME",
                "DOKUMA",
                "DENIM"
            ]
        )
    ]
    .copy()
)

group_completeness_thresholds = (
    fabric_all_plm_completeness
    .groupby("Mal grubu")[
        "technical_filled_count"
    ]
    .quantile(
        [
            0.10,
            0.25,
            0.50,
            0.75
        ]
    )
    .unstack()
    .rename(
        columns={
            0.10: "q10",
            0.25: "q25",
            0.50: "q50",
            0.75: "q75"
        }
    )
    .reset_index()
)

display(group_completeness_thresholds)

,Mal grubu,q10,q25,q50,q75
0,DENIM,6.0,7.0,8.0,9.0
1,DOKUMA,7.0,10.0,14.0,16.0
2,ORME,7.0,13.0,16.0,18.0


In [485]:
profile_information_flags = (
    profile_information_flags
    .merge(
        group_completeness_thresholds,
        on="Mal grubu",
        how="left",
        validate="many_to_one"
    )
)

In [486]:
profile_information_flags["information_level"] = np.select(
    [
        profile_information_flags[
            "technical_filled_count"
        ] <= profile_information_flags["q10"],

        profile_information_flags[
            "technical_filled_count"
        ] <= profile_information_flags["q25"],

        profile_information_flags[
            "technical_filled_count"
        ] >= profile_information_flags["q75"]
    ],
    [
        "VERY_LOW_INFORMATION",
        "LOW_INFORMATION",
        "HIGH_INFORMATION"
    ],
    default="NORMAL_INFORMATION"
)

In [487]:
profile_information_summary = (
    profile_information_flags
    .groupby(
        [
            "Mal grubu",
            "information_level"
        ]
    )
    .agg(
        profile_groups=(
            "duplicate_group_id",
            "nunique"
        ),
        plm_codes=(
            "profile_size",
            "sum"
        ),
        median_profile_size=(
            "profile_size",
            "median"
        ),
        max_profile_size=(
            "profile_size",
            "max"
        ),
        median_filled_fields=(
            "technical_filled_count",
            "median"
        )
    )
    .reset_index()
)

display(profile_information_summary)

,Mal grubu,information_level,profile_groups,plm_codes,median_profile_size,max_profile_size,median_filled_fields
0,DENIM,HIGH_INFORMATION,91,198,2.0,4,9.0
1,DENIM,LOW_INFORMATION,128,391,2.0,14,7.0
2,DENIM,NORMAL_INFORMATION,32,71,2.0,4,8.0
3,DENIM,VERY_LOW_INFORMATION,80,287,2.0,32,6.0
4,DOKUMA,HIGH_INFORMATION,27,57,2.0,3,17.0
5,DOKUMA,LOW_INFORMATION,17,39,2.0,5,8.0
6,DOKUMA,NORMAL_INFORMATION,70,147,2.0,4,13.0
7,DOKUMA,VERY_LOW_INFORMATION,153,396,2.0,20,7.0
8,ORME,HIGH_INFORMATION,92,201,2.0,6,19.0
9,ORME,LOW_INFORMATION,80,190,2.0,16,11.0


In [488]:
display(
    profile_information_flags
    .sort_values(
        [
            "profile_size",
            "technical_filled_count"
        ],
        ascending=[
            False,
            False
        ]
    )
    [
        [
            "duplicate_group_id",
            "Mal grubu",
            "profile_size",
            "structured_filled_count",
            "mv_filled_count",
            "technical_filled_count",
            "information_level"
        ]
    ]
    .head(40)
)

,duplicate_group_id,Mal grubu,profile_size,structured_filled_count,mv_filled_count,technical_filled_count,information_level
538,PLM_DUP_00703,DENIM,32,4,0,4,VERY_LOW_INFORMATION
408,PLM_DUP_00530,ORME,23,2,1,3,VERY_LOW_INFORMATION
866,PLM_DUP_01137,DOKUMA,20,2,0,2,VERY_LOW_INFORMATION
537,PLM_DUP_00700,ORME,18,2,1,3,VERY_LOW_INFORMATION
348,PLM_DUP_00451,ORME,16,8,4,12,LOW_INFORMATION
517,PLM_DUP_00671,DENIM,16,2,0,2,VERY_LOW_INFORMATION
787,PLM_DUP_01025,DENIM,14,5,2,7,LOW_INFORMATION
680,PLM_DUP_00887,DENIM,13,5,2,7,LOW_INFORMATION
959,PLM_DUP_01262,DENIM,12,5,2,7,LOW_INFORMATION
483,PLM_DUP_00625,DENIM,12,6,0,6,VERY_LOW_INFORMATION


In [489]:
high_information_orme_groups = (
    profile_information_flags[
        (
            profile_information_flags[
                "Mal grubu"
            ] == "ORME"
        )
        &
        (
            profile_information_flags[
                "information_level"
            ] == "HIGH_INFORMATION"
        )
    ]
    .sort_values(
        "profile_size",
        ascending=False
    )
)

display(
    high_information_orme_groups.head(10)
)

,duplicate_group_id,Mal grubu,profile_size,structured_filled_count,mv_filled_count,technical_filled_count,technical_completeness_rate,no_mv_information,only_few_structured_fields,very_low_information,q10,q25,q50,q75,information_level
800,PLM_DUP_01044,ORME,6,12,7,19,0.220930,False,False,False,7.0,13.0,16.0,18.0,HIGH_INFORMATION
312,PLM_DUP_00406,ORME,4,11,9,20,0.232558,False,False,False,7.0,13.0,16.0,18.0,HIGH_INFORMATION
340,PLM_DUP_00443,ORME,4,11,11,22,0.255814,False,False,False,7.0,13.0,16.0,18.0,HIGH_INFORMATION
44,PLM_DUP_00058,ORME,3,11,7,18,0.209302,False,False,False,7.0,13.0,16.0,18.0,HIGH_INFORMATION
239,PLM_DUP_00311,ORME,3,10,9,19,0.220930,False,False,False,7.0,13.0,16.0,18.0,HIGH_INFORMATION
622,PLM_DUP_00806,ORME,3,11,7,18,0.209302,False,False,False,7.0,13.0,16.0,18.0,HIGH_INFORMATION
750,PLM_DUP_00977,ORME,3,11,7,18,0.209302,False,False,False,7.0,13.0,16.0,18.0,HIGH_INFORMATION
697,PLM_DUP_00910,ORME,3,11,8,19,0.220930,False,False,False,7.0,13.0,16.0,18.0,HIGH_INFORMATION
283,PLM_DUP_00373,ORME,3,10,9,19,0.220930,False,False,False,7.0,13.0,16.0,18.0,HIGH_INFORMATION
249,PLM_DUP_00322,ORME,3,11,7,18,0.209302,False,False,False,7.0,13.0,16.0,18.0,HIGH_INFORMATION


In [490]:
profile_information_flags["duplicate_evidence_class"] = (
    profile_information_flags["information_level"]
    .map({
        "VERY_LOW_INFORMATION": "INSUFFICIENT_PROFILE",
        "LOW_INFORMATION": "INSUFFICIENT_PROFILE",
        "NORMAL_INFORMATION": "STRONG_PROFILE_MATCH",
        "HIGH_INFORMATION": "VERY_STRONG_PROFILE_MATCH"
    })
)

display(
    profile_information_flags
    .groupby(
        [
            "Mal grubu",
            "duplicate_evidence_class"
        ]
    )
    .agg(
        profile_groups=(
            "duplicate_group_id",
            "nunique"
        ),
        plm_codes=(
            "profile_size",
            "sum"
        )
    )
)

profile_groups  plm_codes
Mal grubu duplicate_evidence_class                            
DENIM     INSUFFICIENT_PROFILE                  208        678
          STRONG_PROFILE_MATCH                   32         71
          VERY_STRONG_PROFILE_MATCH              91        198
DOKUMA    INSUFFICIENT_PROFILE                  170        435
          STRONG_PROFILE_MATCH                   70        147
          VERY_STRONG_PROFILE_MATCH              27         57
ORME      INSUFFICIENT_PROFILE                  149        409
          STRONG_PROFILE_MATCH                  140        309
          VERY_STRONG_PROFILE_MATCH              92        201

In [491]:
def normalize_plm_join_key(series):
    result = (
        series
        .astype("string")
        .str.normalize("NFKC")
        .str.strip()
    )

    numeric_decimal_mask = (
        result.str.fullmatch(
            r"\d+\.0+",
            na=False
        )
    )

    result.loc[numeric_decimal_mask] = (
        result.loc[numeric_decimal_mask]
        .str.replace(
            r"\.0+$",
            "",
            regex=True
        )
    )

    return result

In [492]:
profile_plm_members = (
    exact_profile_plms[
        [
            "plm_code",
            "duplicate_group_id"
        ]
    ]
    .copy()
)

profile_plm_members["plm_join_key"] = (
    normalize_plm_join_key(
        profile_plm_members["plm_code"]
    )
)

profile_plm_members = (
    profile_plm_members
    .merge(
        profile_information_flags[
            [
                "duplicate_group_id",
                "Mal grubu",
                "profile_size",
                "technical_filled_count",
                "information_level",
                "duplicate_evidence_class"
            ]
        ],
        on="duplicate_group_id",
        how="left",
        validate="many_to_one"
    )
)

In [493]:
sap_plm_usage = (
    sap_plm_mapping[
        [
            "Malzeme",
            "PLM Kodu"
        ]
    ]
    .dropna(subset=["PLM Kodu"])
    .copy()
)

sap_plm_usage["plm_join_key"] = (
    normalize_plm_join_key(
        sap_plm_usage["PLM Kodu"]
    )
)

In [494]:
sap_usage_by_plm = (
    sap_plm_usage
    .groupby("plm_join_key")
    .agg(
        sap_material_count=(
            "Malzeme",
            "nunique"
        )
    )
    .reset_index()
)

In [495]:
profile_plm_usage = (
    profile_plm_members
    .merge(
        sap_usage_by_plm,
        on="plm_join_key",
        how="left",
        validate="many_to_one"
    )
)

profile_plm_usage["sap_material_count"] = (
    profile_plm_usage[
        "sap_material_count"
    ]
    .fillna(0)
    .astype(int)
)

In [496]:
profile_usage_summary = (
    profile_plm_usage
    .groupby(
        [
            "duplicate_group_id",
            "Mal grubu",
            "duplicate_evidence_class"
        ],
        as_index=False
    )
    .agg(
        profile_size=(
            "plm_code",
            "nunique"
        ),

        used_plm_codes=(
            "sap_material_count",
            lambda x: (x > 0).sum()
        ),

        total_sap_materials=(
            "sap_material_count",
            "sum"
        ),

        max_sap_materials_on_one_plm=(
            "sap_material_count",
            "max"
        )
    )
)

profile_usage_summary["unused_plm_codes"] = (
    profile_usage_summary["profile_size"]
    - profile_usage_summary["used_plm_codes"]
)

In [497]:
profile_usage_summary["usage_pattern"] = np.select(
    [
        profile_usage_summary["used_plm_codes"] == 0,
        profile_usage_summary["used_plm_codes"] == 1,
        profile_usage_summary["used_plm_codes"] > 1
    ],
    [
        "NONE_USED_IN_SAP",
        "ONE_PLM_USED_IN_SAP",
        "MULTIPLE_PLM_USED_IN_SAP"
    ],
    default="UNKNOWN"
)

In [498]:
display(
    profile_usage_summary[
        profile_usage_summary[
            "Mal grubu"
        ].isin(
            ["ORME", "DOKUMA", "DENIM"]
        )
    ]
    .groupby(
        [
            "Mal grubu",
            "duplicate_evidence_class",
            "usage_pattern"
        ]
    )
    .agg(
        profile_groups=(
            "duplicate_group_id",
            "nunique"
        ),
        total_plm_codes=(
            "profile_size",
            "sum"
        ),
        total_sap_materials=(
            "total_sap_materials",
            "sum"
        )
    )
    .reset_index()
)

,Mal grubu,duplicate_evidence_class,usage_pattern,profile_groups,total_plm_codes,total_sap_materials
0,DENIM,INSUFFICIENT_PROFILE,MULTIPLE_PLM_USED_IN_SAP,13,57,31
1,DENIM,INSUFFICIENT_PROFILE,NONE_USED_IN_SAP,164,497,0
2,DENIM,INSUFFICIENT_PROFILE,ONE_PLM_USED_IN_SAP,31,124,32
3,DENIM,STRONG_PROFILE_MATCH,NONE_USED_IN_SAP,28,63,0
4,DENIM,STRONG_PROFILE_MATCH,ONE_PLM_USED_IN_SAP,4,8,5
5,DENIM,VERY_STRONG_PROFILE_MATCH,NONE_USED_IN_SAP,80,170,0
6,DENIM,VERY_STRONG_PROFILE_MATCH,ONE_PLM_USED_IN_SAP,11,28,11
7,DOKUMA,INSUFFICIENT_PROFILE,NONE_USED_IN_SAP,166,426,0
8,DOKUMA,INSUFFICIENT_PROFILE,ONE_PLM_USED_IN_SAP,4,9,4
9,DOKUMA,STRONG_PROFILE_MATCH,MULTIPLE_PLM_USED_IN_SAP,1,2,3


In [499]:
priority_duplicate_groups = (
    profile_usage_summary[
        (
            profile_usage_summary[
                "duplicate_evidence_class"
            ].isin([
                "STRONG_PROFILE_MATCH",
                "VERY_STRONG_PROFILE_MATCH"
            ])
        )
        &
        (
            profile_usage_summary[
                "usage_pattern"
            ] == "MULTIPLE_PLM_USED_IN_SAP"
        )
    ]
    .copy()
)

display(
    priority_duplicate_groups
    .sort_values(
        [
            "Mal grubu",
            "total_sap_materials"
        ],
        ascending=[True, False]
    )
)

,duplicate_group_id,Mal grubu,duplicate_evidence_class,profile_size,used_plm_codes,total_sap_materials,max_sap_materials_on_one_plm,unused_plm_codes,usage_pattern
229,PLM_DUP_00301,DOKUMA,STRONG_PROFILE_MATCH,2,2,3,2,0,MULTIPLE_PLM_USED_IN_SAP
806,PLM_DUP_01052,ORME,STRONG_PROFILE_MATCH,2,2,5,4,0,MULTIPLE_PLM_USED_IN_SAP
77,PLM_DUP_00096,ORME,STRONG_PROFILE_MATCH,2,2,2,1,0,MULTIPLE_PLM_USED_IN_SAP
78,PLM_DUP_00097,ORME,STRONG_PROFILE_MATCH,2,2,2,1,0,MULTIPLE_PLM_USED_IN_SAP
196,PLM_DUP_00255,ORME,STRONG_PROFILE_MATCH,2,2,2,1,0,MULTIPLE_PLM_USED_IN_SAP
407,PLM_DUP_00529,ORME,STRONG_PROFILE_MATCH,2,2,2,1,0,MULTIPLE_PLM_USED_IN_SAP
450,PLM_DUP_00582,ORME,STRONG_PROFILE_MATCH,2,2,2,1,0,MULTIPLE_PLM_USED_IN_SAP
543,PLM_DUP_00711,ORME,STRONG_PROFILE_MATCH,2,2,2,1,0,MULTIPLE_PLM_USED_IN_SAP
631,PLM_DUP_00816,ORME,STRONG_PROFILE_MATCH,2,2,2,1,0,MULTIPLE_PLM_USED_IN_SAP
851,PLM_DUP_01114,ORME,STRONG_PROFILE_MATCH,2,2,2,1,0,MULTIPLE_PLM_USED_IN_SAP


In [500]:
priority_duplicate_plms = (
    profile_plm_usage[
        profile_plm_usage[
            "duplicate_group_id"
        ].isin(
            priority_duplicate_groups[
                "duplicate_group_id"
            ]
        )
    ]
    .merge(
        plm_duplicate_base[
            [
                "plm_code",
                "Türkçe malzeme açıklaması",
                "Malzeme Türkçe Adı",
                "Malzeme ingilizce adı"
            ]
        ],
        on="plm_code",
        how="left",
        validate="one_to_one"
    )
    .sort_values(
        [
            "duplicate_group_id",
            "sap_material_count",
            "plm_code"
        ],
        ascending=[
            True,
            False,
            True
        ]
    )
)

display(
    priority_duplicate_plms[
        [
            "duplicate_group_id",
            "Mal grubu",
            "duplicate_evidence_class",
            "profile_size",
            "plm_code",
            "sap_material_count",
            "Türkçe malzeme açıklaması",
            "Malzeme Türkçe Adı",
            "Malzeme ingilizce adı"
        ]
    ]
)

,duplicate_group_id,Mal grubu,duplicate_evidence_class,profile_size,plm_code,sap_material_count,Türkçe malzeme açıklaması,Malzeme Türkçe Adı,Malzeme ingilizce adı
3,PLM_DUP_00096,ORME,STRONG_PROFILE_MATCH,2.0,165690,1,NaN,NaN,NaN
4,PLM_DUP_00096,ORME,STRONG_PROFILE_MATCH,2.0,167445,1,NaN,NaN,NaN
6,PLM_DUP_00097,ORME,STRONG_PROFILE_MATCH,2.0,209803,1,NaN,NaN,NaN
12,PLM_DUP_00097,ORME,STRONG_PROFILE_MATCH,2.0,352942,1,NaN,NaN,NaN
8,PLM_DUP_00255,ORME,STRONG_PROFILE_MATCH,2.0,2846,1,-REFERENCENO:3I19686-1I+3I19686-1I,NaN,NaN
18,PLM_DUP_00255,ORME,STRONG_PROFILE_MATCH,2.0,357497,1,-REFERENCENO:3I19686-1I+3I19686-1I,NaN,NaN
21,PLM_DUP_00301,DOKUMA,STRONG_PROFILE_MATCH,2.0,4675,2,-REFERENCENO:VU14269-0H,NaN,NaN
9,PLM_DUP_00301,DOKUMA,STRONG_PROFILE_MATCH,2.0,29789,1,NaN,NaN,NaN
2,PLM_DUP_00529,ORME,STRONG_PROFILE_MATCH,2.0,131452,1,NaN,NaN,NaN
15,PLM_DUP_00529,ORME,STRONG_PROFILE_MATCH,2.0,357240,1,NaN,NaN,NaN


In [501]:
priority_duplicate_sap_materials = (
    sap_plm_usage
    .merge(
        priority_duplicate_plms[
            [
                "duplicate_group_id",
                "Mal grubu",
                "duplicate_evidence_class",
                "plm_join_key",
                "plm_code"
            ]
        ],
        on="plm_join_key",
        how="inner",
        validate="many_to_one"
    )
)

In [502]:
priority_duplicate_sap_materials = (
    priority_duplicate_sap_materials
    .merge(
        sap_plm_mapping[
            [
                "Malzeme",
                "Türkçe malzeme açıklaması",
                "Türkçe malzeme Uzun açıklaması"
            ]
        ]
        .drop_duplicates("Malzeme"),
        on="Malzeme",
        how="left",
        validate="many_to_one"
    )
    .sort_values(
        [
            "duplicate_group_id",
            "plm_code",
            "Malzeme"
        ]
    )
)

display(
    priority_duplicate_sap_materials[
        [
            "duplicate_group_id",
            "Mal grubu",
            "plm_code",
            "Malzeme",
            "Türkçe malzeme açıklaması",
            "Türkçe malzeme Uzun açıklaması"
        ]
    ]
)

,duplicate_group_id,Mal grubu,plm_code,Malzeme,Türkçe malzeme açıklaması,Türkçe malzeme Uzun açıklaması
2,PLM_DUP_00096,ORME,165690,1020000271,30/1 PENYE CT3 GR FLL ELSTN SUP IPLK BO,30/1 PENYE CT3 GR MELANJ 88.0 CO 8.0 PES 4....
3,PLM_DUP_00096,ORME,167445,1020000286,30/1 PENYE LAL GR FLL ELSTN SUP IPLK BO,30/1 PENYE LAL GRI MELANJ 88.0 CO 8.0 PES 4....
16,PLM_DUP_00097,ORME,209803,1020011568,"24/1,DUZ BYA","24/1,Penye,Düz Boyalı"
24,PLM_DUP_00097,ORME,352942,1020018191,"24/1,100.0 Pamuk,SUP,DUZ BYA","24/1,Penye,Süprem,Düz Boyalı"
5,PLM_DUP_00255,ORME,2846,1020000751,30/1 PENYE 3IP MEL ŞARDONLU,30/1 PENYE EVL MAV MELANJ 50 CO 50 PES RD...
17,PLM_DUP_00255,ORME,357497,1020012745,"50.0,ŞRDN 3IP,MEL","Penye,Şardonlu Üç İplik,Melanj"
6,PLM_DUP_00301,DOKUMA,29789,1020000776,VUAL 70 GDX 60/1 60/1 100.0 CO 1/1,VUAL 70 GDX MAT YE L 60/1 60/1 100.0 CO 1/1
10,PLM_DUP_00301,DOKUMA,4675,1020001126,VUAL 70 LNQ PEMBE 60/1 60/1 100 CO 1/1,VUAL 70 LNQ PEMBE EKOSEL 60/1 60/1 100 CO 1/1
14,PLM_DUP_00301,DOKUMA,4675,1020009730,VUAL 70 LNQ PEMBE 60/1 60/1 100%,VUAL 70 LNQ PEMBE EKOSELİ 60/1 6
1,PLM_DUP_00529,ORME,131452,1020000209,30/1 PENYE KOMPAKT JKR SUP DUZ BOYA,30/1 PENYE KOMPAKT FDU K.BEYAZ 74.0 CO 3.0 L...


In [503]:
single_used_cleanup_groups = (
    profile_usage_summary[
        (
            profile_usage_summary[
                "duplicate_evidence_class"
            ].isin([
                "STRONG_PROFILE_MATCH",
                "VERY_STRONG_PROFILE_MATCH"
            ])
        )
        &
        (
            profile_usage_summary[
                "usage_pattern"
            ] == "ONE_PLM_USED_IN_SAP"
        )
    ]
    .copy()
)

display(
    single_used_cleanup_groups
    .groupby("Mal grubu")
    .agg(
        profile_groups=(
            "duplicate_group_id",
            "nunique"
        ),
        total_plm_codes=(
            "profile_size",
            "sum"
        ),
        total_sap_materials=(
            "total_sap_materials",
            "sum"
        )
    )
)

,profile_groups,total_plm_codes,total_sap_materials
Mal grubu,,,
DENIM,15,36,16
DOKUMA,23,46,34
ORME,63,141,98


In [504]:
# Verify the multi-value characteristics we plan to use
important_mv_characteristics = [
    "FIBERCONTENTLISTID",
    "COLORINGID",
    "YARN1TYPEKNITSID"
]

available_mv_characteristics = set(
    mv_duplicate["characteristic"].unique()
)

for characteristic in important_mv_characteristics:
    print(
        characteristic,
        "FOUND" if characteristic in available_mv_characteristics
        else "NOT FOUND"
    )

FIBERCONTENTLISTID FOUND
COLORINGID FOUND
YARN1TYPEKNITSID FOUND


In [505]:
def build_mv_set_feature(
    mv_df,
    characteristic,
    output_column
):
    return (
        mv_df[
            mv_df["characteristic"]
            == characteristic
        ]
        .groupby("plm_code")["value"]
        .agg(
            lambda values:
                tuple(sorted(set(values)))
        )
        .rename(output_column)
        .reset_index()
    )


plm_fiber_feature = build_mv_set_feature(
    mv_duplicate,
    "FIBERCONTENTLISTID",
    "fiber_set"
)

plm_coloring_feature = build_mv_set_feature(
    mv_duplicate,
    "COLORINGID",
    "coloring_set"
)

plm_yarn_type_feature = build_mv_set_feature(
    mv_duplicate,
    "YARN1TYPEKNITSID",
    "yarn_type_set"
)

In [506]:
orme_near_duplicate_features = (
    plm_codes[
        plm_codes["Mal grubu"] == "ORME"
    ][
        [
            "PLM Kodu",
            "Kumaş tipi",
            "Örme alt tipi",
            "Kumaş ağırlığı",
            "Kumaş ağırlığı birimi",
            "1.İplik numarası örme",
            "2.İplik numarası örm",
            "3.İplik numarası örme",
            "Fine",
            "Pus",
            "İlmek uzunluğu/50 iğne"
        ]
    ]
    .copy()
)

orme_near_duplicate_features["plm_code"] = (
    normalize_plm_code_identity(
        orme_near_duplicate_features[
            "PLM Kodu"
        ]
    )
)

In [507]:
structured_text_columns = [
    "Kumaş tipi",
    "Örme alt tipi",
    "1.İplik numarası örme",
    "2.İplik numarası örm",
    "3.İplik numarası örme",
    "Fine",
    "Pus",
    "İlmek uzunluğu/50 iğne"
]

for column in structured_text_columns:
    orme_near_duplicate_features[column] = (
        normalize_duplicate_text(
            orme_near_duplicate_features[column]
        )
        .replace("<MISSING>", pd.NA)
    )

In [508]:
orme_near_duplicate_features[
    "weight"
] = pd.to_numeric(
    orme_near_duplicate_features[
        "Kumaş ağırlığı"
    ],
    errors="coerce"
)

orme_near_duplicate_features[
    "weight_unit"
] = (
    normalize_duplicate_text(
        orme_near_duplicate_features[
            "Kumaş ağırlığı birimi"
        ]
    )
    .replace("<MISSING>", pd.NA)
)

In [509]:
orme_near_duplicate_features[
    "yarn1_count"
] = (
    orme_near_duplicate_features[
        "1.İplik numarası örme"
    ]
    .apply(canonicalize_yarn_count)
)

In [510]:
orme_near_duplicate_features = (
    orme_near_duplicate_features
    .merge(
        plm_fiber_feature,
        on="plm_code",
        how="left",
        validate="one_to_one"
    )
    .merge(
        plm_coloring_feature,
        on="plm_code",
        how="left",
        validate="one_to_one"
    )
    .merge(
        plm_yarn_type_feature,
        on="plm_code",
        how="left",
        validate="one_to_one"
    )
)

assert (
    orme_near_duplicate_features[
        "plm_code"
    ].is_unique
)

print(
    "ORME PLM codes:",
    len(orme_near_duplicate_features)
)

ORME PLM codes: 12812


In [511]:
feature_coverage = pd.DataFrame({
    "feature": [
        "structure",
        "knit_subtype",
        "weight",
        "yarn1_count",
        "fine",
        "pus",
        "fiber",
        "coloring",
        "yarn_type"
    ],
    "coverage": [
        orme_near_duplicate_features[
            "Kumaş tipi"
        ].notna().mean(),

        orme_near_duplicate_features[
            "Örme alt tipi"
        ].notna().mean(),

        orme_near_duplicate_features[
            "weight"
        ].notna().mean(),

        orme_near_duplicate_features[
            "yarn1_count"
        ].notna().mean(),

        orme_near_duplicate_features[
            "Fine"
        ].notna().mean(),

        orme_near_duplicate_features[
            "Pus"
        ].notna().mean(),

        orme_near_duplicate_features[
            "fiber_set"
        ].notna().mean(),

        orme_near_duplicate_features[
            "coloring_set"
        ].notna().mean(),

        orme_near_duplicate_features[
            "yarn_type_set"
        ].notna().mean()
    ]
})

display(feature_coverage)

,feature,coverage
0,structure,0.999844
1,knit_subtype,0.831486
2,weight,0.976350
3,yarn1_count,0.781767
4,fine,0.812285
5,pus,0.543319
6,fiber,0.992351
7,coloring,0.782001
8,yarn_type,0.364268


In [512]:
def generate_plm_pair_channel(
    feature_df,
    blocking_columns,
    channel_name
):
    valid = (
        feature_df[
            ["plm_code"] + blocking_columns
        ]
        .dropna(
            subset=blocking_columns
        )
        .copy()
    )

    pairs = (
        valid
        .merge(
            valid,
            on=blocking_columns,
            how="inner",
            suffixes=("_left", "_right")
        )
    )

    # Keep each unordered pair only once
    pairs = pairs[
        pairs["plm_code_left"]
        < pairs["plm_code_right"]
    ].copy()

    pairs["candidate_channel"] = (
        channel_name
    )

    return pairs[
        [
            "plm_code_left",
            "plm_code_right",
            "candidate_channel"
        ]
    ]

In [513]:
structure_fiber_pairs = (
    generate_plm_pair_channel(
        orme_near_duplicate_features,
        [
            "Kumaş tipi",
            "fiber_set"
        ],
        "structure_fiber"
    )
)

structure_yarn_pairs = (
    generate_plm_pair_channel(
        orme_near_duplicate_features,
        [
            "Kumaş tipi",
            "yarn1_count"
        ],
        "structure_yarn"
    )
)

fiber_yarn_pairs = (
    generate_plm_pair_channel(
        orme_near_duplicate_features,
        [
            "fiber_set",
            "yarn1_count"
        ],
        "fiber_yarn"
    )
)

In [514]:
print(
    "Structure + fiber pairs:",
    len(structure_fiber_pairs)
)

print(
    "Structure + yarn pairs:",
    len(structure_yarn_pairs)
)

print(
    "Fiber + yarn pairs:",
    len(fiber_yarn_pairs)
)

Structure + fiber pairs: 355598
Structure + yarn pairs: 1907759
Fiber + yarn pairs: 498542


In [515]:
orme_near_duplicate_candidates = (
    pd.concat(
        [
            structure_fiber_pairs,
            structure_yarn_pairs,
            fiber_yarn_pairs
        ],
        ignore_index=True
    )
)

orme_candidate_channel_count = (
    orme_near_duplicate_candidates
    .groupby(
        [
            "plm_code_left",
            "plm_code_right"
        ]
    )
    .agg(
        blocking_channels=(
            "candidate_channel",
            "nunique"
        ),
        channel_names=(
            "candidate_channel",
            lambda x:
                tuple(sorted(set(x)))
        )
    )
    .reset_index()
)

print(
    "Unique ORME candidate pairs:",
    len(orme_candidate_channel_count)
)

display(
    orme_candidate_channel_count[
        "blocking_channels"
    ]
    .value_counts()
    .sort_index()
)

Unique ORME candidate pairs: 2580173


blocking_channels
1    2489310
3      90863
Name: count, dtype: int64

In [516]:
# Strong exact-profile groups for ORME
orme_known_duplicate_members = (
    profile_plm_members[
        (
            profile_plm_members["Mal grubu"] == "ORME"
        )
        &
        (
            profile_plm_members[
                "duplicate_evidence_class"
            ].isin([
                "STRONG_PROFILE_MATCH",
                "VERY_STRONG_PROFILE_MATCH"
            ])
        )
    ][
        [
            "duplicate_group_id",
            "plm_code"
        ]
    ]
    .drop_duplicates()
    .copy()
)

In [517]:
orme_known_duplicate_pairs = (
    orme_known_duplicate_members
    .merge(
        orme_known_duplicate_members,
        on="duplicate_group_id",
        suffixes=("_left", "_right")
    )
)

orme_known_duplicate_pairs = (
    orme_known_duplicate_pairs[
        orme_known_duplicate_pairs["plm_code_left"]
        < orme_known_duplicate_pairs["plm_code_right"]
    ]
    [
        [
            "duplicate_group_id",
            "plm_code_left",
            "plm_code_right"
        ]
    ]
    .drop_duplicates()
)

print(
    "Known strong exact duplicate pairs:",
    len(orme_known_duplicate_pairs)
)

Known strong exact duplicate pairs: 352


In [518]:
candidate_pair_keys = set(
    zip(
        orme_candidate_channel_count[
            "plm_code_left"
        ],
        orme_candidate_channel_count[
            "plm_code_right"
        ]
    )
)

orme_known_duplicate_pairs[
    "retrieved_by_current_blocking"
] = [
    (left, right) in candidate_pair_keys
    for left, right in zip(
        orme_known_duplicate_pairs[
            "plm_code_left"
        ],
        orme_known_duplicate_pairs[
            "plm_code_right"
        ]
    )
]

current_blocking_recall = (
    orme_known_duplicate_pairs[
        "retrieved_by_current_blocking"
    ].mean()
)

print(
    "Current blocking recall:",
    round(
        current_blocking_recall * 100,
        2
    ),
    "%"
)

Current blocking recall: 100.0 %


In [519]:
three_channel_candidates = (
    orme_candidate_channel_count[
        orme_candidate_channel_count[
            "blocking_channels"
        ] == 3
    ][
        [
            "plm_code_left",
            "plm_code_right"
        ]
    ]
)

three_channel_keys = set(
    zip(
        three_channel_candidates[
            "plm_code_left"
        ],
        three_channel_candidates[
            "plm_code_right"
        ]
    )
)

orme_known_duplicate_pairs[
    "retrieved_by_three_channels"
] = [
    (left, right) in three_channel_keys
    for left, right in zip(
        orme_known_duplicate_pairs[
            "plm_code_left"
        ],
        orme_known_duplicate_pairs[
            "plm_code_right"
        ]
    )
]

three_channel_recall = (
    orme_known_duplicate_pairs[
        "retrieved_by_three_channels"
    ].mean()
)

print(
    "Three-channel recall:",
    round(
        three_channel_recall * 100,
        2
    ),
    "%"
)

Three-channel recall: 95.17 %


In [520]:
orme_pair_features = (
    orme_candidate_channel_count
    .merge(
        orme_near_duplicate_features[
            [
                "plm_code",
                "weight"
            ]
        ].rename(
            columns={
                "plm_code": "plm_code_left",
                "weight": "weight_left"
            }
        ),
        on="plm_code_left",
        how="left",
        validate="many_to_one"
    )
    .merge(
        orme_near_duplicate_features[
            [
                "plm_code",
                "weight"
            ]
        ].rename(
            columns={
                "plm_code": "plm_code_right",
                "weight": "weight_right"
            }
        ),
        on="plm_code_right",
        how="left",
        validate="many_to_one"
    )
)

In [521]:
orme_pair_features[
    "weight_relative_diff"
] = (
    (
        orme_pair_features["weight_left"]
        -
        orme_pair_features["weight_right"]
    )
    .abs()
    /
    (
        (
            orme_pair_features["weight_left"]
            +
            orme_pair_features["weight_right"]
        )
        / 2
    )
)

In [522]:
orme_pair_features[
    "weight_within_5pct"
] = (
    orme_pair_features[
        "weight_relative_diff"
    ] <= 0.05
)

In [523]:
orme_pair_features[
    "weight_comparable"
] = (
    orme_pair_features[
        "weight_left"
    ].notna()
    &
    orme_pair_features[
        "weight_right"
    ].notna()
)

In [524]:
candidate_strategy_summary = []

# Strategy 1: current union
candidate_strategy_summary.append({
    "strategy": "CURRENT_UNION",
    "candidate_pairs":
        len(orme_pair_features)
})

# Strategy 2: all three core features agree
candidate_strategy_summary.append({
    "strategy": "ALL_THREE_CORE",
    "candidate_pairs":
        (
            orme_pair_features[
                "blocking_channels"
            ] == 3
        ).sum()
})

# Strategy 3: current union + weight within 5%
candidate_strategy_summary.append({
    "strategy": "UNION_WEIGHT_5PCT",
    "candidate_pairs":
        (
            orme_pair_features[
                "weight_within_5pct"
            ]
        ).sum()
})

# Strategy 4: all three core + weight within 5%
candidate_strategy_summary.append({
    "strategy": "ALL_THREE_WEIGHT_5PCT",
    "candidate_pairs":
        (
            (
                orme_pair_features[
                    "blocking_channels"
                ] == 3
            )
            &
            (
                orme_pair_features[
                    "weight_within_5pct"
                ]
            )
        ).sum()
})

candidate_strategy_summary = (
    pd.DataFrame(
        candidate_strategy_summary
    )
)

display(candidate_strategy_summary)

,strategy,candidate_pairs
0,CURRENT_UNION,2580173
1,ALL_THREE_CORE,90863
2,UNION_WEIGHT_5PCT,546389
3,ALL_THREE_WEIGHT_5PCT,30306


In [525]:
def calculate_known_pair_recall(
    candidates,
    known_pairs
):
    candidate_keys = set(
        zip(
            candidates["plm_code_left"],
            candidates["plm_code_right"]
        )
    )

    retrieved = [
        (left, right) in candidate_keys
        for left, right in zip(
            known_pairs["plm_code_left"],
            known_pairs["plm_code_right"]
        )
    ]

    return np.mean(retrieved)

In [526]:
def calculate_known_pair_recall(
    candidates,
    known_pairs
):
    candidate_keys = set(
        zip(
            candidates["plm_code_left"],
            candidates["plm_code_right"]
        )
    )

    retrieved = [
        (left, right) in candidate_keys
        for left, right in zip(
            known_pairs["plm_code_left"],
            known_pairs["plm_code_right"]
        )
    ]

    return np.mean(retrieved)

In [528]:
strategy_candidates = {
    "CURRENT_UNION":
        orme_pair_features,

    "ALL_THREE_CORE":
        orme_pair_features[
            orme_pair_features[
                "blocking_channels"
            ] == 3
        ],

    "UNION_WEIGHT_5PCT":
        orme_pair_features[
            orme_pair_features[
                "weight_within_5pct"
            ]
        ],

    "ALL_THREE_WEIGHT_5PCT":
        orme_pair_features[
            (
                orme_pair_features[
                    "blocking_channels"
                ] == 3
            )
            &
            (
                orme_pair_features[
                    "weight_within_5pct"
                ]
            )
        ]
}

In [529]:
strategy_results = []

for strategy_name, candidates in (
    strategy_candidates.items()
):
    recall = calculate_known_pair_recall(
        candidates,
        orme_known_duplicate_pairs
    )

    strategy_results.append({
        "strategy": strategy_name,
        "candidate_pairs": len(candidates),
        "known_duplicate_recall": recall
    })

strategy_results = pd.DataFrame(
    strategy_results
)

display(
    strategy_results
    .sort_values(
        "candidate_pairs"
    )
)

,strategy,candidate_pairs,known_duplicate_recall
3,ALL_THREE_WEIGHT_5PCT,30306,0.886364
1,ALL_THREE_CORE,90863,0.951705
2,UNION_WEIGHT_5PCT,546389,0.934659
0,CURRENT_UNION,2580173,1.000000


In [530]:
# Weight is acceptable when:
# 1) both values exist and are within 5%
# 2) weight cannot be compared because one/both values are missing

orme_pair_features["weight_missing_neutral"] = (
    orme_pair_features["weight_within_5pct"]
    |
    ~orme_pair_features["weight_comparable"]
)

missing_neutral_candidates = (
    orme_pair_features[
        orme_pair_features[
            "weight_missing_neutral"
        ]
    ]
    .copy()
)

print(
    "Candidate pairs:",
    len(missing_neutral_candidates)
)

print(
    "Known duplicate recall:",
    calculate_known_pair_recall(
        missing_neutral_candidates,
        orme_known_duplicate_pairs
    )
)

Candidate pairs: 569454
Known duplicate recall: 0.9346590909090909


In [531]:
weight_5pct_keys = set(
    zip(
        strategy_candidates[
            "UNION_WEIGHT_5PCT"
        ]["plm_code_left"],
        strategy_candidates[
            "UNION_WEIGHT_5PCT"
        ]["plm_code_right"]
    )
)

missed_by_weight = (
    orme_known_duplicate_pairs[
        [
            "duplicate_group_id",
            "plm_code_left",
            "plm_code_right"
        ]
    ]
    .copy()
)

missed_by_weight[
    "retrieved"
] = [
    (left, right) in weight_5pct_keys
    for left, right in zip(
        missed_by_weight["plm_code_left"],
        missed_by_weight["plm_code_right"]
    )
]

missed_by_weight = (
    missed_by_weight[
        ~missed_by_weight["retrieved"]
    ]
    .drop(columns="retrieved")
)

In [532]:
missed_by_weight = (
    missed_by_weight
    .merge(
        orme_near_duplicate_features[
            [
                "plm_code",
                "weight"
            ]
        ].rename(
            columns={
                "plm_code": "plm_code_left",
                "weight": "weight_left"
            }
        ),
        on="plm_code_left",
        how="left"
    )
    .merge(
        orme_near_duplicate_features[
            [
                "plm_code",
                "weight"
            ]
        ].rename(
            columns={
                "plm_code": "plm_code_right",
                "weight": "weight_right"
            }
        ),
        on="plm_code_right",
        how="left"
    )
)

display(
    missed_by_weight.head(30)
)

print(
    "Missed known pairs:",
    len(missed_by_weight)
)

print(
    "Both weights missing:",
    (
        missed_by_weight["weight_left"].isna()
        &
        missed_by_weight["weight_right"].isna()
    ).sum()
)

print(
    "Only one weight missing:",
    (
        missed_by_weight["weight_left"].isna()
        ^
        missed_by_weight["weight_right"].isna()
    ).sum()
)

print(
    "Both weights present:",
    (
        missed_by_weight["weight_left"].notna()
        &
        missed_by_weight["weight_right"].notna()
    ).sum()
)

,duplicate_group_id,plm_code_left,plm_code_right,weight_left,weight_right
0,PLM_DUP_00153,166237,354736,0.0,0.0
1,PLM_DUP_00153,166237,354737,0.0,0.0
2,PLM_DUP_00160,171640,172229,0.0,0.0
3,PLM_DUP_00311,202958,354685,0.0,0.0
4,PLM_DUP_00311,202958,354687,0.0,0.0
5,PLM_DUP_00654,213718,354366,0.0,0.0
6,PLM_DUP_00654,213718,354898,0.0,0.0
7,PLM_DUP_00090,23638,367599,0.0,0.0
8,PLM_DUP_00021,351605,355410,0.0,0.0
9,PLM_DUP_00228,351719,354896,0.0,0.0


Missed known pairs: 23
Both weights missing: 0
Only one weight missing: 0
Both weights present: 23


In [533]:
strategy_candidates_v2 = {
    "CURRENT_UNION":
        orme_pair_features,

    "ALL_THREE_CORE":
        orme_pair_features[
            orme_pair_features[
                "blocking_channels"
            ] == 3
        ],

    "UNION_WEIGHT_5PCT":
        orme_pair_features[
            orme_pair_features[
                "weight_within_5pct"
            ]
        ],

    "UNION_WEIGHT_5PCT_MISSING_NEUTRAL":
        orme_pair_features[
            orme_pair_features[
                "weight_missing_neutral"
            ]
        ],

    "ALL_THREE_OR_WEIGHT_NEUTRAL":
        orme_pair_features[
            (
                orme_pair_features[
                    "blocking_channels"
                ] == 3
            )
            |
            (
                orme_pair_features[
                    "weight_missing_neutral"
                ]
            )
        ]
}

In [534]:
strategy_results_v2 = []

for strategy_name, candidates in (
    strategy_candidates_v2.items()
):
    strategy_results_v2.append({
        "strategy":
            strategy_name,

        "candidate_pairs":
            len(candidates),

        "known_duplicate_recall":
            calculate_known_pair_recall(
                candidates,
                orme_known_duplicate_pairs
            )
    })

strategy_results_v2 = (
    pd.DataFrame(
        strategy_results_v2
    )
    .sort_values(
        "candidate_pairs"
    )
)

display(strategy_results_v2)

,strategy,candidate_pairs,known_duplicate_recall
1,ALL_THREE_CORE,90863,0.951705
2,UNION_WEIGHT_5PCT,546389,0.934659
3,UNION_WEIGHT_5PCT_MISSING_NEUTRAL,569454,0.934659
4,ALL_THREE_OR_WEIGHT_NEUTRAL,629984,1.000000
0,CURRENT_UNION,2580173,1.000000


In [535]:
# Preserve the raw numeric value for audit
orme_near_duplicate_features["weight_raw"] = (
    orme_near_duplicate_features["weight"]
)

# Zero or negative fabric weights are treated as unavailable
orme_near_duplicate_features["weight_clean"] = (
    orme_near_duplicate_features["weight_raw"]
    .where(
        orme_near_duplicate_features["weight_raw"] > 0,
        np.nan
    )
)

print(
    "Zero / negative weights converted to missing:",
    (
        orme_near_duplicate_features["weight_raw"]
        .notna()
        &
        orme_near_duplicate_features["weight_clean"]
        .isna()
    ).sum()
)

print(
    "Clean weight coverage:",
    round(
        orme_near_duplicate_features[
            "weight_clean"
        ].notna().mean() * 100,
        2
    ),
    "%"
)

Zero / negative weights converted to missing: 447
Clean weight coverage: 94.15 %


In [536]:
weight_lookup = (
    orme_near_duplicate_features[
        [
            "plm_code",
            "weight_clean",
            "weight_unit"
        ]
    ]
    .copy()
)

orme_pair_features_v2 = (
    orme_candidate_channel_count
    .merge(
        weight_lookup.rename(
            columns={
                "plm_code": "plm_code_left",
                "weight_clean": "weight_left",
                "weight_unit": "weight_unit_left"
            }
        ),
        on="plm_code_left",
        how="left",
        validate="many_to_one"
    )
    .merge(
        weight_lookup.rename(
            columns={
                "plm_code": "plm_code_right",
                "weight_clean": "weight_right",
                "weight_unit": "weight_unit_right"
            }
        ),
        on="plm_code_right",
        how="left",
        validate="many_to_one"
    )
)

In [537]:
orme_pair_features_v2["weight_comparable"] = (
    orme_pair_features_v2["weight_left"].notna()
    &
    orme_pair_features_v2["weight_right"].notna()
    &
    (
        orme_pair_features_v2["weight_unit_left"]
        ==
        orme_pair_features_v2["weight_unit_right"]
    )
)

In [538]:
orme_pair_features_v2["weight_relative_diff"] = np.nan

comparable_mask = (
    orme_pair_features_v2["weight_comparable"]
)

orme_pair_features_v2.loc[
    comparable_mask,
    "weight_relative_diff"
] = (
    (
        orme_pair_features_v2.loc[
            comparable_mask,
            "weight_left"
        ]
        -
        orme_pair_features_v2.loc[
            comparable_mask,
            "weight_right"
        ]
    )
    .abs()
    /
    (
        (
            orme_pair_features_v2.loc[
                comparable_mask,
                "weight_left"
            ]
            +
            orme_pair_features_v2.loc[
                comparable_mask,
                "weight_right"
            ]
        )
        / 2
    )
)

In [539]:
orme_pair_features_v2["weight_within_5pct"] = (
    orme_pair_features_v2[
        "weight_relative_diff"
    ] <= 0.05
)

In [540]:
orme_pair_features_v2["weight_missing_neutral"] = (
    ~orme_pair_features_v2["weight_comparable"]
    |
    orme_pair_features_v2["weight_within_5pct"]
)

In [541]:
strategy_candidates_v3 = {
    "CURRENT_UNION":
        orme_pair_features_v2,

    "ALL_THREE_CORE":
        orme_pair_features_v2[
            orme_pair_features_v2[
                "blocking_channels"
            ] == 3
        ],

    "WEIGHT_5PCT_OR_NOT_COMPARABLE":
        orme_pair_features_v2[
            orme_pair_features_v2[
                "weight_missing_neutral"
            ]
        ],

    "ALL_THREE_OR_WEIGHT_NEUTRAL":
        orme_pair_features_v2[
            (
                orme_pair_features_v2[
                    "blocking_channels"
                ] == 3
            )
            |
            (
                orme_pair_features_v2[
                    "weight_missing_neutral"
                ]
            )
        ]
}

In [542]:
strategy_results_v3 = []

for strategy_name, candidates in (
    strategy_candidates_v3.items()
):
    strategy_results_v3.append({
        "strategy": strategy_name,
        "candidate_pairs": len(candidates),
        "known_duplicate_recall":
            calculate_known_pair_recall(
                candidates,
                orme_known_duplicate_pairs
            )
    })

strategy_results_v3 = (
    pd.DataFrame(strategy_results_v3)
    .sort_values("candidate_pairs")
)

display(strategy_results_v3)

,strategy,candidate_pairs,known_duplicate_recall
1,ALL_THREE_CORE,90863,0.951705
2,WEIGHT_5PCT_OR_NOT_COMPARABLE,594936,0.971591
3,ALL_THREE_OR_WEIGHT_NEUTRAL,656465,1.000000
0,CURRENT_UNION,2580173,1.000000


In [543]:
weight_neutral_candidates = (
    strategy_candidates_v3[
        "WEIGHT_5PCT_OR_NOT_COMPARABLE"
    ]
)

weight_neutral_keys = set(
    zip(
        weight_neutral_candidates["plm_code_left"],
        weight_neutral_candidates["plm_code_right"]
    )
)

missed_known_pairs_v3 = (
    orme_known_duplicate_pairs.copy()
)

missed_known_pairs_v3["retrieved"] = [
    (left, right) in weight_neutral_keys
    for left, right in zip(
        missed_known_pairs_v3["plm_code_left"],
        missed_known_pairs_v3["plm_code_right"]
    )
]

missed_known_pairs_v3 = (
    missed_known_pairs_v3[
        ~missed_known_pairs_v3["retrieved"]
    ]
    .drop(columns="retrieved")
)

print(
    "Missed known duplicate pairs:",
    len(missed_known_pairs_v3)
)

Missed known duplicate pairs: 10


In [544]:
missed_weight_diagnostic = (
    missed_known_pairs_v3
    .merge(
        orme_pair_features_v2[
            [
                "plm_code_left",
                "plm_code_right",
                "blocking_channels",
                "weight_left",
                "weight_right",
                "weight_unit_left",
                "weight_unit_right",
                "weight_comparable",
                "weight_relative_diff",
                "weight_within_5pct",
                "weight_missing_neutral"
            ]
        ],
        on=[
            "plm_code_left",
            "plm_code_right"
        ],
        how="left",
        validate="one_to_one"
    )
)

display(missed_weight_diagnostic)

,duplicate_group_id,plm_code_left,plm_code_right,retrieved_by_current_blocking,retrieved_by_three_channels,blocking_channels,weight_left,weight_right,weight_unit_left,weight_unit_right,weight_comparable,weight_relative_diff,weight_within_5pct,weight_missing_neutral
0,PLM_DUP_00632,131423,358214,True,True,3,200.0,200.0,<NA>,<NA>,<NA>,NaN,False,<NA>
1,PLM_DUP_00632,131423,358215,True,True,3,200.0,200.0,<NA>,<NA>,<NA>,NaN,False,<NA>
2,PLM_DUP_00632,131423,358216,True,True,3,200.0,200.0,<NA>,<NA>,<NA>,NaN,False,<NA>
3,PLM_DUP_00529,131452,357240,True,True,3,200.0,200.0,<NA>,<NA>,<NA>,NaN,False,<NA>
4,PLM_DUP_00494,28606,358226,True,True,3,210.0,210.0,<NA>,<NA>,<NA>,NaN,False,<NA>
5,PLM_DUP_00535,356712,98197,True,True,3,280.0,280.0,<NA>,<NA>,<NA>,NaN,False,<NA>
6,PLM_DUP_00632,358214,358215,True,True,3,200.0,200.0,<NA>,<NA>,<NA>,NaN,False,<NA>
7,PLM_DUP_00632,358214,358216,True,True,3,200.0,200.0,<NA>,<NA>,<NA>,NaN,False,<NA>
8,PLM_DUP_00632,358215,358216,True,True,3,200.0,200.0,<NA>,<NA>,<NA>,NaN,False,<NA>
9,PLM_DUP_00064,358344,66906,True,True,3,180.0,180.0,<NA>,<NA>,<NA>,NaN,False,<NA>


In [545]:
raw_weight_lookup = (
    plm_codes[
        [
            "PLM Kodu",
            "Kumaş ağırlığı",
            "Kumaş ağırlığı birimi"
        ]
    ]
    .copy()
)

raw_weight_lookup["plm_code"] = (
    normalize_plm_code_identity(
        raw_weight_lookup["PLM Kodu"]
    )
)

In [546]:
missed_weight_diagnostic = (
    missed_weight_diagnostic
    .merge(
        raw_weight_lookup[
            [
                "plm_code",
                "Kumaş ağırlığı",
                "Kumaş ağırlığı birimi"
            ]
        ].rename(
            columns={
                "plm_code": "plm_code_left",
                "Kumaş ağırlığı": "raw_weight_left",
                "Kumaş ağırlığı birimi": "raw_unit_left"
            }
        ),
        on="plm_code_left",
        how="left",
        validate="many_to_one"
    )
    .merge(
        raw_weight_lookup[
            [
                "plm_code",
                "Kumaş ağırlığı",
                "Kumaş ağırlığı birimi"
            ]
        ].rename(
            columns={
                "plm_code": "plm_code_right",
                "Kumaş ağırlığı": "raw_weight_right",
                "Kumaş ağırlığı birimi": "raw_unit_right"
            }
        ),
        on="plm_code_right",
        how="left",
        validate="many_to_one"
    )
)

display(
    missed_weight_diagnostic[
        [
            "duplicate_group_id",
            "plm_code_left",
            "plm_code_right",
            "raw_weight_left",
            "raw_weight_right",
            "raw_unit_left",
            "raw_unit_right",
            "weight_left",
            "weight_right",
            "weight_comparable",
            "weight_relative_diff"
        ]
    ]
)

,duplicate_group_id,plm_code_left,plm_code_right,raw_weight_left,raw_weight_right,raw_unit_left,raw_unit_right,weight_left,weight_right,weight_comparable,weight_relative_diff
0,PLM_DUP_00632,131423,358214,200.0,200.0,NaN,NaN,200.0,200.0,<NA>,NaN
1,PLM_DUP_00632,131423,358215,200.0,200.0,NaN,NaN,200.0,200.0,<NA>,NaN
2,PLM_DUP_00632,131423,358216,200.0,200.0,NaN,NaN,200.0,200.0,<NA>,NaN
3,PLM_DUP_00529,131452,357240,200.0,200.0,NaN,NaN,200.0,200.0,<NA>,NaN
4,PLM_DUP_00494,28606,358226,210.0,210.0,NaN,NaN,210.0,210.0,<NA>,NaN
5,PLM_DUP_00535,356712,98197,280.0,280.0,NaN,NaN,280.0,280.0,<NA>,NaN
6,PLM_DUP_00632,358214,358215,200.0,200.0,NaN,NaN,200.0,200.0,<NA>,NaN
7,PLM_DUP_00632,358214,358216,200.0,200.0,NaN,NaN,200.0,200.0,<NA>,NaN
8,PLM_DUP_00632,358215,358216,200.0,200.0,NaN,NaN,200.0,200.0,<NA>,NaN
9,PLM_DUP_00064,358344,66906,180.0,180.0,NaN,NaN,180.0,180.0,<NA>,NaN


In [547]:
orme_final_candidate_pairs = (
    orme_pair_features_v2[
        (
            orme_pair_features_v2[
                "blocking_channels"
            ] == 3
        )
        |
        (
            orme_pair_features_v2[
                "weight_missing_neutral"
            ]
        )
    ]
    .copy()
)

assert (
    calculate_known_pair_recall(
        orme_final_candidate_pairs,
        orme_known_duplicate_pairs
    )
    == 1.0
)

print(
    "Final ORME candidate pairs:",
    len(orme_final_candidate_pairs)
)

Final ORME candidate pairs: 656465


In [548]:
# Explicit unit agreement
same_nonmissing_unit = (
    orme_pair_features_v2["weight_unit_left"].notna()
    &
    orme_pair_features_v2["weight_unit_right"].notna()
    &
    orme_pair_features_v2["weight_unit_left"]
        .eq(
            orme_pair_features_v2[
                "weight_unit_right"
            ]
        )
        .fillna(False)
)

# Both units missing:
# numeric weights can still be compared within the same PLM group,
# but this will later be treated as lower-quality evidence.
both_units_missing = (
    orme_pair_features_v2[
        "weight_unit_left"
    ].isna()
    &
    orme_pair_features_v2[
        "weight_unit_right"
    ].isna()
)

orme_pair_features_v2[
    "weight_units_compatible"
] = (
    same_nonmissing_unit
    |
    both_units_missing
)

In [549]:
orme_pair_features_v2[
    "weight_comparable"
] = (
    orme_pair_features_v2[
        "weight_left"
    ].notna()
    &
    orme_pair_features_v2[
        "weight_right"
    ].notna()
    &
    orme_pair_features_v2[
        "weight_units_compatible"
    ]
)

In [550]:
orme_pair_features_v2[
    "weight_evidence_type"
] = np.select(
    [
        same_nonmissing_unit,
        both_units_missing
    ],
    [
        "EXPLICIT_UNIT",
        "IMPLICIT_UNIT_BOTH_MISSING"
    ],
    default="NOT_COMPARABLE"
)

In [551]:
orme_pair_features_v2[
    "weight_relative_diff"
] = np.nan

comparable_mask = (
    orme_pair_features_v2[
        "weight_comparable"
    ]
)

orme_pair_features_v2.loc[
    comparable_mask,
    "weight_relative_diff"
] = (
    (
        orme_pair_features_v2.loc[
            comparable_mask,
            "weight_left"
        ]
        -
        orme_pair_features_v2.loc[
            comparable_mask,
            "weight_right"
        ]
    )
    .abs()
    /
    (
        (
            orme_pair_features_v2.loc[
                comparable_mask,
                "weight_left"
            ]
            +
            orme_pair_features_v2.loc[
                comparable_mask,
                "weight_right"
            ]
        )
        / 2
    )
)

In [552]:
orme_pair_features_v2[
    "weight_within_5pct"
] = (
    orme_pair_features_v2[
        "weight_relative_diff"
    ] <= 0.05
)

In [553]:
orme_pair_features_v2[
    "weight_missing_neutral"
] = (
    ~orme_pair_features_v2[
        "weight_comparable"
    ]
    |
    orme_pair_features_v2[
        "weight_within_5pct"
    ]
)

In [554]:
print(
    "NA in weight_comparable:",
    orme_pair_features_v2[
        "weight_comparable"
    ].isna().sum()
)

print(
    "NA in weight_missing_neutral:",
    orme_pair_features_v2[
        "weight_missing_neutral"
    ].isna().sum()
)

NA in weight_comparable: 0
NA in weight_missing_neutral: 0


In [555]:
strategy_candidates_v4 = {
    "CURRENT_UNION":
        orme_pair_features_v2,

    "ALL_THREE_CORE":
        orme_pair_features_v2[
            orme_pair_features_v2[
                "blocking_channels"
            ] == 3
        ],

    "WEIGHT_5PCT_OR_NOT_COMPARABLE":
        orme_pair_features_v2[
            orme_pair_features_v2[
                "weight_missing_neutral"
            ]
        ],

    "ALL_THREE_OR_WEIGHT_NEUTRAL":
        orme_pair_features_v2[
            (
                orme_pair_features_v2[
                    "blocking_channels"
                ] == 3
            )
            |
            (
                orme_pair_features_v2[
                    "weight_missing_neutral"
                ]
            )
        ]
}

In [556]:
strategy_results_v4 = []

for strategy_name, candidates in (
    strategy_candidates_v4.items()
):
    strategy_results_v4.append({
        "strategy":
            strategy_name,

        "candidate_pairs":
            len(candidates),

        "known_duplicate_recall":
            calculate_known_pair_recall(
                candidates,
                orme_known_duplicate_pairs
            )
    })

strategy_results_v4 = (
    pd.DataFrame(
        strategy_results_v4
    )
    .sort_values(
        "candidate_pairs"
    )
)

display(strategy_results_v4)

,strategy,candidate_pairs,known_duplicate_recall
1,ALL_THREE_CORE,90863,0.951705
2,WEIGHT_5PCT_OR_NOT_COMPARABLE,671933,1.000000
3,ALL_THREE_OR_WEIGHT_NEUTRAL,729723,1.000000
0,CURRENT_UNION,2580173,1.000000


In [557]:
display(
    orme_pair_features_v2[
        orme_pair_features_v2[
            "plm_code_left"
        ].isin(
            missed_known_pairs_v3[
                "plm_code_left"
            ]
        )
        &
        orme_pair_features_v2[
            "plm_code_right"
        ].isin(
            missed_known_pairs_v3[
                "plm_code_right"
            ]
        )
    ][
        [
            "plm_code_left",
            "plm_code_right",
            "weight_left",
            "weight_right",
            "weight_evidence_type",
            "weight_relative_diff",
            "weight_within_5pct"
        ]
    ]
    .head(20)
)

,plm_code_left,plm_code_right,weight_left,weight_right,weight_evidence_type,weight_relative_diff,weight_within_5pct
376321,131423,358214,200.0,200.0,IMPLICIT_UNIT_BOTH_MISSING,0.00000,True
376322,131423,358215,200.0,200.0,IMPLICIT_UNIT_BOTH_MISSING,0.00000,True
376323,131423,358216,200.0,200.0,IMPLICIT_UNIT_BOTH_MISSING,0.00000,True
386465,131452,357240,200.0,200.0,IMPLICIT_UNIT_BOTH_MISSING,0.00000,True
386525,131452,358226,200.0,210.0,IMPLICIT_UNIT_BOTH_MISSING,0.04878,True
1455947,28606,357240,210.0,200.0,IMPLICIT_UNIT_BOTH_MISSING,0.04878,True
1456008,28606,358226,210.0,210.0,IMPLICIT_UNIT_BOTH_MISSING,0.00000,True
1956784,356712,98197,280.0,280.0,IMPLICIT_UNIT_BOTH_MISSING,0.00000,True
2114103,358214,358215,200.0,200.0,IMPLICIT_UNIT_BOTH_MISSING,0.00000,True
2114104,358214,358216,200.0,200.0,IMPLICIT_UNIT_BOTH_MISSING,0.00000,True


In [558]:
orme_final_candidate_pairs = (
    orme_pair_features_v2[
        orme_pair_features_v2[
            "weight_missing_neutral"
        ]
    ]
    .copy()
)

final_blocking_recall = calculate_known_pair_recall(
    orme_final_candidate_pairs,
    orme_known_duplicate_pairs
)

print(
    "Final ORME candidate pairs:",
    len(orme_final_candidate_pairs)
)

print(
    "Known exact-positive recall:",
    round(final_blocking_recall * 100, 2),
    "%"
)

assert final_blocking_recall == 1.0

Final ORME candidate pairs: 671933
Known exact-positive recall: 100.0 %


In [560]:
orme_feature_lookup = (
    orme_near_duplicate_features
    .drop_duplicates("plm_code")
    .set_index("plm_code")
)

assert orme_feature_lookup.index.is_unique

In [561]:
def add_exact_pair_feature(
    pair_df,
    lookup,
    source_column,
    output_column
):
    left = (
        pair_df["plm_code_left"]
        .map(lookup[source_column])
    )

    right = (
        pair_df["plm_code_right"]
        .map(lookup[source_column])
    )

    comparable = (
        left.notna()
        &
        right.notna()
    )

    match = (
        comparable
        &
        left.eq(right).fillna(False)
    )

    pair_df[f"{output_column}_comparable"] = comparable
    pair_df[output_column] = np.where(
        comparable,
        match.astype(float),
        np.nan
    )

In [562]:
def safe_set_jaccard(left_value, right_value):
    valid_types = (
        tuple,
        list,
        set,
        frozenset
    )

    if not isinstance(left_value, valid_types):
        return np.nan

    if not isinstance(right_value, valid_types):
        return np.nan

    left_set = set(left_value)
    right_set = set(right_value)

    if not left_set or not right_set:
        return np.nan

    union = left_set | right_set

    if not union:
        return np.nan

    return (
        len(left_set & right_set)
        / len(union)
    )

In [563]:
orme_pair_scores = (
    orme_final_candidate_pairs[
        [
            "plm_code_left",
            "plm_code_right",
            "blocking_channels",
            "weight_left",
            "weight_right",
            "weight_evidence_type",
            "weight_relative_diff",
            "weight_comparable"
        ]
    ]
    .copy()
)

In [564]:
add_exact_pair_feature(
    orme_pair_scores,
    orme_feature_lookup,
    "Kumaş tipi",
    "structure_match"
)

In [565]:
add_exact_pair_feature(
    orme_pair_scores,
    orme_feature_lookup,
    "Örme alt tipi",
    "knit_subtype_match"
)

In [567]:
print(
    orme_feature_lookup[
        "yarn1_count"
    ]
    .map(type)
    .value_counts()
)

display(
    orme_feature_lookup[
        "yarn1_count"
    ]
    .dropna()
    .head(20)
)

yarn1_count
<class 'str'>                            10016
<class 'pandas._libs.missing.NAType'>     2796
Name: count, dtype: int64


plm_code
100000001        30
100000002        30
100000015        20
100000026        40
100000027        40
100000028        30
100000080        75
100000104        75
100000110        40
100000112    200/40
100000118        30
100000120        30
100000127        30
100000204        30
100000205        30
100000206        20
100000208        30
100000209        30
100000210        30
100000211        70
Name: yarn1_count, dtype: object

In [568]:
def add_exact_pair_feature(
    pair_df,
    lookup,
    source_column,
    output_column
):
    left = (
        pair_df["plm_code_left"]
        .map(lookup[source_column])
        .astype("string")
    )

    right = (
        pair_df["plm_code_right"]
        .map(lookup[source_column])
        .astype("string")
    )

    comparable = (
        left.notna()
        &
        right.notna()
    )

    equal = (
        left.eq(right)
        .fillna(False)
    )

    pair_df[
        f"{output_column}_comparable"
    ] = comparable.astype(bool)

    pair_df[
        output_column
    ] = np.where(
        comparable,
        equal.astype(float),
        np.nan
    )

In [569]:
add_exact_pair_feature(
    orme_pair_scores,
    orme_feature_lookup,
    "yarn1_count",
    "yarn1_count_match"
)

In [570]:
print(
    "Comparable yarn1 pairs:",
    orme_pair_scores[
        "yarn1_count_match_comparable"
    ].sum()
)

print(
    "Matching yarn1 pairs:",
    (
        orme_pair_scores[
            "yarn1_count_match"
        ] == 1
    ).sum()
)

print(
    "Mismatching yarn1 pairs:",
    (
        orme_pair_scores[
            "yarn1_count_match"
        ] == 0
    ).sum()
)

print(
    "Not comparable:",
    orme_pair_scores[
        "yarn1_count_match"
    ].isna().sum()
)

Comparable yarn1 pairs: 630605
Matching yarn1 pairs: 608422
Mismatching yarn1 pairs: 22183
Not comparable: 41328


In [571]:
import re

fiber_aliases = {
    "POLIESTER": "POLYESTER",
    "ELASTHANE/SPANDEX": "ELASTANE",
    "SPANDEX": "ELASTANE",
    "POLYAMIDE6": "POLYAMIDE",
    "POLIAMID6": "POLYAMIDE",
    "TENCEL": "LYOCELL",
    "ACETAT": "ACETATE"
}


def normalize_fiber_name(value):
    value = str(value).strip().upper()
    return fiber_aliases.get(value, value)


def parse_plm_fiber_value(value):
    if value is None or value is pd.NA:
        return None

    text = str(value).strip()

    match = re.match(
        r"^\s*(\d+(?:\.\d+)?)\s+(.+?)\s*$",
        text
    )

    if not match:
        return None

    percentage = float(match.group(1))
    fiber_name = normalize_fiber_name(
        match.group(2)
    )

    return percentage, fiber_name

In [572]:
fiber_mv_rows = (
    mv_duplicate[
        mv_duplicate["characteristic"]
        == "FIBERCONTENTLISTID"
    ][
        [
            "plm_code",
            "value"
        ]
    ]
    .copy()
)

fiber_composition_records = []

for plm_code, group in fiber_mv_rows.groupby(
    "plm_code"
):
    composition = {}

    for value in group["value"]:
        parsed = parse_plm_fiber_value(
            value
        )

        if parsed is None:
            continue

        percentage, fiber_name = parsed

        composition[fiber_name] = (
            composition.get(
                fiber_name,
                0.0
            )
            + percentage
        )

    total_percentage = sum(
        composition.values()
    )

    # Keep a reliability flag instead of silently
    # accepting abnormal composition totals.
    reliable = (
        95 <= total_percentage <= 105
    )

    if reliable:
        normalized_composition = {
            fiber:
                percentage
                / total_percentage
            for fiber, percentage
            in composition.items()
        }

        fiber_name_set = tuple(
            sorted(
                normalized_composition.keys()
            )
        )
    else:
        normalized_composition = None
        fiber_name_set = None

    fiber_composition_records.append({
        "plm_code":
            plm_code,

        "fiber_percentage_total":
            total_percentage,

        "fiber_composition_reliable":
            reliable,

        "fiber_name_set":
            fiber_name_set,

        "fiber_composition":
            normalized_composition
    })


plm_fiber_composition_feature = (
    pd.DataFrame(
        fiber_composition_records
    )
)

In [573]:
print(
    "PLMs with parsed fiber:",
    len(
        plm_fiber_composition_feature
    )
)

print(
    "Reliable compositions:",
    plm_fiber_composition_feature[
        "fiber_composition_reliable"
    ].sum()
)

display(
    plm_fiber_composition_feature[
        "fiber_percentage_total"
    ].describe()
)

PLMs with parsed fiber: 31834
Reliable compositions: 31821


count    31834.000000
mean       100.021518
std          2.806321
min         42.000000
25%        100.000000
50%        100.000000
75%        100.000000
max        536.500000
Name: fiber_percentage_total, dtype: float64

In [574]:
def set_jaccard(left, right):
    valid_types = (
        tuple,
        list,
        set,
        frozenset
    )

    if not isinstance(
        left,
        valid_types
    ):
        return np.nan

    if not isinstance(
        right,
        valid_types
    ):
        return np.nan

    left = set(left)
    right = set(right)

    if not left or not right:
        return np.nan

    return (
        len(left & right)
        /
        len(left | right)
    )

In [575]:
def fiber_composition_similarity(
    left,
    right
):
    if not isinstance(left, dict):
        return np.nan

    if not isinstance(right, dict):
        return np.nan

    fibers = (
        set(left)
        |
        set(right)
    )

    if not fibers:
        return np.nan

    l1_distance = sum(
        abs(
            left.get(fiber, 0.0)
            -
            right.get(fiber, 0.0)
        )
        for fiber in fibers
    )

    similarity = (
        1
        -
        0.5 * l1_distance
    )

    return max(
        0.0,
        min(1.0, similarity)
    )

In [576]:
fiber_composition_lookup = (
    plm_fiber_composition_feature
    .set_index("plm_code")
)

assert (
    fiber_composition_lookup
    .index
    .is_unique
)

In [577]:
fiber_names_left = (
    orme_pair_scores[
        "plm_code_left"
    ]
    .map(
        fiber_composition_lookup[
            "fiber_name_set"
        ]
    )
)

fiber_names_right = (
    orme_pair_scores[
        "plm_code_right"
    ]
    .map(
        fiber_composition_lookup[
            "fiber_name_set"
        ]
    )
)

orme_pair_scores[
    "fiber_name_similarity"
] = [
    set_jaccard(left, right)
    for left, right in zip(
        fiber_names_left,
        fiber_names_right
    )
]

In [578]:
fiber_comp_left = (
    orme_pair_scores[
        "plm_code_left"
    ]
    .map(
        fiber_composition_lookup[
            "fiber_composition"
        ]
    )
)

fiber_comp_right = (
    orme_pair_scores[
        "plm_code_right"
    ]
    .map(
        fiber_composition_lookup[
            "fiber_composition"
        ]
    )
)

orme_pair_scores[
    "fiber_composition_similarity"
] = [
    fiber_composition_similarity(
        left,
        right
    )
    for left, right in zip(
        fiber_comp_left,
        fiber_comp_right
    )
]

In [579]:
print(
    "Comparable fiber-name pairs:",
    orme_pair_scores[
        "fiber_name_similarity"
    ].notna().sum()
)

print(
    "Comparable fiber-composition pairs:",
    orme_pair_scores[
        "fiber_composition_similarity"
    ].notna().sum()
)

print("\nFiber-name similarity:")
display(
    orme_pair_scores[
        "fiber_name_similarity"
    ].describe()
)

print("\nFiber-composition similarity:")
display(
    orme_pair_scores[
        "fiber_composition_similarity"
    ].describe()
)

Comparable fiber-name pairs: 670410
Comparable fiber-composition pairs: 670410

Fiber-name similarity:


count    670410.000000
mean          0.691341
std           0.314954
min           0.000000
25%           0.500000
50%           0.666667
75%           1.000000
max           1.000000
Name: fiber_name_similarity, dtype: float64


Fiber-composition similarity:


count    670410.000000
mean          0.697004
std           0.310249
min           0.000000
25%           0.500000
50%           0.780000
75%           1.000000
max           1.000000
Name: fiber_composition_similarity, dtype: float64

In [581]:
orme_pair_scores["weight_diff_pct"] = (
    orme_pair_scores["weight_relative_diff"] * 100
)

print(
    "weight_diff_pct exists:",
    "weight_diff_pct" in orme_pair_scores.columns
)

weight_diff_pct exists: True


In [582]:
same_fiber_names = (
    orme_pair_scores[
        "fiber_name_similarity"
    ] == 1
)

different_composition = (
    orme_pair_scores[
        "fiber_composition_similarity"
    ] < 0.90
)

same_names_but_different_percentages = (
    orme_pair_scores[
        same_fiber_names
        &
        different_composition
    ]
)

print(
    "Same fiber names but composition < 0.90:",
    len(
        same_names_but_different_percentages
    )
)

display(
    same_names_but_different_percentages[
        [
            "plm_code_left",
            "plm_code_right",
            "fiber_name_similarity",
            "fiber_composition_similarity",
            "yarn1_count_match",
            "weight_diff_pct"
        ]
    ]
    .head(30)
)

Same fiber names but composition < 0.90: 79674


,plm_code_left,plm_code_right,fiber_name_similarity,fiber_composition_similarity,yarn1_count_match,weight_diff_pct
60,100000001,135065,1.0,0.62,1.0,3.278689
113,100000001,191364,1.0,0.84,1.0,3.278689
202,100000001,212385,1.0,0.81,1.0,3.278689
215,100000001,213473,1.0,0.88,1.0,3.174603
233,100000001,214087,1.0,0.64,1.0,3.174603
293,100000001,353722,1.0,0.82,1.0,4.958678
316,100000001,355005,1.0,0.61,1.0,3.278689
372,100000001,357000,1.0,0.62,1.0,3.278689
376,100000001,357246,1.0,0.77,1.0,4.958678
394,100000001,358230,1.0,0.90,1.0,3.278689


In [584]:
display(
    orme_pair_scores.loc[
        same_fiber_names,
        "fiber_composition_similarity"
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

count    299796.000000
mean          0.919325
std           0.136492
min           0.070000
1%            0.450000
5%            0.600000
10%           0.710000
25%           0.880000
50%           1.000000
75%           1.000000
90%           1.000000
95%           1.000000
99%           1.000000
max           1.000000
Name: fiber_composition_similarity, dtype: float64

In [585]:
fiber_composition_threshold_summary = pd.DataFrame({
    "condition": [
        "same fiber names",
        "composition >= 0.99",
        "composition >= 0.95",
        "composition >= 0.90",
        "composition < 0.90",
        "composition < 0.80",
        "composition < 0.70"
    ],
    "pair_count": [
        same_fiber_names.sum(),

        (
            same_fiber_names
            &
            (
                orme_pair_scores[
                    "fiber_composition_similarity"
                ] >= 0.99
            )
        ).sum(),

        (
            same_fiber_names
            &
            (
                orme_pair_scores[
                    "fiber_composition_similarity"
                ] >= 0.95
            )
        ).sum(),

        (
            same_fiber_names
            &
            (
                orme_pair_scores[
                    "fiber_composition_similarity"
                ] >= 0.90
            )
        ).sum(),

        (
            same_fiber_names
            &
            (
                orme_pair_scores[
                    "fiber_composition_similarity"
                ] < 0.90
            )
        ).sum(),

        (
            same_fiber_names
            &
            (
                orme_pair_scores[
                    "fiber_composition_similarity"
                ] < 0.80
            )
        ).sum(),

        (
            same_fiber_names
            &
            (
                orme_pair_scores[
                    "fiber_composition_similarity"
                ] < 0.70
            )
        ).sum()
    ]
})

display(fiber_composition_threshold_summary)

,condition,pair_count
0,same fiber names,299796
1,composition >= 0.99,179395
2,composition >= 0.95,204715
3,composition >= 0.90,220122
4,composition < 0.90,79674
5,composition < 0.80,50326
6,composition < 0.70,25112


In [586]:
orme_pair_scores["fiber_composition_band"] = pd.cut(
    orme_pair_scores["fiber_composition_similarity"],
    bins=[
        -np.inf,
        0.70,
        0.80,
        0.90,
        0.95,
        0.99,
        np.inf
    ],
    labels=[
        "VERY_STRONG_DISAGREEMENT",
        "STRONG_DISAGREEMENT",
        "MODERATE_DISAGREEMENT",
        "WEAK_AGREEMENT",
        "STRONG_AGREEMENT",
        "VERY_STRONG_AGREEMENT"
    ],
    right=False
)

In [588]:
add_exact_pair_feature(
    orme_pair_scores,
    orme_feature_lookup,
    "Örme alt tipi",
    "knit_subtype_match"
)

add_exact_pair_feature(
    orme_pair_scores,
    orme_feature_lookup,
    "Fine",
    "fine_match"
)

add_exact_pair_feature(
    orme_pair_scores,
    orme_feature_lookup,
    "Pus",
    "pus_match"
)

In [589]:
coloring_left = (
    orme_pair_scores["plm_code_left"]
    .map(
        orme_feature_lookup[
            "coloring_set"
        ]
    )
)

coloring_right = (
    orme_pair_scores["plm_code_right"]
    .map(
        orme_feature_lookup[
            "coloring_set"
        ]
    )
)

orme_pair_scores[
    "coloring_similarity"
] = [
    safe_set_jaccard(left, right)
    for left, right in zip(
        coloring_left,
        coloring_right
    )
]

In [590]:
yarn_type_left = (
    orme_pair_scores["plm_code_left"]
    .map(
        orme_feature_lookup[
            "yarn_type_set"
        ]
    )
)

yarn_type_right = (
    orme_pair_scores["plm_code_right"]
    .map(
        orme_feature_lookup[
            "yarn_type_set"
        ]
    )
)

orme_pair_scores[
    "yarn_type_similarity"
] = [
    safe_set_jaccard(left, right)
    for left, right in zip(
        yarn_type_left,
        yarn_type_right
    )
]

In [591]:
orme_pair_feature_coverage = pd.DataFrame({
    "feature": [
        "fiber_composition_similarity",
        "yarn1_count_match",
        "knit_subtype_match",
        "fine_match",
        "pus_match",
        "coloring_similarity",
        "yarn_type_similarity",
        "weight_relative_diff"
    ],
    "coverage": [
        orme_pair_scores[
            "fiber_composition_similarity"
        ].notna().mean(),

        orme_pair_scores[
            "yarn1_count_match"
        ].notna().mean(),

        orme_pair_scores[
            "knit_subtype_match"
        ].notna().mean(),

        orme_pair_scores[
            "fine_match"
        ].notna().mean(),

        orme_pair_scores[
            "pus_match"
        ].notna().mean(),

        orme_pair_scores[
            "coloring_similarity"
        ].notna().mean(),

        orme_pair_scores[
            "yarn_type_similarity"
        ].notna().mean(),

        orme_pair_scores[
            "weight_relative_diff"
        ].notna().mean()
    ]
})

display(
    orme_pair_feature_coverage
    .sort_values(
        "coverage",
        ascending=False
    )
)

,feature,coverage
0,fiber_composition_similarity,0.997733
1,yarn1_count_match,0.938494
3,fine_match,0.884633
2,knit_subtype_match,0.877861
7,weight_relative_diff,0.786825
5,coloring_similarity,0.657939
4,pus_match,0.334475
6,yarn_type_similarity,0.184045


In [592]:
known_exact_pair_index = (
    pd.MultiIndex.from_frame(
        orme_known_duplicate_pairs[
            [
                "plm_code_left",
                "plm_code_right"
            ]
        ]
    )
)

candidate_pair_index = (
    pd.MultiIndex.from_frame(
        orme_pair_scores[
            [
                "plm_code_left",
                "plm_code_right"
            ]
        ]
    )
)

orme_pair_scores[
    "known_exact_positive"
] = (
    candidate_pair_index.isin(
        known_exact_pair_index
    )
)

print(
    orme_pair_scores[
        "known_exact_positive"
    ].value_counts()
)

known_exact_positive
False    671581
True        352
Name: count, dtype: int64


In [593]:
comparison_features = [
    "fiber_composition_similarity",
    "yarn1_count_match",
    "knit_subtype_match",
    "fine_match",
    "pus_match",
    "coloring_similarity",
    "yarn_type_similarity",
    "weight_diff_pct"
]

exact_vs_background = (
    orme_pair_scores
    .groupby(
        "known_exact_positive"
    )[comparison_features]
    .agg(
        [
            "count",
            "mean",
            "median"
        ]
    )
)

display(
    exact_vs_background
)

fiber_composition_similarity                   \
                                            count      mean median   
known_exact_positive                                                 
False                                      670058  0.696844   0.78   
True                                          352  1.000000   1.00   

                     yarn1_count_match                  knit_subtype_match  \
                                 count      mean median              count   
known_exact_positive                                                         
False                           630270  0.964804    1.0             589518   
True                               335  1.000000    1.0                346   

                                      fine_match  ... pus_match  \
                          mean median      count  ...    median   
known_exact_positive                              ...             
False                 0.965399    1.0     594090  ...       0.0   
True                  1.000000    1.0        324  ...       1.0   

                     coloring_similarity                   \
                                   count      mean median   
known_exact_positive                                        
False                             441788  0.245412    0.0   
True                                 303  1.000000    1.0   

                     yarn_type_similarity                  weight_diff_pct  \
                                    count      mean median           count   
known_exact_positive                                                         
False                              123549  0.387806    0.0          528365   
True                                  117  1.000000    1.0             329   

                                          
                          mean    median  
known_exact_positive                      
False                 1.901635  2.020202  
True                  0.000000  0.000000  

[2 rows x 24 columns]

In [594]:
orme_full_profile = (
    plm_duplicate_master[
        plm_duplicate_master["Mal grubu"] == "ORME"
    ]
    .drop_duplicates("plm_code")
    .set_index("plm_code")
    .copy()
)

assert orme_full_profile.index.is_unique

print(
    "ORME profile PLMs:",
    len(orme_full_profile)
)

ORME profile PLMs: 12812


In [595]:
structured_excluded_columns = {
    "PLM Kodu",
    "plm_code",
    "Mal grubu",
    "Malzeme statüs",
    "Türkçe malzeme açıklaması",
    "Malzeme Türkçe Adı",
    "Malzeme ingilizce adı",
    "material_group_code",
    "material_group_name",
    "material_family"
}

structured_profile_columns = [
    column
    for column in plm_codes.columns
    if (
        column in orme_full_profile.columns
        and column not in structured_excluded_columns
    )
]

print(
    "Structured technical fields:",
    len(structured_profile_columns)
)

print(structured_profile_columns)

Structured technical fields: 21
['Örme alt tipi', 'İlmek uzunluğu/50 iğne', '1.İplik numarası örme', '2.İplik numarası örm', '3.İplik numarası örme', 'Kumaş ağırlığı', 'Kumaş ağırlığı birimi', 'Kumaş eni', 'Kumaş eni birimi', 'Pus', 'Fine', 'Kumaş tipi', 'Çözgü sıklığı (tel /in', 'Atkı sıklığı (tel /inç)', 'Dokuma tipi', '1.Çözgü iplik numarası', '1.Atkı iplik numarası', '2.Çözgü iplik numarası', '2.Atkı iplik numarası', '3.Çözgü iplik numarası', '3.Atkı iplik numaras']


In [596]:
mv_profile_columns = sorted(
    set(
        plm_multi_value_valid[
            "PLM Karakteristik Tanımı"
        ]
        .astype("string")
        .str.strip()
        .dropna()
        .unique()
    )
    &
    set(
        orme_full_profile.columns
    )
)

print(
    "Multi-value technical fields:",
    len(mv_profile_columns)
)

Multi-value technical fields: 0


In [598]:
print(
    "plm_duplicate_master shape:",
    plm_duplicate_master.shape
)

print(
    "Total columns:",
    len(plm_duplicate_master.columns)
)

print("\nLast 80 columns:")
for column in plm_duplicate_master.columns[-80:]:
    print(repr(column))

plm_duplicate_master shape: (32361, 88)
Total columns: 88

Last 80 columns:
'Kumaş eni'
'Kumaş eni birimi'
'Pus'
'Fine'
'Mal grubu'
'Kumaş tipi'
'Çözgü sıklığı (tel /in'
'Atkı sıklığı (tel /inç)'
'Dokuma tipi'
'1.Çözgü iplik numarası'
'1.Atkı iplik numarası'
'2.Çözgü iplik numarası'
'2.Atkı iplik numarası'
'3.Çözgü iplik numarası'
'3.Atkı iplik numaras'
'MV__COLORINGID'
'MV__DYETYPEID'
'MV__FIBERCONTENTLISTID'
'MV__KNITTYPEID'
'MV__PHYSICALFINISHFABRICSIDEID'
'MV__WARPCOMPOSITION1LISTID'
'MV__WARPCOMPOSITION2LISTID'
'MV__WARPCOMPOSITION3LISTID'
'MV__WARPCOMPOSITION4LISTID'
'MV__WARPCOMPOSITION5LISTID'
'MV__WARPCOMPOSITION6LISTID'
'MV__WARPSUSCONTENT1'
'MV__WARPSUSCONTENT2'
'MV__WARPSUSCONTENT3'
'MV__WARPYARN1TYPEID'
'MV__WARPYARN2TYPEID'
'MV__WARPYARN3TYPEID'
'MV__WARPYARN4TYPEID'
'MV__WARPYARN5TYPEID'
'MV__WARPYARNCOLORING1ID'
'MV__WARPYARNCOLORING2ID'
'MV__WARPYARNCOLORING3ID'
'MV__WARPYARNCOLORING4ID'
'MV__WARPYARNCOLORING5ID'
'MV__WEFTCOMPOSITION1LISTID'
'MV__WEFTCOMPOSITION2LISTID

In [599]:
profile_column_names = [
    str(column)
    for column in plm_duplicate_master.columns
]

for characteristic in [
    "FIBERCONTENTLISTID",
    "COLORINGID",
    "YARN1TYPEKNITSID"
]:
    matches = [
        column
        for column in profile_column_names
        if characteristic in column
    ]

    print(
        characteristic,
        "->",
        matches
    )

FIBERCONTENTLISTID -> ['MV__FIBERCONTENTLISTID']
COLORINGID -> ['MV__COLORINGID']
YARN1TYPEKNITSID -> ['MV__YARN1TYPEKNITSID']


In [600]:
mv_characteristics = (
    plm_multi_value_valid[
        "PLM Karakteristik Tanımı"
    ]
    .astype("string")
    .str.strip()
    .dropna()
    .unique()
)

profile_columns = [
    str(column)
    for column in plm_duplicate_master.columns
]

In [601]:
mv_column_mapping = {}

for characteristic in mv_characteristics:

    exact_matches = [
        column
        for column in profile_columns
        if column == characteristic
    ]

    if exact_matches:
        mv_column_mapping[
            characteristic
        ] = exact_matches[0]
        continue

    fuzzy_matches = [
        column
        for column in profile_columns
        if (
            column.endswith(
                characteristic
            )
            or column.startswith(
                characteristic
            )
        )
    ]

    if len(fuzzy_matches) == 1:
        mv_column_mapping[
            characteristic
        ] = fuzzy_matches[0]

In [602]:
print(
    "MV characteristics in source:",
    len(mv_characteristics)
)

print(
    "MV columns matched in profile:",
    len(mv_column_mapping)
)

display(
    pd.DataFrame(
        mv_column_mapping.items(),
        columns=[
            "characteristic",
            "profile_column"
        ]
    )
    .head(30)
)

MV characteristics in source: 64
MV columns matched in profile: 64


,characteristic,profile_column
0,PHYSICALFINISHFABRICSIDEID,MV__PHYSICALFINISHFABRICSIDEID
1,FIBERCONTENTLISTID,MV__FIBERCONTENTLISTID
2,COLORINGID,MV__COLORINGID
3,KNITTYPEID,MV__KNITTYPEID
4,YARNCOLORING2KNITSID,MV__YARNCOLORING2KNITSID
5,YARNCOLORING1KNITSID,MV__YARNCOLORING1KNITSID
6,YARN2TYPEKNITSID,MV__YARN2TYPEKNITSID
7,YARN1TYPEKNITSID,MV__YARN1TYPEKNITSID
8,DYETYPEID,MV__DYETYPEID
9,YARNCOMPOSITION2LISTID,MV__YARNCOMPOSITION2LISTID


In [603]:
mv_profile_columns = list(
    mv_column_mapping.values()
)

print(
    "Multi-value technical fields:",
    len(mv_profile_columns)
)

Multi-value technical fields: 64


In [604]:
mv_orme_coverage_records = []

for column in mv_profile_columns:

    values = (
        orme_full_profile[column]
        .astype("string")
        .str.strip()
    )

    available = (
        values.notna()
        &
        ~values.isin(
            {
                "<MISSING>",
                "<ABSENT>"
            }
        )
    )

    mv_orme_coverage_records.append({
        "feature": column,
        "coverage": available.mean()
    })


mv_orme_coverage = pd.DataFrame(
    mv_orme_coverage_records,
    columns=[
        "feature",
        "coverage"
    ]
)

display(
    mv_orme_coverage
    .sort_values(
        "coverage",
        ascending=False
    )
    .head(30)
)

,feature,coverage
1,MV__FIBERCONTENTLISTID,0.992351
0,MV__PHYSICALFINISHFABRICSIDEID,0.816734
2,MV__COLORINGID,0.782001
10,MV__YARNCOMPOSITION1LISTID,0.777786
9,MV__YARNCOMPOSITION2LISTID,0.585467
7,MV__YARN1TYPEKNITSID,0.364268
3,MV__KNITTYPEID,0.336013
5,MV__YARNCOLORING1KNITSID,0.271230
8,MV__DYETYPEID,0.242273
11,MV__YARNCOMPOSITION3LISTID,0.215423


In [605]:
direct_mv_features = {
    "FIBERCONTENTLISTID",
    "COLORINGID",
    "YARN1TYPEKNITSID"
}

In [606]:
direct_mv_characteristics = {
    "FIBERCONTENTLISTID",
    "COLORINGID",
    "YARN1TYPEKNITSID"
}

direct_mv_profile_columns = {
    mv_column_mapping[
        characteristic
    ]
    for characteristic
    in direct_mv_characteristics
    if characteristic
    in mv_column_mapping
}

In [607]:
additional_mv_features = (
    mv_orme_coverage[
        (
            mv_orme_coverage[
                "coverage"
            ] >= 0.01
        )
        &
        (
            ~mv_orme_coverage[
                "feature"
            ].isin(
                direct_mv_profile_columns
            )
        )
    ]["feature"]
    .tolist()
)

print(
    "Direct MV columns excluded:",
    direct_mv_profile_columns
)

print(
    "Additional MV features:",
    len(additional_mv_features)
)

Direct MV columns excluded: {'MV__YARN1TYPEKNITSID', 'MV__FIBERCONTENTLISTID', 'MV__COLORINGID'}
Additional MV features: 13


In [608]:
display(
    mv_orme_coverage[
        mv_orme_coverage["feature"].isin(
            additional_mv_features
        )
    ]
    .sort_values(
        "coverage",
        ascending=False
    )
)

,feature,coverage
0,MV__PHYSICALFINISHFABRICSIDEID,0.816734
10,MV__YARNCOMPOSITION1LISTID,0.777786
9,MV__YARNCOMPOSITION2LISTID,0.585467
3,MV__KNITTYPEID,0.336013
5,MV__YARNCOLORING1KNITSID,0.271230
8,MV__DYETYPEID,0.242273
11,MV__YARNCOMPOSITION3LISTID,0.215423
6,MV__YARN2TYPEKNITSID,0.189510
4,MV__YARNCOLORING2KNITSID,0.155011
17,MV__YARN3TYPEKNITSID,0.074383


In [609]:
orme_additional_structured_features = [
    column
    for column in [
        "İlmek uzunluğu/50 iğne",
        "2.İplik numarası örm",
        "3.İplik numarası örme"
    ]
    if column in orme_full_profile.columns
]

print(
    "Additional structured features:",
    orme_additional_structured_features
)

Additional structured features: ['İlmek uzunluğu/50 iğne', '2.İplik numarası örm', '3.İplik numarası örme']


In [611]:
def add_agreement_summary_fast(
    pair_df,
    profile_df,
    feature_columns,
    prefix
):
    comparable_count = np.zeros(
        len(pair_df),
        dtype=np.int16
    )

    match_count = np.zeros(
        len(pair_df),
        dtype=np.int16
    )

    mismatch_count = np.zeros(
        len(pair_df),
        dtype=np.int16
    )

    if not feature_columns:
        pair_df[
            f"{prefix}_comparable_count"
        ] = comparable_count

        pair_df[
            f"{prefix}_match_count"
        ] = match_count

        pair_df[
            f"{prefix}_mismatch_count"
        ] = mismatch_count

        pair_df[
            f"{prefix}_agreement_rate"
        ] = np.nan

        return

    profile = (
        profile_df[
            feature_columns
        ]
        .copy()
    )

    code_to_position = pd.Series(
        np.arange(
            len(profile),
            dtype=np.int32
        ),
        index=profile.index
    )

    left_positions = (
        pair_df["plm_code_left"]
        .map(code_to_position)
    )

    right_positions = (
        pair_df["plm_code_right"]
        .map(code_to_position)
    )

    if (
        left_positions.isna().any()
        or right_positions.isna().any()
    ):
        raise ValueError(
            "Some PLM codes are missing "
            "from the ORME profile."
        )

    left_positions = (
        left_positions
        .astype(np.int32)
        .to_numpy()
    )

    right_positions = (
        right_positions
        .astype(np.int32)
        .to_numpy()
    )

    for column in feature_columns:

        values = (
            profile[column]
            .astype("string")
            .str.strip()
        )

        values = values.mask(
            values.isin([
                "<MISSING>",
                "<ABSENT>"
            ])
        )

        codes = (
            values
            .astype("category")
            .cat.codes
            .to_numpy()
        )

        left_values = codes[
            left_positions
        ]

        right_values = codes[
            right_positions
        ]

        comparable = (
            (left_values >= 0)
            &
            (right_values >= 0)
        )

        match = (
            comparable
            &
            (
                left_values
                ==
                right_values
            )
        )

        mismatch = (
            comparable
            &
            ~match
        )

        comparable_count += (
            comparable.astype(
                np.int16
            )
        )

        match_count += (
            match.astype(
                np.int16
            )
        )

        mismatch_count += (
            mismatch.astype(
                np.int16
            )
        )

    pair_df[
        f"{prefix}_comparable_count"
    ] = comparable_count

    pair_df[
        f"{prefix}_match_count"
    ] = match_count

    pair_df[
        f"{prefix}_mismatch_count"
    ] = mismatch_count

    pair_df[
        f"{prefix}_agreement_rate"
    ] = np.divide(
        match_count,
        comparable_count,
        out=np.full(
            len(pair_df),
            np.nan
        ),
        where=(
            comparable_count > 0
        )
    )